# queue-boarding/v1.0.0 — GitHub 독립 학습·추론 노트북

이 노트북은 GitHub의 `analysis/` 폴더를 요구하지 않습니다. 학습·추론 Python 코드와 의존 코드는
압축이나 Base64 번들이 아니라 **파일별 일반 코드 셀**로 원문 그대로 포함되어 있습니다.
추론만 할 때 필요한 외부 파일은
`queue_boarding_v1_0_0.pkl` 하나입니다.

> `queue_boarding_v1_0_0.metadata.json`은 감사·사람용 요약이며 추론 필수 파일이 아닙니다.
> 실행 계약, 피처 목록, state profile, 모델 객체는 PKL 내부에 들어 있습니다.

> **데이터 정책:** 2026-08-18 최종 평가 데이터는 재학습·보정·모델 선택에 사용하지 않습니다.


In [ ]:
%pip -q install pandas numpy scikit-learn lightgbm joblib


In [ ]:
from pathlib import Path
import hashlib, io, json, sys
import joblib
import numpy as np
import pandas as pd

RUNTIME_ROOT = Path('/content/queue_boarding_runtime')
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
for relative in ('analysis', 'gbis_client', 'collector/gbis_collector'):
    (RUNTIME_ROOT / relative).mkdir(parents=True, exist_ok=True)
print('plain-source runtime:', RUNTIME_ROOT)


## 1. 원본 Python 소스

각 코드 셀은 제목에 표시된 파일의 원문입니다. 셀을 실행하면 `/content/queue_boarding_runtime`에
동일한 디렉터리 구조로 저장됩니다. 숨겨진 archive나 압축 해제 단계는 없습니다.


### `analysis/all_prearrival_seat_regression.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/all_prearrival_seat_regression.py
from __future__ import annotations

import argparse
import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

from model_feasibility import (
    FeatureSet,
    _as_records,
    build_model_table,
    build_visits,
    json_ready,
    load_data,
    make_preprocessor,
    prepare_subset,
)
from tminus_feasibility import ROUTE_ID, ROUTE_NAME, prepare_raw_locations


ALL_PREARRIVAL = FeatureSet(
    name="all_prearrival_target_vehicle",
    numeric=(
        "snapshot_time_sin",
        "snapshot_time_cos",
        "route_progress",
        "snapshot_route_progress",
        "x",
        "y",
        "snapshot_capacity",
        "target_load_ratio",
        "target_stop_gap",
    ),
    categorical=(
        "station_seq_cat",
        "direction",
        "snapshot_day_of_week",
        "snapshot_low_plate_cat",
        "target_state_cat",
    ),
)


DYNAMIC_ALL_PREARRIVAL = FeatureSet(
    name="all_prearrival_dynamic",
    numeric=(
        *ALL_PREARRIVAL.numeric,
        "snapshot_remaining_seats",
        "seat_delta_previous_stop",
        "seat_change_per_stop",
        "rolling_seat_change_per_stop_3",
        "minutes_since_previous_stop",
        "rolling_minutes_per_stop_3",
        "estimated_minutes_to_arrival",
        "projected_arrival_seats",
        "seats_per_remaining_stop",
        "load_gap_interaction",
        "currently_low_5",
        "currently_low_10",
    ),
    categorical=(
        *ALL_PREARRIVAL.categorical,
        "snapshot_station_seq_cat",
        "snapshot_time_bin_30",
    ),
)


ENGINEERED_ALL_PREARRIVAL = FeatureSet(
    name="all_prearrival_previous_bus",
    numeric=(
        *DYNAMIC_ALL_PREARRIVAL.numeric,
        "previous_bus_departure_seats",
        "previous_bus_departure_capacity",
        "previous_bus_departure_load_ratio",
        "previous_bus_same_capacity",
        "previous_bus_departure_age_minutes",
        "previous_bus_projected_headway_minutes",
        "previous_bus_freshness_exp",
        "previous_bus_departure_missing",
        "previous_bus_headway_minutes",
        "previous_bus_headway_log1p",
        "previous_bus_is_bunched",
        "previous_bus_headway_missing",
        "previous_3_bus_departure_mean",
        "previous_3_bus_departure_std",
        "previous_bus_departure_trend",
        "previous_3_bus_departure_load_ratio_mean",
        "previous_3_bus_departure_load_ratio_std",
        "previous_3_bus_departure_load_ratio_trend",
        "previous_bus_was_full",
        "previous_bus_was_low_5",
        "previous_bus_was_full_normalized",
        "previous_bus_was_low_10pct",
    ),
    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,
)


def departure_histories(
    visits: pd.DataFrame,
    *,
    turnaround_seq: int,
) -> dict[tuple[int, str], pd.DataFrame]:
    departures = visits.loc[
        visits["departure_seats"].ge(0)
        & visits["departure_seen"].notna()
        & visits["station_seq"].notna()
    ].copy()
    departures["station_seq"] = departures["station_seq"].astype(int)
    departures["direction"] = np.where(
        departures["station_seq"].le(turnaround_seq), "to_city", "return"
    )
    departures["departure_capacity"] = np.where(
        departures["low_plate"].eq(2), 70.0, 45.0
    )
    departures["departure_load_ratio"] = (
        1 - departures["departure_seats"] / departures["departure_capacity"]
    ).clip(0, 1)
    departures = departures.sort_values(
        ["station_seq", "direction", "departure_seen", "vehicle_id"]
    )
    grouped = departures.groupby(["station_seq", "direction"], sort=False)
    raw_headway = (
        grouped["departure_seen"].diff().dt.total_seconds() / 60
    )
    departures["previous_bus_is_bunched"] = raw_headway.between(
        0, 2, inclusive="both"
    ).astype(float)
    departures["previous_bus_headway_minutes"] = raw_headway.where(
        raw_headway.between(0, 120, inclusive="both")
    )
    departures["previous_bus_headway_log1p"] = np.log1p(
        departures["previous_bus_headway_minutes"]
    )
    departures["previous_bus_headway_missing"] = departures[
        "previous_bus_headway_minutes"
    ].isna().astype(float)
    departures["previous_bus_departure_trend"] = grouped["departure_seats"].diff()
    departures["previous_3_bus_departure_mean"] = grouped["departure_seats"].transform(
        lambda values: values.rolling(3, min_periods=1).mean()
    )
    departures["previous_3_bus_departure_std"] = grouped["departure_seats"].transform(
        lambda values: values.rolling(3, min_periods=2).std(ddof=0)
    )
    departures["previous_3_bus_departure_load_ratio_mean"] = grouped[
        "departure_load_ratio"
    ].transform(lambda values: values.rolling(3, min_periods=1).mean())
    departures["previous_3_bus_departure_load_ratio_std"] = grouped[
        "departure_load_ratio"
    ].transform(lambda values: values.rolling(3, min_periods=2).std(ddof=0))
    departures["previous_3_bus_departure_load_ratio_trend"] = grouped[
        "departure_load_ratio"
    ].transform(
        lambda values: values.rolling(3, min_periods=2).apply(
            lambda window: (window[-1] - window[0]) / (len(window) - 1),
            raw=True,
        )
    )
    return {
        (int(station_seq), str(direction)): group.reset_index(drop=True)
        for (station_seq, direction), group in departures.groupby(
            ["station_seq", "direction"], sort=False
        )
    }


def add_previous_bus_features(
    frame: pd.DataFrame,
    history: pd.DataFrame | None,
    *,
    max_age_minutes: float = 180.0,
    freshness_time_constant_minutes: float = 30.0,
) -> pd.DataFrame:
    if freshness_time_constant_minutes <= 0:
        raise ValueError("freshness_time_constant_minutes는 0보다 커야 합니다.")
    output = frame.copy()
    feature_columns = [
        "previous_bus_departure_seats",
        "previous_bus_departure_capacity",
        "previous_bus_departure_load_ratio",
        "previous_bus_same_capacity",
        "previous_bus_departure_age_minutes",
        "previous_bus_projected_headway_minutes",
        "previous_bus_freshness_exp",
        "previous_bus_departure_missing",
        "previous_bus_headway_minutes",
        "previous_bus_headway_log1p",
        "previous_bus_is_bunched",
        "previous_bus_headway_missing",
        "previous_3_bus_departure_mean",
        "previous_3_bus_departure_std",
        "previous_bus_departure_trend",
        "previous_3_bus_departure_load_ratio_mean",
        "previous_3_bus_departure_load_ratio_std",
        "previous_3_bus_departure_load_ratio_trend",
        "previous_bus_was_full",
        "previous_bus_was_low_5",
        "previous_bus_was_full_normalized",
        "previous_bus_was_low_10pct",
    ]
    for column in feature_columns:
        output[column] = np.nan
    output["previous_bus_departure_missing"] = 1.0
    output["previous_bus_vehicle_id"] = pd.Series(
        pd.NA, index=output.index, dtype="string"
    )
    output["previous_bus_trip_id"] = pd.Series(
        pd.NA, index=output.index, dtype="string"
    )
    if history is None or history.empty or output.empty:
        return output

    sort_columns = ["departure_seen"]
    if "vehicle_id" in history.columns:
        sort_columns.append("vehicle_id")
    history = history.sort_values(sort_columns, kind="stable").reset_index(drop=True)

    # side="left"를 사용해 스냅샷과 같은 시각의 출발도 제외한다. 실제 API
    # 요청 시점에 이미 완료된 출발 관측만 피처로 허용하기 위해서다.
    positions = np.asarray(
        history["departure_seen"].searchsorted(
            output["snapshot_time"], side="left"
        ),
        dtype=int,
    ) - 1
    # 정상 노선 순서에서는 현재 운행이 목표 정류장을 이미 출발했을 수 없지만,
    # trip 분할 오류나 순서 이상에도 자기 라벨을 참조하지 않도록 명시적으로
    # 같은 trip을 건너뛰고 그보다 앞선 적격 출발까지 역검색한다.
    if "trip_id" in output.columns and "trip_id" in history.columns:
        target_trip_ids = output["trip_id"].astype(str).to_numpy()
        history_trip_ids = history["trip_id"].astype(str).to_numpy()
        for index, position in enumerate(positions):
            while (
                position >= 0
                and history_trip_ids[position] == target_trip_ids[index]
            ):
                position -= 1
            positions[index] = position
    valid = positions >= 0
    if not valid.any():
        return output
    target_indices = np.flatnonzero(valid)
    selected = history.iloc[positions[valid]].reset_index(drop=True)
    age = (
        output.iloc[target_indices]["snapshot_time"].reset_index(drop=True)
        - selected["departure_seen"]
    ).dt.total_seconds() / 60
    fresh = age.between(0, max_age_minutes, inclusive="both").to_numpy()
    target_indices = target_indices[fresh]
    selected = selected.loc[fresh].reset_index(drop=True)
    age = age.loc[fresh].reset_index(drop=True)
    if len(target_indices) == 0:
        return output

    seats = selected["departure_seats"].to_numpy(dtype=float)
    capacity = selected["departure_capacity"].to_numpy(dtype=float)
    load_ratio = np.clip(1 - seats / capacity, 0, 1)
    output.iloc[target_indices, output.columns.get_loc("previous_bus_departure_seats")] = seats
    output.iloc[
        target_indices, output.columns.get_loc("previous_bus_departure_capacity")
    ] = capacity
    output.iloc[
        target_indices, output.columns.get_loc("previous_bus_departure_load_ratio")
    ] = load_ratio
    if "snapshot_capacity" in output.columns:
        snapshot_capacity = pd.to_numeric(
            output.iloc[target_indices]["snapshot_capacity"], errors="coerce"
        ).to_numpy(dtype=float)
        comparable = np.isfinite(capacity) & np.isfinite(snapshot_capacity)
        same_capacity = np.full(len(target_indices), np.nan)
        same_capacity[comparable] = np.isclose(
            capacity[comparable], snapshot_capacity[comparable]
        ).astype(float)
        output.iloc[
            target_indices, output.columns.get_loc("previous_bus_same_capacity")
        ] = same_capacity
    output.iloc[
        target_indices, output.columns.get_loc("previous_bus_departure_age_minutes")
    ] = age.to_numpy(dtype=float)
    if "estimated_minutes_to_arrival" in output.columns:
        estimated_eta = pd.to_numeric(
            output.iloc[target_indices]["estimated_minutes_to_arrival"],
            errors="coerce",
        ).reset_index(drop=True)
        estimated_eta = estimated_eta.where(estimated_eta.ge(0))
        output.iloc[
            target_indices,
            output.columns.get_loc("previous_bus_projected_headway_minutes"),
        ] = (age + estimated_eta).to_numpy(dtype=float)
    output.iloc[
        target_indices, output.columns.get_loc("previous_bus_freshness_exp")
    ] = np.exp(-age.to_numpy(dtype=float) / freshness_time_constant_minutes)
    output.iloc[
        target_indices, output.columns.get_loc("previous_bus_departure_missing")
    ] = 0.0
    if "vehicle_id" in selected.columns:
        output.iloc[
            target_indices, output.columns.get_loc("previous_bus_vehicle_id")
        ] = selected["vehicle_id"].astype(str).to_numpy()
    if "trip_id" in selected.columns:
        output.iloc[
            target_indices, output.columns.get_loc("previous_bus_trip_id")
        ] = selected["trip_id"].astype(str).to_numpy()
    for target_column, source_column in [
        ("previous_bus_headway_minutes", "previous_bus_headway_minutes"),
        ("previous_bus_headway_log1p", "previous_bus_headway_log1p"),
        ("previous_bus_is_bunched", "previous_bus_is_bunched"),
        ("previous_bus_headway_missing", "previous_bus_headway_missing"),
        ("previous_3_bus_departure_mean", "previous_3_bus_departure_mean"),
        ("previous_3_bus_departure_std", "previous_3_bus_departure_std"),
        ("previous_bus_departure_trend", "previous_bus_departure_trend"),
        (
            "previous_3_bus_departure_load_ratio_mean",
            "previous_3_bus_departure_load_ratio_mean",
        ),
        (
            "previous_3_bus_departure_load_ratio_std",
            "previous_3_bus_departure_load_ratio_std",
        ),
        (
            "previous_3_bus_departure_load_ratio_trend",
            "previous_3_bus_departure_load_ratio_trend",
        ),
    ]:
        if source_column not in selected.columns:
            continue
        output.iloc[target_indices, output.columns.get_loc(target_column)] = selected[
            source_column
        ].to_numpy(dtype=float)
    output.iloc[
        target_indices, output.columns.get_loc("previous_bus_was_full")
    ] = (seats == 0).astype(float)
    output.iloc[
        target_indices, output.columns.get_loc("previous_bus_was_low_5")
    ] = (seats <= 5).astype(float)
    output.iloc[
        target_indices, output.columns.get_loc("previous_bus_was_full_normalized")
    ] = np.isclose(load_ratio, 1.0).astype(float)
    output.iloc[
        target_indices, output.columns.get_loc("previous_bus_was_low_10pct")
    ] = ((1 - load_ratio) <= 0.10).astype(float)
    return output


def route_progress(
    station_seq: pd.Series,
    direction: pd.Series,
    turnaround_seq: int,
    max_seq: int,
) -> pd.Series:
    to_city = station_seq / max(turnaround_seq, 1)
    returning = (max_seq - station_seq) / max(max_seq - turnaround_seq, 1)
    return pd.Series(
        np.where(direction.eq("to_city"), to_city, returning),
        index=station_seq.index,
    ).clip(0, 1)


def build_all_prearrival_table(
    events: pd.DataFrame,
    visits: pd.DataFrame,
    by_vehicle: dict[str, pd.DataFrame],
    *,
    turnaround_seq: int,
    max_seq: int,
) -> pd.DataFrame:
    """각 도착 사건에 대해 상류 정류장마다 마지막 관측 하나를 만든다."""
    trip_starts = visits.groupby("trip_id", sort=False)["first_seen"].min()
    histories = departure_histories(visits, turnaround_seq=turnaround_seq)
    frames: list[pd.DataFrame] = []

    for event in events.itertuples(index=False):
        vehicle_rows = by_vehicle.get(str(event.vehicle_id))
        trip_start = trip_starts.get(event.trip_id)
        if vehicle_rows is None or pd.isna(trip_start):
            continue
        candidates = vehicle_rows.loc[
            vehicle_rows["observed_at"].ge(trip_start)
            & vehicle_rows["observed_at"].lt(event.event_time)
            & vehicle_rows["direction"].eq(event.direction)
            & vehicle_rows["station_seq"].lt(event.station_seq)
        ].copy()
        if candidates.empty:
            continue

        # 같은 상류 정류장에 오래 머문 차량이 과대표집되지 않도록 정류장마다
        # 목표 도착 전에 이용 가능한 마지막 스냅샷 하나만 남긴다.
        candidates = (
            candidates.sort_values(["station_seq", "observed_at", "run_id"])
            .groupby("station_seq", sort=False)
            .tail(1)
            .sort_values("observed_at")
        )
        minute = candidates["observed_at"].dt.hour * 60 + candidates["observed_at"].dt.minute
        angle = 2 * np.pi * minute / (24 * 60)
        frame = pd.DataFrame(
            {
                "event_id": (
                    str(event.trip_id)
                    + ":"
                    + str(int(event.station_seq))
                    + ":"
                    + str(event.event_time)
                ),
                "trip_id": str(event.trip_id),
                "vehicle_id": str(event.vehicle_id),
                "date": str(event.date),
                "event_time": event.event_time,
                "snapshot_time": candidates["observed_at"].to_numpy(),
                "label_quality": str(event.label_quality),
                "label_seats": float(event.label_seats),
                "capacity": float(event.capacity),
                "station_seq_cat": str(event.station_seq_cat),
                "direction": str(event.direction),
                "route_progress": float(event.route_progress),
                "x": float(event.x) if pd.notna(event.x) else np.nan,
                "y": float(event.y) if pd.notna(event.y) else np.nan,
                "snapshot_station_seq": candidates["station_seq"].to_numpy(dtype=int),
                "snapshot_remaining_seats": candidates["remaining_seats"].to_numpy(dtype=float),
                "snapshot_capacity": candidates["capacity"].to_numpy(dtype=float),
                "target_load_ratio": candidates["load_ratio"].to_numpy(dtype=float),
                "target_stop_gap": float(event.station_seq) - candidates["station_seq"].to_numpy(dtype=float),
                "snapshot_time_sin": np.sin(angle).to_numpy(dtype=float),
                "snapshot_time_cos": np.cos(angle).to_numpy(dtype=float),
                "snapshot_day_of_week": candidates["observed_at"].dt.dayofweek.astype(str).to_numpy(),
                "snapshot_low_plate_cat": candidates["low_plate"].astype(int).astype(str).to_numpy(),
                "target_state_cat": candidates["state_code"].astype(int).astype(str).to_numpy(),
            }
        )
        frame["snapshot_route_progress"] = route_progress(
            frame["snapshot_station_seq"],
            frame["direction"],
            turnaround_seq,
            max_seq,
        )
        frame["minutes_to_arrival"] = (
            frame["event_time"] - frame["snapshot_time"]
        ).dt.total_seconds() / 60
        stops_moved = frame["snapshot_station_seq"].diff()
        frame["seat_delta_previous_stop"] = frame["snapshot_remaining_seats"].diff()
        frame["seat_change_per_stop"] = (
            frame["seat_delta_previous_stop"] / stops_moved.where(stops_moved.gt(0))
        )
        frame["rolling_seat_change_per_stop_3"] = (
            frame["seat_change_per_stop"].rolling(3, min_periods=1).mean()
        )
        frame["minutes_since_previous_stop"] = (
            frame["snapshot_time"].diff().dt.total_seconds() / 60
        )
        frame["rolling_minutes_per_stop_3"] = (
            (frame["minutes_since_previous_stop"] / stops_moved.where(stops_moved.gt(0)))
            .rolling(3, min_periods=1)
            .median()
        )
        frame["estimated_minutes_to_arrival"] = (
            frame["rolling_minutes_per_stop_3"] * frame["target_stop_gap"]
        )
        frame["projected_arrival_seats"] = (
            frame["snapshot_remaining_seats"]
            + frame["rolling_seat_change_per_stop_3"] * frame["target_stop_gap"]
        ).clip(lower=0, upper=float(event.capacity))
        frame["seats_per_remaining_stop"] = (
            frame["snapshot_remaining_seats"] / frame["target_stop_gap"]
        )
        frame["load_gap_interaction"] = (
            frame["target_load_ratio"] * frame["target_stop_gap"]
        )
        frame["currently_low_5"] = frame["snapshot_remaining_seats"].le(5).astype(float)
        frame["currently_low_10"] = frame["snapshot_remaining_seats"].le(10).astype(float)
        frame["snapshot_station_seq_cat"] = frame["snapshot_station_seq"].astype(str)
        snapshot_minute = (
            frame["snapshot_time"].dt.hour * 60 + frame["snapshot_time"].dt.minute
        )
        frame["snapshot_time_bin_30"] = (snapshot_minute // 30).astype(str)
        frame = add_previous_bus_features(
            frame,
            histories.get((int(event.station_seq), str(event.direction))),
        )
        frame = frame.loc[
            frame["minutes_to_arrival"].gt(0) & frame["target_stop_gap"].gt(0)
        ]
        frames.append(frame)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True).sort_values(
        ["event_time", "event_id", "snapshot_time"]
    ).reset_index(drop=True)


def event_weights(data: pd.DataFrame) -> np.ndarray:
    counts = data.groupby("event_id")["event_id"].transform("size").to_numpy(dtype=float)
    weights = 1.0 / counts
    return weights / weights.mean()


def event_bias(data: pd.DataFrame, predictions: np.ndarray) -> float:
    residual = data[["event_id", "label_seats"]].copy()
    residual["residual"] = residual["label_seats"].to_numpy(dtype=float) - predictions
    per_event = residual.groupby("event_id", sort=False)["residual"].median()
    return float(per_event.median())


def clip_seats(predictions: np.ndarray, capacity: pd.Series) -> np.ndarray:
    return np.minimum(np.maximum(predictions, 0), capacity.to_numpy(dtype=float))


def event_balanced_metrics(
    data: pd.DataFrame,
    predictions: np.ndarray,
    *,
    model: str,
    split: str,
) -> dict[str, Any]:
    scored = data[["event_id", "label_seats"]].copy()
    scored["prediction"] = clip_seats(predictions, data["capacity"])
    scored["absolute_error"] = (
        scored["label_seats"] - scored["prediction"]
    ).abs()
    scored["within_3"] = scored["absolute_error"].le(3)
    scored["within_5"] = scored["absolute_error"].le(5)
    per_event = scored.groupby("event_id", sort=False).agg(
        label_seats=("label_seats", "first"),
        mae=("absolute_error", "mean"),
        within_3=("within_3", "mean"),
        within_5=("within_5", "mean"),
    )
    low = per_event["label_seats"].le(5)
    y = scored["label_seats"].to_numpy(dtype=float)
    pred = scored["prediction"].to_numpy(dtype=float)
    return {
        "split": split,
        "model": model,
        "rows": int(len(scored)),
        "events": int(len(per_event)),
        "low_0_5_events": int(low.sum()),
        "row_mae_seats": float(mean_absolute_error(y, pred)),
        "row_rmse_seats": float(math.sqrt(mean_squared_error(y, pred))),
        "row_r2": float(r2_score(y, pred)),
        "event_balanced_mae_seats": float(per_event["mae"].mean()),
        "event_balanced_within_3": float(per_event["within_3"].mean()),
        "event_balanced_within_5": float(per_event["within_5"].mean()),
        "low_0_5_event_balanced_mae_seats": (
            float(per_event.loc[low, "mae"].mean()) if low.any() else np.nan
        ),
        "low_0_5_event_balanced_within_3": (
            float(per_event.loc[low, "within_3"].mean()) if low.any() else np.nan
        ),
    }


def make_model(seed: int) -> Pipeline:
    return Pipeline(
        [
            ("features", clone(make_preprocessor(ALL_PREARRIVAL))),
            (
                "regressor",
                HistGradientBoostingRegressor(
                    loss="absolute_error",
                    learning_rate=0.05,
                    max_iter=300,
                    max_leaf_nodes=15,
                    min_samples_leaf=20,
                    l2_regularization=1.0,
                    random_state=seed,
                ),
            ),
        ]
    )


def fit_and_predict(
    train: pd.DataFrame,
    calibration: pd.DataFrame,
    test: pd.DataFrame,
    *,
    delta_target: bool,
    seed: int,
) -> tuple[np.ndarray, float, float]:
    model = make_model(seed)
    train_y = train["label_seats"].to_numpy(dtype=float)
    if delta_target:
        train_y = train_y - train["snapshot_remaining_seats"].to_numpy(dtype=float)
    model.fit(
        train[ALL_PREARRIVAL.columns],
        train_y,
        regressor__sample_weight=event_weights(train),
    )
    cal_prediction = model.predict(calibration[ALL_PREARRIVAL.columns])
    test_prediction = model.predict(test[ALL_PREARRIVAL.columns])
    if delta_target:
        cal_prediction += calibration["snapshot_remaining_seats"].to_numpy(dtype=float)
        test_prediction += test["snapshot_remaining_seats"].to_numpy(dtype=float)
    bias = event_bias(calibration, cal_prediction)
    cal_prediction = clip_seats(cal_prediction + bias, calibration["capacity"])
    calibration_mae = event_balanced_metrics(
        calibration,
        cal_prediction,
        model="calibration",
        split="calibration",
    )["event_balanced_mae_seats"]
    return test_prediction + bias, float(bias), float(calibration_mae)


def bucket_metrics(
    test: pd.DataFrame,
    predictions: np.ndarray,
    column: str,
    bins: list[float],
    labels: list[str],
) -> pd.DataFrame:
    bucket = pd.cut(test[column], bins=bins, labels=labels, include_lowest=True)
    rows: list[dict[str, Any]] = []
    for label in labels:
        mask = bucket.eq(label).to_numpy()
        if not mask.any():
            continue
        result = event_balanced_metrics(
            test.loc[mask], predictions[mask], model="selected", split=str(label)
        )
        result["bucket"] = str(label)
        rows.append(result)
    return pd.DataFrame(rows)


def main() -> int:
    parser = argparse.ArgumentParser(
        description="모든 상류 정류장 스냅샷에서 도착 잔여좌석 회귀"
    )
    parser.add_argument("--db", type=Path, default=Path("data/gbis.sqlite3"))
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("analysis/all_prearrival_seat_results"),
    )
    parser.add_argument("--include-inferred", action="store_true")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    args.output_dir.mkdir(parents=True, exist_ok=True)

    locations, stations = load_data(args.db, ROUTE_ID)
    visits = build_visits(locations, stations)
    model_table, turnaround_seq = build_model_table(
        visits, stations, label_target="arrival"
    )
    source = prepare_subset(model_table)
    qualities = ["A", "B"] if args.include_inferred else ["A"]
    source = source.loc[source["label_quality"].isin(qualities)].copy()
    _, by_vehicle = prepare_raw_locations(locations, turnaround_seq)
    snapshots = build_all_prearrival_table(
        source,
        visits,
        by_vehicle,
        turnaround_seq=turnaround_seq,
        max_seq=int(stations["station_seq"].max()),
    )
    train = snapshots.loc[snapshots["date"].isin(["2026-08-04", "2026-08-05"])]
    calibration = snapshots.loc[snapshots["date"].eq("2026-08-06")]
    test = snapshots.loc[snapshots["date"].eq("2026-08-07")]
    if min(train["event_id"].nunique(), calibration["event_id"].nunique(), test["event_id"].nunique()) == 0:
        raise ValueError("시간 분할 중 하나에 도착 사건이 없습니다.")

    rows: list[dict[str, Any]] = []
    predictions: dict[str, np.ndarray] = {}
    persistence = test["snapshot_remaining_seats"].to_numpy(dtype=float)
    predictions["current_seats_persistence"] = persistence
    baseline = event_balanced_metrics(
        test,
        persistence,
        model="current_seats_persistence",
        split="test_2026-08-07",
    )
    baseline["selection_calibration_mae"] = event_balanced_metrics(
        calibration,
        calibration["snapshot_remaining_seats"].to_numpy(dtype=float),
        model="current_seats_persistence",
        split="calibration",
    )["event_balanced_mae_seats"]
    baseline["bias_correction_seats"] = 0.0
    rows.append(baseline)

    for model_name, delta in [("direct_hgb", False), ("delta_hgb", True)]:
        prediction, bias, calibration_mae = fit_and_predict(
            train, calibration, test, delta_target=delta, seed=args.seed
        )
        predictions[model_name] = clip_seats(prediction, test["capacity"])
        result = event_balanced_metrics(
            test,
            prediction,
            model=model_name,
            split="test_2026-08-07",
        )
        result["selection_calibration_mae"] = calibration_mae
        result["bias_correction_seats"] = bias
        rows.append(result)

    metrics = pd.DataFrame(rows)
    selected_row = metrics.loc[metrics["model"].ne("current_seats_persistence")].sort_values(
        ["selection_calibration_mae", "event_balanced_mae_seats"]
    ).iloc[0]
    selected_name = str(selected_row["model"])
    selected_prediction = predictions[selected_name]
    by_minutes = bucket_metrics(
        test,
        selected_prediction,
        "minutes_to_arrival",
        [-np.inf, 5, 10, 20, 30, 60, np.inf],
        ["0-5", "5-10", "10-20", "20-30", "30-60", "60+"],
    )
    by_stops = bucket_metrics(
        test,
        selected_prediction,
        "target_stop_gap",
        [0, 1, 2, 5, 10, 20, np.inf],
        ["1", "2", "3-5", "6-10", "11-20", "21+"],
    )

    metrics.to_csv(args.output_dir / "model_metrics.csv", index=False)
    by_minutes.to_csv(args.output_dir / "metrics_by_minutes.csv", index=False)
    by_stops.to_csv(args.output_dir / "metrics_by_stops.csv", index=False)
    result = {
        "route_id": ROUTE_ID,
        "route_name": ROUTE_NAME,
        "target": "arrival_seats before boarding",
        "label_quality": qualities,
        "snapshot_unit": "last available observation at every upstream station",
        "feature_leakage_rule": (
            "actual minutes_to_arrival is evaluation-only; model uses current time, "
            "current seats, current/target position, vehicle state, and target metadata"
        ),
        "split": {
            "train": ["2026-08-04", "2026-08-05"],
            "calibration_and_selection": ["2026-08-06"],
            "untouched_test": ["2026-08-07"],
        },
        "rows": {
            "train": int(len(train)),
            "calibration": int(len(calibration)),
            "test": int(len(test)),
        },
        "events": {
            "train": int(train["event_id"].nunique()),
            "calibration": int(calibration["event_id"].nunique()),
            "test": int(test["event_id"].nunique()),
        },
        "models": _as_records(metrics),
        "selected_on_calibration": json_ready(selected_row.to_dict()),
        "performance_by_minutes": _as_records(by_minutes),
        "performance_by_stops": _as_records(by_stops),
        "limitations": [
            "실제 도착까지 남은 시간은 평가 구간화에만 사용했으며 모델 피처에는 넣지 않았다.",
            "한 도착 사건에 여러 상류 정류장 스냅샷이 있으므로 사건별 동일 가중 지표를 주 지표로 사용했다.",
            "A등급 직접 도착 관측만 기본 평가에 사용한다.",
            "최종 테스트가 하루라 장기 운영 성능 확정치가 아니다.",
        ],
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(result), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps(json_ready(result), ensure_ascii=False, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/bus_model_2_same_metric_experiment.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/bus_model_2_same_metric_experiment.py
"""Evaluate Bus_model_2 methodology on Yeonwu's identical pooled task.

Bus_model_2 originally predicts a later raw observation.  This script keeps its
LightGBM/log1p/Huber/low-seat-weight design, but evaluates arrival-seat labels,
routes, rolling dates, event weighting, and metrics exactly like Yeonwu.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

from yeonwu_peer_feature_experiment import (
    DEFAULT_ROUTES,
    ROOT,
    cache_observation_cutoff,
    load_embedded_modules,
    load_fresh_features,
    refresh_cache,
)


BUS2_FEATURES = (
    "route_code",
    "snapshot_station_seq_cat",
    "snapshot_station_seq_numeric",
    "snapshot_time_minute",
    "snapshot_day_of_week",
    "station_time_pop",
    "target_stop_gap",
    "seat_delta",
    "seats_per_stop",
)
CATEGORICAL = ("route_code", "snapshot_station_seq_cat")


def add_bus2_features(data: pd.DataFrame) -> pd.DataFrame:
    """Leak-free version of Bus_model_2 station_time_pop and its interactions.

    Every row reads station-time history only from earlier weekday calendar days.
    No raw-row lag is used: repeated observations at one stop would otherwise be
    mistaken for a previous stop, and future arrival data could leak into input.
    """
    output = data.copy()
    output["snapshot_time"] = pd.to_datetime(output["snapshot_time"])
    output["snapshot_hour"] = output["snapshot_time"].dt.hour.astype("int8")
    output["snapshot_time_minute"] = (
        output["snapshot_time"].dt.hour * 60 + output["snapshot_time"].dt.minute
    ).astype("int16")
    output["snapshot_station_seq_numeric"] = pd.to_numeric(
        output["snapshot_station_seq_cat"], errors="coerce").fillna(-1).astype("int16")
    output["station_time_pop"] = np.nan
    global_mean = float(output["snapshot_remaining_seats"].median())
    keys = ["route_code", "snapshot_station_seq_cat", "snapshot_hour"]
    fallback_keys = ["route_code", "snapshot_hour"]
    weekday = pd.to_datetime(output["date"]).dt.dayofweek.lt(5)
    for date in sorted(output["date"].unique()):
        target_mask = output["date"].eq(date)
        history = output.loc[output["date"].lt(date) & weekday]
        stats = history.groupby(keys, observed=True)["snapshot_remaining_seats"].mean()
        fallback = history.groupby(fallback_keys, observed=True)["snapshot_remaining_seats"].mean()
        target = output.loc[target_mask]
        primary = stats.reindex(target.set_index(keys).index).to_numpy()
        backup = fallback.reindex(target.set_index(fallback_keys).index).to_numpy()
        output.loc[target_mask, "station_time_pop"] = pd.Series(primary, index=target.index).fillna(
            pd.Series(backup, index=target.index)
        ).fillna(global_mean)
    output["seat_delta"] = (
        output["snapshot_remaining_seats"].to_numpy(float)
        - output["station_time_pop"].to_numpy(float)
    )
    output["seats_per_stop"] = output["snapshot_remaining_seats"].to_numpy(float) / np.maximum(
        output["target_stop_gap"].to_numpy(float), 1.0
    )
    return output


def encoded_features(train: pd.DataFrame, target: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Use Bus_model_2's train-category mapping. Unknown validation categories=-1."""
    left, right = train.loc[:, BUS2_FEATURES].copy(), target.loc[:, BUS2_FEATURES].copy()
    for column in CATEGORICAL:
        categories = pd.Index(left[column].astype("string").dropna().unique())
        left[column] = pd.Categorical(left[column].astype("string"), categories=categories).codes
        right[column] = pd.Categorical(right[column].astype("string"), categories=categories).codes
    # Yeonwu stores day-of-week as a categorical string.  Bus_model_2 uses the
    # original numeric `dayofweek`, so restore that numeric representation.
    for column in set(BUS2_FEATURES) - set(CATEGORICAL):
        left[column] = pd.to_numeric(left[column], errors="coerce").astype("float64")
        right[column] = pd.to_numeric(right[column], errors="coerce").astype("float64")
    return left, right


def bus2_prediction(
    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]], *, critical_threshold: int,
    critical_weight: float, seed: int,
) -> pd.DataFrame:
    """Exact Bus_model_2 estimator policy, evaluated over Yeonwu rolling folds."""
    frames: list[pd.DataFrame] = []
    for fold_number, (date, train, validation) in enumerate(folds):
        x_train, x_validation = encoded_features(train, validation)
        model = LGBMRegressor(
            objective="huber", alpha=0.9, n_estimators=300, learning_rate=0.05,
            random_state=seed + fold_number, n_jobs=-1, verbosity=-1,
        )
        y_train = np.log1p(train["label_seats"].to_numpy(float))
        weights = np.where(
            train["label_seats"].to_numpy(float) <= critical_threshold,
            critical_weight, 1.0,
        )
        print(f"[Bus_model_2 {date}] train={len(train):,}; valid={len(validation):,}", flush=True)
        model.fit(x_train, y_train, sample_weight=weights, categorical_feature=list(CATEGORICAL))
        prediction = np.clip(np.expm1(model.predict(x_validation)), 0, validation["capacity"].to_numpy(float))
        frame = validation.copy()
        frame["prediction"] = prediction
        frame["candidate"] = "bus_model_2_leak_free"
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def score_partitions(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    from main_model_feature_augmentation import required_metrics
    from latest_main_model_feature_recheck import LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE

    partitions = {
        "complete_selection": predictions["date"].isin(LATEST_COMPLETE_DATES),
        "partial_day_check": predictions["date"].eq(LATEST_PARTIAL_DATE),
        "all_oof": predictions["date"].isin((*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE)),
    }
    pooled_rows, route_rows = [], []
    for partition, mask in partitions.items():
        frame = predictions.loc[mask]
        pooled_rows.append({"partition": partition, "candidate": "bus_model_2_leak_free", **required_metrics(frame)})
        for (route_id, route_name), route in frame.groupby(["route_id", "route_name"], sort=False):
            route_rows.append({"partition": partition, "route_id": route_id, "route_name": route_name, "candidate": "bus_model_2_leak_free", **required_metrics(route)})
    pooled, by_route = pd.DataFrame(pooled_rows), pd.DataFrame(route_rows)
    columns = ["event_balanced_mae", "low_0_10_mae", "full_accuracy", "full_recall", "full_precision", "full_f1"]
    macro = by_route.groupby(["partition", "candidate"], sort=False)[columns].mean().reset_index()
    macro.insert(2, "routes", len(DEFAULT_ROUTES))
    return pooled, by_route, macro


def main() -> int:
    parser = argparse.ArgumentParser(description="Bus_model_2, Yeonwu-equivalent pooled evaluation")
    parser.add_argument("--database", type=Path, default=ROOT / "data/gbis_api_cache.sqlite3")
    parser.add_argument("--cache-dir", type=Path, default=ROOT / "data/analysis_cache/bus_model_2_routes")
    parser.add_argument("--featured-cache", type=Path, default=ROOT / "data/analysis_cache/bus_model_2_features.pkl")
    parser.add_argument("--output-dir", type=Path, default=ROOT / "analysis/bus_model_2_same_metric_results")
    parser.add_argument("--refresh", action="store_true")
    parser.add_argument("--dotenv", type=Path, default=ROOT / ".env")
    parser.add_argument("--api-base-url", default="https://161.33.212.6")
    parser.add_argument("--api-key-env", default="GBIS_API_KEY")
    parser.add_argument("--rebuild-features", action="store_true")
    parser.add_argument("--critical-threshold", type=int, default=3)
    parser.add_argument("--critical-weight", type=float, default=5.0)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    load_embedded_modules()
    refresh_cache(args)
    cutoff = cache_observation_cutoff(args.database)
    print(f"source cutoff: {cutoff}", flush=True)
    data, route_metadata = load_fresh_features(args, cutoff)
    data = add_bus2_features(data)
    from latest_main_model_feature_recheck import rolling_folds
    predictions = bus2_prediction(
        rolling_folds(data), critical_threshold=args.critical_threshold,
        critical_weight=args.critical_weight, seed=args.seed,
    )
    predictions["route_id"] = predictions["event_id"].str.split("::", n=1).str[0]
    predictions["route_name"] = predictions["route_id"].map(DEFAULT_ROUTES)
    pooled, by_route, macro = score_partitions(predictions)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    predictions.to_pickle(args.output_dir / "oof_predictions.pkl")
    pooled.to_csv(args.output_dir / "metrics_pooled.csv", index=False)
    by_route.to_csv(args.output_dir / "metrics_by_route.csv", index=False)
    macro.to_csv(args.output_dir / "metrics_route_macro.csv", index=False)
    (args.output_dir / "summary.json").write_text(json.dumps({
        "source_cutoff": cutoff,
        "included_routes": DEFAULT_ROUTES,
        "excluded_routes": "all non-active routes; no current complete-data coverage",
        "complete_selection_dates": ["2026-08-11", "2026-08-12"],
        "partial_day_check": "2026-08-13; excluded from selection",
        "model": "Bus_model_2 LightGBMRegressor(objective=huber, alpha=0.9, n_estimators=300, learning_rate=0.05), log1p target",
        "critical_weighting": {"threshold": args.critical_threshold, "weight": args.critical_weight},
        "feature_rule": "station_time_pop uses only earlier weekday dates; no raw-row lag features",
        "features": list(BUS2_FEATURES),
        "route_metadata": route_metadata,
        "pooled_metrics": pooled.to_dict(orient="records"),
        "route_macro_metrics": macro.to_dict(orient="records"),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\ncomplete-day pooled metrics")
    print(pooled.loc[pooled.partition.eq("complete_selection")].to_string(index=False))
    print("\ncomplete-day route-macro metrics")
    print(macro.loc[macro.partition.eq("complete_selection")].to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/confidence_interval_comparison.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/confidence_interval_comparison.py
"""Fair pooled comparison of point predictions and calibrated uncertainty.

All candidates use one source/feature cache, identical temporal folds, the same
q05/q50/q95 LightGBM + split-conformal policy, and the same P(Y<=3) head.
The latest calendar day is reported as partial and excluded from selection.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline

from uncertainty_heads import (
    event_weights,
    fit_uncertainty_bundle,
    predict_uncertainty,
    reliability_table,
    uncertainty_metrics,
)
from yeonwu_peer_feature_experiment import (
    DEFAULT_ROUTES,
    ROOT,
    add_peer_features,
    cache_fingerprint,
    load_embedded_modules,
)


MODEL_NAMES = (
    "state_profile",
    "pooled_main",
    "bus_model_2",
    "sanghyuk_ridge",
    "minseok_random_forest",
)
FROZEN_MANIFEST = ROOT / "analysis/frozen_development_manifest.json"


def load_frozen_features(args: argparse.Namespace) -> tuple[pd.DataFrame, dict[str, Any], dict[str, Any]]:
    """Fail closed unless source stat, feature metadata, and freeze manifest agree."""
    if args.refresh or args.rebuild_features:
        raise ValueError("frozen development policy forbids refresh/rebuild")
    manifest = json.loads(args.frozen_manifest.read_text(encoding="utf-8"))
    expected_feature = ROOT / manifest["feature_cache"]
    expected_metadata = ROOT / manifest["feature_metadata"]
    if args.featured_cache.resolve() != expected_feature.resolve():
        raise ValueError(f"feature cache is not the frozen cache: {args.featured_cache}")
    metadata = json.loads(expected_metadata.read_text(encoding="utf-8"))
    for key in ("source_cutoff", "source_fingerprint"):
        if metadata.get(key) != manifest.get(key):
            raise ValueError(
                f"frozen feature metadata mismatch for {key}: "
                f"{metadata.get(key)} != {manifest.get(key)}"
            )
    actual_fingerprint = cache_fingerprint(args.database, manifest["source_cutoff"])
    if actual_fingerprint != manifest["source_fingerprint"]:
        raise ValueError(
            "frozen source cache fingerprint changed; refusing to inspect or rebuild it"
        )
    data = pd.read_pickle(expected_feature)
    if data.attrs.get("source_cutoff") != manifest["source_cutoff"]:
        raise ValueError("frozen feature pickle cutoff does not match manifest")
    route_metadata = data.attrs.get("route_metadata", metadata.get("route_metadata"))
    for route_id, route in route_metadata.items():
        if route.get("source_cutoff") != manifest["source_cutoff"]:
            raise ValueError(f"route {route_id} feature cutoff mismatch")
    return data, route_metadata, manifest


def prepare_comparison_data(data: pd.DataFrame) -> pd.DataFrame:
    from bus_model_2_same_metric_experiment import add_bus2_features

    output = add_bus2_features(add_peer_features(data))
    observed = pd.to_datetime(output["snapshot_time"])
    output["weekday_numeric"] = observed.dt.dayofweek.astype("int8")
    output["is_weekend"] = observed.dt.dayofweek.ge(5).astype("int8")
    output["is_rush_hour"] = observed.dt.hour.isin([6, 7, 8, 9, 16, 17, 18, 19, 20]).astype("int8")
    return output


def candidate_features() -> dict[str, Any]:
    from bus_model_2_same_metric_experiment import BUS2_FEATURES, CATEGORICAL as BUS2_CATEGORICAL
    from model_feasibility import FeatureSet
    from pooled_main_model_overfit_ablation import pooled_candidates
    from yeonwu_peer_feature_experiment import candidates

    state = candidates()["state_profile"]
    main, _ = pooled_candidates()["pooled_37_no_ceiling"]
    bus_categorical = tuple(BUS2_CATEGORICAL)
    bus_numeric = tuple(item for item in BUS2_FEATURES if item not in bus_categorical)
    shared_numeric = (
        "snapshot_remaining_seats",
        "snapshot_station_seq_numeric",
        "target_stop_gap",
        "seat_delta_previous_stop",
        "minutes_since_previous_stop",
        "snapshot_time_sin",
        "snapshot_time_cos",
        "weekday_numeric",
        "is_weekend",
        "is_rush_hour",
    )
    # Weather and arbitrary vehicle IDs from the original Ridge notebook are
    # intentionally omitted: weather is absent from the common cache and IDs do
    # not generalize to unseen pooled-route vehicles.
    ridge = FeatureSet(
        "sanghyuk_ridge_common_task",
        shared_numeric,
        ("route_code", "direction", "snapshot_low_plate_cat", "station_seq_cat"),
    )
    rf = FeatureSet(
        "minseok_random_forest_common_task",
        shared_numeric,
        ("route_code", "snapshot_station_seq_cat", "station_seq_cat"),
    )
    return {
        "state_profile": state,
        "pooled_main": main,
        "bus_model_2": FeatureSet("bus_model_2", bus_numeric, bus_categorical),
        "sanghyuk_ridge": ridge,
        "minseok_random_forest": rf,
    }


def _point_prediction(
    name: str,
    train: pd.DataFrame,
    validation: pd.DataFrame,
    features: Any,
    *,
    validation_date: str,
    seed: int,
    smoke: bool,
) -> np.ndarray:
    from model_feasibility import make_preprocessor

    if name in {"state_profile", "pooled_main"}:
        if smoke:
            # The smoke path validates orchestration without constructing the
            # resource-heavy 3-component production ensemble.
            model = Pipeline([("features", make_preprocessor(features)), ("regressor", Ridge(alpha=10.0))])
            model.fit(train[list(features.columns)], train["label_seats"], regressor__sample_weight=event_weights(train))
            return np.clip(model.predict(validation[list(features.columns)]), 0, validation["capacity"])
        from latest_main_model_overfit_ablation import apply_candidate, train_schema

        raw = train_schema(name, features, [(validation_date, train, validation)], seed=seed)
        scored, _ = apply_candidate(name, features, False, raw)
        return scored["prediction"].to_numpy(dtype=float)
    if name == "bus_model_2":
        from state_profile_bus_model_2_horizon5_experiment import predict_bus2

        return predict_bus2(
            train,
            validation,
            critical_threshold=3,
            critical_weight=5.0,
            seed=seed,
        )["prediction"].to_numpy(dtype=float)
    if name == "sanghyuk_ridge":
        estimator = Ridge(alpha=10.0)
    elif name == "minseok_random_forest":
        estimator = RandomForestRegressor(
            n_estimators=10 if smoke else 300,
            max_depth=18,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=seed,
            n_jobs=-1,
        )
    else:
        raise ValueError(name)
    model = Pipeline([("features", make_preprocessor(features)), ("regressor", estimator)])
    fit_kwargs = {} if name == "sanghyuk_ridge" else {"regressor__sample_weight": event_weights(train)}
    model.fit(train[list(features.columns)], train["label_seats"], **fit_kwargs)
    return np.clip(
        model.predict(validation[list(features.columns)]),
        0,
        validation["capacity"].to_numpy(dtype=float),
    )


def _sample_events(data: pd.DataFrame, per_route: int, seed: int) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for offset, (_, route) in enumerate(data.groupby("route_id", sort=False)):
        events = route.groupby("event_id", sort=False)["label_seats"].first()
        low = events[events.le(3)].index.tolist()
        rng = np.random.default_rng(seed + offset)
        remaining = events.index.difference(low).to_numpy()
        take = max(per_route - min(len(low), per_route // 3), 1)
        chosen = [*low[: max(per_route // 3, 1)], *rng.choice(remaining, min(take, len(remaining)), replace=False)]
        frames.append(route.loc[route["event_id"].isin(chosen)])
    return pd.concat(frames).sort_index()


def _selection_dates(data: pd.DataFrame, requested: list[str] | None) -> tuple[list[str], str]:
    partial_date = str(data["date"].max())
    if requested:
        dates = requested
    else:
        candidates: list[str] = []
        for date, frame in data.loc[data["date"].lt(partial_date)].groupby("date", sort=True):
            if set(frame["route_id"].astype(str)) == set(DEFAULT_ROUTES) and pd.Timestamp(date).dayofweek < 5:
                candidates.append(str(date))
        dates = candidates[-2:]
    if not dates:
        raise ValueError("no completed pooled selection dates")
    return dates, partial_date


def _metric_row(data: pd.DataFrame) -> dict[str, Any]:
    truth = data["label_seats"].to_numpy(dtype=float)
    prediction = data["prediction"].to_numpy(dtype=float)
    weights = event_weights(data)
    low = truth <= 10
    full_true, full_pred = truth == 0, prediction <= 0.5
    point = {
        "rows": int(len(data)),
        "events": int(data["event_id"].nunique()),
        "event_balanced_mae": float(np.average(np.abs(truth - prediction), weights=weights)),
        "low_0_10_events": int(data.loc[low, "event_id"].nunique()),
        "low_0_10_mae": float(np.average(np.abs(truth[low] - prediction[low]), weights=weights[low])) if low.any() else np.nan,
        "full_accuracy": float(accuracy_score(full_true, full_pred, sample_weight=weights)),
        "full_recall": float(recall_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
        "full_precision": float(precision_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
        "full_f1": float(f1_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
        "full_threshold_seats": 0.5,
    }
    return {**point, **uncertainty_metrics(data)}


def metric_tables(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    pooled_rows, route_rows, gap_rows = [], [], []
    gap = pd.cut(
        predictions["target_stop_gap"],
        bins=[0, 2, 5, np.inf],
        labels=["stops_1_2", "stops_3_5", "stops_6_plus"],
    )
    predictions = predictions.assign(stop_gap_band=gap)
    for candidate, frame in predictions.groupby("candidate", sort=False):
        pooled_rows.append({"candidate": candidate, **_metric_row(frame)})
        for (route_id, route_name), route in frame.groupby(["route_id", "route_name"], sort=False):
            route_rows.append({"candidate": candidate, "route_id": route_id, "route_name": route_name, **_metric_row(route)})
        for band, subset in frame.groupby("stop_gap_band", observed=True):
            gap_rows.append({"candidate": candidate, "stop_gap_band": str(band), **_metric_row(subset)})
    pooled, by_route, by_gap = pd.DataFrame(pooled_rows), pd.DataFrame(route_rows), pd.DataFrame(gap_rows)
    metric_columns = [
        "event_balanced_mae", "low_0_10_mae", "full_accuracy", "full_recall",
        "full_precision", "full_f1", "interval_90_coverage", "interval_90_mean_width",
        "interval_90_score", "low_3_brier",
        "lower_bound_90_coverage", "lower_bound_90_mean_point_distance",
        "lower_bound_90_pinball", "point_below_lower_bound_rate",
    ]
    macro = by_route.groupby("candidate", sort=False)[metric_columns].mean().reset_index()
    macro.insert(1, "routes", len(DEFAULT_ROUTES))
    return pooled, by_route, macro, by_gap


def main() -> int:
    parser = argparse.ArgumentParser(description="Pooled model confidence-interval comparison")
    parser.add_argument("--database", type=Path, default=ROOT / "data/gbis_api_cache.sqlite3")
    parser.add_argument("--cache-dir", type=Path, default=ROOT / "data/analysis_cache/state_profile_main_routes")
    parser.add_argument("--featured-cache", type=Path, default=ROOT / "data/analysis_cache/state_profile_main_features.pkl")
    parser.add_argument("--frozen-manifest", type=Path, default=FROZEN_MANIFEST)
    parser.add_argument("--output-dir", type=Path, default=ROOT / "analysis/confidence_interval_comparison_results")
    parser.add_argument("--models", nargs="+", choices=MODEL_NAMES)
    parser.add_argument("--selection-dates", nargs="+")
    parser.add_argument("--refresh", action="store_true")
    parser.add_argument("--dotenv", type=Path, default=ROOT / ".env")
    parser.add_argument("--api-base-url", default="https://161.33.212.6")
    parser.add_argument("--api-key-env", default="GBIS_API_KEY")
    parser.add_argument("--rebuild-features", action="store_true")
    parser.add_argument("--uncertainty-estimators", type=int, default=400)
    parser.add_argument(
        "--uncertainty-targets",
        nargs="+",
        choices=("absolute", "delta", "delta_per_stop"),
        default=["absolute"],
    )
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--smoke-test", action="store_true")
    parser.add_argument("--smoke-events-per-route", type=int, default=80)
    args = parser.parse_args()
    if args.models is None:
        args.models = ["sanghyuk_ridge"] if args.smoke_test else list(MODEL_NAMES)
    load_embedded_modules()
    data, route_metadata, frozen_manifest = load_frozen_features(args)
    cutoff = frozen_manifest["source_cutoff"]
    data = prepare_comparison_data(data)
    selection_dates, partial_date = _selection_dates(data, args.selection_dates)
    features_by_name = candidate_features()
    outputs: list[pd.DataFrame] = []
    calibrations: list[dict[str, Any]] = []
    for fold_number, validation_date in enumerate(selection_dates):
        prior_dates = sorted(
            date for date in data.loc[data["date"].lt(validation_date), "date"].unique()
            if pd.Timestamp(date).dayofweek < 5
        )
        if len(prior_dates) < 2:
            raise ValueError(f"{validation_date}: insufficient prior dates for calibration")
        calibration_date = prior_dates[-1]
        proper_train = data.loc[data["date"].lt(calibration_date) & pd.to_datetime(data["date"]).dt.dayofweek.lt(5)].copy()
        calibration = data.loc[data["date"].eq(calibration_date)].copy()
        point_train = data.loc[data["date"].lt(validation_date) & pd.to_datetime(data["date"]).dt.dayofweek.lt(5)].copy()
        validation = data.loc[data["date"].eq(validation_date)].copy()
        for frame_name, frame in (("train", proper_train), ("calibration", calibration), ("point_train", point_train), ("validation", validation)):
            missing = sorted(set(DEFAULT_ROUTES) - set(frame["route_id"].astype(str)))
            if frame.empty or missing:
                raise ValueError(f"{validation_date} {frame_name} invalid; missing_routes={missing}")
        if args.smoke_test:
            proper_train = _sample_events(proper_train, args.smoke_events_per_route, args.seed)
            calibration = _sample_events(calibration, args.smoke_events_per_route, args.seed + 1)
            point_train = _sample_events(point_train, args.smoke_events_per_route, args.seed + 2)
            validation = _sample_events(validation, args.smoke_events_per_route, args.seed + 3)
        for name in args.models:
            features = features_by_name[name]
            print(f"[{validation_date}][{name}] point", flush=True)
            prediction = _point_prediction(
                name, point_train, validation, features,
                validation_date=validation_date, seed=args.seed + fold_number * 20,
                smoke=args.smoke_test,
            )
            for target_kind in args.uncertainty_targets:
                candidate_name = (
                    f"{name}__{target_kind}"
                    if len(args.uncertainty_targets) > 1
                    else name
                )
                print(
                    f"[{validation_date}][{name}][{target_kind}] uncertainty",
                    flush=True,
                )
                bundle = fit_uncertainty_bundle(
                    proper_train,
                    calibration,
                    numeric=tuple(features.numeric),
                    categorical=tuple(features.categorical),
                    seed=args.seed + 100 + fold_number * 20,
                    n_estimators=10 if args.smoke_test else args.uncertainty_estimators,
                    min_group_events=5 if args.smoke_test else 30,
                    target_kind=target_kind,
                )
                frame = validation.copy()
                frame["prediction"] = prediction
                frame["candidate"] = candidate_name
                frame["base_model"] = name
                frame["uncertainty_target"] = target_kind
                frame["validation_date"] = validation_date
                frame["calibration_date"] = calibration_date
                uncertainty = predict_uncertainty(bundle, validation)
                frame[list(uncertainty.columns)] = uncertainty
                outputs.append(frame)
                calibrations.append({
                    "candidate": candidate_name,
                    "base_model": name,
                    "uncertainty_target": target_kind,
                    "validation_date": validation_date,
                    "calibration_date": calibration_date,
                    "global_adjustment": bundle.global_adjustment,
                    "group_adjustments": bundle.conformal_adjustments,
                    "lower_global_adjustment": bundle.lower_global_adjustment,
                    "lower_group_adjustments": bundle.lower_conformal_adjustments,
                })
    predictions = pd.concat(outputs, ignore_index=True)
    pooled, by_route, macro, by_gap = metric_tables(predictions)
    reliability = pd.concat(
        [reliability_table(frame).assign(candidate=name) for name, frame in predictions.groupby("candidate", sort=False)],
        ignore_index=True,
    )
    args.output_dir.mkdir(parents=True, exist_ok=True)
    predictions.to_pickle(args.output_dir / "predictions.pkl")
    pooled.to_csv(args.output_dir / "metrics_pooled.csv", index=False)
    by_route.to_csv(args.output_dir / "metrics_by_route.csv", index=False)
    macro.to_csv(args.output_dir / "metrics_route_macro.csv", index=False)
    by_gap.to_csv(args.output_dir / "metrics_by_stop_gap.csv", index=False)
    reliability.to_csv(args.output_dir / "low_3_reliability.csv", index=False)
    summary = {
        "source_cutoff": cutoff,
        "source_fingerprint": frozen_manifest["source_fingerprint"],
        "frozen_manifest": frozen_manifest,
        "included_routes": DEFAULT_ROUTES,
        "excluded_routes": "all non-active routes; insufficient current complete-day pooled coverage",
        "selection_dates": selection_dates,
        "partial_day_check": {"date": partial_date, "included_in_selection": False},
        "models": args.models,
        "uncertainty_targets": args.uncertainty_targets,
        "uncertainty_policy": "LightGBM q05/q50/q95 two-sided plus q10 one-sided lower; event-balanced split conformal; P(Y<=3) with Platt calibration",
        "calibrations": calibrations,
        "team_model_adaptation": {
            "bus_model_2": "original Huber/log/low-seat weighting; arrival labels and leak-free historical aggregates",
            "sanghyuk_ridge": "Ridge(alpha=10); common-cache features; weather and arbitrary vehicle ID excluded",
            "minseok_random_forest": "RF depth/leaf/max_features policy; common arrival task and leak-free features",
        },
        "route_cache_metadata": route_metadata,
        "pooled_metrics": pooled.to_dict(orient="records"),
        "route_macro_metrics": macro.to_dict(orient="records"),
        "smoke_test": args.smoke_test,
    }
    (args.output_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print("\npooled metrics")
    print(pooled.to_string(index=False))
    print("\nroute-macro metrics")
    print(macro.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/high_risk_feasibility.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/high_risk_feasibility.py
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from model_feasibility import (
    PLANNING,
    REALTIME,
    build_model_table,
    build_visits,
    cluster_bootstrap_summary,
    evaluate_final_split,
    json_ready,
    load_data,
    prepare_subset,
    trip_level_diagnostics,
)


ROUTE_ID = "219000013"
ROUTE_NAME = "1000"

# 이 범위는 최종 테스트일(8월 7일)을 보기 전에 8월 4~5일 만차 분포로 고정했다.
# 오전은 고양→서울의 만차 집중 구간, 오후는 서울 회차 후 고양 방향 구간이다.
GATE_CONFIG = {
    "weekday_only": True,
    "morning": {"hours": [6, 7, 8], "station_seq_min": 11, "station_seq_max": 20},
    "evening": {"hours": [17, 18, 19], "station_seq_min": 29, "station_seq_max": 40},
}


def high_risk_mask(data: pd.DataFrame) -> pd.Series:
    weekday = data["event_time"].dt.dayofweek.lt(5)
    morning = data["hour"].isin(GATE_CONFIG["morning"]["hours"]) & data[
        "station_seq"
    ].between(
        GATE_CONFIG["morning"]["station_seq_min"],
        GATE_CONFIG["morning"]["station_seq_max"],
    )
    evening = data["hour"].isin(GATE_CONFIG["evening"]["hours"]) & data[
        "station_seq"
    ].between(
        GATE_CONFIG["evening"]["station_seq_min"],
        GATE_CONFIG["evening"]["station_seq_max"],
    )
    return weekday & (morning | evening)


def gate_by_date(data: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    mask = high_risk_mask(data)
    for date, day in data.groupby("date", sort=True):
        gated = day.loc[mask.loc[day.index]]
        positives = int(day["is_full"].sum())
        gated_positives = int(gated["is_full"].sum())
        rows.append(
            {
                "date": date,
                "all_rows": int(len(day)),
                "all_positives": positives,
                "all_prevalence": float(day["is_full"].mean()),
                "gated_rows": int(len(gated)),
                "gated_positives": gated_positives,
                "gate_coverage": float(len(gated) / len(day)),
                "gated_prevalence": float(gated["is_full"].mean()) if len(gated) else 0.0,
                "gate_positive_recall": (
                    float(gated_positives / positives) if positives else None
                ),
            }
        )
    return pd.DataFrame(rows)


def selected_row(metrics: pd.DataFrame, feature_set: str) -> pd.Series:
    candidates = metrics.loc[
        metrics["feature_set"].eq(feature_set)
        & metrics["model"].ne("historical_rate")
    ]
    return candidates.sort_values(
        ["selection_average_precision", "brier"], ascending=[False, True]
    ).iloc[0]


def comparison_table(
    full_metrics: pd.DataFrame,
    gated_metrics: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    for feature_set in [PLANNING.name, REALTIME.name]:
        for scope, metrics in [("full_route", full_metrics), ("high_risk_gate", gated_metrics)]:
            selected = selected_row(metrics, feature_set)
            rows.append(
                {
                    "scope": scope,
                    "feature_set": feature_set,
                    "selected_model": selected["model"],
                    "test_rows": int(selected["rows"]),
                    "test_positives": int(selected["positives"]),
                    "prevalence": float(selected["prevalence"]),
                    "selection_average_precision": float(
                        selected["selection_average_precision"]
                    ),
                    "test_average_precision": float(selected["average_precision"]),
                    "test_brier": float(selected["brier"]),
                    "precision": float(selected["precision_at_threshold"]),
                    "recall": float(selected["recall_at_threshold"]),
                    "alert_rate": float(selected["alert_rate"]),
                }
            )
    return pd.DataFrame(rows)


def make_comparison_plot(comparison: pd.DataFrame, output: Path) -> None:
    labels = [
        f"{row.feature_set}\n{row.scope}"
        for row in comparison.itertuples(index=False)
    ]
    x = range(len(comparison))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    axes[0].bar(x, comparison["precision"] * 100, color=["#94a3b8", "#2563eb"] * 2)
    axes[0].set_xticks(list(x), labels, rotation=15, ha="right")
    axes[0].set_ylabel("Precision (%)")
    axes[0].set_title("Precision at calibration-selected threshold")
    axes[0].axhline(30, color="#dc2626", linestyle="--", linewidth=1, label="30% target")
    axes[0].legend()

    axes[1].bar(x, comparison["recall"] * 100, color=["#94a3b8", "#2563eb"] * 2)
    axes[1].set_xticks(list(x), labels, rotation=15, ha="right")
    axes[1].set_ylabel("Recall (%)")
    axes[1].set_title("Recall at calibration-selected threshold")
    axes[1].axhline(50, color="#dc2626", linestyle="--", linewidth=1, label="50% target")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(output, dpi=160)
    plt.close(fig)


def main() -> int:
    parser = argparse.ArgumentParser(description="1000번 고위험 구간 전용 모델 검증")
    parser.add_argument("--db", type=Path, default=Path("data/gbis.sqlite3"))
    parser.add_argument(
        "--output-dir", type=Path, default=Path("analysis/high_risk_results")
    )
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    args.output_dir.mkdir(parents=True, exist_ok=True)

    locations, stations = load_data(args.db, ROUTE_ID)
    visits = build_visits(locations, stations)
    table, _ = build_model_table(visits, stations)
    data = prepare_subset(table)
    gated = data.loc[high_risk_mask(data)].copy().reset_index(drop=True)

    full_metrics, _, _ = evaluate_final_split(
        data, [PLANNING, REALTIME], args.seed
    )
    gated_metrics, gated_fitted, gated_predictions = evaluate_final_split(
        gated, [PLANNING, REALTIME], args.seed
    )
    strict_gated = gated.loc[gated["label_quality"].eq("A")].copy()
    strict_metrics, _, _ = evaluate_final_split(
        strict_gated, [PLANNING, REALTIME], args.seed
    )
    gate_daily = gate_by_date(data)
    comparison = comparison_table(full_metrics, gated_metrics)

    test = gated.loc[gated["date"].eq("2026-08-07")].copy()
    selected_details: dict[str, Any] = {}
    baseline_predictions = gated_predictions[("planning", "historical_rate")]
    for feature_set in [PLANNING.name, REALTIME.name]:
        selected = selected_row(gated_metrics, feature_set)
        key = (feature_set, str(selected["model"]))
        selected_details[feature_set] = {
            "metrics": json_ready(selected.to_dict()),
            "trip_diagnostics": trip_level_diagnostics(
                test,
                gated_predictions[key],
                float(selected["threshold"]),
            ),
            "trip_cluster_bootstrap": cluster_bootstrap_summary(
                test,
                gated_predictions[key],
                baseline_predictions,
                args.seed,
            ),
        }

    gate_daily.to_csv(args.output_dir / "gate_by_date.csv", index=False)
    gated_metrics.to_csv(args.output_dir / "gated_final_metrics.csv", index=False)
    strict_metrics.to_csv(args.output_dir / "gated_strict_A_metrics.csv", index=False)
    comparison.to_csv(args.output_dir / "scope_comparison.csv", index=False)
    make_comparison_plot(comparison, args.output_dir / "precision_recall_comparison.png")

    test_gate = gate_daily.loc[gate_daily["date"].eq("2026-08-07")].iloc[0]
    planning_selected = selected_row(gated_metrics, PLANNING.name)
    realtime_selected = selected_row(gated_metrics, REALTIME.name)
    result = {
        "route_id": ROUTE_ID,
        "route_name": ROUTE_NAME,
        "gate_config": GATE_CONFIG,
        "gate_selection_period": ["2026-08-04", "2026-08-05"],
        "calibration_date": "2026-08-06",
        "untouched_test_date": "2026-08-07",
        "test_gate": json_ready(test_gate.to_dict()),
        "selected_models": selected_details,
        "strict_A_selected": {
            feature_set: json_ready(selected_row(strict_metrics, feature_set).to_dict())
            for feature_set in [PLANNING.name, REALTIME.name]
        },
        "product_kpi": {"minimum_precision": 0.30, "minimum_recall": 0.50},
        "verdict": {
            "planning_passes_kpi": bool(
                planning_selected["precision_at_threshold"] >= 0.30
                and planning_selected["recall_at_threshold"] >= 0.50
            ),
            "realtime_passes_kpi": bool(
                realtime_selected["precision_at_threshold"] >= 0.30
                and realtime_selected["recall_at_threshold"] >= 0.50
            ),
            "planning": (
                "게이트는 양성률을 높였지만 개별 차량을 구분할 동적 신호가 없어 "
                "계획형 정밀도는 개선되지 않았다."
            ),
            "realtime": (
                "상류 좌석을 포함하면 정밀도가 약 두 배 개선되지만 현재 제품 KPI에는 미달한다."
            ),
        },
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(result), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps(json_ready(result), ensure_ascii=False, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/hypothesis_model_search.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/hypothesis_model_search.py
from __future__ import annotations

import argparse
import hashlib
import json
import sqlite3
import time
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any, Callable, Iterable

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.pipeline import Pipeline

from all_prearrival_seat_regression import (
    DYNAMIC_ALL_PREARRIVAL,
    ENGINEERED_ALL_PREARRIVAL,
    build_all_prearrival_table,
    clip_seats,
    event_bias,
    event_weights,
)
from model_feasibility import (
    FeatureSet,
    _as_records,
    build_model_table,
    build_visits,
    json_ready,
    make_preprocessor,
)
from tminus_feasibility import ROUTE_ID, ROUTE_NAME, prepare_raw_locations


CACHE_VERSION = 6
DEFAULT_DEVELOPMENT_DATES = (
    "2026-08-04",
    "2026-08-05",
    "2026-08-06",
    "2026-08-07",
    "2026-08-10",
)
DEFAULT_STRESS_DATES = ("2026-08-08", "2026-08-09")
DEFAULT_TEST_DATE = "2026-08-11"
HISTORICAL_FEATURES = (
    "historical_pair_delta",
    "historical_pair_log_count",
    "historical_pair_time_delta",
    "historical_pair_time_log_count",
)
ROUTE_PROFILE_FEATURES = (
    "route_profile_delta",
    "route_profile_seats",
    "route_profile_uncertainty",
    "route_profile_fallback_share",
)
CAPACITY_44_FEATURES = (
    "observed_ceiling_capacity",
    "observed_ceiling_load_ratio",
    "observed_ceiling_load_gap",
)
OBSERVED_SEAT_CEILING_PARAM = "observed_seat_ceiling_clip"
OBSERVED_SEAT_CEILING_EVIDENCE_CUTOFF = "2026-08-04"
PREVIOUS_BUS_SEAT_FEATURES = (
    "previous_bus_departure_seats",
    "previous_bus_departure_age_minutes",
    "previous_bus_departure_missing",
)
NORMALIZED_PREVIOUS_BUS_FEATURES = (
    "previous_bus_departure_capacity",
    "previous_bus_departure_load_ratio",
    "previous_bus_same_capacity",
    "previous_bus_departure_age_minutes",
    "previous_bus_projected_headway_minutes",
    "previous_bus_freshness_exp",
    "previous_bus_departure_missing",
    "previous_3_bus_departure_load_ratio_mean",
    "previous_3_bus_departure_load_ratio_std",
    "previous_3_bus_departure_load_ratio_trend",
    "previous_bus_was_full_normalized",
    "previous_bus_was_low_10pct",
)
PAIR_ALL_PREARRIVAL = FeatureSet(
    name="all_prearrival_source_target_pair",
    numeric=DYNAMIC_ALL_PREARRIVAL.numeric,
    categorical=(
        *DYNAMIC_ALL_PREARRIVAL.categorical,
        "snapshot_target_pair_cat",
    ),
)
HISTORICAL_ALL_PREARRIVAL = FeatureSet(
    name="all_prearrival_historical_profiles",
    numeric=(*PAIR_ALL_PREARRIVAL.numeric, *HISTORICAL_FEATURES),
    categorical=PAIR_ALL_PREARRIVAL.categorical,
)
ROUTE_PROFILE_ALL_PREARRIVAL = FeatureSet(
    name="all_prearrival_route_profile",
    numeric=(*DYNAMIC_ALL_PREARRIVAL.numeric, *ROUTE_PROFILE_FEATURES),
    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,
)
CAPACITY_44_ALL_PREARRIVAL = FeatureSet(
    name="all_prearrival_observed_44_70_capacity",
    numeric=(*DYNAMIC_ALL_PREARRIVAL.numeric, *CAPACITY_44_FEATURES),
    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,
)
PREVIOUS_BUS_SEAT_ALL_PREARRIVAL = FeatureSet(
    name="all_prearrival_previous_bus_seat",
    numeric=(*DYNAMIC_ALL_PREARRIVAL.numeric, *PREVIOUS_BUS_SEAT_FEATURES),
    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,
)
PREVIOUS_BUS_SEAT_CAPACITY_ALL_PREARRIVAL = FeatureSet(
    name="all_prearrival_previous_bus_seat_observed_44_70_capacity",
    numeric=(
        *DYNAMIC_ALL_PREARRIVAL.numeric,
        *PREVIOUS_BUS_SEAT_FEATURES,
        *CAPACITY_44_FEATURES,
    ),
    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,
)
NORMALIZED_PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL = FeatureSet(
    name="all_prearrival_normalized_previous_bus_observed_44_70_capacity",
    numeric=(
        *DYNAMIC_ALL_PREARRIVAL.numeric,
        *NORMALIZED_PREVIOUS_BUS_FEATURES,
        *CAPACITY_44_FEATURES,
    ),
    categorical=DYNAMIC_ALL_PREARRIVAL.categorical,
)
PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL = FeatureSet(
    name="all_prearrival_previous_bus_observed_44_70_capacity",
    numeric=(*ENGINEERED_ALL_PREARRIVAL.numeric, *CAPACITY_44_FEATURES),
    categorical=ENGINEERED_ALL_PREARRIVAL.categorical,
)


@dataclass
class Candidate:
    name: str
    stage: str
    why: str
    if_works: str
    if_fails: str
    model_kind: str
    target_kind: str = "delta"
    feature_variant: str = "dynamic"
    low_weight: float = 1.0
    weighting_kind: str = "event"
    far_weight: float = 1.0
    far_threshold: int = 6
    gap_weight_power: float = 0.0
    params: dict[str, Any] = field(default_factory=dict)


@dataclass
class EnsembleSpec:
    """앙상블 구성과 미래 추론에 쓰는 배포용 bias를 보관한다.

    ``bias``는 모든 개발 OOF residual로 추정한 배포용 보정값이다. 모델
    선택용 OOF prediction에는 이 값을 적용하지 않고, 각 날짜보다 앞선 OOF
    fold에서만 추정한 ``oof_biases``를 적용한다.
    """

    name: str
    kind: str
    components: tuple[str, ...]
    params: dict[str, float]
    bias: float = 0.0
    oof_biases: dict[str, float] = field(default_factory=dict)


def load_analysis_data(
    db_path: Path,
    route_id: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """수집기 DB와 API 로컬 캐시를 같은 분석 스키마로 읽는다."""
    uri = f"file:{db_path.resolve()}?mode=ro"
    with sqlite3.connect(uri, uri=True) as connection:
        tables = {
            str(row[0])
            for row in connection.execute(
                "SELECT name FROM sqlite_master WHERE type = 'table'"
            )
        }
        if "bus_locations" in tables:
            location_query = """
                SELECT run_id, observed_at_kst, query_time, route_id, vehicle_id,
                       plate_no, station_id, station_seq, remaining_seats,
                       low_plate, state_code
                FROM bus_locations
                WHERE route_id = ?
                ORDER BY vehicle_id, observed_at_kst, run_id
            """
        elif "location_history" in tables:
            location_query = """
                SELECT rowid AS run_id, observed_at AS observed_at_kst,
                       query_time, route_id, vehicle_id, plate_no, station_id,
                       station_seq, remaining_seats, low_plate, state_code
                FROM location_history
                WHERE route_id = ?
                ORDER BY vehicle_id, observed_at, rowid
            """
        else:
            raise ValueError(
                f"{db_path}에 bus_locations 또는 location_history가 없습니다."
            )
        locations = pd.read_sql_query(
            location_query, connection, params=(route_id,)
        )
        stations = pd.read_sql_query(
            """
            SELECT route_id, station_id, station_seq, station_name, mobile_no,
                   region_name, x, y, center_yn
            FROM route_stations
            WHERE route_id = ?
            ORDER BY station_seq
            """,
            connection,
            params=(route_id,),
        )
    if locations.empty or stations.empty:
        raise ValueError(f"노선 {route_id}의 위치 또는 정류장 데이터가 없습니다.")
    locations["observed_at"] = pd.to_datetime(locations["observed_at_kst"])
    return locations, stations


def cache_fingerprint(db_path: Path) -> dict[str, Any]:
    stat = db_path.stat()
    return {
        "cache_version": CACHE_VERSION,
        "source": str(db_path.resolve()),
        "source_size": int(stat.st_size),
        "source_mtime_ns": int(stat.st_mtime_ns),
        "route_id": ROUTE_ID,
        "label_quality": "A",
    }


def cache_payload_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as payload:
        for chunk in iter(lambda: payload.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def cache_payload_hashes_match(
    metadata: dict[str, Any],
    snapshot_path: Path,
    flow_path: Path,
) -> bool:
    """Accept a cache commit marker only when both payload bytes match it."""

    payload_hashes = metadata.get("payload_sha256")
    if not isinstance(payload_hashes, dict):
        return False
    expected_snapshot = str(payload_hashes.get("snapshots", ""))
    expected_flow = str(payload_hashes.get("stop_flows", ""))
    if len(expected_snapshot) != 64 or len(expected_flow) != 64:
        return False
    try:
        return (
            cache_payload_sha256(snapshot_path) == expected_snapshot
            and cache_payload_sha256(flow_path) == expected_flow
        )
    except OSError:
        return False


def audit_route_output_support(
    db_path: Path,
    route_id: str,
    *,
    evidence_cutoff: str = OBSERVED_SEAT_CEILING_EVIDENCE_CUTOFF,
) -> dict[str, Any]:
    """Audit the raw route feed before enabling the empirical point projection.

    Candidate eligibility uses only rows on or before the frozen evidence
    cutoff. Later rows are a deployment-safety monitor and must never influence
    selection; runtime inference falls back per row when it observes a changed
    support.
    """
    uri = f"file:{db_path.resolve()}?mode=ro"
    with sqlite3.connect(uri, uri=True) as connection:
        tables = {
            str(row[0])
            for row in connection.execute(
                "SELECT name FROM sqlite_master WHERE type = 'table'"
            )
        }
        if "bus_locations" in tables:
            table = "bus_locations"
            observed_column = "observed_at_kst"
        elif "location_history" in tables:
            table = "location_history"
            observed_column = "observed_at"
        else:
            raise ValueError(
                f"{db_path}에 bus_locations 또는 location_history가 없습니다."
            )
        raw = pd.read_sql_query(
            f"""
            SELECT vehicle_id, low_plate, remaining_seats,
                   {observed_column} AS observed_at
            FROM {table}
            WHERE route_id = ? AND remaining_seats >= 0
            """,
            connection,
            params=(route_id,),
        )
    if raw.empty:
        raise ValueError("Route 1000 output-support audit에 유효 좌석 관측이 없습니다.")

    category = pd.to_numeric(raw["low_plate"], errors="coerce")
    integral = category.notna() & np.isclose(category % 1, 0.0)
    supported = integral & category.isin([0.0, 2.0])
    unknown_rows = int((~supported).sum())
    seats = raw["remaining_seats"].to_numpy(dtype=float)
    ordinary = supported.to_numpy() & category.eq(0).to_numpy()
    double_decker = supported.to_numpy() & category.eq(2).to_numpy()
    violation = (ordinary & (seats > 44.0)) | (double_decker & (seats > 70.0))
    dates = raw["observed_at"].astype(str).str.slice(0, 10)
    evidence = dates.le(evidence_cutoff).to_numpy()
    unsupported = (~supported).to_numpy()

    def category_profile(code: int, ceiling: float, mask: np.ndarray) -> dict[str, Any]:
        scoped = raw.loc[mask]
        evidence_mask = mask & evidence
        evidence_seats = seats[evidence_mask]
        return {
            "category": code,
            "ceiling": ceiling,
            "rows": int(mask.sum()),
            "vehicles": int(scoped["vehicle_id"].nunique()),
            "maximum": float(seats[mask].max()) if mask.any() else None,
            "ceiling_hits": int(np.sum(seats[mask] == ceiling)),
            "evidence_rows": int(evidence_mask.sum()),
            "evidence_maximum": (
                float(evidence_seats.max()) if evidence_mask.any() else None
            ),
            "evidence_ceiling_hits": int(np.sum(evidence_seats == ceiling)),
        }

    profiles = {
        "0": category_profile(0, 44.0, ordinary),
        "2": category_profile(2, 70.0, double_decker),
    }
    selection_eligible = bool(
        not (unsupported & evidence).any()
        and not (violation & evidence).any()
        and all(
            profile["evidence_ceiling_hits"] > 0
            for profile in profiles.values()
        )
    )
    return {
        "scope": f"route_{route_id}_current_fleet_only",
        "source": str(db_path.resolve()),
        "evidence_cutoff": evidence_cutoff,
        "valid_seat_rows": int(len(raw)),
        "observed_at_min": str(raw["observed_at"].min()),
        "observed_at_max": str(raw["observed_at"].max()),
        "unknown_or_unsupported_category_rows": unknown_rows,
        "above_ceiling_rows": int(violation.sum()),
        "selection_evidence_unknown_or_unsupported_rows": int(
            (unsupported & evidence).sum()
        ),
        "selection_evidence_above_ceiling_rows": int(
            (violation & evidence).sum()
        ),
        "post_cutoff_unknown_or_unsupported_rows": int(
            (unsupported & ~evidence).sum()
        ),
        "post_cutoff_above_ceiling_rows": int((violation & ~evidence).sum()),
        "by_category": profiles,
        "selection_candidate_eligible": selection_eligible,
        "current_deployment_support_safe": bool(
            unknown_rows == 0 and not violation.any()
        ),
        "selection_rule": "use_only_rows_on_or_before_evidence_cutoff",
        "post_cutoff_rule": "monitor_only_never_select; runtime row fallback or fail",
    }


def load_or_build_snapshots(
    db_path: Path,
    cache_path: Path,
    *,
    rebuild: bool,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    metadata_path = cache_path.with_suffix(cache_path.suffix + ".json")
    flow_cache_path = cache_path.with_name(
        cache_path.stem + "_stop_flows" + cache_path.suffix
    )
    fingerprint = cache_fingerprint(db_path)
    if (
        not rebuild
        and cache_path.exists()
        and flow_cache_path.exists()
        and metadata_path.exists()
    ):
        existing = json.loads(metadata_path.read_text(encoding="utf-8"))
        if (
            existing.get("fingerprint") == fingerprint
            and cache_payload_hashes_match(
                existing, cache_path, flow_cache_path
            )
        ):
            return pd.read_pickle(cache_path), pd.read_pickle(flow_cache_path), {
                "cache_hit": True,
                "cache_path": str(cache_path),
                **existing,
            }

    locations, stations = load_analysis_data(db_path, ROUTE_ID)
    visits = build_visits(locations, stations)
    table, turnaround_seq = build_model_table(
        visits, stations, label_target="arrival"
    )
    source = table.loc[
        table["is_peak"]
        & table["label_quality"].eq("A")
        & table["label_seats"].notna()
    ].copy()
    _, by_vehicle = prepare_raw_locations(locations, turnaround_seq)
    snapshots = build_all_prearrival_table(
        source,
        visits,
        by_vehicle,
        turnaround_seq=turnaround_seq,
        max_seq=int(stations["station_seq"].max()),
    )
    stop_flows = visits.loc[
        (~visits["is_pass_node"])
        & visits["observed_arrival_seats"].ge(0)
        & visits["departure_seats"].ge(0)
        & visits["arrival_seen"].notna()
        & visits["departure_seen"].notna()
    ].copy()
    stop_flows["direction"] = np.where(
        stop_flows["station_seq"].le(turnaround_seq), "to_city", "return"
    )
    stop_flows["date"] = stop_flows["arrival_seen"].dt.date.astype(str)
    stop_flows["time_bin_2h"] = (
        stop_flows["arrival_seen"].dt.hour // 2 * 2
    ).astype(int)
    stop_flows["stop_net"] = (
        stop_flows["departure_seats"]
        - stop_flows["observed_arrival_seats"]
    ).astype(float)
    stop_flows = stop_flows[
        [
            "vehicle_id",
            "trip_id",
            "station_seq",
            "direction",
            "date",
            "arrival_seen",
            "departure_seen",
            "time_bin_2h",
            "observed_arrival_seats",
            "departure_seats",
            "stop_net",
        ]
    ].sort_values(["arrival_seen", "station_seq", "vehicle_id"])
    stop_flows.attrs["pass_node_sequences"] = sorted(
        visits.loc[visits["is_pass_node"], "station_seq"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    snapshots.to_pickle(cache_path)
    stop_flows.to_pickle(flow_cache_path)
    metadata = {
        "fingerprint": fingerprint,
        "payload_sha256": {
            "snapshots": cache_payload_sha256(cache_path),
            "stop_flows": cache_payload_sha256(flow_cache_path),
        },
        "source_route_rows": int(len(locations)),
        "source_observed_at_min": locations["observed_at"].min().isoformat(),
        "source_observed_at_max": locations["observed_at"].max().isoformat(),
        "rows": int(len(snapshots)),
        "events": int(snapshots["event_id"].nunique()),
        "stop_flow_rows": int(len(stop_flows)),
        "dates": sorted(snapshots["date"].unique().tolist()),
    }
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return snapshots, stop_flows, {
        "cache_hit": False,
        "cache_path": str(cache_path),
        **metadata,
    }


def smoothed_history_feature(
    train: pd.DataFrame,
    target: pd.DataFrame,
    keys: tuple[str, ...],
    *,
    alpha: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    delta = (
        train["label_seats"].to_numpy(dtype=float)
        - train["snapshot_remaining_seats"].to_numpy(dtype=float)
    )
    total_sum = float(delta.sum())
    total_count = len(delta)
    global_mean = total_sum / max(total_count, 1)
    working = train.loc[:, list(keys)].copy()
    working["_delta"] = delta
    stats = working.groupby(list(keys), dropna=False)["_delta"].agg(["sum", "count"])

    train_index = pd.MultiIndex.from_frame(train.loc[:, list(keys)])
    group_sum = stats["sum"].reindex(train_index).to_numpy(dtype=float)
    group_count = stats["count"].reindex(train_index).to_numpy(dtype=float)
    global_loo = np.where(
        total_count > 1,
        (total_sum - delta) / (total_count - 1),
        global_mean,
    )
    train_value = (
        group_sum - delta + alpha * global_loo
    ) / np.maximum(group_count - 1 + alpha, 1)
    train_count = np.maximum(group_count - 1, 0)

    target_index = pd.MultiIndex.from_frame(target.loc[:, list(keys)])
    target_sum = stats["sum"].reindex(target_index).to_numpy(dtype=float)
    target_count = stats["count"].reindex(target_index).to_numpy(dtype=float)
    seen = np.isfinite(target_sum) & np.isfinite(target_count)
    target_value = np.full(len(target), global_mean, dtype=float)
    target_value[seen] = (
        target_sum[seen] + alpha * global_mean
    ) / (target_count[seen] + alpha)
    target_count = np.where(seen, target_count, 0.0)
    return train_value, np.log1p(train_count), target_value, np.log1p(target_count)


def add_historical_profiles(
    train: pd.DataFrame,
    target: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """학습 행은 leave-one-row-out, 미래 행은 train-only 통계로 만든다."""
    train_output = add_source_target_pair(train)
    target_output = add_source_target_pair(target)
    specs = [
        (
            "historical_pair",
            ("station_seq_cat", "snapshot_station_seq_cat", "direction"),
            20.0,
        ),
        (
            "historical_pair_time",
            (
                "station_seq_cat",
                "snapshot_station_seq_cat",
                "direction",
                "snapshot_time_bin_30",
            ),
            30.0,
        ),
    ]
    for prefix, keys, alpha in specs:
        train_value, train_count, target_value, target_count = (
            smoothed_history_feature(train, target, keys, alpha=alpha)
        )
        train_output[f"{prefix}_delta"] = train_value
        train_output[f"{prefix}_log_count"] = train_count
        target_output[f"{prefix}_delta"] = target_value
        target_output[f"{prefix}_log_count"] = target_count
    return train_output, target_output


def add_source_target_pair(data: pd.DataFrame) -> pd.DataFrame:
    output = data.copy()
    output["snapshot_target_pair_cat"] = (
        output["snapshot_station_seq_cat"].astype(str)
        + "->"
        + output["station_seq_cat"].astype(str)
        + "|"
        + output["direction"].astype(str)
    )
    return output


def add_observed_capacity_features(data: pd.DataFrame) -> pd.DataFrame:
    output = data.copy()
    capacity = np.where(
        output["snapshot_low_plate_cat"].astype(str).eq("2"), 70.0, 44.0
    )
    output["observed_ceiling_capacity"] = capacity
    output["observed_ceiling_load_ratio"] = (
        1
        - output["snapshot_remaining_seats"].to_numpy(dtype=float) / capacity
    ).clip(0, 1)
    output["observed_ceiling_load_gap"] = (
        output["observed_ceiling_load_ratio"]
        * output["target_stop_gap"].to_numpy(dtype=float)
    )
    return output


def observed_seat_ceiling(data: pd.DataFrame) -> np.ndarray:
    """Return Route 1000's repeatedly observed seat-count support.

    Route 1000's ordinary vehicles expose at most 44 passenger seats even though
    the legacy nominal ``capacity`` column is 45.  Category 2 vehicles expose 70.
    This is a route-scoped output constraint, not a GBIS-wide physical invariant:
    other locally cached routes contain ordinary vehicles with 45+ seats.  Fail
    closed if Route 1000 ever exposes a new vehicle category instead of silently
    applying the current fleet mapping to it.
    """
    if "snapshot_low_plate_cat" not in data.columns:
        raise ValueError(
            "observed seat-ceiling clipping requires snapshot_low_plate_cat"
        )
    category = data["snapshot_low_plate_cat"].astype("string")
    unsupported = sorted(category.dropna().loc[~category.isin(["0", "2"])].unique())
    if category.isna().any() or unsupported:
        raise ValueError(
            "Route 1000 observed seat-ceiling clipping supports only vehicle "
            f"categories 0 and 2; unsupported={unsupported}, "
            f"missing={int(category.isna().sum())}"
        )
    return np.where(category.eq("2"), 70.0, 44.0)


def postprocess_ensemble_prediction(
    spec: EnsembleSpec,
    prediction: np.ndarray,
    data: pd.DataFrame,
) -> np.ndarray:
    """Apply the exact deployment-time support constraints for an ensemble."""
    output = clip_seats(np.asarray(prediction, dtype=float), data["capacity"])
    if float(spec.params.get(OBSERVED_SEAT_CEILING_PARAM, 0.0)) != 0.0:
        if "snapshot_remaining_seats" not in data.columns:
            raise ValueError(
                "observed seat-ceiling clipping requires snapshot_remaining_seats"
            )
        empirical_ceiling = observed_seat_ceiling(data)
        current = data["snapshot_remaining_seats"].to_numpy(dtype=float)
        nominal = data["capacity"].to_numpy(dtype=float)
        # A newly observed counterexample means the current-fleet assumption has
        # changed. Disable the projection for that row rather than forcing a
        # stale 44/70 ceiling. Later observations remain monitor-only and never
        # retroactively change the pre-cutoff H8 candidate definition.
        effective_ceiling = np.where(
            current > empirical_ceiling, nominal, empirical_ceiling
        )
        output = np.minimum(output, effective_ceiling)
    return output


def route_profile_values(
    flows: pd.DataFrame,
    target: pd.DataFrame,
    *,
    alpha: float = 10.0,
) -> pd.DataFrame:
    """과거 직접 도착→출발 순좌석변화를 목표 직전까지 누적한다."""
    output = pd.DataFrame(index=target.index)
    for column in ROUTE_PROFILE_FEATURES:
        output[column] = 0.0
    if target.empty or flows.empty:
        output["route_profile_seats"] = target[
            "snapshot_remaining_seats"
        ].to_numpy(dtype=float)
        output["route_profile_fallback_share"] = 1.0
        return output

    weekday_flows = flows.loc[
        pd.to_datetime(flows["date"]).dt.dayofweek.lt(5)
    ].copy()
    if weekday_flows.empty:
        output["route_profile_seats"] = target[
            "snapshot_remaining_seats"
        ].to_numpy(dtype=float)
        output["route_profile_fallback_share"] = 1.0
        return output

    global_mean = float(weekday_flows["stop_net"].mean())
    global_variance = float(weekday_flows["stop_net"].var(ddof=0))
    station_stats = weekday_flows.groupby(
        ["station_seq", "direction"], sort=False
    )["stop_net"].agg(["mean", "var", "count"])
    cell_stats = weekday_flows.groupby(
        ["station_seq", "direction", "time_bin_2h"], sort=False
    )["stop_net"].agg(["mean", "var", "count"])

    station_mean = station_stats["mean"].to_dict()
    station_variance = station_stats["var"].fillna(global_variance).to_dict()
    cell_mean: dict[tuple[int, str, int], float] = {}
    cell_variance: dict[tuple[int, str, int], float] = {}
    pass_node_sequences = {
        int(value) for value in flows.attrs.get("pass_node_sequences", [])
    }
    for key, row in cell_stats.iterrows():
        station_key = (int(key[0]), str(key[1]))
        prior_mean = float(station_mean.get(station_key, global_mean))
        prior_variance = float(
            station_variance.get(station_key, global_variance)
        )
        count = float(row["count"])
        cell_mean[(int(key[0]), str(key[1]), int(key[2]))] = float(
            (count * float(row["mean"]) + alpha * prior_mean)
            / (count + alpha)
        )
        raw_variance = (
            float(row["var"]) if pd.notna(row["var"]) else prior_variance
        )
        cell_variance[(int(key[0]), str(key[1]), int(key[2]))] = float(
            (count * raw_variance + alpha * prior_variance)
            / (count + alpha)
        )

    deltas = np.zeros(len(target), dtype=float)
    uncertainties = np.zeros(len(target), dtype=float)
    fallback_shares = np.zeros(len(target), dtype=float)
    for position, row in enumerate(target.itertuples(index=False)):
        day_of_week = int(row.snapshot_day_of_week)
        if day_of_week >= 5:
            fallback_shares[position] = 1.0
            continue
        start = int(row.snapshot_station_seq)
        if str(row.target_state_cat) != "1":
            start += 1
        stop = int(row.station_seq_cat)
        if start >= stop:
            continue
        time_bin = int(row.snapshot_time.hour // 2 * 2)
        means: list[float] = []
        variances: list[float] = []
        fallback_count = 0
        for station_seq in range(start, stop):
            if station_seq in pass_node_sequences:
                means.append(0.0)
                variances.append(0.0)
                continue
            cell_key = (station_seq, str(row.direction), time_bin)
            station_key = (station_seq, str(row.direction))
            if cell_key in cell_mean:
                means.append(cell_mean[cell_key])
                variances.append(cell_variance[cell_key])
            elif station_key in station_mean:
                means.append(float(station_mean[station_key]))
                variances.append(
                    float(station_variance.get(station_key, global_variance))
                )
                fallback_count += 1
            else:
                means.append(global_mean)
                variances.append(global_variance)
                fallback_count += 1
        deltas[position] = float(np.sum(means))
        uncertainties[position] = float(np.sqrt(np.maximum(np.sum(variances), 0)))
        fallback_shares[position] = fallback_count / max(len(means), 1)

    current = target["snapshot_remaining_seats"].to_numpy(dtype=float)
    capacity = target["capacity"].to_numpy(dtype=float)
    output["route_profile_delta"] = deltas
    output["route_profile_seats"] = np.minimum(
        np.maximum(current + deltas, 0), capacity
    )
    output["route_profile_uncertainty"] = uncertainties
    output["route_profile_fallback_share"] = fallback_shares
    return output


def add_route_profile_frames(
    train: pd.DataFrame,
    target: pd.DataFrame,
    flows: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_output = train.copy()
    for column in ROUTE_PROFILE_FEATURES:
        train_output[column] = 0.0
    for date, indices in train.groupby("date", sort=False).groups.items():
        # Service-time simulation: a training row may only use flows observed on
        # strictly earlier dates.  Leave-one-date-out would let older rows see
        # later training dates and creates a train/serve distribution mismatch.
        crossfit_flows = flows.loc[flows["date"].lt(str(date))]
        values = route_profile_values(crossfit_flows, train.loc[indices])
        train_output.loc[indices, list(ROUTE_PROFILE_FEATURES)] = values[
            list(ROUTE_PROFILE_FEATURES)
        ].to_numpy()
    target_output = target.copy()
    for date, indices in target.groupby("date", sort=False).groups.items():
        historical_flows = flows.loc[flows["date"].lt(str(date))]
        values = route_profile_values(historical_flows, target.loc[indices])
        target_output.loc[indices, list(ROUTE_PROFILE_FEATURES)] = values[
            list(ROUTE_PROFILE_FEATURES)
        ].to_numpy()
    return train_output, target_output


def feature_frames(
    train: pd.DataFrame,
    target: pd.DataFrame,
    variant: str,
    flows: pd.DataFrame | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, FeatureSet]:
    if variant == "dynamic":
        return train, target, DYNAMIC_ALL_PREARRIVAL
    if variant == "source_target_pair":
        return (
            add_source_target_pair(train),
            add_source_target_pair(target),
            PAIR_ALL_PREARRIVAL,
        )
    if variant == "capacity_44_70":
        return (
            add_observed_capacity_features(train),
            add_observed_capacity_features(target),
            CAPACITY_44_ALL_PREARRIVAL,
        )
    if variant == "previous_bus_seat":
        return train, target, PREVIOUS_BUS_SEAT_ALL_PREARRIVAL
    if variant == "previous_bus_seat_capacity_44_70":
        return (
            add_observed_capacity_features(train),
            add_observed_capacity_features(target),
            PREVIOUS_BUS_SEAT_CAPACITY_ALL_PREARRIVAL,
        )
    if variant == "previous_bus_normalized_capacity_44_70":
        return (
            add_observed_capacity_features(train),
            add_observed_capacity_features(target),
            NORMALIZED_PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL,
        )
    if variant == "previous_bus_capacity_44_70":
        return (
            add_observed_capacity_features(train),
            add_observed_capacity_features(target),
            PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL,
        )
    if variant == "previous_bus":
        return train, target, ENGINEERED_ALL_PREARRIVAL
    if variant == "historical":
        historical_train, historical_target = add_historical_profiles(train, target)
        return historical_train, historical_target, HISTORICAL_ALL_PREARRIVAL
    if variant == "route_profile":
        if flows is None:
            raise ValueError("route_profile feature에는 과거 stop flow가 필요합니다.")
        profile_train, profile_target = add_route_profile_frames(
            train, target, flows
        )
        return profile_train, profile_target, ROUTE_PROFILE_ALL_PREARRIVAL
    raise ValueError(f"알 수 없는 feature variant: {variant}")


def make_regressor(candidate: Candidate, seed: int) -> Pipeline:
    _, _, feature_set = feature_frames(
        pd.DataFrame(columns=DYNAMIC_ALL_PREARRIVAL.columns),
        pd.DataFrame(columns=DYNAMIC_ALL_PREARRIVAL.columns),
        "dynamic",
    )
    if candidate.feature_variant == "previous_bus":
        feature_set = ENGINEERED_ALL_PREARRIVAL
    elif candidate.feature_variant == "previous_bus_seat":
        feature_set = PREVIOUS_BUS_SEAT_ALL_PREARRIVAL
    elif candidate.feature_variant == "previous_bus_seat_capacity_44_70":
        feature_set = PREVIOUS_BUS_SEAT_CAPACITY_ALL_PREARRIVAL
    elif candidate.feature_variant == "previous_bus_normalized_capacity_44_70":
        feature_set = NORMALIZED_PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL
    elif candidate.feature_variant == "previous_bus_capacity_44_70":
        feature_set = PREVIOUS_BUS_CAPACITY_ALL_PREARRIVAL
    elif candidate.feature_variant == "source_target_pair":
        feature_set = PAIR_ALL_PREARRIVAL
    elif candidate.feature_variant == "capacity_44_70":
        feature_set = CAPACITY_44_ALL_PREARRIVAL
    elif candidate.feature_variant == "historical":
        feature_set = HISTORICAL_ALL_PREARRIVAL
    elif candidate.feature_variant == "route_profile":
        feature_set = ROUTE_PROFILE_ALL_PREARRIVAL
    preprocessor = clone(make_preprocessor(feature_set))
    if candidate.model_kind == "hgb":
        model = HistGradientBoostingRegressor(
            random_state=seed,
            **candidate.params,
        )
    elif candidate.model_kind == "extra_trees":
        model = ExtraTreesRegressor(
            n_jobs=-1,
            random_state=seed,
            **candidate.params,
        )
    elif candidate.model_kind == "random_forest":
        model = RandomForestRegressor(
            n_jobs=-1,
            random_state=seed,
            **candidate.params,
        )
    elif candidate.model_kind == "lightgbm":
        model = lgb.LGBMRegressor(
            n_jobs=-1,
            random_state=seed,
            verbosity=-1,
            **candidate.params,
        )
    else:
        raise ValueError(f"학습 모델이 아닌 candidate입니다: {candidate.model_kind}")
    return Pipeline([("features", preprocessor), ("regressor", model)])


def tree_node_count(model: Any | None) -> int:
    """Count nodes in a fitted supported tree model or pipeline.

    The deployment bundle is fitted on more rows than any rolling-origin fold,
    so its node count must be measured from that final fitted object.  This
    helper also handles HistGradientBoosting's private predictor layout, which
    does not expose ``estimators_``.
    """
    if model is None:
        return 0
    regressor = (
        model.named_steps.get("regressor", model)
        if hasattr(model, "named_steps")
        else model
    )
    if hasattr(regressor, "estimators_"):
        estimators = np.asarray(regressor.estimators_, dtype=object).ravel()
        return int(
            sum(
                estimator.tree_.node_count
                for estimator in estimators
                if hasattr(estimator, "tree_")
            )
        )
    if hasattr(regressor, "booster_"):
        return int(
            sum(
                2 * int(tree["num_leaves"]) - 1
                for tree in regressor.booster_.dump_model()["tree_info"]
            )
        )
    if hasattr(regressor, "_predictors"):
        return int(
            sum(
                len(predictor.nodes)
                for stage in regressor._predictors
                for predictor in stage
            )
        )
    return 0


def encode_target(data: pd.DataFrame, kind: str) -> np.ndarray:
    label = data["label_seats"].to_numpy(dtype=float)
    current = data["snapshot_remaining_seats"].to_numpy(dtype=float)
    if kind == "direct":
        return label
    if kind == "delta":
        return label - current
    if kind == "delta_per_stop":
        gap = np.maximum(data["target_stop_gap"].to_numpy(dtype=float), 1.0)
        return (label - current) / gap
    if kind == "delta_per_sqrt_stop":
        gap = np.maximum(data["target_stop_gap"].to_numpy(dtype=float), 1.0)
        return (label - current) / np.sqrt(gap)
    if kind == "route_residual":
        return label - data["route_profile_seats"].to_numpy(dtype=float)
    if kind == "route_residual_per_stop":
        gap = np.maximum(data["target_stop_gap"].to_numpy(dtype=float), 1.0)
        baseline = data["route_profile_seats"].to_numpy(dtype=float)
        return (label - baseline) / gap
    raise ValueError(f"알 수 없는 target kind: {kind}")


def decode_target(raw: np.ndarray, data: pd.DataFrame, kind: str) -> np.ndarray:
    current = data["snapshot_remaining_seats"].to_numpy(dtype=float)
    if kind == "direct":
        prediction = raw
    elif kind == "delta":
        prediction = current + raw
    elif kind == "delta_per_stop":
        prediction = current + raw * data["target_stop_gap"].to_numpy(dtype=float)
    elif kind == "delta_per_sqrt_stop":
        gap = np.maximum(data["target_stop_gap"].to_numpy(dtype=float), 1.0)
        prediction = current + raw * np.sqrt(gap)
    elif kind == "route_residual":
        prediction = data["route_profile_seats"].to_numpy(dtype=float) + raw
    elif kind == "route_residual_per_stop":
        prediction = data["route_profile_seats"].to_numpy(dtype=float) + (
            raw * data["target_stop_gap"].to_numpy(dtype=float)
        )
    else:
        raise ValueError(f"알 수 없는 target kind: {kind}")
    return clip_seats(np.asarray(prediction, dtype=float), data["capacity"])


def formula_prediction(candidate: Candidate, data: pd.DataFrame) -> np.ndarray:
    current = data["snapshot_remaining_seats"].to_numpy(dtype=float)
    if candidate.name == "persistence":
        return current
    shrinkage = float(candidate.params["shrinkage"])
    projected = data["projected_arrival_seats"].fillna(
        data["snapshot_remaining_seats"]
    ).to_numpy(dtype=float)
    return clip_seats(
        current + shrinkage * (projected - current), data["capacity"]
    )


def event_summary(data: pd.DataFrame, predictions: np.ndarray) -> dict[str, Any]:
    scored = data[
        [
            "event_id",
            "date",
            "trip_id",
            "label_seats",
            "snapshot_remaining_seats",
            "target_stop_gap",
            "capacity",
        ]
    ].copy()
    scored["prediction"] = clip_seats(predictions, data["capacity"])
    scored["absolute_error"] = (
        scored["label_seats"] - scored["prediction"]
    ).abs()
    scored["within_3"] = scored["absolute_error"].le(3)
    per_event = scored.groupby("event_id", sort=False).agg(
        trip_id=("trip_id", "first"),
        label_seats=("label_seats", "first"),
        mae=("absolute_error", "mean"),
        within_3=("within_3", "mean"),
        bias=("prediction", "mean"),
    )
    per_event["bias"] -= per_event["label_seats"]
    per_trip = per_event.groupby("trip_id", sort=False)["mae"].mean()
    low = per_event["label_seats"].le(5)
    full = per_event["label_seats"].eq(0)

    emerging_rows = scored.loc[
        scored["label_seats"].le(5)
        & scored["snapshot_remaining_seats"].gt(5)
    ]
    emerging = emerging_rows.groupby("event_id", sort=False)["absolute_error"].mean()
    far_rows = scored.loc[scored["target_stop_gap"].ge(6)]
    far = far_rows.groupby("event_id", sort=False)["absolute_error"].mean()
    near_rows = scored.loc[scored["target_stop_gap"].le(2)]
    near = near_rows.groupby("event_id", sort=False)["absolute_error"].mean()
    return {
        "rows": int(len(scored)),
        "events": int(len(per_event)),
        "low_0_5_events": int(low.sum()),
        "full_events": int(full.sum()),
        "event_balanced_mae": float(per_event["mae"].mean()),
        "trip_balanced_mae": float(per_trip.mean()),
        "event_balanced_mae_std": float(per_event["mae"].std(ddof=0)),
        "event_balanced_mae_p90": float(per_event["mae"].quantile(0.9)),
        "event_balanced_within_3": float(per_event["within_3"].mean()),
        "event_balanced_bias": float(per_event["bias"].mean()),
        "low_0_5_mae": float(per_event.loc[low, "mae"].mean()) if low.any() else np.nan,
        "full_mae": float(per_event.loc[full, "mae"].mean()) if full.any() else np.nan,
        "emerging_low_events": int(emerging.index.nunique()),
        "emerging_low_mae": float(emerging.mean()) if len(emerging) else np.nan,
        "near_1_2_stop_mae": float(near.mean()) if len(near) else np.nan,
        "far_6_plus_stop_mae": float(far.mean()) if len(far) else np.nan,
    }


def scored_frame(
    data: pd.DataFrame,
    predictions: np.ndarray,
    *,
    candidate: str,
) -> pd.DataFrame:
    columns = [
        "event_id",
        "date",
        "trip_id",
        "snapshot_time",
        "label_seats",
        "snapshot_remaining_seats",
        "target_stop_gap",
        "minutes_to_arrival",
        "capacity",
    ]
    if "snapshot_low_plate_cat" in data.columns:
        columns.append("snapshot_low_plate_cat")
    output = data[columns].copy()
    output["candidate"] = candidate
    output["prediction"] = clip_seats(predictions, data["capacity"])
    return output


def candidate_weights(
    data: pd.DataFrame,
    low_weight: float,
    *,
    weighting_kind: str = "event",
    far_weight: float = 1.0,
    far_threshold: int = 6,
    gap_weight_power: float = 0.0,
) -> np.ndarray:
    if not np.isfinite(gap_weight_power) or gap_weight_power < 0:
        raise ValueError("gap_weight_power는 유한한 0 이상이어야 합니다.")
    event_balanced = event_weights(data)
    rows_per_event = data.groupby("event_id")["event_id"].transform(
        "size"
    ).to_numpy(dtype=float)
    events_per_trip = data.groupby("trip_id")["event_id"].transform(
        "nunique"
    ).to_numpy(dtype=float)
    trip_balanced = 1.0 / (rows_per_event * events_per_trip)
    if weighting_kind == "event":
        weights = event_balanced
    elif weighting_kind == "trip":
        weights = trip_balanced
    elif weighting_kind == "event_trip_hybrid":
        weights = np.sqrt(event_balanced * trip_balanced)
    else:
        raise ValueError(f"알 수 없는 weighting kind: {weighting_kind}")
    if low_weight != 1:
        weights = weights * np.where(data["label_seats"].le(5), low_weight, 1.0)
    if far_weight != 1:
        weights = weights * np.where(
            data["target_stop_gap"].ge(far_threshold), far_weight, 1.0
        )
    if gap_weight_power != 0:
        gap = np.maximum(
            data["target_stop_gap"].to_numpy(dtype=float), 1.0
        )
        weights = weights * np.power(gap, gap_weight_power)
    if weighting_kind == "trip":
        trip_totals = pd.Series(weights, index=data.index).groupby(
            data["trip_id"]
        ).transform("sum").to_numpy(dtype=float)
        weights = weights / trip_totals
    return weights / weights.mean()


def development_folds(
    data: pd.DataFrame,
    dates: tuple[str, ...],
) -> list[tuple[str, pd.DataFrame, pd.DataFrame]]:
    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]] = []
    for position, validation_date in enumerate(dates[1:], start=1):
        train_dates = dates[:position]
        train = data.loc[data["date"].isin(train_dates)].copy()
        validation = data.loc[data["date"].eq(validation_date)].copy()
        if train.empty or validation.empty:
            raise ValueError(f"개발 fold {validation_date}가 비어 있습니다.")
        folds.append((validation_date, train, validation))
    return folds


STRICT_FORWARD_BIAS_RULE = (
    "strict_forward_prior_oof_folds_first_fold_zero"
)
DEPLOYMENT_BIAS_SOURCE = (
    "all_development_oof_residuals_for_future_predictions_only"
)


def validate_test_date(
    test_date: str,
    *,
    development_dates: tuple[str, ...] = DEFAULT_DEVELOPMENT_DATES,
    stress_dates: tuple[str, ...] = DEFAULT_STRESS_DATES,
) -> None:
    """잠금 테스트가 개발 또는 스트레스 관측 기간을 침범하지 않게 한다."""
    try:
        parsed_test = pd.Timestamp(test_date).date()
        parsed_development = tuple(
            pd.Timestamp(value).date() for value in development_dates
        )
        parsed_stress = tuple(pd.Timestamp(value).date() for value in stress_dates)
    except (TypeError, ValueError) as error:
        raise ValueError(f"유효하지 않은 최종 테스트 날짜입니다: {test_date}") from error

    if parsed_test in parsed_development:
        raise ValueError(
            f"최종 테스트 날짜 {test_date}가 개발 날짜와 겹칩니다."
        )
    if parsed_test in parsed_stress:
        raise ValueError(
            f"최종 테스트 날짜 {test_date}가 스트레스 날짜와 겹칩니다."
        )
    if parsed_test <= max(parsed_development):
        raise ValueError(
            "최종 테스트 날짜는 마지막 개발 날짜 "
            f"{max(parsed_development).isoformat()}보다 뒤여야 합니다: {test_date}"
        )


def strict_forward_bias_predictions(
    data: pd.DataFrame,
    predictions: np.ndarray,
    *,
    validation_dates: Iterable[str] | None = None,
) -> tuple[np.ndarray, dict[str, float]]:
    """각 OOF 날짜를 오직 이전 OOF residual로 bias 보정한다.

    첫 fold는 이전 residual이 없으므로 0을 사용한다. 반환 prediction은 원래
    행 순서를 유지하며 좌석 범위로 clip된다. 현재 또는 미래 fold의 label은
    해당 fold의 보정값 계산에 절대 사용되지 않는다.
    """
    raw = np.asarray(predictions, dtype=float)
    if len(raw) != len(data):
        raise ValueError("prediction과 OOF data의 길이가 다릅니다.")
    if "date" not in data or data["date"].isna().any():
        raise ValueError("strict-forward bias에는 결측 없는 date가 필요합니다.")

    row_dates = data["date"].astype(str).to_numpy()
    ordered_dates = (
        tuple(str(value) for value in validation_dates)
        if validation_dates is not None
        else tuple(sorted(set(row_dates)))
    )
    if len(set(ordered_dates)) != len(ordered_dates):
        raise ValueError("validation_dates에 중복 날짜가 있습니다.")

    adjusted = raw.copy()
    covered = np.zeros(len(data), dtype=bool)
    history_positions: list[int] = []
    biases: dict[str, float] = {}
    for validation_date in ordered_dates:
        current = row_dates == validation_date
        if not current.any():
            raise ValueError(f"OOF validation 날짜 {validation_date}가 비어 있습니다.")
        if history_positions:
            history = np.asarray(history_positions, dtype=int)
            bias = event_bias(data.iloc[history], raw[history])
        else:
            bias = 0.0
        biases[validation_date] = float(bias)
        adjusted[current] = clip_seats(
            raw[current] + bias,
            data.loc[current, "capacity"],
        )
        positions = np.flatnonzero(current)
        history_positions.extend(positions.tolist())
        covered[current] = True

    if not covered.all():
        missing = sorted(set(row_dates[~covered]))
        raise ValueError(f"validation_dates에 포함되지 않은 OOF 날짜가 있습니다: {missing}")
    return adjusted, biases


def run_candidate(
    candidate: Candidate,
    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]],
    flows: pd.DataFrame,
    *,
    seed: int,
) -> tuple[dict[str, Any], pd.DataFrame, list[dict[str, Any]]]:
    raw_frames: list[pd.DataFrame] = []
    train_maes: list[float] = []
    fit_seconds: list[float] = []
    prediction_ms_per_1000: list[float] = []
    fold_tree_nodes: list[int] = []
    for fold_number, (validation_date, train, validation) in enumerate(folds):
        fold_flows = flows.loc[flows["date"].lt(validation_date)]
        if candidate.model_kind == "formula":
            raw_validation = formula_prediction(candidate, validation)
        elif candidate.model_kind == "route_formula":
            raw_validation = route_profile_values(
                fold_flows, validation
            )["route_profile_seats"].to_numpy(dtype=float)
        else:
            prepared_train, prepared_validation, feature_set = feature_frames(
                train,
                validation,
                candidate.feature_variant,
                flows=fold_flows,
            )
            model = make_regressor(candidate, seed + fold_number)
            target = encode_target(prepared_train, candidate.target_kind)
            fit_started = time.perf_counter()
            model.fit(
                prepared_train[feature_set.columns],
                target,
                regressor__sample_weight=candidate_weights(
                    prepared_train,
                    candidate.low_weight,
                    weighting_kind=candidate.weighting_kind,
                    far_weight=candidate.far_weight,
                    far_threshold=candidate.far_threshold,
                    gap_weight_power=candidate.gap_weight_power,
                ),
            )
            fit_seconds.append(time.perf_counter() - fit_started)
            predict_started = time.perf_counter()
            validation_raw_model = model.predict(
                prepared_validation[feature_set.columns]
            )
            prediction_seconds = time.perf_counter() - predict_started
            prediction_ms_per_1000.append(
                prediction_seconds * 1_000_000 / max(len(prepared_validation), 1)
            )
            raw_validation = decode_target(
                validation_raw_model,
                prepared_validation,
                candidate.target_kind,
            )
            train_prediction = decode_target(
                model.predict(prepared_train[feature_set.columns]),
                prepared_train,
                candidate.target_kind,
            )
            train_maes.append(
                event_summary(prepared_train, train_prediction)["event_balanced_mae"]
            )
            fold_tree_nodes.append(tree_node_count(model))
        raw_frames.append(
            scored_frame(
                validation,
                raw_validation,
                candidate=candidate.name,
            )
        )

    oof = pd.concat(raw_frames, ignore_index=True)
    raw_oof_prediction = oof["prediction"].to_numpy(dtype=float)
    deployment_bias = event_bias(oof, raw_oof_prediction)
    strict_prediction, oof_biases = strict_forward_bias_predictions(
        oof,
        raw_oof_prediction,
        validation_dates=[fold[0] for fold in folds],
    )
    oof["raw_prediction"] = raw_oof_prediction
    oof["prediction"] = strict_prediction
    oof["deployment_prediction"] = clip_seats(
        raw_oof_prediction + deployment_bias, oof["capacity"]
    )
    fold_rows: list[dict[str, Any]] = []
    for validation_date in [fold[0] for fold in folds]:
        fold_data = oof.loc[oof["date"].eq(validation_date)]
        metrics = event_summary(
            fold_data, fold_data["prediction"].to_numpy(dtype=float)
        )
        metrics.update(
            {
                "candidate": candidate.name,
                "stage": candidate.stage,
                "validation_date": validation_date,
                "oof_bias_correction": oof_biases[validation_date],
                "oof_bias_correction_rule": STRICT_FORWARD_BIAS_RULE,
            }
        )
        fold_rows.append(metrics)

    aggregate = event_summary(oof, oof["prediction"].to_numpy(dtype=float))
    daily_mae = np.asarray(
        [row["event_balanced_mae"] for row in fold_rows], dtype=float
    )
    aggregate.update(
        {
            "candidate": candidate.name,
            "stage": candidate.stage,
            "model_kind": candidate.model_kind,
            "target_kind": candidate.target_kind,
            "feature_variant": candidate.feature_variant,
            "low_weight": candidate.low_weight,
            "weighting_kind": candidate.weighting_kind,
            "far_weight": candidate.far_weight,
            "far_threshold": candidate.far_threshold,
            "gap_weight_power": candidate.gap_weight_power,
            # 기존 필드명은 저장 artifact/SeatServiceModel 계약이다. 이 값은
            # OOF 선택 점수가 아니라 미래 prediction에만 적용한다.
            "bias_correction": deployment_bias,
            "bias_correction_source": DEPLOYMENT_BIAS_SOURCE,
            "oof_bias_correction_rule": STRICT_FORWARD_BIAS_RULE,
            "oof_bias_corrections": json.dumps(
                oof_biases, ensure_ascii=False, sort_keys=True
            ),
            "mean_daily_mae": float(daily_mae.mean()),
            "std_daily_mae": float(daily_mae.std(ddof=0)),
            "robust_score": float(daily_mae.mean() + 0.5 * daily_mae.std(ddof=0)),
            "service_score": float(
                daily_mae.mean()
                + 0.5 * daily_mae.std(ddof=0)
                + 0.05 * aggregate["low_0_5_mae"]
            ),
            "mean_train_mae": (
                float(np.mean(train_maes)) if train_maes else np.nan
            ),
            "overfit_gap": (
                float(aggregate["event_balanced_mae"] - np.mean(train_maes))
                if train_maes
                else np.nan
            ),
            "mean_fit_seconds": (
                float(np.mean(fit_seconds)) if fit_seconds else np.nan
            ),
            "prediction_ms_per_1000": (
                float(np.mean(prediction_ms_per_1000))
                if prediction_ms_per_1000
                else np.nan
            ),
            "mean_fold_tree_nodes": (
                float(np.mean(fold_tree_nodes)) if fold_tree_nodes else 0.0
            ),
            "why": candidate.why,
            "if_works": candidate.if_works,
            "if_fails": candidate.if_fails,
            "params": json.dumps(candidate.params, ensure_ascii=False, sort_keys=True),
        }
    )
    return aggregate, oof, fold_rows


def best_name(results: pd.DataFrame, names: Iterable[str]) -> str:
    subset = results.loc[results["candidate"].isin(list(names))]
    if subset.empty:
        raise ValueError("선택할 candidate 결과가 없습니다.")
    return str(subset.sort_values(["service_score", "robust_score"]).iloc[0]["candidate"])


def combine_oof(
    name: str,
    left: pd.DataFrame,
    right: pd.DataFrame,
    *,
    left_weight: np.ndarray | float,
) -> pd.DataFrame:
    keys = ["event_id", "snapshot_time"]
    output = left.copy()
    target_index = pd.MultiIndex.from_frame(output[keys])
    left_prediction = output["prediction"].to_numpy(dtype=float)
    left_deployment = output.get(
        "deployment_prediction", output["prediction"]
    ).to_numpy(dtype=float)
    aligned_right = (
        right.set_index(keys)["prediction"]
        .reindex(target_index)
        .to_numpy(dtype=float)
    )
    right_deployment = right.get(
        "deployment_prediction", right["prediction"]
    ).to_numpy(dtype=float)
    aligned_right_deployment = (
        right.assign(_deployment_prediction=right_deployment)
        .set_index(keys)["_deployment_prediction"]
        .reindex(target_index)
        .to_numpy(dtype=float)
    )
    weights = np.asarray(left_weight, dtype=float)
    output["prediction"] = (
        weights * left_prediction + (1 - weights) * aligned_right
    )
    output["deployment_prediction"] = (
        weights * left_deployment + (1 - weights) * aligned_right_deployment
    )
    output["candidate"] = name
    return output


def combine_weighted_oof(
    name: str,
    components: dict[str, pd.DataFrame],
    weights: dict[str, float],
) -> pd.DataFrame:
    first_name = next(iter(components))
    output = components[first_name].copy()
    keys = ["event_id", "snapshot_time"]
    target_index = pd.MultiIndex.from_frame(output[keys])
    prediction = np.zeros(len(output), dtype=float)
    deployment_prediction = np.zeros(len(output), dtype=float)
    for component, frame in components.items():
        aligned = frame.set_index(keys)["prediction"].reindex(target_index)
        prediction += float(weights[component]) * aligned.to_numpy(dtype=float)
        deployment_column = frame.get("deployment_prediction", frame["prediction"])
        aligned_deployment = (
            frame.assign(
                _deployment_prediction=deployment_column.to_numpy(dtype=float)
            )
            .set_index(keys)["_deployment_prediction"]
            .reindex(target_index)
        )
        deployment_prediction += (
            float(weights[component])
            * aligned_deployment.to_numpy(dtype=float)
        )
    output["prediction"] = prediction
    output["deployment_prediction"] = deployment_prediction
    output["candidate"] = name
    return output


def score_ensemble(
    spec: EnsembleSpec,
    output: pd.DataFrame,
) -> tuple[dict[str, Any], pd.DataFrame]:
    output = output.copy()
    selection_base = output["prediction"].to_numpy(dtype=float)
    strict_prediction, oof_biases = strict_forward_bias_predictions(
        output, selection_base
    )
    deployment_base = output.get(
        "deployment_prediction", output["prediction"]
    ).to_numpy(dtype=float)
    deployment_bias = event_bias(output, deployment_base)
    spec.bias = deployment_bias
    spec.oof_biases = oof_biases
    output["raw_prediction"] = selection_base
    output["prediction"] = postprocess_ensemble_prediction(
        spec, strict_prediction, output
    )
    output["deployment_prediction"] = postprocess_ensemble_prediction(
        spec, deployment_base + deployment_bias, output
    )
    aggregate = event_summary(output, output["prediction"].to_numpy(dtype=float))
    daily = []
    for _, group in output.groupby("date", sort=True):
        daily.append(event_summary(group, group["prediction"].to_numpy(dtype=float)))
    daily_mae = np.asarray([row["event_balanced_mae"] for row in daily])
    aggregate.update(
        {
            "candidate": spec.name,
            "stage": "H7_ensemble",
            "model_kind": "ensemble",
            "target_kind": "blend",
            "feature_variant": "mixed",
            "low_weight": np.nan,
            "weighting_kind": "mixed",
            "far_weight": np.nan,
            "far_threshold": np.nan,
            "bias_correction": deployment_bias,
            "bias_correction_source": DEPLOYMENT_BIAS_SOURCE,
            "oof_bias_correction_rule": STRICT_FORWARD_BIAS_RULE,
            "oof_bias_corrections": json.dumps(
                oof_biases, ensure_ascii=False, sort_keys=True
            ),
            "mean_daily_mae": float(daily_mae.mean()),
            "std_daily_mae": float(daily_mae.std(ddof=0)),
            "robust_score": float(daily_mae.mean() + 0.5 * daily_mae.std(ddof=0)),
            "service_score": float(
                daily_mae.mean()
                + 0.5 * daily_mae.std(ddof=0)
                + 0.05 * aggregate["low_0_5_mae"]
            ),
            "mean_train_mae": np.nan,
            "overfit_gap": np.nan,
            "mean_fit_seconds": np.nan,
            "prediction_ms_per_1000": np.nan,
            "mean_fold_tree_nodes": np.nan,
            "why": (
                "서로 다른 귀납 편향 또는 근거리 persistence를 결합하면 "
                "날짜별 오차 분산을 줄일 수 있다."
            ),
            "if_works": "개별 최선보다 robust score와 날짜별 분산이 함께 감소한다.",
            "if_fails": "기저 모델 오차가 강하게 상관되거나 거리별 게이트가 불필요하다.",
            "params": json.dumps(spec.params, ensure_ascii=False, sort_keys=True),
        }
    )
    return aggregate, output


def select_with_final_node_budget(
    all_selection: pd.DataFrame,
    ensemble_specs: dict[str, EnsembleSpec],
    *,
    ensure_final_fit: Callable[[str], None],
    final_component_tree_nodes: dict[str, int],
    maximum_final_fit_tree_nodes: int,
) -> str:
    """Select by score while enforcing the budget on final fitted models.

    Candidate folds are smaller than the full development fit, so their mean
    node counts are only diagnostics.  Components are fitted lazily in score
    order and cached by the caller; selection stops at the first bundle whose
    measured final node count satisfies the hard deployment constraint.
    """
    if maximum_final_fit_tree_nodes < 0:
        raise ValueError("최종 fit tree node 상한은 0 이상이어야 합니다.")
    required = {"candidate", "service_score", "robust_score"}
    missing = sorted(required - set(all_selection.columns))
    if missing:
        raise ValueError(f"배포 후보 선택 열이 누락되었습니다: {missing}")

    all_selection["final_fit_tree_nodes"] = np.nan
    all_selection["deployment_eligible"] = pd.Series(
        pd.NA, index=all_selection.index, dtype="boolean"
    )
    all_selection["deployment_check"] = "not_evaluated_after_selection"
    for row_index in all_selection.sort_values(
        ["service_score", "robust_score"]
    ).index:
        name = str(all_selection.at[row_index, "candidate"])
        components = (
            ensemble_specs[name].components
            if name in ensemble_specs
            else (name,)
        )
        for component in components:
            ensure_final_fit(component)
            if component not in final_component_tree_nodes:
                raise ValueError(
                    f"최종 fit node 수가 기록되지 않았습니다: {component}"
                )
        bundle_nodes = int(
            sum(final_component_tree_nodes[component] for component in components)
        )
        eligible = bundle_nodes <= maximum_final_fit_tree_nodes
        all_selection.at[row_index, "final_fit_tree_nodes"] = bundle_nodes
        all_selection.at[row_index, "deployment_eligible"] = eligible
        all_selection.at[row_index, "deployment_check"] = (
            "passed" if eligible else "failed"
        )
        if eligible:
            return name
    raise ValueError("최종 fit tree node 제약을 만족하는 candidate가 없습니다.")


def make_candidates() -> tuple[list[Candidate], dict[str, dict[str, Any]]]:
    baseline_hgb = {
        "loss": "absolute_error",
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 15,
        "min_samples_leaf": 20,
        "l2_regularization": 1.0,
        "early_stopping": False,
    }
    formulas = [
        Candidate(
            name="persistence",
            stage="H0_physical",
            why="정류장 사이 좌석 변화 중앙값이 0이므로 현재 좌석 유지가 강한 기준선이다.",
            if_works="근거리에서 학습 모델과 비슷하거나 더 정확하다.",
            if_fails="도착 전 누적 승하차 변화가 무시할 수 없다는 뜻이다.",
            model_kind="formula",
        ),
        Candidate(
            name="route_stop_profile",
            stage="H0_physical",
            why=(
                "선행 연구의 2단계 구조처럼 정류장별 직접 도착→출발 순좌석변화를 "
                "shrinkage한 뒤 목표 직전까지 누적한다."
            ),
            if_works="특히 6정류장 이상과 신규 저잔여에서 persistence를 안정적으로 이긴다.",
            if_fails="승차 변화만으로는 하차와 차량별 수요 차이를 설명하지 못한다.",
            model_kind="route_formula",
            feature_variant="route_profile",
        ),
    ]
    for shrinkage in (0.25, 0.5, 1.0):
        formulas.append(
            Candidate(
                name=f"trajectory_shrink_{int(shrinkage * 100):03d}",
                stage="H0_physical",
                why="최근 좌석 변화 추세에는 신호가 있지만 그대로 외삽하면 잡음이 증폭될 수 있다.",
                if_works="0과 1 사이 shrinkage가 persistence와 완전 외삽을 모두 이긴다.",
                if_fails="짧은 관측 궤적의 변화율이 반복 가능한 수요 신호가 아니다.",
                model_kind="formula",
                params={"shrinkage": shrinkage},
            )
        )
    transformations = [
        Candidate(
            name="hgb_direct_reference",
            stage="H1_target",
            why="절대 도착 좌석을 직접 학습하면 정류장·시간대의 전역 수준을 활용할 수 있다.",
            if_works="현재 좌석 기준 변화량보다 반복 시간대 패턴이 더 강하다.",
            if_fails="현재 좌석을 기준점으로 두지 않아 차량별 상태 차이가 커진다.",
            model_kind="hgb",
            target_kind="direct",
            params=baseline_hgb,
        ),
        Candidate(
            name="hgb_delta_reference",
            stage="H1_target",
            why="실시간 현재 좌석이 강한 기준점이므로 남은 변화량만 학습하면 문제가 단순해진다.",
            if_works="direct 및 persistence보다 날짜 외 MAE가 낮다.",
            if_fails="변화량 분산이 절대 좌석보다 크거나 시간대 패턴을 잃는다.",
            model_kind="hgb",
            target_kind="delta",
            params=baseline_hgb,
        ),
        Candidate(
            name="hgb_delta_per_stop",
            stage="H1_target",
            why="예측 거리가 제각각이므로 정류장당 변화율을 학습하면 먼 거리 외삽이 쉬워질 수 있다.",
            if_works="특히 6정류장 이상 MAE가 감소한다.",
            if_fails="승하차 변화가 정류장 수에 선형 비례하지 않고 특정 정류장에 집중된다.",
            model_kind="hgb",
            target_kind="delta_per_stop",
            params=baseline_hgb,
        ),
    ]
    return formulas + transformations, {"baseline_hgb": baseline_hgb}


def candidate_from(
    parent: Candidate,
    *,
    name: str,
    stage: str,
    why: str,
    if_works: str,
    if_fails: str,
    **changes: Any,
) -> Candidate:
    return replace(
        parent,
        name=name,
        stage=stage,
        why=why,
        if_works=if_works,
        if_fails=if_fails,
        **changes,
    )


def fit_final_candidate(
    candidate: Candidate,
    train: pd.DataFrame,
    targets: dict[str, pd.DataFrame],
    flows: pd.DataFrame,
    *,
    bias: float,
    seed: int,
) -> tuple[dict[str, np.ndarray], Pipeline | None, dict[str, Any]]:
    if candidate.model_kind == "formula":
        return {
            key: clip_seats(
                formula_prediction(candidate, target) + bias, target["capacity"]
            )
            for key, target in targets.items()
        }, None, {"feature_set": "formula"}
    if candidate.model_kind == "route_formula":
        return {
            key: clip_seats(
                route_profile_values(flows, target)[
                    "route_profile_seats"
                ].to_numpy(dtype=float)
                + bias,
                target["capacity"],
            )
            for key, target in targets.items()
        }, None, {"feature_set": ROUTE_PROFILE_ALL_PREARRIVAL.name}

    combined_target = pd.concat(
        [target.assign(_target_split=key) for key, target in targets.items()],
        ignore_index=True,
    )
    prepared_train, prepared_target, feature_set = feature_frames(
        train,
        combined_target,
        candidate.feature_variant,
        flows=flows,
    )
    model = make_regressor(candidate, seed)
    model.fit(
        prepared_train[feature_set.columns],
        encode_target(prepared_train, candidate.target_kind),
        regressor__sample_weight=candidate_weights(
            prepared_train,
            candidate.low_weight,
            weighting_kind=candidate.weighting_kind,
            far_weight=candidate.far_weight,
            far_threshold=candidate.far_threshold,
            gap_weight_power=candidate.gap_weight_power,
        ),
    )
    predictions: dict[str, np.ndarray] = {}
    for key in targets:
        mask = prepared_target["_target_split"].eq(key)
        split = prepared_target.loc[mask]
        decoded = decode_target(
            model.predict(split[feature_set.columns]), split, candidate.target_kind
        )
        predictions[key] = clip_seats(decoded + bias, split["capacity"])
    return predictions, model, {
        "feature_set": feature_set.name,
        "feature_columns": feature_set.columns,
        "target_kind": candidate.target_kind,
        "feature_variant": candidate.feature_variant,
    }


def ensemble_prediction(
    spec: EnsembleSpec,
    component_predictions: dict[str, np.ndarray],
    target: pd.DataFrame,
) -> np.ndarray:
    if spec.kind == "linear":
        left, right = spec.components
        weight = float(spec.params["left_weight"])
        prediction = (
            weight * component_predictions[left]
            + (1 - weight) * component_predictions[right]
        )
    elif spec.kind == "distance_gate":
        left, right = spec.components
        near = target["target_stop_gap"].le(2).to_numpy()
        weight = np.where(
            near,
            float(spec.params["near_left_weight"]),
            float(spec.params["far_left_weight"]),
        )
        prediction = (
            weight * component_predictions[left]
            + (1 - weight) * component_predictions[right]
        )
    elif spec.kind == "stop_gap_gate":
        left, right = spec.components
        threshold = float(spec.params["threshold"])
        left_weight = np.where(
            target["target_stop_gap"].lt(threshold).to_numpy(),
            float(spec.params["below_left_weight"]),
            float(spec.params["above_left_weight"]),
        )
        prediction = (
            left_weight * component_predictions[left]
            + (1 - left_weight) * component_predictions[right]
        )
    elif spec.kind == "weighted":
        prediction = np.zeros(len(target), dtype=float)
        for component in spec.components:
            prediction += (
                float(spec.params[component]) * component_predictions[component]
            )
    else:
        raise ValueError(f"알 수 없는 ensemble: {spec.kind}")
    return postprocess_ensemble_prediction(spec, prediction + spec.bias, target)


def stop_band_metrics(
    data: pd.DataFrame,
    predictions: np.ndarray,
    model: str,
) -> pd.DataFrame:
    bands = pd.cut(
        data["target_stop_gap"],
        bins=[0, 2, 5, 10, np.inf],
        labels=["1-2", "3-5", "6-10", "11+"],
    )
    rows: list[dict[str, Any]] = []
    for band in ["1-2", "3-5", "6-10", "11+"]:
        mask = bands.eq(band).to_numpy()
        if not mask.any():
            continue
        result = event_summary(data.loc[mask], predictions[mask])
        result.update({"model": model, "stop_band": band})
        rows.append(result)
    return pd.DataFrame(rows)


def cluster_bootstrap_mae_delta(
    data: pd.DataFrame,
    selected: np.ndarray,
    baseline: np.ndarray,
    *,
    seed: int,
    repeats: int = 2_000,
) -> dict[str, Any]:
    scored = data[["event_id", "trip_id", "label_seats"]].copy()
    scored["selected_error"] = np.abs(
        data["label_seats"].to_numpy(dtype=float) - selected
    )
    scored["baseline_error"] = np.abs(
        data["label_seats"].to_numpy(dtype=float) - baseline
    )
    per_event = scored.groupby("event_id", sort=False).agg(
        trip_id=("trip_id", "first"),
        selected=("selected_error", "mean"),
        baseline=("baseline_error", "mean"),
    )
    per_event["delta"] = per_event["baseline"] - per_event["selected"]
    trip_groups = [
        group["delta"].to_numpy(dtype=float)
        for _, group in per_event.groupby("trip_id", sort=False)
    ]
    if not trip_groups:
        raise ValueError("bootstrap할 trip이 없습니다.")
    rng = np.random.default_rng(seed)
    boot = np.empty(repeats, dtype=float)
    for index in range(repeats):
        selected_indices = rng.integers(0, len(trip_groups), size=len(trip_groups))
        sampled = np.concatenate([trip_groups[position] for position in selected_indices])
        boot[index] = sampled.mean()
    lower, median, upper = np.quantile(boot, [0.025, 0.5, 0.975])
    return {
        "metric": "baseline MAE minus selected MAE; positive favors selected",
        "unit": "trip_id",
        "trips": int(len(trip_groups)),
        "events": int(len(per_event)),
        "observed_delta": float(per_event["delta"].mean()),
        "lower_95": float(lower),
        "median": float(median),
        "upper_95": float(upper),
        "probability_selected_better": float((boot > 0).mean()),
    }


def main() -> int:
    parser = argparse.ArgumentParser(
        description="가설 기반 도착 잔여좌석 모델·파라미터·앙상블 탐색"
    )
    parser.add_argument(
        "--db",
        type=Path,
        default=Path("data/gbis_api_cache.sqlite3"),
    )
    parser.add_argument(
        "--snapshot-cache",
        type=Path,
        default=Path("data/analysis_cache/all_prearrival_A.pkl"),
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("analysis/model_search_results"),
    )
    parser.add_argument("--test-date", default=DEFAULT_TEST_DATE)
    parser.add_argument("--rebuild-cache", action="store_true")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    validate_test_date(args.test_date)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    experiment_source_sha256 = hashlib.sha256(
        Path(__file__).read_bytes()
    ).hexdigest()
    inference_source_sha256 = {
        filename: cache_payload_sha256(Path(__file__).resolve().parent / filename)
        for filename in (
            "seat_service_model.py",
            "all_prearrival_seat_regression.py",
        )
    }

    snapshots, stop_flows, cache_info = load_or_build_snapshots(
        args.db, args.snapshot_cache, rebuild=args.rebuild_cache
    )
    output_support_audit = audit_route_output_support(args.db, ROUTE_ID)
    development_dates = tuple(
        date for date in DEFAULT_DEVELOPMENT_DATES if date in set(snapshots["date"])
    )
    if development_dates != DEFAULT_DEVELOPMENT_DATES:
        raise ValueError(
            f"개발 날짜가 부족합니다: {development_dates}"
        )
    development = snapshots.loc[
        snapshots["date"].isin(development_dates)
    ].copy()
    test = snapshots.loc[snapshots["date"].eq(args.test_date)].copy()
    stress = snapshots.loc[
        snapshots["date"].isin(DEFAULT_STRESS_DATES)
    ].copy()
    if test.empty:
        raise ValueError(f"최종 테스트 날짜 {args.test_date}가 비어 있습니다.")
    folds = development_folds(development, development_dates)

    candidates, settings = make_candidates()
    registry = {candidate.name: candidate for candidate in candidates}
    result_rows: list[dict[str, Any]] = []
    fold_rows: list[dict[str, Any]] = []
    oof_predictions: dict[str, pd.DataFrame] = {}

    def evaluate(items: Iterable[Candidate]) -> None:
        for candidate in items:
            if candidate.name in oof_predictions:
                continue
            result, oof, candidate_folds = run_candidate(
                candidate, folds, stop_flows, seed=args.seed
            )
            result_rows.append(result)
            fold_rows.extend(candidate_folds)
            oof_predictions[candidate.name] = oof
            registry[candidate.name] = candidate

    evaluate(candidates)
    results = pd.DataFrame(result_rows)
    target_names = [
        "hgb_direct_reference",
        "hgb_delta_reference",
        "hgb_delta_per_stop",
    ]
    target_winner_name = best_name(results, target_names)
    target_winner = registry[target_winner_name]

    capacity_candidates = [
        candidate_from(
            target_winner,
            name="hgb_flexible",
            stage="H2_capacity",
            why="기준 HGB의 15개 leaf가 정류장·거리별 비선형성을 과소적합할 수 있다.",
            if_works="train 오차는 낮아지면서 날짜 외 robust score도 개선된다.",
            if_fails="현재 데이터량에서 추가 분기는 날짜 고유 패턴에 과적합된다.",
            params={
                "loss": "absolute_error",
                "learning_rate": 0.035,
                "max_iter": 450,
                "max_leaf_nodes": 31,
                "min_samples_leaf": 10,
                "l2_regularization": 0.5,
                "early_stopping": False,
            },
        ),
        candidate_from(
            target_winner,
            name="hgb_overfit_probe",
            stage="H2_capacity",
            why="먼저 충분히 큰 모델로 학습 가능한 신호의 상한과 과적합 간격을 측정한다.",
            if_works="검증도 개선되어 기존 모델이 명백히 과소적합이었다.",
            if_fails="train 개선 대비 검증 악화로 데이터 반복성이 부족함을 확인한다.",
            params={
                "loss": "absolute_error",
                "learning_rate": 0.04,
                "max_iter": 500,
                "max_leaf_nodes": 63,
                "min_samples_leaf": 5,
                "l2_regularization": 0.0,
                "early_stopping": False,
            },
        ),
        candidate_from(
            target_winner,
            name="hgb_regularized",
            stage="H2_capacity",
            why="날짜가 네 개뿐이므로 더 큰 leaf와 L2가 일별 변동을 줄일 수 있다.",
            if_works="평균 MAE가 비슷해도 날짜별 표준편차와 robust score가 감소한다.",
            if_fails="이미 기준 모델의 규제가 충분해 추가 규제가 수요 신호까지 지운다.",
            params={
                "loss": "absolute_error",
                "learning_rate": 0.04,
                "max_iter": 350,
                "max_leaf_nodes": 15,
                "min_samples_leaf": 50,
                "l2_regularization": 5.0,
                "early_stopping": False,
            },
        ),
        candidate_from(
            target_winner,
            name="hgb_squared_tail",
            stage="H2_capacity",
            why="제곱손실은 큰 오차를 더 벌주므로 저잔여와 먼 거리 꼬리오차를 줄일 수 있다.",
            if_works="p90·저잔여 MAE가 크게 감소하고 평균 MAE 손실이 작다.",
            if_fails="일반 사례를 평균 쪽으로 끌어 MAE와 보정이 악화된다.",
            params={
                "loss": "squared_error",
                "learning_rate": 0.035,
                "max_iter": 450,
                "max_leaf_nodes": 31,
                "min_samples_leaf": 15,
                "l2_regularization": 1.0,
                "early_stopping": False,
            },
        ),
    ]
    evaluate(capacity_candidates)
    results = pd.DataFrame(result_rows)
    hgb_pool = target_names + [candidate.name for candidate in capacity_candidates]
    capacity_winner_name = best_name(results, hgb_pool)
    capacity_winner = registry[capacity_winner_name]

    weighting_candidates = [
        candidate_from(
            capacity_winner,
            name="hgb_low_weight_2",
            stage="H3_rare_weight",
            why="0~5석 사건이 약 2%라 전체 MAE 학습에서 기울기가 묻힐 수 있다.",
            if_works="전체 robust score를 거의 유지하며 저잔여·신규 저잔여 MAE가 감소한다.",
            if_fails="현재 좌석 자체가 이미 위험을 설명하거나 양성 표본이 너무 적다.",
            low_weight=2.0,
        ),
        candidate_from(
            capacity_winner,
            name="hgb_low_weight_4",
            stage="H3_rare_weight",
            why="더 강한 비용 민감 학습으로 서비스 핵심 꼬리구간의 상한을 확인한다.",
            if_works="저잔여 성능 개선이 가중치 2보다 커서 별도 전문가 모델 가치가 있다.",
            if_fails="전체 오차와 확률 보정만 악화되어 회귀 가중치보다 별도 위험 헤드가 낫다.",
            low_weight=4.0,
        ),
        candidate_from(
            capacity_winner,
            name="hgb_low_weight_8",
            stage="H3_rare_weight",
            why="가중치 4에서 저잔여 개선이 계속됐으므로 비용 민감도의 포화점을 확인한다.",
            if_works="저잔여·신규 저잔여 MAE가 더 줄고 전체 robust score 손실이 제한적이다.",
            if_fails="희귀 사건을 과도하게 반복해 전체 오차와 보정편향이 급격히 커진다.",
            low_weight=8.0,
        ),
    ]
    evaluate(weighting_candidates)
    results = pd.DataFrame(result_rows)
    weighting_pool = [capacity_winner_name] + [
        candidate.name for candidate in weighting_candidates
    ]
    weighting_winner_name = best_name(results, weighting_pool)
    weighting_winner = registry[weighting_winner_name]

    feature_candidates = [
        candidate_from(
            weighting_winner,
            name="hgb_capacity_44_70",
            stage="H4_features",
            why=(
                "일반차 37대의 관측 빈자리 상한은 모두 44석인데 기존 load ratio는 "
                "45석을 사용해 1석 편향이 있다."
            ),
            if_works="44/70 관측 상한 적재율이 용량 regime과 저잔여 예측을 개선한다.",
            if_fails="1석 비율 차이는 트리 분할에 미미하거나 45석 명목 정원이 더 적절하다.",
            feature_variant="capacity_44_70",
        ),
        candidate_from(
            weighting_winner,
            name="hgb_route_profile",
            stage="H4_features",
            why=(
                "정류장×2시간대 직접 도착→출발 순좌석변화 누적값을 HGB가 "
                "현재 차량 궤적과 함께 보정하면 구조와 residual을 분리할 수 있다."
            ),
            if_works="route formula보다 전체·장거리 MAE가 더 낮고 날짜별 방향도 안정적이다.",
            if_fails="짧은 기간의 flow profile 오차를 모델이 다시 과적합한다.",
            feature_variant="route_profile",
        ),
        candidate_from(
            weighting_winner,
            name="hgb_source_target_pair",
            stage="H4_features",
            why="선행 연구에서 현재 정류장과 목표 정류장의 명시적 쌍이 재귀식보다 하류 예측에 강했다.",
            if_works="별도 pair 범주가 6정류장 이상 MAE를 낮춰 상호작용 학습 부족을 보완한다.",
            if_fails="기존 HGB가 두 정류장 피처의 상호작용을 이미 학습했거나 pair 표본이 희소하다.",
            feature_variant="source_target_pair",
        ),
        candidate_from(
            weighting_winner,
            name="hgb_previous_bus_seat",
            stage="H4_features",
            why=(
                "같은 목표 정류장의 직전 버스 출발 잔여좌석 자체가 국소 수요의 "
                "가장 직접적인 시계열 신호인지 다른 통계와 분리해 확인한다."
            ),
            if_works="단일 좌석 신호만으로 전체 또는 저잔여 MAE가 안정적으로 감소한다.",
            if_fails="현재 버스의 실시간 좌석 궤적에 비해 직전 버스 값의 추가 정보가 작다.",
            feature_variant="previous_bus_seat",
        ),
        candidate_from(
            weighting_winner,
            name="hgb_previous_bus",
            stage="H4_features",
            why="같은 정류장의 직전 버스 출발 좌석은 국소 수요와 이월 혼잡의 대리변수다.",
            if_works="특히 만차·저잔여 MAE가 개선되고 일별 효과가 같은 방향이다.",
            if_fails="직전 버스와 현재 버스 사이 수요 변화가 커 일반 좌석에는 잡음이다.",
            feature_variant="previous_bus",
        ),
        candidate_from(
            weighting_winner,
            name="hgb_previous_bus_seat_capacity_44_70",
            stage="H4_features",
            why=(
                "직전 버스 출발 잔여좌석의 효과를 검증된 44/70석 차량 용량 "
                "보정과 결합해 순수한 증분 기여를 비교한다."
            ),
            if_works="용량 보정 단독보다 날짜별·저잔여 MAE가 함께 감소한다.",
            if_fails="직전 버스 좌석은 용량 regime을 보정한 뒤에도 잡음이다.",
            feature_variant="previous_bus_seat_capacity_44_70",
        ),
        candidate_from(
            weighting_winner,
            name="hgb_previous_bus_normalized_capacity_44_70",
            stage="H4_features",
            why=(
                "절대 출발좌석은 45/70석 정원 불일치와 오래된 참조에서 실패했다. "
                "적재율·정원 일치·freshness·최근 3대 적재율만 분리하면 국소 수요 "
                "신호를 좌석 단위 잡음 없이 사용할 수 있다."
            ),
            if_works=(
                "용량 보정 단독보다 전체·저잔여·장거리 MAE가 네 날짜에서 "
                "일관되게 감소한다."
            ),
            if_fails=(
                "이전 차량 혼잡은 현재 차량 도착 좌석의 점 추정보다 위험 시간대 "
                "분류에만 유효하거나 참조 차량 교체가 여전히 너무 잦다."
            ),
            feature_variant="previous_bus_normalized_capacity_44_70",
        ),
        candidate_from(
            weighting_winner,
            name="hgb_previous_bus_capacity_44_70",
            stage="H4_features",
            why=(
                "직전 버스의 좌석·나이·배차·최근 3대 통계를 44/70석 용량 "
                "보정과 함께 사용해 정보 결합 효과를 확인한다."
            ),
            if_works="직전 버스 단독 및 용량 보정 단독보다 안정적으로 개선된다.",
            if_fails="짧은 수집 기간의 배차·추세 통계가 과적합을 늘린다.",
            feature_variant="previous_bus_capacity_44_70",
        ),
        candidate_from(
            weighting_winner,
            name="hgb_historical_profiles",
            stage="H4_features",
            why="선행 연구처럼 정류장쌍·시간대별 평균 변화량을 shrinkage하면 반복 수요를 포착할 수 있다.",
            if_works="먼 거리와 날짜 외 MAE가 개선되며 충분한 count 그룹에서 효과가 크다.",
            if_fails="수집 기간이 짧아 동일 정류장쌍·시간대 반복이 부족하거나 주간 변동이 크다.",
            feature_variant="historical",
        ),
    ]
    evaluate(feature_candidates)
    results = pd.DataFrame(result_rows)
    feature_pool = [weighting_winner_name] + [
        candidate.name for candidate in feature_candidates
    ]
    feature_winner_name = best_name(results, feature_pool)
    feature_winner = registry[feature_winner_name]

    generalization_candidates = [
        candidate_from(
            feature_winner,
            name="hgb_trip_balanced",
            stage="H5_generalization",
            why=(
                "한 운행에서 많은 목표 사건과 스냅샷이 만들어지는 trip이 학습을 "
                "지배하면 특정 혼잡 운행을 반복 암기할 수 있다."
            ),
            if_works="trip당 총 가중치를 같게 했을 때 일별 분산과 과적합 간격이 감소한다.",
            if_fails="서비스 목표가 사건 평균이므로 trip 균형이 정보량 많은 운행을 과도하게 줄인다.",
            weighting_kind="trip",
        ),
        candidate_from(
            feature_winner,
            name="hgb_event_trip_hybrid",
            stage="H5_generalization",
            why="사건 균형과 trip 균형의 기하평균은 두 반복 구조 사이의 완만한 절충이다.",
            if_works="순수 trip 균형보다 전체 MAE를 유지하면서 날짜별 분산을 줄인다.",
            if_fails="기존 사건 균형이 이미 충분해 가중치 변경이 유효 신호만 약화한다.",
            weighting_kind="event_trip_hybrid",
        ),
        candidate_from(
            feature_winner,
            name="hgb_far_weight_2",
            stage="H5_generalization",
            why="6정류장 이상 MAE가 근거리의 3배 이상이라 평균 목적함수에서 장거리 기울기가 부족할 수 있다.",
            if_works="근거리 손실을 제한하면서 6정류장 이상과 신규 저잔여 MAE가 감소한다.",
            if_fails="장거리 오차는 표본 가중 부족이 아니라 본질적 수요 불확실성 때문이다.",
            far_weight=2.0,
            far_threshold=6,
        ),
        candidate_from(
            feature_winner,
            name="hgb_far_weight_4",
            stage="H5_generalization",
            why="장거리 가중치 2의 개선이 부족할 때 전문가 모델의 상한을 확인한다.",
            if_works="장거리 꼬리오차가 추가로 줄어 거리 gate의 구성요소가 된다.",
            if_fails="근거리와 전체 MAE 손실이 커져 장거리 데이터 자체가 부족하다는 뜻이다.",
            far_weight=4.0,
            far_threshold=6,
        ),
        candidate_from(
            feature_winner,
            name="hgb_route_residual",
            stage="H5_generalization",
            why=(
                "정류장 flow 누적식은 persistence를 이겼지만 단순 피처 추가는 실패했다. "
                "물리 예측을 기준점으로 두고 잔차만 학습하면 구조와 보정을 분리할 수 있다."
            ),
            if_works="route feature 모델보다 특히 6정류장 이상 MAE가 낮아진다.",
            if_fails="짧은 기간의 stop-flow baseline 편향이 현재 좌석 기준점보다 불안정하다.",
            feature_variant="route_profile",
            target_kind="route_residual",
        ),
        candidate_from(
            feature_winner,
            name="hgb_route_residual_per_stop",
            stage="H5_generalization",
            why="물리 baseline의 남은 오차도 거리와 함께 누적되므로 정류장당 residual로 정규화한다.",
            if_works="절대 residual보다 장거리 외삽과 일별 안정성이 개선된다.",
            if_fails="baseline 오차가 특정 정류장에 집중돼 선형 거리 정규화가 맞지 않는다.",
            feature_variant="route_profile",
            target_kind="route_residual_per_stop",
        ),
    ]
    evaluate(generalization_candidates)
    results = pd.DataFrame(result_rows)
    generalization_pool = [feature_winner_name] + [
        candidate.name for candidate in generalization_candidates
    ]
    generalization_winner_name = best_name(results, generalization_pool)
    generalization_winner = registry[generalization_winner_name]

    tree_candidates = [
        candidate_from(
            generalization_winner,
            name="extra_trees_diversity",
            stage="H6_family",
            why="무작위 분할 ExtraTrees는 boosting과 다른 오차를 내므로 단독 또는 앙상블에 유용할 수 있다.",
            if_works="단독 MAE가 경쟁력 있거나 HGB와 블렌드할 때 분산이 감소한다.",
            if_fails="희소 one-hot 공간에서 외삽이 약하고 날짜 변화에 불안정하다.",
            model_kind="extra_trees",
            params={
                "n_estimators": 350,
                "max_depth": None,
                "min_samples_leaf": 3,
                "max_features": 0.7,
            },
        ),
        candidate_from(
            generalization_winner,
            name="random_forest_diversity",
            stage="H6_family",
            why="bootstrap 평균화는 데이터가 짧을 때 HGB보다 날짜별 분산을 줄일 수 있다.",
            if_works="robust score와 p90 오차가 HGB보다 안정적이다.",
            if_fails="평균화 편향 때문에 좌석 변화량의 극단을 과소추정한다.",
            model_kind="random_forest",
            params={
                "n_estimators": 300,
                "max_depth": 22,
                "min_samples_leaf": 3,
                "max_features": 0.7,
            },
        ),
        candidate_from(
            generalization_winner,
            name="extra_trees_compact",
            stage="H6_family",
            why=(
                "무제한 ExtraTrees의 앙상블 이득이 깊은 말단보다 모델 다양성에서 온다면 "
                "깊이·트리 수를 줄여도 성능을 유지할 수 있다."
            ),
            if_works="OOF service score가 1% 이내이고 node 수와 추론시간이 크게 감소한다.",
            if_fails="희귀 저잔여 패턴을 포착하는 데 깊은 개별 트리가 실제로 필요하다.",
            model_kind="extra_trees",
            params={
                "n_estimators": 160,
                "max_depth": 18,
                "min_samples_leaf": 5,
                "max_features": 0.7,
            },
        ),
        candidate_from(
            generalization_winner,
            name="extra_trees_tiny",
            stage="H6_family",
            why="서비스 가능한 크기의 하한을 확인해 정확도-용량 Pareto 경계를 만든다.",
            if_works="100개 얕은 트리도 HGB 블렌드의 오차 다양성을 충분히 제공한다.",
            if_fails="규제가 강해지며 HGB와 다른 유용한 잔차까지 사라진다.",
            model_kind="extra_trees",
            params={
                "n_estimators": 100,
                "max_depth": 14,
                "min_samples_leaf": 8,
                "max_features": 0.7,
            },
        ),
        candidate_from(
            generalization_winner,
            name="extra_trees_compact_70",
            stage="H6_family",
            why=(
                "compact ExtraTrees의 앙상블 이득은 확인됐지만 160개 트리가 "
                "100만-node 배포 상한을 넘었다. 깊이 18·leaf 5를 유지하고 "
                "트리 수만 70으로 줄여 유용한 상호작용과 용량 비용을 분리한다."
            ),
            if_works=(
                "HGB 50% 혼합이 tiny stack보다 낮은 service score를 내면서 "
                "총 tree node가 100만 이하다."
            ),
            if_fails=(
                "compact의 이득이 많은 트리의 분산 감소에서 왔거나 깊은 개별 "
                "트리가 날짜 고유 패턴에 과적합된다."
            ),
            model_kind="extra_trees",
            params={
                "n_estimators": 70,
                "max_depth": 18,
                "min_samples_leaf": 5,
                "max_features": 0.7,
            },
        ),
        candidate_from(
            generalization_winner,
            name="extra_trees_compact_48",
            stage="H6_family",
            why=(
                "70-tree 후보는 fold 평균이 아니라 최종 재학습 artifact에서 "
                "100만-node 상한을 넘었다. 같은 깊이 18·leaf 5를 유지하고 "
                "트리 수만 48로 줄여 실제 최종 번들 제약을 검증한다."
            ),
            if_works=(
                "최종 fit HGB·ExtraTrees·LightGBM node 합이 100만 이하이면서 "
                "70-tree stack과 OOF service score 차이가 실질적으로 작다."
            ),
            if_fails=(
                "트리 감소로 ExtraTrees의 분산이 커져 앙상블 이득이 사라지거나 "
                "전체 데이터의 tree 성장이 예상보다 커 상한을 다시 넘는다."
            ),
            model_kind="extra_trees",
            params={
                "n_estimators": 48,
                "max_depth": 18,
                "min_samples_leaf": 5,
                "max_features": 0.7,
            },
        ),
        candidate_from(
            generalization_winner,
            name="lightgbm_leafwise_l1",
            stage="H6_family",
            why=(
                "HGB의 63-leaf 과적합 probe가 검증도 개선했으므로 leaf-wise boosting이 "
                "같은 비선형 신호를 더 효율적으로 포착할 수 있다."
            ),
            if_works="HGB보다 낮은 robust score를 작은 artifact와 빠른 추론으로 달성한다.",
            if_fails="날짜 수가 적어 leaf-wise 성장이 날짜 고유 패턴을 더 과적합한다.",
            model_kind="lightgbm",
            params={
                "objective": "regression_l1",
                "n_estimators": 600,
                "learning_rate": 0.025,
                "num_leaves": 63,
                "min_child_samples": 10,
                "subsample": 0.85,
                "subsample_freq": 1,
                "colsample_bytree": 0.8,
                "reg_alpha": 0.1,
                "reg_lambda": 1.0,
            },
        ),
        candidate_from(
            generalization_winner,
            name="lightgbm_sqrt_gap_l1",
            stage="H6_family",
            why=(
                "개발 데이터의 좌석 변화량 분산이 gap의 약 0.5제곱으로 "
                "증가한다. delta/sqrt(gap)으로 이분산을 줄이고 sample weight에 "
                "sqrt(gap)을 곱해 최종 좌석 MAE 목적을 보존한다."
            ),
            if_works=(
                "기존 delta/gap LightGBM보다 전체·장거리 MAE가 감소하고, "
                "ExtraTrees/HGB와 혼합할 때 service score도 낮아진다."
            ),
            if_fails=(
                "변화가 특정 정류장에 집중돼 단일 sqrt 거리 스케일이 맞지 "
                "않거나 장거리 가중이 저잔여 표본을 희석한다."
            ),
            model_kind="lightgbm",
            target_kind="delta_per_sqrt_stop",
            gap_weight_power=0.5,
            params={
                "objective": "regression_l1",
                "n_estimators": 600,
                "learning_rate": 0.025,
                "num_leaves": 63,
                "min_child_samples": 10,
                "subsample": 0.85,
                "subsample_freq": 1,
                "colsample_bytree": 0.8,
                "reg_alpha": 0.1,
                "reg_lambda": 1.0,
            },
        ),
        candidate_from(
            generalization_winner,
            name="lightgbm_regularized_l1",
            stage="H6_family",
            why="더 규제된 LightGBM으로 leaf-wise 모델의 날짜 변동을 줄일 수 있는지 분리 검증한다.",
            if_works="깊은 LightGBM보다 train 오차는 높아도 날짜별 분산과 service score가 낮다.",
            if_fails="추가 규제가 희귀 저잔여 분기까지 지운다.",
            model_kind="lightgbm",
            params={
                "objective": "regression_l1",
                "n_estimators": 500,
                "learning_rate": 0.03,
                "num_leaves": 31,
                "min_child_samples": 30,
                "subsample": 0.9,
                "subsample_freq": 1,
                "colsample_bytree": 0.8,
                "reg_alpha": 0.2,
                "reg_lambda": 3.0,
            },
        ),
    ]
    evaluate(tree_candidates)
    results = pd.DataFrame(result_rows)

    best_individual_name = best_name(
        results,
        [candidate.name for candidate in registry.values()],
    )
    hgb_names = [
        candidate.name
        for candidate in registry.values()
        if candidate.model_kind == "hgb"
    ]
    best_hgb_name = best_name(results, hgb_names)
    tree_names = [candidate.name for candidate in tree_candidates]
    best_tree_name = best_name(results, tree_names)

    ensemble_specs: dict[str, EnsembleSpec] = {}
    ensemble_rows: list[dict[str, Any]] = []
    ensemble_oof: dict[str, pd.DataFrame] = {}
    for tree_name in tree_names:
        tree_result = results.loc[results["candidate"].eq(tree_name)].iloc[0]
        for weight in np.linspace(0, 1, 11):
            name = (
                f"blend_{tree_name}_hgb_{int(weight * 100):03d}"
            )
            spec = EnsembleSpec(
                name=name,
                kind="linear",
                components=(best_hgb_name, tree_name),
                params={"left_weight": float(weight)},
            )
            output = combine_oof(
                name,
                oof_predictions[best_hgb_name],
                oof_predictions[tree_name],
                left_weight=float(weight),
            )
            row, output = score_ensemble(spec, output)
            row["mean_fold_tree_nodes"] = float(
                tree_result["mean_fold_tree_nodes"]
            )
            row["prediction_ms_per_1000"] = float(
                results.loc[
                    results["candidate"].isin([best_hgb_name, tree_name]),
                    "prediction_ms_per_1000",
                ].sum()
            )
            ensemble_specs[name] = spec
            ensemble_rows.append(row)
            ensemble_oof[name] = output

    route_oof = oof_predictions["route_stop_profile"]
    for weight in np.linspace(0, 1, 11):
        name = f"blend_hgb_route_profile_{int(weight * 100):03d}"
        spec = EnsembleSpec(
            name=name,
            kind="linear",
            components=(best_hgb_name, "route_stop_profile"),
            params={"left_weight": float(weight)},
        )
        output = combine_oof(
            name,
            oof_predictions[best_hgb_name],
            route_oof,
            left_weight=float(weight),
        )
        row, output = score_ensemble(spec, output)
        row["mean_fold_tree_nodes"] = float(
            results.loc[
                results["candidate"].eq(best_hgb_name),
                "mean_fold_tree_nodes",
            ].iloc[0]
        )
        row["prediction_ms_per_1000"] = float(
            results.loc[
                results["candidate"].eq(best_hgb_name),
                "prediction_ms_per_1000",
            ].iloc[0]
        )
        ensemble_specs[name] = spec
        ensemble_rows.append(row)
        ensemble_oof[name] = output

    model_oof = oof_predictions[best_individual_name]
    persistence_oof = oof_predictions["persistence"]
    for near_weight in (0.0, 0.25, 0.5, 0.75, 1.0):
        for far_weight in (0.5, 0.75, 1.0):
            name = (
                f"distance_gate_near_{int(near_weight * 100):03d}"
                f"_far_{int(far_weight * 100):03d}"
            )
            weights = np.where(
                model_oof["target_stop_gap"].le(2).to_numpy(),
                near_weight,
                far_weight,
            )
            spec = EnsembleSpec(
                name=name,
                kind="distance_gate",
                components=(best_individual_name, "persistence"),
                params={
                    "near_left_weight": near_weight,
                    "far_left_weight": far_weight,
                },
            )
            output = combine_oof(
                name,
                model_oof,
                persistence_oof,
                left_weight=weights,
            )
            row, output = score_ensemble(spec, output)
            ensemble_specs[name] = spec
            ensemble_rows.append(row)
            ensemble_oof[name] = output

    for specialist_name in (
        "hgb_far_weight_2",
        "hgb_far_weight_4",
        "hgb_route_residual",
        "hgb_route_residual_per_stop",
    ):
        for threshold in (6, 11):
            for specialist_weight in (0.25, 0.5, 0.75, 1.0):
                name = (
                    f"stop_gate_{specialist_name}_gap_{threshold}_"
                    f"weight_{int(specialist_weight * 100):03d}"
                )
                base_weight = np.where(
                    oof_predictions[feature_winner_name]["target_stop_gap"]
                    .lt(threshold)
                    .to_numpy(),
                    1.0,
                    1.0 - specialist_weight,
                )
                spec = EnsembleSpec(
                    name=name,
                    kind="stop_gap_gate",
                    components=(feature_winner_name, specialist_name),
                    params={
                        "threshold": float(threshold),
                        "below_left_weight": 1.0,
                        "above_left_weight": float(1.0 - specialist_weight),
                    },
                )
                output = combine_oof(
                    name,
                    oof_predictions[feature_winner_name],
                    oof_predictions[specialist_name],
                    left_weight=base_weight,
                )
                row, output = score_ensemble(spec, output)
                row["mean_fold_tree_nodes"] = float(
                    results.loc[
                        results["candidate"].isin(spec.components),
                        "mean_fold_tree_nodes",
                    ].sum()
                )
                row["prediction_ms_per_1000"] = float(
                    results.loc[
                        results["candidate"].isin(spec.components),
                        "prediction_ms_per_1000",
                    ].sum()
                )
                ensemble_specs[name] = spec
                ensemble_rows.append(row)
                ensemble_oof[name] = output

    stack_components = (
        best_hgb_name,
        "extra_trees_tiny",
        "lightgbm_leafwise_l1",
    )
    for hgb_units in range(11):
        for extra_units in range(11 - hgb_units):
            lightgbm_units = 10 - hgb_units - extra_units
            raw_weights = {
                stack_components[0]: hgb_units / 10,
                stack_components[1]: extra_units / 10,
                stack_components[2]: lightgbm_units / 10,
            }
            weights = {
                component: weight
                for component, weight in raw_weights.items()
                if weight > 0
            }
            components = tuple(weights)
            name = (
                f"stack3_hgb_{hgb_units * 10:03d}_"
                f"extra_{extra_units * 10:03d}_"
                f"lgb_{lightgbm_units * 10:03d}"
            )
            spec = EnsembleSpec(
                name=name,
                kind="weighted",
                components=components,
                params=weights,
            )
            output = combine_weighted_oof(
                name,
                {component: oof_predictions[component] for component in components},
                weights,
            )
            row, output = score_ensemble(spec, output)
            row["mean_fold_tree_nodes"] = float(
                results.loc[
                    results["candidate"].isin(components),
                    "mean_fold_tree_nodes",
                ].sum()
            )
            row["prediction_ms_per_1000"] = float(
                results.loc[
                    results["candidate"].isin(components),
                    "prediction_ms_per_1000",
                ].sum()
            )
            ensemble_specs[name] = spec
            ensemble_rows.append(row)
            ensemble_oof[name] = output

    # 압축 ExtraTrees와 sqrt-gap LightGBM은 각각 별도의 가설에서 나온
    # 후보다. 전 조합 grid를 다시 훑지 않고, 기존 최적점 주변에서 두
    # 구성요소의 증분 기여를 분리하는 세 조합만 검증한다.
    focused_stack_components = (
        best_hgb_name,
        "extra_trees_compact_70",
        "lightgbm_sqrt_gap_l1",
    )
    focused_stack_weights = (
        (0.5, 0.4, 0.1),
        (0.4, 0.5, 0.1),
        (0.4, 0.4, 0.2),
    )
    for hgb_weight, extra_weight, lightgbm_weight in focused_stack_weights:
        weights = dict(
            zip(
                focused_stack_components,
                (hgb_weight, extra_weight, lightgbm_weight),
                strict=True,
            )
        )
        name = (
            f"stack_sqrt_hgb_{int(hgb_weight * 100):03d}_"
            f"extra_{int(extra_weight * 100):03d}_"
            f"lgb_{int(lightgbm_weight * 100):03d}"
        )
        spec = EnsembleSpec(
            name=name,
            kind="weighted",
            components=focused_stack_components,
            params=weights,
        )
        output = combine_weighted_oof(
            name,
            {
                component: oof_predictions[component]
                for component in focused_stack_components
            },
            weights,
        )
        row, output = score_ensemble(spec, output)
        row["mean_fold_tree_nodes"] = float(
            results.loc[
                results["candidate"].isin(focused_stack_components),
                "mean_fold_tree_nodes",
            ].sum()
        )
        row["prediction_ms_per_1000"] = float(
            results.loc[
                results["candidate"].isin(focused_stack_components),
                "prediction_ms_per_1000",
            ].sum()
        )
        ensemble_specs[name] = spec
        ensemble_rows.append(row)
        ensemble_oof[name] = output

    # 최종 재학습 artifact에서 70-tree stack이 100만-node를 넘는 것을
    # 확인한 뒤, 가중치를 다시 탐색하지 않고 트리 수만 48로 줄인 동일
    # 40:40:20 조합을 배포 제약 대안으로 검증한다.
    compact_48_components = (
        best_hgb_name,
        "extra_trees_compact_48",
        "lightgbm_sqrt_gap_l1",
    )
    compact_48_weights = dict(
        zip(compact_48_components, (0.4, 0.4, 0.2), strict=True)
    )
    compact_48_name = "stack_sqrt48_hgb_040_extra_040_lgb_020"
    compact_48_spec = EnsembleSpec(
        name=compact_48_name,
        kind="weighted",
        components=compact_48_components,
        params=compact_48_weights,
    )
    compact_48_output = combine_weighted_oof(
        compact_48_name,
        {
            component: oof_predictions[component]
            for component in compact_48_components
        },
        compact_48_weights,
    )
    compact_48_row, compact_48_output = score_ensemble(
        compact_48_spec, compact_48_output
    )
    compact_48_row["mean_fold_tree_nodes"] = float(
        results.loc[
            results["candidate"].isin(compact_48_components),
            "mean_fold_tree_nodes",
        ].sum()
    )
    compact_48_row["prediction_ms_per_1000"] = float(
        results.loc[
            results["candidate"].isin(compact_48_components),
            "prediction_ms_per_1000",
        ].sum()
    )
    ensemble_specs[compact_48_name] = compact_48_spec
    ensemble_rows.append(compact_48_row)
    ensemble_oof[compact_48_name] = compact_48_output

    # H8: Route 1000's current fleet repeatedly exposes 0..44/0..70 point
    # support. This is not a GBIS-wide capacity contract, so the raw-feed audit
    # must pass before the no-parameter output projection is even registered.
    if output_support_audit["selection_candidate_eligible"]:
        observed_cap_name = f"{compact_48_name}_observed_cap44_70"
        observed_cap_params = {
            **compact_48_weights,
            OBSERVED_SEAT_CEILING_PARAM: 1.0,
        }
        observed_cap_spec = EnsembleSpec(
            name=observed_cap_name,
            kind="weighted",
            components=compact_48_components,
            params=observed_cap_params,
        )
        observed_cap_output = combine_weighted_oof(
            observed_cap_name,
            {
                component: oof_predictions[component]
                for component in compact_48_components
            },
            compact_48_weights,
        )
        observed_cap_row, observed_cap_output = score_ensemble(
            observed_cap_spec, observed_cap_output
        )
        observed_cap_row.update(
            {
                "stage": "H8_output_support",
                "why": (
                    "1000번 현행 일반차/2층차의 경험적 점 지지집합은 개발 선택 "
                    "전인 2026-08-04까지 이미 0~44/0~70석으로 확인됐으므로 "
                    "명목 45석에서 생기는 범위 밖 점 예측을 제거한다."
                ),
                "if_works": (
                    "44석 초과 일반차 예측만 줄고 모든 날짜와 스트레스 "
                    "구간에서 오차가 비악화한다."
                ),
                "if_fails": (
                    "현행 fleet 지지집합이 바뀌었거나 상한 근처 예측이 이미 "
                    "충분히 보정되어 있다."
                ),
                "mean_fold_tree_nodes": compact_48_row[
                    "mean_fold_tree_nodes"
                ],
                "prediction_ms_per_1000": compact_48_row[
                    "prediction_ms_per_1000"
                ],
                "output_support_scope": "route_219000013_current_fleet",
                "output_support_evidence_cutoff": (
                    OBSERVED_SEAT_CEILING_EVIDENCE_CUTOFF
                ),
            }
        )
        ensemble_specs[observed_cap_name] = observed_cap_spec
        ensemble_rows.append(observed_cap_row)
        ensemble_oof[observed_cap_name] = observed_cap_output

    ensemble_results = pd.DataFrame(ensemble_rows)
    all_selection = pd.concat([results, ensemble_results], ignore_index=True)
    research_best_name = str(
        all_selection.sort_values(["service_score", "robust_score"]).iloc[0][
            "candidate"
        ]
    )
    maximum_final_fit_tree_nodes = 1_000_000
    final_targets = {"test": test, "stress": stress}
    development_flows = stop_flows.loc[
        stop_flows["date"].lt(args.test_date)
    ].copy()
    final_predictions: dict[str, dict[str, np.ndarray]] = {}
    fitted_models: dict[str, Pipeline] = {}
    final_metadata: dict[str, dict[str, Any]] = {}
    final_component_tree_nodes: dict[str, int] = {}

    def ensure_final_fit(name: str) -> None:
        if name in final_predictions:
            return
        candidate = registry[name]
        result = results.loc[results["candidate"].eq(name)].iloc[0]
        predictions, model, model_metadata = fit_final_candidate(
            candidate,
            development,
            final_targets,
            development_flows,
            bias=float(result["bias_correction"]),
            seed=args.seed,
        )
        final_predictions[name] = predictions
        final_component_tree_nodes[name] = tree_node_count(model)
        if model is not None:
            fitted_models[name] = model
        model_metadata.update(
            {
                "candidate": asdict(candidate),
                "bias_correction": float(result["bias_correction"]),
                "bias_correction_source": str(
                    result["bias_correction_source"]
                ),
                "oof_bias_correction_rule": str(
                    result["oof_bias_correction_rule"]
                ),
                "final_fit_tree_nodes": final_component_tree_nodes[name],
            }
        )
        final_metadata[name] = model_metadata

    selected_name = select_with_final_node_budget(
        all_selection,
        ensemble_specs,
        ensure_final_fit=ensure_final_fit,
        final_component_tree_nodes=final_component_tree_nodes,
        maximum_final_fit_tree_nodes=maximum_final_fit_tree_nodes,
    )

    component_names = {"persistence", best_individual_name}
    if selected_name in ensemble_specs:
        component_names.update(ensemble_specs[selected_name].components)
    else:
        component_names.add(selected_name)
    for name in component_names:
        ensure_final_fit(name)

    for old_artifact in args.output_dir.glob("*.joblib"):
        old_artifact.unlink()
    for old_metadata in args.output_dir.glob("*.metadata.json"):
        old_metadata.unlink()
    artifact_sizes: dict[str, int] = {}
    requires_route_profile = False
    for name in sorted(component_names):
        candidate = registry[name]
        requires_route_profile = (
            requires_route_profile or candidate.feature_variant == "route_profile"
        )
        model = fitted_models.get(name)
        if model is not None:
            artifact_path = args.output_dir / f"{name}.joblib"
            joblib.dump(model, artifact_path, compress=3)
            artifact_sizes[name] = int(artifact_path.stat().st_size)
        (args.output_dir / f"{name}.metadata.json").write_text(
            json.dumps(
                json_ready(final_metadata[name]), ensure_ascii=False, indent=2
            ),
            encoding="utf-8",
        )
    if requires_route_profile:
        flow_artifact = args.output_dir / "route_profile_flows.joblib"
        joblib.dump(development_flows, flow_artifact, compress=3)
        artifact_sizes["route_profile_flows"] = int(flow_artifact.stat().st_size)

    saved_artifact_sha256 = {
        path.name: cache_payload_sha256(path)
        for path in sorted(
            [
                *args.output_dir.glob("*.joblib"),
                *args.output_dir.glob("*.metadata.json"),
            ],
            key=lambda value: value.name,
        )
    }

    if selected_name in ensemble_specs:
        selected_spec = ensemble_specs[selected_name]
        selected_test = ensemble_prediction(
            selected_spec,
            {name: final_predictions[name]["test"] for name in selected_spec.components},
            test,
        )
        selected_stress = ensemble_prediction(
            selected_spec,
            {name: final_predictions[name]["stress"] for name in selected_spec.components},
            stress,
        ) if not stress.empty else np.asarray([], dtype=float)
    else:
        selected_spec = None
        selected_test = final_predictions[selected_name]["test"]
        selected_stress = final_predictions[selected_name]["stress"]

    final_rows: list[dict[str, Any]] = []
    final_output_names = ["persistence", best_individual_name, selected_name]
    for name in dict.fromkeys(final_output_names):
        if name == selected_name:
            prediction = selected_test
        else:
            prediction = final_predictions[name]["test"]
        row = event_summary(test, prediction)
        row.update({"model": name, "split": args.test_date})
        final_rows.append(row)
    final_metrics = pd.DataFrame(final_rows)

    stress_metrics = pd.DataFrame()
    if not stress.empty:
        stress_row = event_summary(stress, selected_stress)
        stress_row.update(
            {
                "model": selected_name,
                "split": "weekend_2026-08-08_09",
            }
        )
        stress_metrics = pd.DataFrame([stress_row])

    by_stop = stop_band_metrics(test, selected_test, selected_name)
    persistence_test = final_predictions["persistence"]["test"]
    bootstrap = cluster_bootstrap_mae_delta(
        test,
        selected_test,
        persistence_test,
        seed=args.seed,
    )
    bootstrap_best_individual = cluster_bootstrap_mae_delta(
        test,
        selected_test,
        final_predictions[best_individual_name]["test"],
        seed=args.seed,
    )
    prediction_output = test[
        [
            "event_id",
            "date",
            "event_time",
            "snapshot_time",
            "trip_id",
            "station_seq_cat",
            "snapshot_station_seq",
            "target_stop_gap",
            "minutes_to_arrival",
            "label_seats",
            "snapshot_remaining_seats",
            "previous_bus_departure_seats",
        ]
    ].copy()
    prediction_output["persistence"] = persistence_test
    prediction_output["selected_prediction"] = selected_test
    prediction_output["absolute_error"] = np.abs(
        prediction_output["label_seats"] - selected_test
    )

    selected_oof = (
        ensemble_oof[selected_name]
        if selected_name in ensemble_oof
        else oof_predictions[selected_name]
    ).copy()
    persistence_oof_aligned = (
        oof_predictions["persistence"]
        .set_index(["event_id", "snapshot_time"])["prediction"]
        .reindex(
            pd.MultiIndex.from_frame(
                selected_oof[["event_id", "snapshot_time"]]
            )
        )
        .to_numpy(dtype=float)
    )
    selected_oof["persistence_prediction"] = persistence_oof_aligned
    selected_oof["absolute_error"] = np.abs(
        selected_oof["label_seats"] - selected_oof["prediction"]
    )

    deployment_columns = all_selection[
        [
            "candidate",
            "final_fit_tree_nodes",
            "deployment_eligible",
            "deployment_check",
        ]
    ]
    results = results.merge(deployment_columns, on="candidate", how="left")
    ensemble_results = ensemble_results.merge(
        deployment_columns, on="candidate", how="left"
    )
    results["stage_rank"] = results.groupby("stage")["service_score"].rank(
        method="min"
    )
    results["verdict"] = np.where(
        results["stage_rank"].eq(1),
        "stage_best",
        "not_selected",
    )
    fold_metrics = pd.DataFrame(fold_rows)
    results.to_csv(args.output_dir / "hypothesis_results.csv", index=False)
    fold_metrics.to_csv(args.output_dir / "fold_metrics.csv", index=False)
    ensemble_results.to_csv(args.output_dir / "ensemble_results.csv", index=False)
    final_metrics.to_csv(args.output_dir / "final_test_metrics.csv", index=False)
    stress_metrics.to_csv(args.output_dir / "weekend_stress_metrics.csv", index=False)
    by_stop.to_csv(args.output_dir / "final_metrics_by_stop_band.csv", index=False)
    prediction_output.to_csv(
        args.output_dir / "final_test_predictions.csv", index=False
    )
    selected_oof.to_csv(
        args.output_dir / "selected_oof_predictions.csv", index=False
    )

    selected_development = all_selection.loc[
        all_selection["candidate"].eq(selected_name)
    ].iloc[0]
    selected_components = (
        selected_spec.components
        if selected_spec is not None
        else (selected_name,)
    )
    selected_artifact_sizes = {
        name: artifact_sizes[name]
        for name in selected_components
        if name in artifact_sizes
    }
    deployment_evaluations = all_selection.loc[
        all_selection["deployment_check"].ne(
            "not_evaluated_after_selection"
        ),
        [
            "candidate",
            "final_fit_tree_nodes",
            "deployment_eligible",
            "deployment_check",
        ],
    ]
    summary = {
        "route_id": ROUTE_ID,
        "route_name": ROUTE_NAME,
        "target": "arrival_seats before boarding",
        "label_quality": ["A"],
        "data_cache": cache_info,
        "protocol": {
            "experiment_source_sha256": experiment_source_sha256,
            "inference_source_sha256": inference_source_sha256,
            "hypothesis_and_parameter_selection": {
                "rolling_origin_validation_dates": list(development_dates[1:]),
                "training_rule": "all earlier development dates",
                "oof_bias_rule": STRICT_FORWARD_BIAS_RULE,
                "deployment_bias_source": DEPLOYMENT_BIAS_SOURCE,
                "selection_score": (
                    "mean daily event-balanced MAE + 0.5 * day std + "
                    "0.05 * low-seat MAE"
                ),
            },
            "locked_confirmation_test": {
                "date": args.test_date,
                "selection_use": "none",
                "caveat": (
                    "partial-day data; label counts were inspected before scoring, "
                    "so this is not a pristine holdout"
                ),
            },
            "weekend_distribution_shift_stress": list(DEFAULT_STRESS_DATES),
            "leakage_rules": [
                "actual minutes_to_arrival is evaluation-only",
                "historical target profiles are train-only and leave-one-row-out in training",
                "previous-bus departures must be strictly earlier than snapshot_time",
                "OOF bias for each validation date uses only earlier OOF residuals",
                "the final test date is not used for candidate, parameter, or blend selection",
                (
                    "the Route 1000 44/70 output support was evidenced no later "
                    f"than {OBSERVED_SEAT_CEILING_EVIDENCE_CUTOFF}"
                ),
            ],
        },
        "output_support_constraint": {
            "selected_model_enabled": bool(
                selected_spec is not None
                and float(
                    selected_spec.params.get(OBSERVED_SEAT_CEILING_PARAM, 0.0)
                )
                != 0.0
            ),
            "scope": "route_219000013_current_fleet_only",
            "mapping": {"snapshot_low_plate_cat_0": 44, "category_2": 70},
            "evidence_cutoff": OBSERVED_SEAT_CEILING_EVIDENCE_CUTOFF,
            "unknown_category_rule": "fail_closed",
            "not_a_gbis_wide_capacity_contract": True,
            "raw_source_audit": output_support_audit,
            "source_fingerprint": cache_info.get("fingerprint"),
            "point_projection_only": (
                "prediction intervals retain the existing nominal-capacity support"
            ),
        },
        "data": {
            "development_rows": int(len(development)),
            "development_events": int(development["event_id"].nunique()),
            "test_rows": int(len(test)),
            "test_events": int(test["event_id"].nunique()),
            "test_low_0_5_events": int(
                test.loc[test["label_seats"].le(5), "event_id"].nunique()
            ),
            "stress_rows": int(len(stress)),
            "stress_events": int(stress["event_id"].nunique()),
        },
        "stage_winners": {
            "target_representation": target_winner_name,
            "capacity_and_loss": capacity_winner_name,
            "rare_weighting": weighting_winner_name,
            "feature_engineering": feature_winner_name,
            "generalization": generalization_winner_name,
            "best_hgb": best_hgb_name,
            "best_tree": best_tree_name,
            "best_individual": best_individual_name,
            "research_best_without_deployment_constraint": research_best_name,
            "selected_final": selected_name,
        },
        "deployment_constraint": {
            "rule": (
                "lazy score-ordered selection using actual full-development "
                "fitted component node counts"
            ),
            "maximum_final_fit_tree_nodes": maximum_final_fit_tree_nodes,
            "selected_final_fit_tree_nodes": int(
                selected_development["final_fit_tree_nodes"]
            ),
            "selected_component_tree_nodes": {
                name: final_component_tree_nodes[name]
                for name in selected_components
            },
            "selected_artifact_sizes_bytes": selected_artifact_sizes,
            "all_saved_artifact_sizes_bytes": artifact_sizes,
            "saved_artifact_sha256": saved_artifact_sha256,
            "saved_artifact_manifest_rule": (
                "all joblib and component metadata files present after stale "
                "artifacts are removed and before summary publication"
            ),
            "evaluated_candidates": _as_records(deployment_evaluations),
        },
        "selected_development_metrics": json_ready(selected_development.to_dict()),
        "selected_ensemble": (
            asdict(selected_spec) if selected_spec is not None else None
        ),
        "final_test_metrics": _as_records(final_metrics),
        "weekend_stress_metrics": _as_records(stress_metrics),
        "final_metrics_by_stop_band": _as_records(by_stop),
        "bootstrap_improvement_vs_persistence": bootstrap,
        "bootstrap_improvement_vs_best_individual": bootstrap_best_individual,
        "limitations": [
            "잠금 확인셋은 2026-08-11 오전 일부이며 평가 전 라벨 건수는 확인했다.",
            "2026-08-12 이후의 완전히 미개봉 평일 데이터가 아직 없다.",
            "학기·날씨·승하차·대기열 데이터는 아직 없다.",
            "저잔여 사건 수가 작아 서비스 경고 확률은 별도 장기 보정이 필요하다.",
            "모델 선택용 rolling 검증일은 4개라 날짜 분산 추정이 아직 불안정하다.",
        ],
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(json_ready(summary), ensure_ascii=False, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/latest_main_model_feature_recheck.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/latest_main_model_feature_recheck.py
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

import pandas as pd

from hypothesis_model_search import add_observed_capacity_features
from linear_feature_experiment import json_ready
from main_model_feature_augmentation import (
    feature_sets,
    paired_trip_bootstrap,
    prepare_augmented,
    required_metrics,
    run_feature_set,
)


ROLLING_DATES = (
    "2026-08-04",
    "2026-08-05",
    "2026-08-06",
    "2026-08-07",
    "2026-08-10",
    "2026-08-11",
    "2026-08-12",
    "2026-08-13",
)
LATEST_COMPLETE_DATES = ("2026-08-11", "2026-08-12")
LATEST_PARTIAL_DATE = "2026-08-13"
DEFAULT_FEATURE_SETS = (
    "baseline_official",
    "target_low_10_only",
    "path_low_10_sum_only",
    "low_rate_without_count",
    "correlation_pruned",
    "augmented_unique",
)


def rolling_folds(data: pd.DataFrame) -> list[tuple[str, pd.DataFrame, pd.DataFrame]]:
    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]] = []
    for position, validation_date in enumerate(ROLLING_DATES[1:], start=1):
        train = data.loc[data["date"].isin(ROLLING_DATES[:position])].copy()
        validation = data.loc[data["date"].eq(validation_date)].copy()
        if train.empty or validation.empty:
            raise ValueError(f"rolling fold is empty: {validation_date}")
        folds.append((validation_date, train, validation))
    return folds


def partition_metrics(predictions: pd.DataFrame) -> pd.DataFrame:
    partitions = {
        "latest_complete_08_11_12": predictions["date"].isin(LATEST_COMPLETE_DATES),
        "latest_partial_08_13": predictions["date"].eq(LATEST_PARTIAL_DATE),
        "latest_all_08_11_13": predictions["date"].isin(
            (*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE)
        ),
    }
    rows: list[dict[str, Any]] = []
    for partition, mask in partitions.items():
        for candidate, frame in predictions.loc[mask].groupby("candidate", sort=False):
            rows.append(
                {"partition": partition, "feature_set": candidate, **required_metrics(frame)}
            )
    return pd.DataFrame(rows)


def main() -> int:
    parser = argparse.ArgumentParser(
        description="최신 완전일과 부분일에서 주 모델 신규 피처를 rolling-origin 재검증"
    )
    parser.add_argument(
        "--snapshot-cache",
        type=Path,
        default=Path("data/analysis_cache/route_specific_features/219000013_snapshots.pkl"),
    )
    parser.add_argument(
        "--flow-cache",
        type=Path,
        default=Path("data/analysis_cache/route_specific_features/219000013_flows.pkl"),
    )
    parser.add_argument(
        "--augmented-cache",
        type=Path,
        default=Path("data/analysis_cache/main_model_augmented_features_latest.pkl"),
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("analysis/latest_main_model_feature_results"),
    )
    parser.add_argument("--feature-sets", nargs="*", default=list(DEFAULT_FEATURE_SETS))
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    available = feature_sets()
    unknown = sorted(set(args.feature_sets) - set(available))
    if unknown:
        raise ValueError(f"unknown feature sets: {unknown}")
    data = prepare_augmented(
        args.snapshot_cache, args.flow_cache, args.augmented_cache
    )
    data = add_observed_capacity_features(data)
    missing_dates = sorted(set(ROLLING_DATES) - set(data["date"]))
    if missing_dates:
        raise ValueError(f"missing rolling dates: {missing_dates}")
    folds = rolling_folds(data)

    predictions: list[pd.DataFrame] = []
    daily: list[pd.DataFrame] = []
    for name in args.feature_sets:
        output, _, candidate_daily, _ = run_feature_set(
            name, available[name], folds, seed=args.seed
        )
        predictions.append(output)
        daily.append(candidate_daily)
    prediction_table = pd.concat(predictions, ignore_index=True)
    daily_table = pd.concat(daily, ignore_index=True)
    metrics = partition_metrics(prediction_table)

    bootstrap_rows: list[dict[str, Any]] = []
    for partition, dates in {
        "latest_complete_08_11_12": LATEST_COMPLETE_DATES,
        "latest_partial_08_13": (LATEST_PARTIAL_DATE,),
        "latest_all_08_11_13": (*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE),
    }.items():
        selected = prediction_table.loc[prediction_table["date"].isin(dates)]
        for candidate in args.feature_sets:
            if candidate == "baseline_official":
                continue
            for low_only in (False, True):
                row = paired_trip_bootstrap(
                    selected,
                    candidate,
                    low_only=low_only,
                    seed=args.seed,
                )
                bootstrap_rows.append({"partition": partition, **row})
    bootstrap = pd.DataFrame(bootstrap_rows)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(args.output_dir / "metrics_by_partition.csv", index=False)
    daily_table.to_csv(args.output_dir / "daily_metrics.csv", index=False)
    bootstrap.to_csv(args.output_dir / "paired_trip_bootstrap.csv", index=False)
    prediction_table.to_pickle(args.output_dir / "oof_predictions.pkl")
    summary = {
        "protocol": {
            "rolling_dates": ROLLING_DATES,
            "latest_complete_dates": LATEST_COMPLETE_DATES,
            "latest_partial_date": LATEST_PARTIAL_DATE,
            "training_rule": "each fold uses only prior listed weekdays",
            "feature_history_rule": "same-route strictly earlier calendar dates",
            "selection_caveat": "all dates have been inspected in prior experiments; comparison is retrospective, not pristine prospective validation",
        },
        "source": {
            "snapshot_cache": str(args.snapshot_cache),
            "rows": int(len(data)),
            "events": int(data["event_id"].nunique()),
            "min_date": str(data["date"].min()),
            "max_date": str(data["date"].max()),
            "max_snapshot_time": pd.Timestamp(data["snapshot_time"].max()).isoformat(),
            "preceding_bus_segment_audit": data.attrs.get("preceding_bus_segment_audit"),
        },
        "feature_sets": args.feature_sets,
        "metrics": metrics.to_dict(orient="records"),
        "bootstrap": bootstrap.to_dict(orient="records"),
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(metrics.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/latest_main_model_overfit_ablation.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/latest_main_model_overfit_ablation.py
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from hypothesis_model_search import (
    OBSERVED_SEAT_CEILING_PARAM,
    EnsembleSpec,
    add_observed_capacity_features,
    combine_weighted_oof,
    postprocess_ensemble_prediction,
    strict_forward_bias_predictions,
)
from latest_main_model_feature_recheck import (
    LATEST_COMPLETE_DATES,
    LATEST_PARTIAL_DATE,
    rolling_folds,
)
from main_model_feature_augmentation import (
    component_candidates,
    feature_sets,
    prepare_augmented,
    required_metrics,
    run_component,
)
from linear_feature_experiment import json_ready
from model_feasibility import FeatureSet


OBSERVED_CEILING_FEATURES = (
    "observed_ceiling_capacity",
    "observed_ceiling_load_ratio",
    "observed_ceiling_load_gap",
)
COORDINATE_FEATURES = ("x", "y")
PRIMARY_NAME = "primary_40_empirical_cap"


def ablation_candidates() -> dict[str, tuple[FeatureSet, bool]]:
    primary = feature_sets()["importance_pruned"]

    def without(name: str, removed: tuple[str, ...]) -> FeatureSet:
        numeric = tuple(column for column in primary.numeric if column not in removed)
        return FeatureSet(name=name, numeric=numeric, categorical=primary.categorical)

    no_observed = without("no_observed_features_37", OBSERVED_CEILING_FEATURES)
    no_coordinates = without("no_coordinates_38", COORDINATE_FEATURES)
    clean = without(
        "clean_35",
        (*OBSERVED_CEILING_FEATURES, *COORDINATE_FEATURES),
    )
    return {
        PRIMARY_NAME: (primary, True),
        "primary_40_nominal_cap": (primary, False),
        "no_observed_features_37_empirical_cap": (no_observed, True),
        "no_observed_features_37_nominal_cap": (no_observed, False),
        "no_coordinates_38_empirical_cap": (no_coordinates, True),
        "clean_35_empirical_cap": (clean, True),
        "clean_35_nominal_cap": (clean, False),
    }


def train_schema(
    schema_name: str,
    features: FeatureSet,
    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]],
    *,
    seed: int,
) -> pd.DataFrame:
    components: dict[str, pd.DataFrame] = {}
    # Keep the exact v1.0.0 model families and parameters.
    for candidate in component_candidates():
        print(f"[{schema_name}] {candidate.name}", flush=True)
        output, _, _ = run_component(
            candidate,
            features,
            folds,
            feature_set_name=schema_name,
            seed=seed,
        )
        components[candidate.name] = output

    weights = {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2}
    blended = combine_weighted_oof(schema_name, components, weights)
    strict, _ = strict_forward_bias_predictions(blended, blended["prediction"])
    blended["prediction"] = strict
    return blended


def apply_candidate(
    name: str,
    features: FeatureSet,
    empirical_cap: bool,
    schema_prediction: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    blended = schema_prediction.copy()
    weights = {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2}
    params: dict[str, float] = dict(weights)
    if empirical_cap:
        params[OBSERVED_SEAT_CEILING_PARAM] = 1.0
    spec = EnsembleSpec(
        name=name,
        kind="weighted",
        components=tuple(weights),
        params=params,
    )
    blended["prediction"] = postprocess_ensemble_prediction(
        spec, blended["prediction"].to_numpy(dtype=float), blended
    )
    blended["candidate"] = name
    blended["empirical_cap"] = empirical_cap
    blended["feature_count"] = len(features.columns)
    daily = pd.DataFrame(
        [
            {
                "candidate": name,
                "validation_date": date,
                "empirical_cap": empirical_cap,
                "feature_count": len(features.columns),
                **required_metrics(frame),
            }
            for date, frame in blended.groupby("date", sort=True)
        ]
    )
    return blended, daily


def partition_metrics(predictions: pd.DataFrame) -> pd.DataFrame:
    partitions = {
        "latest_complete_08_11_12": predictions["date"].isin(LATEST_COMPLETE_DATES),
        "latest_partial_08_13": predictions["date"].eq(LATEST_PARTIAL_DATE),
        "latest_all_08_11_13": predictions["date"].isin(
            (*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE)
        ),
    }
    rows: list[dict[str, Any]] = []
    for partition, mask in partitions.items():
        for candidate, frame in predictions.loc[mask].groupby("candidate", sort=False):
            rows.append(
                {
                    "partition": partition,
                    "candidate": candidate,
                    "feature_count": int(frame["feature_count"].iloc[0]),
                    "empirical_cap": bool(frame["empirical_cap"].iloc[0]),
                    **required_metrics(frame),
                }
            )
    return pd.DataFrame(rows)


def paired_trip_bootstrap(
    predictions: pd.DataFrame,
    candidate: str,
    *,
    low_only: bool,
    repeats: int = 2_000,
    seed: int = 42,
) -> dict[str, Any]:
    selected = predictions.loc[
        predictions["candidate"].isin([PRIMARY_NAME, candidate])
    ].copy()
    if low_only:
        selected = selected.loc[selected["label_seats"].le(10)].copy()
    selected["absolute_error"] = (
        selected["label_seats"] - selected["prediction"]
    ).abs()
    per_event = (
        selected.groupby(["trip_id", "event_id", "candidate"], observed=True)[
            "absolute_error"
        ]
        .mean()
        .unstack("candidate")
        .dropna(subset=[PRIMARY_NAME, candidate])
        .reset_index()
    )
    per_event["improvement"] = per_event[PRIMARY_NAME] - per_event[candidate]
    trip_groups = [
        group["improvement"].to_numpy(dtype=float)
        for _, group in per_event.groupby("trip_id", sort=False)
    ]
    rng = np.random.default_rng(seed)
    draws = np.empty(repeats, dtype=float)
    for index in range(repeats):
        sampled = rng.integers(0, len(trip_groups), size=len(trip_groups))
        draws[index] = np.concatenate([trip_groups[item] for item in sampled]).mean()
    return {
        "baseline": PRIMARY_NAME,
        "candidate": candidate,
        "metric": "low_0_10_mae" if low_only else "event_balanced_mae",
        "candidate_improvement": float(per_event["improvement"].mean()),
        "ci_95_lower": float(np.quantile(draws, 0.025)),
        "ci_95_upper": float(np.quantile(draws, 0.975)),
        "probability_candidate_better": float((draws > 0).mean()),
        "trips": int(per_event["trip_id"].nunique()),
        "events": int(len(per_event)),
    }


def main() -> int:
    parser = argparse.ArgumentParser(
        description="1000번 경험적 상한 피처·출력 보정과 좌표 피처 제거 ablation"
    )
    parser.add_argument(
        "--snapshot-cache",
        type=Path,
        default=Path(
            "data/analysis_cache/route_specific_features/219000013_snapshots.pkl"
        ),
    )
    parser.add_argument(
        "--flow-cache",
        type=Path,
        default=Path("data/analysis_cache/route_specific_features/219000013_flows.pkl"),
    )
    parser.add_argument(
        "--augmented-cache",
        type=Path,
        default=Path("data/analysis_cache/main_model_augmented_features_latest.pkl"),
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("analysis/latest_main_model_overfit_ablation_results"),
    )
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    data = prepare_augmented(
        args.snapshot_cache, args.flow_cache, args.augmented_cache
    )
    data = add_observed_capacity_features(data)
    folds = rolling_folds(data)
    candidates = ablation_candidates()

    prediction_frames: list[pd.DataFrame] = []
    daily_frames: list[pd.DataFrame] = []
    schema_predictions: dict[tuple[str, ...], pd.DataFrame] = {}
    for name, (features, empirical_cap) in candidates.items():
        schema_key = tuple(features.columns)
        if schema_key not in schema_predictions:
            schema_predictions[schema_key] = train_schema(
                features.name,
                features,
                folds,
                seed=args.seed,
            )
        prediction, daily = apply_candidate(
            name,
            features,
            empirical_cap,
            schema_predictions[schema_key],
        )
        prediction_frames.append(prediction)
        daily_frames.append(daily)
    predictions = pd.concat(prediction_frames, ignore_index=True)
    daily = pd.concat(daily_frames, ignore_index=True)
    metrics = partition_metrics(predictions)

    latest_complete = predictions.loc[
        predictions["date"].isin(LATEST_COMPLETE_DATES)
    ]
    bootstrap = pd.DataFrame(
        [
            paired_trip_bootstrap(
                latest_complete,
                candidate,
                low_only=low_only,
                seed=args.seed,
            )
            for candidate in candidates
            if candidate != PRIMARY_NAME
            for low_only in (False, True)
        ]
    )

    args.output_dir.mkdir(parents=True, exist_ok=True)
    predictions.to_pickle(args.output_dir / "oof_predictions.pkl")
    daily.to_csv(args.output_dir / "daily_metrics.csv", index=False)
    metrics.to_csv(args.output_dir / "metrics_by_partition.csv", index=False)
    bootstrap.to_csv(args.output_dir / "paired_trip_bootstrap.csv", index=False)
    summary = {
        "protocol": {
            "rolling_folds": [fold[0] for fold in folds],
            "latest_complete_dates": LATEST_COMPLETE_DATES,
            "latest_partial_date": LATEST_PARTIAL_DATE,
            "training_rule": "each fold uses only earlier listed weekdays",
            "feature_history_rule": "same-route strictly earlier calendar dates",
            "model_rule": "fixed v1.0.0 HGB/ExtraTrees/LightGBM parameters and 0.4/0.4/0.2 weights",
        },
        "removed_feature_blocks": {
            "observed_ceiling": OBSERVED_CEILING_FEATURES,
            "coordinates": COORDINATE_FEATURES,
        },
        "candidates": {
            name: {
                "feature_count": len(features.columns),
                "empirical_cap": empirical_cap,
                "numeric": features.numeric,
                "categorical": features.categorical,
            }
            for name, (features, empirical_cap) in candidates.items()
        },
        "metrics": metrics.to_dict(orient="records"),
        "bootstrap": bootstrap.to_dict(orient="records"),
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(metrics.to_string(index=False))
    print("\npaired bootstrap, latest complete")
    print(bootstrap.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/lightgbm_feature_experiment.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/lightgbm_feature_experiment.py
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

import lightgbm as lgb
import numpy as np
import pandas as pd

from linear_feature_experiment import (
    DEVELOPMENT_DATES,
    FEATURE_BLOCKS,
    VALIDATION_DATES,
    add_preceding_bus_segment_features,
    add_strict_prior_flow_features,
    add_strict_prior_low_rate_features,
    cumulative_feature_sets,
    event_weights,
    json_ready,
    metric_row,
)


REGRESSOR_PARAMS: dict[str, Any] = {
    "objective": "regression_l1",
    "n_estimators": 500,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "min_child_samples": 30,
    "reg_alpha": 0.2,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

CLASSIFIER_PARAMS: dict[str, Any] = {
    "objective": "binary",
    "n_estimators": 300,
    "learning_rate": 0.03,
    "num_leaves": 15,
    "min_child_samples": 30,
    "reg_alpha": 0.2,
    "reg_lambda": 3.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}


def run_rolling_origin(
    data: pd.DataFrame,
    *,
    feature_sets: dict[str, tuple[str, ...]] | None = None,
    regressor_params: dict[str, Any] | None = None,
    classifier_params: dict[str, Any] | None = None,
    development_dates: tuple[str, ...] = DEVELOPMENT_DATES,
    validation_dates: tuple[str, ...] = VALIDATION_DATES,
    return_predictions: bool = False,
) -> (
    tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]
    | tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]
):
    """Run the same causal feature ablation with fixed LightGBM models."""
    selected_feature_sets = feature_sets or cumulative_feature_sets()
    regression_config = {**REGRESSOR_PARAMS, **(regressor_params or {})}
    classification_config = {**CLASSIFIER_PARAMS, **(classifier_params or {})}
    metrics: list[dict[str, Any]] = []
    predictions: list[pd.DataFrame] = []
    importances: list[dict[str, Any]] = []

    for validation_date in validation_dates:
        train_dates = [date for date in development_dates if date < validation_date]
        train = data.loc[data["date"].isin(train_dates)].copy()
        validation = data.loc[data["date"].eq(validation_date)].copy()
        if train.empty or validation.empty:
            raise ValueError(f"rolling fold가 비었습니다: {validation_date}")
        train_weight = event_weights(train)
        train_delta = (
            train["label_seats"].to_numpy(float)
            - train["snapshot_remaining_seats"].to_numpy(float)
        )
        train_full = train["label_seats"].eq(0).astype(int).to_numpy()
        unique_full_classes = np.unique(train_full)
        single_full_class = (
            int(unique_full_classes[0]) if len(unique_full_classes) == 1 else None
        )

        persistence = validation[
            [
                "event_id",
                "trip_id",
                "date",
                "snapshot_time",
                "label_seats",
                "snapshot_remaining_seats",
                "capacity",
                "target_stop_gap",
            ]
        ].copy()
        persistence["candidate"] = "persistence"
        persistence["prediction"] = persistence["snapshot_remaining_seats"]
        persistence["full_probability"] = persistence[
            "snapshot_remaining_seats"
        ].eq(0).astype(float)
        metrics.append(
            {
                "candidate": "persistence",
                "validation_date": validation_date,
                **metric_row(persistence),
            }
        )
        predictions.append(persistence)

        weighted_positive = float(train_weight[train_full == 1].sum())
        weighted_negative = float(train_weight[train_full == 0].sum())
        scale_pos_weight = weighted_negative / max(weighted_positive, 1e-12)

        for candidate, columns in selected_feature_sets.items():
            x_train = train.loc[:, list(columns)]
            x_validation = validation.loc[:, list(columns)]
            regressor = lgb.LGBMRegressor(**regression_config)
            regressor.fit(x_train, train_delta, sample_weight=train_weight)
            raw_delta = regressor.predict(x_validation)
            prediction = np.clip(
                validation["snapshot_remaining_seats"].to_numpy(float) + raw_delta,
                0,
                validation["capacity"].to_numpy(float),
            )

            classifier: lgb.LGBMClassifier | None = None
            if single_full_class is None:
                classifier = lgb.LGBMClassifier(
                    **classification_config,
                    scale_pos_weight=scale_pos_weight,
                )
                classifier.fit(x_train, train_full, sample_weight=train_weight)
                full_probability = classifier.predict_proba(x_validation)[:, 1]
                classifier_training_mode = "lightgbm"
            else:
                # Some route-local folds contain no historical full events. A
                # binary tree cannot be fitted without both classes, so preserve
                # the information available at that service date with a constant
                # probability rather than leaking a later positive label.
                full_probability = np.full(
                    len(validation), float(single_full_class), dtype=float
                )
                classifier_training_mode = f"constant_{single_full_class}"

            scored = validation[
                [
                    "event_id",
                    "trip_id",
                    "date",
                    "snapshot_time",
                    "label_seats",
                    "snapshot_remaining_seats",
                    "capacity",
                    "target_stop_gap",
                ]
            ].copy()
            scored["candidate"] = candidate
            scored["prediction"] = prediction
            scored["full_probability"] = full_probability
            metrics.append(
                {
                    "candidate": candidate,
                    "validation_date": validation_date,
                    "classifier_training_mode": classifier_training_mode,
                    "train_full_events": int(
                        train.loc[train["label_seats"].eq(0), "event_id"].nunique()
                    ),
                    **metric_row(scored),
                }
            )
            predictions.append(scored)

            models: list[tuple[str, Any]] = [("regressor", regressor)]
            if classifier is not None:
                models.append(("classifier", classifier))
            for model_name, model in models:
                gains = model.booster_.feature_importance(importance_type="gain")
                for feature, gain in zip(columns, gains):
                    importances.append(
                        {
                            "candidate": candidate,
                            "validation_date": validation_date,
                            "model": model_name,
                            "feature": feature,
                            "gain": float(gain),
                        }
                    )

    prediction_table = pd.concat(predictions, ignore_index=True)
    aggregate = pd.DataFrame(
        [
            {"candidate": candidate, **metric_row(frame)}
            for candidate, frame in prediction_table.groupby(
                "candidate", sort=False
            )
        ]
    )
    result = (aggregate, pd.DataFrame(metrics), pd.DataFrame(importances))
    if return_predictions:
        return (*result, prediction_table)
    return result


def comparison_with_linear(
    lightgbm_metrics: pd.DataFrame,
    linear_metrics_path: Path,
) -> pd.DataFrame:
    if not linear_metrics_path.is_file():
        return pd.DataFrame()
    linear = pd.read_csv(linear_metrics_path)
    metric_columns = [
        "event_balanced_mae",
        "low_0_10_mae",
        "low_0_5_mae",
        "full_accuracy",
        "full_recall",
        "full_precision",
        "full_f1",
    ]
    comparison = linear[["candidate", *metric_columns]].merge(
        lightgbm_metrics[["candidate", *metric_columns]],
        on="candidate",
        suffixes=("_linear", "_lightgbm"),
        validate="one_to_one",
    )
    for metric in metric_columns:
        comparison[f"{metric}_lightgbm_minus_linear"] = (
            comparison[f"{metric}_lightgbm"] - comparison[f"{metric}_linear"]
        )
    return comparison


def main() -> int:
    parser = argparse.ArgumentParser(
        description="사용자 제안 피처의 strict-prior LightGBM ablation"
    )
    parser.add_argument(
        "--cache",
        type=Path,
        default=Path("data/analysis_cache/all_prearrival_A.pkl"),
    )
    parser.add_argument(
        "--flow-cache",
        type=Path,
        default=Path("data/analysis_cache/all_prearrival_A_stop_flows.pkl"),
    )
    parser.add_argument(
        "--linear-metrics",
        type=Path,
        default=Path("analysis/linear_feature_results/metrics.csv"),
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("analysis/lightgbm_feature_results"),
    )
    args = parser.parse_args()

    data = pd.read_pickle(args.cache)
    flows = pd.read_pickle(args.flow_cache)
    data = add_strict_prior_low_rate_features(data)
    data = add_strict_prior_flow_features(data, flows)
    data, preceding_audit = add_preceding_bus_segment_features(data, flows)
    development = data.loc[data["date"].isin(DEVELOPMENT_DATES)].copy()
    metrics, daily_metrics, importances = run_rolling_origin(development)
    comparison = comparison_with_linear(metrics, args.linear_metrics)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(args.output_dir / "metrics.csv", index=False)
    daily_metrics.to_csv(args.output_dir / "daily_metrics.csv", index=False)
    importances.to_csv(args.output_dir / "feature_importance.csv", index=False)
    if not comparison.empty:
        comparison.to_csv(args.output_dir / "linear_comparison.csv", index=False)
    coverage = {
        column: float(development[column].notna().mean())
        for _, columns in FEATURE_BLOCKS
        for column in columns
    }
    summary = {
        "protocol": {
            "development_dates": DEVELOPMENT_DATES,
            "validation_dates": VALIDATION_DATES,
            "historical_feature_rule": "strictly earlier calendar dates only",
            "regression_target": "arrival_seats - snapshot_remaining_seats",
            "regression_params": REGRESSOR_PARAMS,
            "classification_params": CLASSIFIER_PARAMS,
            "classification_balance": "per-fold event-weighted scale_pos_weight",
            "classification_threshold": 0.5,
            "evaluation_weighting": "equal total weight per arrival event",
        },
        "feature_sets": cumulative_feature_sets(),
        "preceding_bus_segment_audit": preceding_audit,
        "feature_coverage": coverage,
        "metrics": metrics.to_dict(orient="records"),
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(metrics.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/linear_feature_experiment.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/linear_feature_experiment.py
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


DEVELOPMENT_DATES = (
    "2026-08-04",
    "2026-08-05",
    "2026-08-06",
    "2026-08-07",
    "2026-08-10",
)
VALIDATION_DATES = DEVELOPMENT_DATES[1:]

CURRENT_FEATURES = ("snapshot_remaining_seats",)
TARGET_LOW_RATE_FEATURES = (
    "target_low_5_rate",
    "target_low_10_rate",
    "target_low_rate_log_count",
)
PATH_LOW_RATE_FEATURES = (
    "path_low_5_mean",
    "path_low_5_median",
    "path_low_5_sum",
    "path_low_10_mean",
    "path_low_10_median",
    "path_low_10_sum",
    "path_low_rate_stop_count",
)
PATH_FLOW_FEATURES = (
    "path_flow_mean",
    "path_flow_median",
    "path_flow_sum",
    "path_flow_std",
    "path_flow_fallback_share",
)
PRECEDING_BUS_FEATURES = (
    "preceding_bus_segment_delta",
    "preceding_bus_segment_delta_per_stop",
    "preceding_bus_segment_start_seats",
    "preceding_bus_target_arrival_seats",
    "previous_bus_departure_age_minutes",
    "preceding_bus_segment_missing",
)

FEATURE_BLOCKS = (
    ("current_only", CURRENT_FEATURES),
    ("target_low_rate", TARGET_LOW_RATE_FEATURES),
    ("path_low_rate", PATH_LOW_RATE_FEATURES),
    ("path_flow", PATH_FLOW_FEATURES),
    ("preceding_bus", PRECEDING_BUS_FEATURES),
)


def event_weights(data: pd.DataFrame) -> np.ndarray:
    counts = data.groupby("event_id")["event_id"].transform("size").to_numpy(float)
    weights = 1.0 / counts
    return weights / weights.mean()


def _binary_rate_lookup(
    events: pd.DataFrame,
    outcome: str,
    *,
    alpha: float,
) -> Callable[[int, str, int], tuple[float, float]]:
    if events.empty:
        return lambda station, direction, hour: (0.0, 0.0)

    global_rate = float(events[outcome].mean())
    direction_stats = events.groupby("direction", observed=True)[outcome].agg(
        ["sum", "count"]
    )
    direction_rate = {
        str(key): (float(row["sum"]) + alpha * global_rate)
        / (float(row["count"]) + alpha)
        for key, row in direction_stats.iterrows()
    }

    station_stats = events.groupby(
        ["target_station_seq", "direction"], observed=True
    )[outcome].agg(["sum", "count"])
    station_rate: dict[tuple[int, str], float] = {}
    for key, row in station_stats.iterrows():
        normalized = (int(key[0]), str(key[1]))
        prior = direction_rate.get(normalized[1], global_rate)
        station_rate[normalized] = (
            float(row["sum"]) + alpha * prior
        ) / (float(row["count"]) + alpha)

    cell_stats = events.groupby(
        ["target_station_seq", "direction", "arrival_hour"], observed=True
    )[outcome].agg(["sum", "count"])
    cell_rate: dict[tuple[int, str, int], tuple[float, float]] = {}
    for key, row in cell_stats.iterrows():
        normalized = (int(key[0]), str(key[1]), int(key[2]))
        prior = station_rate.get(normalized[:2], direction_rate.get(normalized[1], global_rate))
        cell_rate[normalized] = (
            (float(row["sum"]) + alpha * prior)
            / (float(row["count"]) + alpha),
            float(row["count"]),
        )

    def lookup(station: int, direction: str, hour: int) -> tuple[float, float]:
        cell = cell_rate.get((int(station), str(direction), int(hour)))
        if cell is not None:
            return cell
        station_value = station_rate.get(
            (int(station), str(direction)),
            direction_rate.get(str(direction), global_rate),
        )
        return float(station_value), 0.0

    return lookup


def _event_history(data: pd.DataFrame) -> pd.DataFrame:
    events = data.drop_duplicates("event_id").copy()
    events["target_station_seq"] = pd.to_numeric(
        events["station_seq_cat"], errors="raise"
    ).astype(int)
    events["arrival_hour"] = events["event_time"].dt.hour.astype(int)
    events["is_low_5"] = events["label_seats"].le(5).astype(float)
    events["is_low_10"] = events["label_seats"].le(10).astype(float)
    return events


def add_strict_prior_low_rate_features(
    data: pd.DataFrame,
    *,
    alpha: float = 20.0,
) -> pd.DataFrame:
    """Attach station/time low-seat rates using strictly earlier dates only."""
    output = data.copy()
    columns = (*TARGET_LOW_RATE_FEATURES, *PATH_LOW_RATE_FEATURES)
    for column in columns:
        output[column] = np.nan
    events = _event_history(data)

    for date, indices in output.groupby("date", sort=True).groups.items():
        history = events.loc[events["date"].lt(str(date))]
        low_5 = _binary_rate_lookup(history, "is_low_5", alpha=alpha)
        low_10 = _binary_rate_lookup(history, "is_low_10", alpha=alpha)
        records: list[dict[str, float]] = []
        for row in output.loc[indices].itertuples(index=False):
            target = int(row.station_seq_cat)
            current = int(row.snapshot_station_seq)
            hour = int(pd.Timestamp(row.snapshot_time).hour)
            direction = str(row.direction)
            target_low_5, target_count = low_5(target, direction, hour)
            target_low_10, _ = low_10(target, direction, hour)
            path_stations = list(range(current + 1, target + 1))
            path_5 = np.asarray(
                [low_5(station, direction, hour)[0] for station in path_stations],
                dtype=float,
            )
            path_10 = np.asarray(
                [low_10(station, direction, hour)[0] for station in path_stations],
                dtype=float,
            )
            records.append(
                {
                    "target_low_5_rate": target_low_5,
                    "target_low_10_rate": target_low_10,
                    "target_low_rate_log_count": np.log1p(target_count),
                    "path_low_5_mean": float(path_5.mean()) if len(path_5) else 0.0,
                    "path_low_5_median": float(np.median(path_5)) if len(path_5) else 0.0,
                    "path_low_5_sum": float(path_5.sum()),
                    "path_low_10_mean": float(path_10.mean()) if len(path_10) else 0.0,
                    "path_low_10_median": float(np.median(path_10)) if len(path_10) else 0.0,
                    "path_low_10_sum": float(path_10.sum()),
                    "path_low_rate_stop_count": float(len(path_stations)),
                }
            )
        output.loc[indices, list(columns)] = pd.DataFrame(
            records, index=indices
        )[list(columns)].to_numpy()
    return output


def _flow_lookup(
    flows: pd.DataFrame,
    *,
    alpha: float,
) -> Callable[[int, str, int], tuple[float, bool]]:
    weekday = flows.loc[pd.to_datetime(flows["date"]).dt.dayofweek.lt(5)]
    if weekday.empty:
        return lambda station, direction, time_bin: (0.0, True)
    global_mean = float(weekday["stop_net"].mean())
    station_stats = weekday.groupby(
        ["station_seq", "direction"], observed=True
    )["stop_net"].agg(["mean", "count"])
    station_mean = {
        (int(key[0]), str(key[1])): float(row["mean"])
        for key, row in station_stats.iterrows()
    }
    cell_stats = weekday.groupby(
        ["station_seq", "direction", "time_bin_2h"], observed=True
    )["stop_net"].agg(["sum", "count"])
    cell_mean: dict[tuple[int, str, int], float] = {}
    for key, row in cell_stats.iterrows():
        normalized = (int(key[0]), str(key[1]), int(key[2]))
        prior = station_mean.get(normalized[:2], global_mean)
        cell_mean[normalized] = (
            float(row["sum"]) + alpha * prior
        ) / (float(row["count"]) + alpha)

    def lookup(station: int, direction: str, time_bin: int) -> tuple[float, bool]:
        key = (int(station), str(direction), int(time_bin))
        if key in cell_mean:
            return cell_mean[key], False
        station_key = key[:2]
        if station_key in station_mean:
            return station_mean[station_key], True
        return global_mean, True

    return lookup


def add_strict_prior_flow_features(
    data: pd.DataFrame,
    flows: pd.DataFrame,
    *,
    alpha: float = 10.0,
) -> pd.DataFrame:
    """Attach path summaries of past station-level net seat changes."""
    output = data.copy()
    for column in PATH_FLOW_FEATURES:
        output[column] = np.nan
    pass_nodes = {int(value) for value in flows.attrs.get("pass_node_sequences", [])}
    for date, indices in output.groupby("date", sort=True).groups.items():
        history = flows.loc[flows["date"].lt(str(date))].copy()
        lookup = _flow_lookup(history, alpha=alpha)
        records: list[dict[str, float]] = []
        for row in output.loc[indices].itertuples(index=False):
            current = int(row.snapshot_station_seq)
            target = int(row.station_seq_cat)
            start = current if str(row.target_state_cat) == "1" else current + 1
            stations = [station for station in range(start, target) if station not in pass_nodes]
            time_bin = int(pd.Timestamp(row.snapshot_time).hour // 2 * 2)
            looked_up = [lookup(station, str(row.direction), time_bin) for station in stations]
            values = np.asarray([item[0] for item in looked_up], dtype=float)
            fallback = np.asarray([item[1] for item in looked_up], dtype=float)
            records.append(
                {
                    "path_flow_mean": float(values.mean()) if len(values) else 0.0,
                    "path_flow_median": float(np.median(values)) if len(values) else 0.0,
                    "path_flow_sum": float(values.sum()),
                    "path_flow_std": float(values.std(ddof=0)) if len(values) else 0.0,
                    "path_flow_fallback_share": float(fallback.mean()) if len(fallback) else 1.0,
                }
            )
        output.loc[indices, list(PATH_FLOW_FEATURES)] = pd.DataFrame(
            records, index=indices
        )[list(PATH_FLOW_FEATURES)].to_numpy()
    return output


def add_preceding_bus_segment_features(
    data: pd.DataFrame,
    flows: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, int | float]]:
    """Join the prior bus's directly observed source-departure to target-arrival delta."""
    output = data.copy()
    for column in PRECEDING_BUS_FEATURES:
        if column not in output.columns:
            output[column] = np.nan
    output["preceding_bus_segment_missing"] = 1.0

    ordered = flows.sort_values(["departure_seen", "arrival_seen"]).drop_duplicates(
        ["trip_id", "station_seq"], keep="last"
    )
    departure = {
        (str(row.trip_id), int(row.station_seq)): (
            float(row.departure_seats),
            pd.Timestamp(row.departure_seen),
        )
        for row in ordered.itertuples(index=False)
    }
    arrival = {
        (str(row.trip_id), int(row.station_seq)): (
            float(row.observed_arrival_seats),
            pd.Timestamp(row.arrival_seen),
        )
        for row in ordered.itertuples(index=False)
    }

    matched = 0
    future_reference = 0
    invalid_order = 0
    for index, row in output.iterrows():
        if pd.isna(row.get("previous_bus_trip_id")):
            continue
        trip_id = str(row["previous_bus_trip_id"])
        current = int(row["snapshot_station_seq"])
        target = int(row["station_seq_cat"])
        source_value = departure.get((trip_id, current))
        target_value = arrival.get((trip_id, target))
        if source_value is None or target_value is None or target <= current:
            continue
        source_seats, source_time = source_value
        target_seats, target_time = target_value
        snapshot_time = pd.Timestamp(row["snapshot_time"])
        if target_time >= snapshot_time:
            future_reference += 1
            continue
        if source_time > target_time:
            invalid_order += 1
            continue
        delta = target_seats - source_seats
        output.at[index, "preceding_bus_segment_delta"] = delta
        output.at[index, "preceding_bus_segment_delta_per_stop"] = delta / (
            target - current
        )
        output.at[index, "preceding_bus_segment_start_seats"] = source_seats
        output.at[index, "preceding_bus_target_arrival_seats"] = target_seats
        output.at[index, "preceding_bus_segment_missing"] = 0.0
        matched += 1
    return output, {
        "rows": int(len(output)),
        "matched_rows": matched,
        "coverage": matched / max(len(output), 1),
        "future_reference_rows": future_reference,
        "invalid_order_rows": invalid_order,
    }


def cumulative_feature_sets() -> dict[str, tuple[str, ...]]:
    output: dict[str, tuple[str, ...]] = {}
    selected: list[str] = []
    for name, columns in FEATURE_BLOCKS:
        selected.extend(columns)
        output[name] = tuple(selected)
    return output


def regression_pipeline() -> Pipeline:
    return Pipeline(
        [
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler()),
            ("model", Ridge(alpha=10.0)),
        ]
    )


def classification_pipeline() -> Pipeline:
    return Pipeline(
        [
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    C=0.5,
                    class_weight="balanced",
                    max_iter=2_000,
                    random_state=42,
                ),
            ),
        ]
    )


def _weighted_mean(values: np.ndarray, weights: np.ndarray) -> float:
    return float(np.average(values, weights=weights))


def metric_row(data: pd.DataFrame) -> dict[str, Any]:
    weights = event_weights(data)
    true = data["label_seats"].to_numpy(float)
    prediction = data["prediction"].to_numpy(float)
    errors = np.abs(true - prediction)
    low_10 = true <= 10
    low_5 = true <= 5
    full_true = true == 0
    full_pred = data["full_probability"].to_numpy(float) >= 0.5
    return {
        "rows": int(len(data)),
        "events": int(data["event_id"].nunique()),
        "full_events": int(data.loc[full_true, "event_id"].nunique()),
        "low_0_10_events": int(data.loc[low_10, "event_id"].nunique()),
        "event_balanced_mae": _weighted_mean(errors, weights),
        "low_0_10_mae": (
            _weighted_mean(errors[low_10], weights[low_10]) if low_10.any() else np.nan
        ),
        "low_0_5_mae": (
            _weighted_mean(errors[low_5], weights[low_5]) if low_5.any() else np.nan
        ),
        "full_accuracy": float(accuracy_score(full_true, full_pred, sample_weight=weights)),
        "full_recall": float(
            recall_score(full_true, full_pred, sample_weight=weights, zero_division=0)
        ),
        "full_precision": float(
            precision_score(full_true, full_pred, sample_weight=weights, zero_division=0)
        ),
        "full_f1": float(f1_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
    }


def run_rolling_origin(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    metrics: list[dict[str, Any]] = []
    predictions: list[pd.DataFrame] = []
    coefficients: list[dict[str, Any]] = []
    feature_sets = cumulative_feature_sets()
    for validation_date in VALIDATION_DATES:
        train_dates = [date for date in DEVELOPMENT_DATES if date < validation_date]
        train = data.loc[data["date"].isin(train_dates)].copy()
        validation = data.loc[data["date"].eq(validation_date)].copy()
        if train.empty or validation.empty:
            raise ValueError(f"rolling fold가 비었습니다: {validation_date}")
        train_weight = event_weights(train)
        train_delta = (
            train["label_seats"].to_numpy(float)
            - train["snapshot_remaining_seats"].to_numpy(float)
        )
        train_full = train["label_seats"].eq(0).astype(int).to_numpy()
        if len(np.unique(train_full)) < 2:
            raise ValueError(f"만차 분류 학습에 두 클래스가 없습니다: {validation_date}")

        persistence = validation[
            [
                "event_id",
                "trip_id",
                "date",
                "snapshot_time",
                "label_seats",
                "snapshot_remaining_seats",
                "capacity",
                "target_stop_gap",
            ]
        ].copy()
        persistence["candidate"] = "persistence"
        persistence["prediction"] = persistence["snapshot_remaining_seats"]
        persistence["full_probability"] = persistence[
            "snapshot_remaining_seats"
        ].eq(0).astype(float)
        metrics.append(
            {
                "candidate": "persistence",
                "validation_date": validation_date,
                **metric_row(persistence),
            }
        )
        predictions.append(persistence)

        for candidate, columns in feature_sets.items():
            regressor = regression_pipeline()
            regressor.fit(
                train[list(columns)],
                train_delta,
                model__sample_weight=train_weight,
            )
            raw_delta = regressor.predict(validation[list(columns)])
            prediction = np.clip(
                validation["snapshot_remaining_seats"].to_numpy(float) + raw_delta,
                0,
                validation["capacity"].to_numpy(float),
            )
            classifier = classification_pipeline()
            classifier.fit(
                train[list(columns)],
                train_full,
                model__sample_weight=train_weight,
            )
            full_probability = classifier.predict_proba(validation[list(columns)])[:, 1]
            scored = validation[
                [
                    "event_id",
                    "trip_id",
                    "date",
                    "snapshot_time",
                    "label_seats",
                    "snapshot_remaining_seats",
                    "capacity",
                    "target_stop_gap",
                ]
            ].copy()
            scored["candidate"] = candidate
            scored["prediction"] = prediction
            scored["full_probability"] = full_probability
            row = metric_row(scored)
            metrics.append({"candidate": candidate, "validation_date": validation_date, **row})
            predictions.append(scored)

            transformed_names = regressor.named_steps["impute"].get_feature_names_out(columns)
            ridge_coefficients = regressor.named_steps["model"].coef_
            for feature, coefficient in zip(transformed_names, ridge_coefficients):
                coefficients.append(
                    {
                        "candidate": candidate,
                        "validation_date": validation_date,
                        "feature": str(feature),
                        "standardized_coefficient": float(coefficient),
                    }
                )

    prediction_table = pd.concat(predictions, ignore_index=True)
    aggregate: list[dict[str, Any]] = []
    for candidate, frame in prediction_table.groupby("candidate", sort=False):
        aggregate.append({"candidate": candidate, **metric_row(frame)})
    return pd.DataFrame(aggregate), pd.DataFrame(metrics), pd.DataFrame(coefficients)


def json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    return value


def main() -> int:
    parser = argparse.ArgumentParser(
        description="사용자 제안 피처의 strict-prior 선형 ablation"
    )
    parser.add_argument(
        "--cache",
        type=Path,
        default=Path("data/analysis_cache/all_prearrival_A.pkl"),
    )
    parser.add_argument(
        "--flow-cache",
        type=Path,
        default=Path("data/analysis_cache/all_prearrival_A_stop_flows.pkl"),
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("analysis/linear_feature_results"),
    )
    args = parser.parse_args()

    data = pd.read_pickle(args.cache)
    flows = pd.read_pickle(args.flow_cache)
    data = add_strict_prior_low_rate_features(data)
    data = add_strict_prior_flow_features(data, flows)
    data, preceding_audit = add_preceding_bus_segment_features(data, flows)
    development = data.loc[data["date"].isin(DEVELOPMENT_DATES)].copy()
    metrics, daily_metrics, coefficients = run_rolling_origin(development)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(args.output_dir / "metrics.csv", index=False)
    daily_metrics.to_csv(args.output_dir / "daily_metrics.csv", index=False)
    coefficients.to_csv(args.output_dir / "coefficients.csv", index=False)
    coverage = {
        column: float(development[column].notna().mean())
        for _, columns in FEATURE_BLOCKS
        for column in columns
    }
    summary = {
        "protocol": {
            "development_dates": DEVELOPMENT_DATES,
            "validation_dates": VALIDATION_DATES,
            "historical_feature_rule": "strictly earlier calendar dates only",
            "regression": "Ridge(alpha=10) on arrival delta, clipped to [0, capacity]",
            "classification": "balanced LogisticRegression(C=0.5), threshold=0.5",
            "evaluation_weighting": "equal total weight per arrival event",
        },
        "feature_sets": cumulative_feature_sets(),
        "preceding_bus_segment_audit": preceding_audit,
        "feature_coverage": coverage,
        "metrics": metrics.to_dict(orient="records"),
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(metrics.to_string(index=False))
    print(json.dumps(preceding_audit, ensure_ascii=False, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/main_model_feature_augmentation.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/main_model_feature_augmentation.py
from __future__ import annotations

import argparse
import json
import time
from dataclasses import asdict
from pathlib import Path
from typing import Any

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline

from all_prearrival_seat_regression import DYNAMIC_ALL_PREARRIVAL, event_weights
from hypothesis_model_search import (
    CAPACITY_44_ALL_PREARRIVAL,
    DEFAULT_DEVELOPMENT_DATES,
    DEFAULT_STRESS_DATES,
    DEFAULT_TEST_DATE,
    OBSERVED_SEAT_CEILING_PARAM,
    Candidate,
    EnsembleSpec,
    add_observed_capacity_features,
    candidate_weights,
    combine_weighted_oof,
    decode_target,
    development_folds,
    encode_target,
    event_bias,
    event_summary,
    postprocess_ensemble_prediction,
    scored_frame,
    strict_forward_bias_predictions,
    tree_node_count,
)
from linear_feature_experiment import (
    PATH_FLOW_FEATURES,
    PATH_LOW_RATE_FEATURES,
    PRECEDING_BUS_FEATURES,
    TARGET_LOW_RATE_FEATURES,
    add_preceding_bus_segment_features,
    add_strict_prior_flow_features,
    add_strict_prior_low_rate_features,
    json_ready,
)
from model_feasibility import FeatureSet, make_preprocessor


BASE_NUMERIC = CAPACITY_44_ALL_PREARRIVAL.numeric
BASE_CATEGORICAL = CAPACITY_44_ALL_PREARRIVAL.categorical
ALL_PROPOSED = (
    *TARGET_LOW_RATE_FEATURES,
    *PATH_LOW_RATE_FEATURES,
    *PATH_FLOW_FEATURES,
    *PRECEDING_BUS_FEATURES,
)

# This is not merely correlated: it is exactly target_stop_gap. Current seats
# are also already deployed and therefore are not repeated in ALL_PROPOSED.
EXACT_DUPLICATE_PROPOSED = ("path_low_rate_stop_count",)
UNIQUE_PROPOSED = tuple(
    column for column in ALL_PROPOSED if column not in EXACT_DUPLICATE_PROPOSED
)

# Qualitative meaning review plus the development-only Pearson/Spearman audit:
# retain a target risk, normalized/cumulative path risk, normalized/cumulative
# stop flow, dispersion/reliability, and one normalized preceding-bus signal.
CORRELATION_PRUNED_PROPOSED = (
    "target_low_10_rate",
    "target_low_rate_log_count",
    "path_low_10_mean",
    "path_low_10_sum",
    "path_flow_mean",
    "path_flow_sum",
    "path_flow_std",
    "path_flow_fallback_share",
    "preceding_bus_segment_delta_per_stop",
    "previous_bus_departure_age_minutes",
    "preceding_bus_segment_missing",
)

# Existing features removed only where their meaning is already represented and
# both Pearson and Spearman are near deterministic on development data. Keep the
# observed-ceiling variants because they encode the corrected 44/70-seat support.
REDUNDANT_EXISTING = (
    "snapshot_capacity",
    "target_load_ratio",
    "seat_change_per_stop",
    "load_gap_interaction",
    "currently_low_5",
    "currently_low_10",
    "x",
    "y",
)


def feature_sets() -> dict[str, FeatureSet]:
    lean_numeric = tuple(
        column for column in BASE_NUMERIC if column not in REDUNDANT_EXISTING
    )
    low_rate_block = (
        "target_low_10_rate",
        "target_low_rate_log_count",
        "path_low_10_mean",
        "path_low_10_sum",
    )
    path_flow_block = (
        "path_flow_mean",
        "path_flow_sum",
        "path_flow_std",
        "path_flow_fallback_share",
    )
    importance_pruned = tuple(
        column
        for column in CORRELATION_PRUNED_PROPOSED
        if column
        not in {"preceding_bus_segment_delta_per_stop", "preceding_bus_segment_missing"}
    )
    return {
        "baseline_official": FeatureSet(
            "baseline_official", BASE_NUMERIC, BASE_CATEGORICAL
        ),
        "augmented_unique": FeatureSet(
            "augmented_unique", (*BASE_NUMERIC, *UNIQUE_PROPOSED), BASE_CATEGORICAL
        ),
        "correlation_pruned": FeatureSet(
            "correlation_pruned",
            (*BASE_NUMERIC, *CORRELATION_PRUNED_PROPOSED),
            BASE_CATEGORICAL,
        ),
        "lean_merged": FeatureSet(
            "lean_merged",
            (*lean_numeric, *CORRELATION_PRUNED_PROPOSED),
            BASE_CATEGORICAL,
        ),
        "baseline_no_redundant": FeatureSet(
            "baseline_no_redundant", lean_numeric, BASE_CATEGORICAL
        ),
        "low_rate_block": FeatureSet(
            "low_rate_block", (*BASE_NUMERIC, *low_rate_block), BASE_CATEGORICAL
        ),
        "path_flow_block": FeatureSet(
            "path_flow_block", (*BASE_NUMERIC, *path_flow_block), BASE_CATEGORICAL
        ),
        "path_flow_sum_only": FeatureSet(
            "path_flow_sum_only", (*BASE_NUMERIC, "path_flow_sum"), BASE_CATEGORICAL
        ),
        "low_rate_plus_flow_sum": FeatureSet(
            "low_rate_plus_flow_sum",
            (*BASE_NUMERIC, *low_rate_block, "path_flow_sum"),
            BASE_CATEGORICAL,
        ),
        "importance_pruned": FeatureSet(
            "importance_pruned",
            (*BASE_NUMERIC, *importance_pruned),
            BASE_CATEGORICAL,
        ),
        "lean_importance_pruned": FeatureSet(
            "lean_importance_pruned",
            (*lean_numeric, *importance_pruned),
            BASE_CATEGORICAL,
        ),
        "target_low_10_only": FeatureSet(
            "target_low_10_only",
            (*BASE_NUMERIC, "target_low_10_rate"),
            BASE_CATEGORICAL,
        ),
        "target_low_10_with_count": FeatureSet(
            "target_low_10_with_count",
            (*BASE_NUMERIC, "target_low_10_rate", "target_low_rate_log_count"),
            BASE_CATEGORICAL,
        ),
        "path_low_10_mean_only": FeatureSet(
            "path_low_10_mean_only",
            (*BASE_NUMERIC, "path_low_10_mean"),
            BASE_CATEGORICAL,
        ),
        "path_low_10_sum_only": FeatureSet(
            "path_low_10_sum_only",
            (*BASE_NUMERIC, "path_low_10_sum"),
            BASE_CATEGORICAL,
        ),
        "low_rate_without_count": FeatureSet(
            "low_rate_without_count",
            (
                *BASE_NUMERIC,
                "target_low_10_rate",
                "path_low_10_mean",
                "path_low_10_sum",
            ),
            BASE_CATEGORICAL,
        ),
        "previous_bus_age_only": FeatureSet(
            "previous_bus_age_only",
            (*BASE_NUMERIC, "previous_bus_departure_age_minutes"),
            BASE_CATEGORICAL,
        ),
    }


def component_candidates() -> tuple[Candidate, ...]:
    common = {
        "stage": "main_feature_augmentation",
        "why": "fixed reproduction of the deployed component with a candidate schema",
        "if_works": "strict OOF improves without changing model family or weights",
        "if_fails": "the added schema is redundant or unstable",
        "feature_variant": "custom",
        "low_weight": 2.0,
        "weighting_kind": "event",
    }
    return (
        Candidate(
            name="hgb",
            model_kind="hgb",
            target_kind="delta_per_stop",
            params={
                "loss": "absolute_error",
                "learning_rate": 0.04,
                "max_iter": 500,
                "max_leaf_nodes": 63,
                "min_samples_leaf": 5,
                "l2_regularization": 0.0,
                "early_stopping": False,
            },
            **common,
        ),
        Candidate(
            name="extra_trees",
            model_kind="extra_trees",
            target_kind="delta_per_stop",
            params={
                "n_estimators": 48,
                "max_depth": 18,
                "min_samples_leaf": 5,
                "max_features": 0.7,
            },
            **common,
        ),
        Candidate(
            name="lightgbm",
            model_kind="lightgbm",
            target_kind="delta_per_sqrt_stop",
            gap_weight_power=0.5,
            params={
                "objective": "regression_l1",
                "n_estimators": 600,
                "learning_rate": 0.025,
                "num_leaves": 63,
                "min_child_samples": 10,
                "subsample": 0.85,
                "subsample_freq": 1,
                "colsample_bytree": 0.8,
                "reg_alpha": 0.1,
                "reg_lambda": 1.0,
            },
            **common,
        ),
    )


def make_model(candidate: Candidate, features: FeatureSet, seed: int) -> Pipeline:
    if candidate.model_kind == "hgb":
        estimator = HistGradientBoostingRegressor(
            random_state=seed, **candidate.params
        )
    elif candidate.model_kind == "extra_trees":
        estimator = ExtraTreesRegressor(
            n_jobs=-1, random_state=seed, **candidate.params
        )
    elif candidate.model_kind == "lightgbm":
        estimator = lgb.LGBMRegressor(
            n_jobs=-1, random_state=seed, verbosity=-1, **candidate.params
        )
    else:
        raise ValueError(candidate.model_kind)
    return Pipeline(
        [("features", make_preprocessor(features)), ("regressor", estimator)]
    )


def prepare_augmented(cache: Path, flow_cache: Path, output_cache: Path) -> pd.DataFrame:
    if output_cache.is_file():
        return pd.read_pickle(output_cache)
    data = pd.read_pickle(cache)
    flows = pd.read_pickle(flow_cache)
    data = add_strict_prior_low_rate_features(data)
    data = add_strict_prior_flow_features(data, flows)
    data, audit = add_preceding_bus_segment_features(data, flows)
    data.attrs["preceding_bus_segment_audit"] = audit
    output_cache.parent.mkdir(parents=True, exist_ok=True)
    data.to_pickle(output_cache)
    return data


def correlation_audit(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    development = data.loc[data["date"].isin(DEFAULT_DEVELOPMENT_DATES)].copy()
    numeric = list(dict.fromkeys((*BASE_NUMERIC, *ALL_PROPOSED)))
    rows: list[dict[str, Any]] = []
    for method in ("pearson", "spearman"):
        correlation = development[numeric].corr(method=method)
        for position, left in enumerate(numeric):
            for right in numeric[:position]:
                value = float(correlation.at[left, right])
                if np.isfinite(value) and abs(value) >= 0.85:
                    rows.append(
                        {
                            "method": method,
                            "left": left,
                            "right": right,
                            "correlation": value,
                            "absolute_correlation": abs(value),
                            "contains_proposed_feature": bool(
                                left in UNIQUE_PROPOSED or right in UNIQUE_PROPOSED
                            ),
                        }
                    )
    missing = pd.DataFrame(
        [
            {
                "feature": column,
                "missing_rate": float(development[column].isna().mean()),
                "non_missing_unique": int(development[column].nunique(dropna=True)),
            }
            for column in numeric
        ]
    )
    return pd.DataFrame(rows), missing


def qualitative_catalog() -> pd.DataFrame:
    decisions = {
        "target_low_5_rate": ("remove_correlation", "same target risk as low10; lower support and Pearson/Spearman 0.924/0.952"),
        "target_low_10_rate": ("keep", "directly aligned with the required low<=10 metric"),
        "target_low_rate_log_count": ("keep", "history reliability, not a risk level duplicate"),
        "path_low_5_mean": ("remove_correlation", "same path risk as low10 mean; Pearson/Spearman 0.942/0.975"),
        "path_low_5_median": ("remove_correlation", "same distribution center as retained mean"),
        "path_low_5_sum": ("remove_correlation", "same cumulative risk as low10 sum; Pearson/Spearman 0.972/0.988"),
        "path_low_10_mean": ("keep", "distance-normalized path risk"),
        "path_low_10_median": ("remove_correlation", "same path center as mean; Pearson/Spearman 0.900/0.936"),
        "path_low_10_sum": ("keep", "cumulative path risk distinct from normalized mean"),
        "path_low_rate_stop_count": ("remove_exact", "exactly equals target_stop_gap"),
        "path_flow_mean": ("keep", "distance-normalized historical net seat flow"),
        "path_flow_median": ("remove_correlation", "same flow center as mean; Pearson/Spearman 0.971/0.949"),
        "path_flow_sum": ("keep", "expected cumulative net seat change"),
        "path_flow_std": ("keep", "path-flow heterogeneity"),
        "path_flow_fallback_share": ("keep", "historical lookup reliability"),
        "preceding_bus_segment_delta": ("remove_correlation", "same segment signal as normalized delta; Spearman 0.968"),
        "preceding_bus_segment_delta_per_stop": ("keep_pending_importance", "normalized preceding-bus segment signal"),
        "preceding_bus_segment_start_seats": ("remove_semantic", "overlaps existing previous-bus departure seats"),
        "preceding_bus_target_arrival_seats": ("remove_semantic", "arithmetic combination of segment start and delta"),
        "previous_bus_departure_age_minutes": ("keep_pending_importance", "prior-bus freshness; available in raw data but not the deployed schema"),
        "preceding_bus_segment_missing": ("keep_pending_importance", "required missingness signal for 95% sparse segment feature"),
    }
    rows = [
        {
            "source": "proposed",
            "feature": feature,
            "decision": decisions[feature][0],
            "reason": decisions[feature][1],
        }
        for feature in ALL_PROPOSED
    ]
    existing = {
        "snapshot_capacity": "same capacity regime as observed_ceiling_capacity; Pearson/Spearman 1.000",
        "target_load_ratio": "legacy-capacity version of observed_ceiling_load_ratio; correlation above 0.998",
        "seat_change_per_stop": "near-deterministic proxy of seat_delta_previous_stop; correlation above 0.993",
        "load_gap_interaction": "legacy-capacity version of observed_ceiling_load_gap; correlation above 0.986",
        "currently_low_5": "deterministic threshold of snapshot_remaining_seats and low prior permutation importance",
        "currently_low_10": "deterministic threshold of snapshot_remaining_seats and low prior permutation importance",
        "x": "fixed-route location proxy of route_progress; correlation above 0.988",
        "y": "fixed-route location proxy of route_progress; absolute correlation above 0.972",
    }
    rows.extend(
        {
            "source": "existing",
            "feature": feature,
            "decision": "candidate_remove_rejected_by_ablation",
            "reason": reason,
        }
        for feature, reason in existing.items()
    )
    return pd.DataFrame(rows)


def paired_trip_bootstrap(
    predictions: pd.DataFrame,
    candidate: str,
    *,
    low_only: bool,
    repeats: int = 2_000,
    seed: int = 42,
) -> dict[str, Any]:
    selected = predictions.loc[
        predictions["candidate"].isin(["baseline_official", candidate])
    ].copy()
    selected["absolute_error"] = (
        selected["label_seats"] - selected["prediction"]
    ).abs()
    if low_only:
        selected = selected.loc[selected["label_seats"].le(10)].copy()
    per_event = (
        selected.groupby(["trip_id", "event_id", "candidate"], observed=True)[
            "absolute_error"
        ]
        .mean()
        .unstack("candidate")
        .dropna(subset=["baseline_official", candidate])
        .reset_index()
    )
    per_event["improvement"] = (
        per_event["baseline_official"] - per_event[candidate]
    )
    trip_groups = [
        group["improvement"].to_numpy(float)
        for _, group in per_event.groupby("trip_id", sort=False)
    ]
    rng = np.random.default_rng(seed)
    draws = np.empty(repeats, dtype=float)
    for index in range(repeats):
        sampled = rng.integers(0, len(trip_groups), size=len(trip_groups))
        draws[index] = np.concatenate([trip_groups[item] for item in sampled]).mean()
    return {
        "candidate": candidate,
        "metric": "low_0_10_mae" if low_only else "event_balanced_mae",
        "candidate_improvement": float(per_event["improvement"].mean()),
        "ci_95_lower": float(np.quantile(draws, 0.025)),
        "ci_95_upper": float(np.quantile(draws, 0.975)),
        "probability_candidate_better": float((draws > 0).mean()),
        "trips": int(per_event["trip_id"].nunique()),
        "events": int(len(per_event)),
    }


def _feature_importance_rows(
    model: Pipeline,
    features: FeatureSet,
    *,
    feature_set_name: str,
    component: str,
    validation_date: str,
) -> list[dict[str, Any]]:
    estimator = model.named_steps["regressor"]
    if hasattr(estimator, "booster_"):
        raw = estimator.booster_.feature_importance(importance_type="gain")
    elif hasattr(estimator, "feature_importances_"):
        raw = estimator.feature_importances_
    else:
        return []
    # Numeric values are emitted first by make_preprocessor. Missing indicators
    # and one-hot columns follow; this audit deliberately reports the direct
    # numeric contribution used for pruning proposed numeric features.
    direct = np.asarray(raw, dtype=float)[: len(features.numeric)]
    total = max(float(np.asarray(raw, dtype=float).sum()), 1e-12)
    return [
        {
            "feature_set": feature_set_name,
            "component": component,
            "validation_date": validation_date,
            "feature": feature,
            "importance": float(value),
            "normalized_importance": float(value / total),
        }
        for feature, value in zip(features.numeric, direct, strict=True)
    ]


def run_component(
    candidate: Candidate,
    features: FeatureSet,
    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]],
    *,
    feature_set_name: str,
    seed: int,
) -> tuple[pd.DataFrame, list[dict[str, Any]], list[dict[str, Any]]]:
    raw_frames: list[pd.DataFrame] = []
    importance: list[dict[str, Any]] = []
    diagnostics: list[dict[str, Any]] = []
    for fold_number, (validation_date, train, validation) in enumerate(folds):
        model = make_model(candidate, features, seed + fold_number)
        started = time.perf_counter()
        model.fit(
            train[list(features.columns)],
            encode_target(train, candidate.target_kind),
            regressor__sample_weight=candidate_weights(
                train,
                candidate.low_weight,
                weighting_kind=candidate.weighting_kind,
                gap_weight_power=candidate.gap_weight_power,
            ),
        )
        prediction = decode_target(
            model.predict(validation[list(features.columns)]),
            validation,
            candidate.target_kind,
        )
        raw_frames.append(scored_frame(validation, prediction, candidate=candidate.name))
        importance.extend(
            _feature_importance_rows(
                model,
                features,
                feature_set_name=feature_set_name,
                component=candidate.name,
                validation_date=validation_date,
            )
        )
        diagnostics.append(
            {
                "feature_set": feature_set_name,
                "component": candidate.name,
                "validation_date": validation_date,
                "fit_seconds": time.perf_counter() - started,
                "tree_nodes": tree_node_count(model),
            }
        )
    output = pd.concat(raw_frames, ignore_index=True)
    raw = output["prediction"].to_numpy(float)
    strict, biases = strict_forward_bias_predictions(
        output, raw, validation_dates=[fold[0] for fold in folds]
    )
    output["raw_prediction"] = raw
    output["prediction"] = strict
    output["deployment_prediction"] = raw + event_bias(output, raw)
    for row in diagnostics:
        row["strict_oof_bias"] = biases[row["validation_date"]]
    return output, importance, diagnostics


def required_metrics(data: pd.DataFrame) -> dict[str, Any]:
    prediction = data["prediction"].to_numpy(float)
    base = event_summary(data, prediction)
    weights = event_weights(data)
    truth = data["label_seats"].to_numpy(float)
    error = np.abs(truth - prediction)
    low10 = truth <= 10
    full_true = truth == 0
    full_pred = prediction <= 0.5
    base.update(
        {
            "low_0_10_events": int(data.loc[low10, "event_id"].nunique()),
            "low_0_10_mae": float(np.average(error[low10], weights=weights[low10])),
            "full_accuracy": float(accuracy_score(full_true, full_pred, sample_weight=weights)),
            "full_recall": float(recall_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
            "full_precision": float(precision_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
            "full_f1": float(f1_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
            "full_threshold_seats": 0.5,
        }
    )
    return base


def run_feature_set(
    name: str,
    features: FeatureSet,
    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]],
    *,
    seed: int,
) -> tuple[pd.DataFrame, dict[str, Any], pd.DataFrame, pd.DataFrame]:
    components: dict[str, pd.DataFrame] = {}
    importance: list[dict[str, Any]] = []
    diagnostics: list[dict[str, Any]] = []
    for candidate in component_candidates():
        print(f"[{name}] {candidate.name}", flush=True)
        output, component_importance, component_diagnostics = run_component(
            candidate,
            features,
            folds,
            feature_set_name=name,
            seed=seed,
        )
        components[candidate.name] = output
        importance.extend(component_importance)
        diagnostics.extend(component_diagnostics)
    weights = {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2}
    blended = combine_weighted_oof(name, components, weights)
    strict, biases = strict_forward_bias_predictions(blended, blended["prediction"])
    spec = EnsembleSpec(
        name=name,
        kind="weighted",
        components=tuple(weights),
        params={**weights, OBSERVED_SEAT_CEILING_PARAM: 1.0},
    )
    blended["prediction"] = postprocess_ensemble_prediction(spec, strict, blended)
    deployment_base = blended["deployment_prediction"].to_numpy(float)
    deployment_bias = event_bias(blended, deployment_base)
    blended["deployment_prediction"] = postprocess_ensemble_prediction(
        spec, deployment_base + deployment_bias, blended
    )
    blended["ensemble_deployment_bias"] = deployment_bias
    blended["candidate"] = name
    metrics = required_metrics(blended)
    daily = pd.DataFrame(
        [
            {"feature_set": name, "validation_date": date, **required_metrics(frame)}
            for date, frame in blended.groupby("date", sort=True)
        ]
    )
    metrics.update(
        {
            "feature_set": name,
            "numeric_features": len(features.numeric),
            "categorical_features": len(features.categorical),
            "total_features": len(features.columns),
            "strict_ensemble_biases": json.dumps(biases, sort_keys=True),
            "deployment_bias": deployment_bias,
        }
    )
    return blended, metrics, daily, pd.DataFrame(importance + diagnostics)


def fit_final_models(
    name: str,
    features: FeatureSet,
    development: pd.DataFrame,
    targets: dict[str, pd.DataFrame],
    oof: pd.DataFrame,
    output_dir: Path,
    *,
    seed: int,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    component_predictions: dict[str, dict[str, np.ndarray]] = {}
    component_nodes: dict[str, int] = {}
    model_dir = output_dir / "candidate_models" / name
    model_dir.mkdir(parents=True, exist_ok=True)
    for candidate in component_candidates():
        model = make_model(candidate, features, seed)
        model.fit(
            development[list(features.columns)],
            encode_target(development, candidate.target_kind),
            regressor__sample_weight=candidate_weights(
                development,
                candidate.low_weight,
                weighting_kind=candidate.weighting_kind,
                gap_weight_power=candidate.gap_weight_power,
            ),
        )
        component_predictions[candidate.name] = {
            key: decode_target(
                model.predict(frame[list(features.columns)]),
                frame,
                candidate.target_kind,
            )
            for key, frame in targets.items()
        }
        component_nodes[candidate.name] = tree_node_count(model)
        joblib.dump(model, model_dir / f"{candidate.name}.joblib")

    weights = {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2}
    spec = EnsembleSpec(
        name=name,
        kind="weighted",
        components=tuple(weights),
        params={**weights, OBSERVED_SEAT_CEILING_PARAM: 1.0},
    )
    # This is the same all-development OOF deployment correction computed from
    # component deployment predictions in run_feature_set. It never changes a
    # development-fold selection prediction.
    deployment_bias = float(oof["ensemble_deployment_bias"].iloc[0])
    metric_rows: list[dict[str, Any]] = []
    for target_name, target in targets.items():
        prediction = sum(
            weights[component] * component_predictions[component][target_name]
            for component in weights
        )
        prediction = postprocess_ensemble_prediction(
            spec, prediction + deployment_bias, target
        )
        scored = scored_frame(target, prediction, candidate=name)
        metric_rows.append(
            {"feature_set": name, "split": target_name, **required_metrics(scored)}
        )
    metadata = {
        "feature_set": name,
        "feature_columns": features.columns,
        "numeric_features": features.numeric,
        "categorical_features": features.categorical,
        "component_candidates": [asdict(item) for item in component_candidates()],
        "weights": weights,
        "deployment_bias": deployment_bias,
        "component_tree_nodes": component_nodes,
        "total_tree_nodes": int(sum(component_nodes.values())),
    }
    (model_dir / "metadata.json").write_text(
        json.dumps(json_ready(metadata), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return pd.DataFrame(metric_rows), metadata


def main() -> int:
    parser = argparse.ArgumentParser(
        description="기존 주 모델에 strict-prior 제안 피처를 결합하고 중복/저효율 schema를 비교"
    )
    parser.add_argument(
        "--cache", type=Path, default=Path("data/analysis_cache/all_prearrival_A.pkl")
    )
    parser.add_argument(
        "--flow-cache",
        type=Path,
        default=Path("data/analysis_cache/all_prearrival_A_stop_flows.pkl"),
    )
    parser.add_argument(
        "--augmented-cache",
        type=Path,
        default=Path("data/analysis_cache/main_model_augmented_features.pkl"),
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("analysis/main_model_feature_augmentation_results"),
    )
    parser.add_argument("--feature-sets", nargs="*", default=list(feature_sets()))
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--skip-final-fit", action="store_true")
    args = parser.parse_args()

    data = prepare_augmented(args.cache, args.flow_cache, args.augmented_cache)
    data = add_observed_capacity_features(data)
    selected_sets = feature_sets()
    unknown = sorted(set(args.feature_sets) - set(selected_sets))
    if unknown:
        raise ValueError(f"unknown feature sets: {unknown}")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    correlations, missing = correlation_audit(data)
    correlations.to_csv(args.output_dir / "high_correlation_pairs.csv", index=False)
    missing.to_csv(args.output_dir / "feature_missingness.csv", index=False)
    qualitative_catalog().to_csv(
        args.output_dir / "qualitative_overlap_catalog.csv", index=False
    )

    development = data.loc[data["date"].isin(DEFAULT_DEVELOPMENT_DATES)].copy()
    folds = development_folds(development, DEFAULT_DEVELOPMENT_DATES)
    all_oof: list[pd.DataFrame] = []
    all_metrics: list[dict[str, Any]] = []
    all_daily: list[pd.DataFrame] = []
    all_diagnostics: list[pd.DataFrame] = []
    for name in args.feature_sets:
        output, metrics, daily, diagnostics = run_feature_set(
            name, selected_sets[name], folds, seed=args.seed
        )
        all_oof.append(output)
        all_metrics.append(metrics)
        all_daily.append(daily)
        all_diagnostics.append(diagnostics)
    oof = pd.concat(all_oof, ignore_index=True)
    metrics = pd.DataFrame(all_metrics)
    daily = pd.concat(all_daily, ignore_index=True)
    diagnostics = pd.concat(all_diagnostics, ignore_index=True)
    oof.to_pickle(args.output_dir / "oof_predictions.pkl")
    metrics.to_csv(args.output_dir / "development_metrics.csv", index=False)
    daily.to_csv(args.output_dir / "daily_metrics.csv", index=False)
    diagnostics.to_csv(args.output_dir / "importance_and_fit_diagnostics.csv", index=False)
    bootstrap = pd.DataFrame()
    if "baseline_official" in set(oof["candidate"]):
        candidates = [
            name for name in args.feature_sets if name != "baseline_official"
        ]
        bootstrap = pd.DataFrame(
            [
                paired_trip_bootstrap(oof, candidate, low_only=low_only, seed=args.seed)
                for candidate in candidates
                for low_only in (False, True)
            ]
        )
        bootstrap.to_csv(args.output_dir / "paired_trip_bootstrap.csv", index=False)

    final_metrics = pd.DataFrame()
    final_metadata: dict[str, Any] = {}
    if not args.skip_final_fit:
        targets = {
            "locked_test_2026_08_11": data.loc[data["date"].eq(DEFAULT_TEST_DATE)].copy(),
            "weekend_stress": data.loc[data["date"].isin(DEFAULT_STRESS_DATES)].copy(),
        }
        final_rows: list[pd.DataFrame] = []
        for name in args.feature_sets:
            print(f"[{name}] final fit", flush=True)
            rows, metadata = fit_final_models(
                name,
                selected_sets[name],
                development,
                targets,
                oof.loc[oof["candidate"].eq(name)].copy(),
                args.output_dir,
                seed=args.seed,
            )
            final_rows.append(rows)
            final_metadata[name] = metadata
        final_metrics = pd.concat(final_rows, ignore_index=True)
        final_metrics.to_csv(args.output_dir / "confirmation_metrics.csv", index=False)

    summary = {
        "protocol": {
            "development_dates": DEFAULT_DEVELOPMENT_DATES,
            "locked_test_date": DEFAULT_TEST_DATE,
            "stress_dates": DEFAULT_STRESS_DATES,
            "feature_history_rule": "same-route strictly earlier calendar dates",
            "selection_rule": "development rolling-origin only; locked test and stress are report-only",
            "full_classification_rule": "regression point prediction <= 0.5 seats",
            "correlation_rule": "development-only Pearson and Spearman; report abs(correlation)>=0.85",
            "model_rule": "fixed official HGB/ExtraTrees/LightGBM families, parameters and 0.4/0.4/0.2 weights",
        },
        "feature_sets": {
            name: {"numeric": value.numeric, "categorical": value.categorical}
            for name, value in selected_sets.items()
        },
        "preceding_bus_segment_audit": data.attrs.get("preceding_bus_segment_audit"),
        "development_metrics": metrics.to_dict(orient="records"),
        "paired_trip_bootstrap": bootstrap.to_dict(orient="records"),
        "confirmation_metrics": final_metrics.to_dict(orient="records"),
        "final_metadata": final_metadata,
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(metrics.to_string(index=False))
    if not final_metrics.empty:
        print("\nconfirmation")
        print(final_metrics.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/model_correction_layer_experiment.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/model_correction_layer_experiment.py
"""Forward evaluation of classification, interval, and model correction layers.

The experiment is deliberately post-hoc and leak-free:

* point/interval predictions are loaded from the frozen, shared OOF comparison;
* 2026-08-13 predictions/residuals calibrate correction layers;
* every reported correction result is evaluated on 2026-08-14 only;
* hurdle and ordinal heads train before 2026-08-13, calibrate on 2026-08-13,
  and are evaluated on 2026-08-14;
* no source or feature cache is refreshed or rebuilt.
"""
from __future__ import annotations

import argparse
import json
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Any, Iterable

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline

from confidence_interval_comparison import (
    FROZEN_MANIFEST,
    candidate_features,
    load_frozen_features,
    prepare_comparison_data,
)
from model_feasibility import FeatureSet, make_preprocessor
from uncertainty_heads import (
    _model_pipeline,
    event_weights,
    weighted_quantile,
)
from yeonwu_peer_feature_experiment import DEFAULT_ROUTES, ROOT, load_embedded_modules


CALIBRATION_DATE = "2026-08-13"
EVALUATION_DATE = "2026-08-14"
ORDINAL_BUCKETS = (0, 3, 5, 10)
BASE_MODELS = (
    "state_profile",
    "pooled_main",
    "bus_model_2",
    "sanghyuk_ridge",
    "minseok_random_forest",
)
PREDICTION_EXPORT_COLUMNS = (
    "event_id",
    "trip_id",
    "date",
    "snapshot_time",
    "route_id",
    "route_name",
    "label_seats",
    "snapshot_remaining_seats",
    "target_stop_gap",
    "capacity",
    "prediction",
    "base_model",
    "correction_method",
    "risk_head",
    "threshold_policy",
    "full_probability",
    "full_prediction",
)


def _json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): _json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def _classification_metrics(
    data: pd.DataFrame,
    prediction: np.ndarray,
    *,
    full_probability: np.ndarray | None = None,
    full_prediction: np.ndarray | None = None,
) -> dict[str, float | int]:
    truth = data["label_seats"].to_numpy(dtype=float)
    weights = event_weights(data)
    prediction = np.asarray(prediction, dtype=float)
    low = truth <= 10
    full_true = truth == 0
    full_pred = prediction <= 0.5 if full_prediction is None else np.asarray(full_prediction, dtype=bool)
    result: dict[str, float | int] = {
        "rows": int(len(data)),
        "events": int(data["event_id"].nunique()),
        "low_0_10_events": int(data.loc[low, "event_id"].nunique()),
        "full_events": int(data.loc[full_true, "event_id"].nunique()),
        "event_balanced_mae": float(np.average(np.abs(truth - prediction), weights=weights)),
        "low_0_10_mae": (
            float(np.average(np.abs(truth[low] - prediction[low]), weights=weights[low]))
            if low.any()
            else np.nan
        ),
        "full_accuracy": float(accuracy_score(full_true, full_pred, sample_weight=weights)),
        "full_recall": float(recall_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
        "full_precision": float(precision_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
        "full_f1": float(f1_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
    }
    if full_probability is not None:
        probability = np.asarray(full_probability, dtype=float)
        result["full_brier"] = float(
            np.average((probability - full_true.astype(float)) ** 2, weights=weights)
        )
    return result


def _metric_tables(
    predictions: pd.DataFrame,
    *,
    candidate_columns: tuple[str, ...],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    pooled_rows: list[dict[str, Any]] = []
    route_rows: list[dict[str, Any]] = []
    for keys, frame in predictions.groupby(list(candidate_columns), sort=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        identity = dict(zip(candidate_columns, keys, strict=True))
        probability = frame["full_probability"].to_numpy(float) if "full_probability" in frame else None
        full_prediction = frame["full_prediction"].to_numpy(bool) if "full_prediction" in frame else None
        pooled_rows.append(
            {
                **identity,
                **_classification_metrics(
                    frame,
                    frame["prediction"].to_numpy(float),
                    full_probability=probability,
                    full_prediction=full_prediction,
                ),
            }
        )
        for (route_id, route_name), route in frame.groupby(["route_id", "route_name"], sort=False):
            route_probability = route["full_probability"].to_numpy(float) if "full_probability" in route else None
            route_full = route["full_prediction"].to_numpy(bool) if "full_prediction" in route else None
            route_rows.append(
                {
                    **identity,
                    "route_id": route_id,
                    "route_name": route_name,
                    **_classification_metrics(
                        route,
                        route["prediction"].to_numpy(float),
                        full_probability=route_probability,
                        full_prediction=route_full,
                    ),
                }
            )
    pooled = pd.DataFrame(pooled_rows)
    by_route = pd.DataFrame(route_rows)
    metric_columns = [
        "event_balanced_mae",
        "low_0_10_mae",
        "full_accuracy",
        "full_recall",
        "full_precision",
        "full_f1",
    ]
    if "full_brier" in by_route:
        metric_columns.append("full_brier")
    macro = by_route.groupby(list(candidate_columns), sort=False)[metric_columns].mean().reset_index()
    macro.insert(len(candidate_columns), "routes", len(DEFAULT_ROUTES))
    return pooled, by_route, macro


def _weighted_median(values: np.ndarray, weights: np.ndarray) -> float:
    return weighted_quantile(np.asarray(values, float), np.asarray(weights, float), 0.5)


def _meta_frame(data: pd.DataFrame) -> pd.DataFrame:
    output = pd.DataFrame(index=data.index)
    output["base_prediction"] = data["prediction"].to_numpy(float)
    output["snapshot_remaining_seats"] = data["snapshot_remaining_seats"].to_numpy(float)
    output["target_stop_gap"] = data["target_stop_gap"].to_numpy(float)
    output["prediction_minus_current"] = output["base_prediction"] - output["snapshot_remaining_seats"]
    output["route_code"] = data["route_id"].astype(str).to_numpy()
    return output


def _apply_route_shrinkage(
    calibration: pd.DataFrame,
    evaluation: pd.DataFrame,
    *,
    prior_events: float = 50.0,
) -> tuple[np.ndarray, dict[str, Any]]:
    weights = event_weights(calibration)
    residual = calibration["label_seats"].to_numpy(float) - calibration["prediction"].to_numpy(float)
    global_bias = _weighted_median(residual, weights)
    route_biases: dict[str, float] = {}
    audit: dict[str, Any] = {}
    for route_id, route in calibration.groupby("route_id", sort=False):
        mask = calibration.index.isin(route.index)
        local = _weighted_median(residual[mask], weights[mask])
        events = int(route["event_id"].nunique())
        strength = events / (events + prior_events)
        shrunk = strength * local + (1.0 - strength) * global_bias
        route_biases[str(route_id)] = float(shrunk)
        audit[str(route_id)] = {
            "events": events,
            "raw_median_residual": float(local),
            "shrinkage_strength": float(strength),
            "shrunk_bias": float(shrunk),
        }
    correction = evaluation["route_id"].astype(str).map(route_biases).fillna(global_bias).to_numpy(float)
    prediction = evaluation["prediction"].to_numpy(float) + correction
    return prediction, {"global_bias": global_bias, "routes": audit}


def _apply_ridge_residual(
    calibration: pd.DataFrame,
    evaluation: pd.DataFrame,
    *,
    alpha: float = 30.0,
) -> np.ndarray:
    features = FeatureSet(
        "ridge_route_residual",
        ("base_prediction", "snapshot_remaining_seats", "target_stop_gap", "prediction_minus_current"),
        ("route_code",),
    )
    train_x, test_x = _meta_frame(calibration), _meta_frame(evaluation)
    residual = calibration["label_seats"].to_numpy(float) - calibration["prediction"].to_numpy(float)
    model = Pipeline(
        [("features", make_preprocessor(features)), ("regressor", Ridge(alpha=alpha))]
    )
    model.fit(train_x[list(features.columns)], residual, regressor__sample_weight=event_weights(calibration))
    correction = np.clip(model.predict(test_x[list(features.columns)]), -8.0, 8.0)
    return evaluation["prediction"].to_numpy(float) + correction


def _apply_hgb_residual(
    calibration: pd.DataFrame,
    evaluation: pd.DataFrame,
    *,
    seed: int,
) -> np.ndarray:
    train_x, test_x = _meta_frame(calibration), _meta_frame(evaluation)
    route_values = sorted(set(train_x["route_code"]) | set(test_x["route_code"]))
    route_map = {value: index for index, value in enumerate(route_values)}
    for frame in (train_x, test_x):
        frame["route_code"] = frame["route_code"].map(route_map).fillna(-1).astype(float)
    residual = calibration["label_seats"].to_numpy(float) - calibration["prediction"].to_numpy(float)
    model = HistGradientBoostingRegressor(
        loss="absolute_error",
        learning_rate=0.04,
        max_iter=160,
        max_leaf_nodes=15,
        min_samples_leaf=80,
        l2_regularization=4.0,
        early_stopping=False,
        random_state=seed,
    )
    model.fit(train_x, residual, sample_weight=event_weights(calibration))
    correction = np.clip(model.predict(test_x), -8.0, 8.0)
    return evaluation["prediction"].to_numpy(float) + correction


def residual_correction_experiment(
    existing: pd.DataFrame,
    *,
    seed: int,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    outputs: list[pd.DataFrame] = []
    audits: dict[str, Any] = {}
    for model_name in BASE_MODELS:
        candidate = f"{model_name}__delta_per_stop"
        calibration = existing.loc[
            existing["candidate"].eq(candidate) & existing["date"].eq(CALIBRATION_DATE)
        ].copy()
        evaluation = existing.loc[
            existing["candidate"].eq(candidate) & existing["date"].eq(EVALUATION_DATE)
        ].copy()
        if calibration.empty or evaluation.empty:
            raise ValueError(f"missing existing OOF predictions for {candidate}")
        methods: dict[str, np.ndarray] = {
            "none": evaluation["prediction"].to_numpy(float),
        }
        route_prediction, route_audit = _apply_route_shrinkage(calibration, evaluation)
        methods["route_shrunk_median_bias"] = route_prediction
        methods["ridge_route_residual"] = _apply_ridge_residual(calibration, evaluation)
        methods["hgb_route_residual"] = _apply_hgb_residual(calibration, evaluation, seed=seed)
        audits[model_name] = route_audit
        for method, prediction in methods.items():
            frame = evaluation.copy()
            frame["base_model"] = model_name
            frame["correction_method"] = method
            frame["prediction"] = np.clip(prediction, 0.0, frame["capacity"].to_numpy(float))
            outputs.append(frame)
    return pd.concat(outputs, ignore_index=True), audits


def _platt_calibrate(
    model: Pipeline,
    calibration: pd.DataFrame,
    evaluation: pd.DataFrame,
    features: list[str],
    *,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    cal_raw = np.clip(model.predict_proba(calibration[features])[:, 1], 1e-6, 1 - 1e-6)
    eval_raw = np.clip(model.predict_proba(evaluation[features])[:, 1], 1e-6, 1 - 1e-6)
    target = calibration["label_seats"].eq(0).astype(int).to_numpy()
    if np.unique(target).size < 2:
        constant = float(np.average(target, weights=event_weights(calibration)))
        return np.full(len(calibration), constant), np.full(len(evaluation), constant)
    calibrator = LogisticRegression(C=100.0, max_iter=1000, random_state=seed)
    calibrator.fit(
        np.log(cal_raw / (1.0 - cal_raw)).reshape(-1, 1),
        target,
        sample_weight=event_weights(calibration),
    )
    return (
        calibrator.predict_proba(np.log(cal_raw / (1.0 - cal_raw)).reshape(-1, 1))[:, 1],
        calibrator.predict_proba(np.log(eval_raw / (1.0 - eval_raw)).reshape(-1, 1))[:, 1],
    )


def _threshold_score(
    data: pd.DataFrame,
    probability: np.ndarray,
    threshold: float,
    *,
    macro: bool,
) -> float:
    truth = data["label_seats"].eq(0).to_numpy()
    predicted = probability >= threshold
    if not macro:
        return float(f1_score(truth, predicted, sample_weight=event_weights(data), zero_division=0))
    scores: list[float] = []
    for _, route in data.assign(_truth=truth, _pred=predicted).groupby("route_id", sort=False):
        scores.append(
            float(
                f1_score(
                    route["_truth"],
                    route["_pred"],
                    sample_weight=event_weights(route),
                    zero_division=0,
                )
            )
        )
    return float(np.mean(scores))


def _select_threshold(
    data: pd.DataFrame,
    probability: np.ndarray,
    *,
    macro: bool,
) -> tuple[float, float]:
    candidates = np.unique(
        np.r_[np.linspace(0.01, 0.99, 99), np.quantile(probability, np.linspace(0.5, 0.999, 80))]
    )
    scored = [(_threshold_score(data, probability, float(value), macro=macro), float(value)) for value in candidates]
    score, threshold = max(scored, key=lambda item: (item[0], -item[1]))
    return threshold, score


def _fit_hurdle(
    train: pd.DataFrame,
    calibration: pd.DataFrame,
    evaluation: pd.DataFrame,
    feature_set: FeatureSet,
    *,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    features = list(feature_set.columns)
    model = _model_pipeline(
        tuple(feature_set.numeric),
        tuple(feature_set.categorical),
        lgb.LGBMClassifier(
            objective="binary",
            n_estimators=350,
            learning_rate=0.035,
            num_leaves=31,
            min_child_samples=25,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
        ),
    )
    target = train["label_seats"].eq(0).astype(int).to_numpy()
    model.fit(train[features], target, model__sample_weight=event_weights(train))
    return _platt_calibrate(model, calibration, evaluation, features, seed=seed + 1)


def _ordinal_class(values: pd.Series) -> np.ndarray:
    return np.select(
        [values.eq(0), values.le(3), values.le(5), values.le(10)],
        [0, 1, 2, 3],
        default=4,
    ).astype(int)


def _fit_ordinal(
    train: pd.DataFrame,
    calibration: pd.DataFrame,
    evaluation: pd.DataFrame,
    feature_set: FeatureSet,
    *,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    features = list(feature_set.columns)
    model = _model_pipeline(
        tuple(feature_set.numeric),
        tuple(feature_set.categorical),
        lgb.LGBMClassifier(
            objective="multiclass",
            num_class=5,
            n_estimators=400,
            learning_rate=0.035,
            num_leaves=31,
            min_child_samples=25,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
        ),
    )
    model.fit(
        train[features],
        _ordinal_class(train["label_seats"]),
        model__sample_weight=event_weights(train),
    )
    cal_raw = np.clip(model.predict_proba(calibration[features]), 1e-8, 1.0)
    eval_raw = np.clip(model.predict_proba(evaluation[features]), 1e-8, 1.0)
    calibrator = LogisticRegression(C=10.0, max_iter=1500, random_state=seed + 1)
    calibrator.fit(
        np.log(cal_raw),
        _ordinal_class(calibration["label_seats"]),
        sample_weight=event_weights(calibration),
    )
    cal_probability = calibrator.predict_proba(np.log(cal_raw))
    eval_probability = calibrator.predict_proba(np.log(eval_raw))
    # P(Y=0) and cumulative P(Y<=10); the five-class formulation shares signal
    # among adjacent low-seat categories instead of fitting four independent heads.
    return (
        cal_probability[:, 0],
        eval_probability[:, 0],
        cal_probability[:, :4].sum(axis=1),
        eval_probability[:, :4].sum(axis=1),
    )


def risk_head_experiment(
    data: pd.DataFrame,
    existing: pd.DataFrame,
    *,
    seed: int,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    feature_set = candidate_features()["state_profile"]
    weekdays = pd.to_datetime(data["date"]).dt.dayofweek.lt(5)
    train = data.loc[data["date"].lt(CALIBRATION_DATE) & weekdays].copy()
    calibration = data.loc[data["date"].eq(CALIBRATION_DATE)].copy()
    evaluation = data.loc[data["date"].eq(EVALUATION_DATE)].copy()
    for name, frame in (("train", train), ("calibration", calibration), ("evaluation", evaluation)):
        missing = sorted(set(DEFAULT_ROUTES) - set(frame["route_id"].astype(str)))
        if frame.empty or missing:
            raise ValueError(f"{name} split invalid; missing_routes={missing}")

    hurdle_cal, hurdle_eval = _fit_hurdle(
        train, calibration, evaluation, feature_set, seed=seed
    )
    ordinal_cal, ordinal_eval, ordinal_low10_cal, ordinal_low10_eval = _fit_ordinal(
        train, calibration, evaluation, feature_set, seed=seed + 20
    )
    point_candidate = "state_profile__delta_per_stop"
    point = existing.loc[
        existing["candidate"].eq(point_candidate) & existing["date"].eq(EVALUATION_DATE)
    ].copy()
    key_columns = ["event_id", "snapshot_time"]
    point_index = pd.MultiIndex.from_frame(point[key_columns])
    eval_index = pd.MultiIndex.from_frame(evaluation[key_columns])
    if point_index.has_duplicates or eval_index.has_duplicates:
        raise ValueError("risk-head alignment keys are not unique")
    position = pd.Series(np.arange(len(evaluation)), index=eval_index).reindex(point_index).to_numpy()
    if np.isnan(position).any():
        raise ValueError("risk-head evaluation rows do not align with point predictions")
    position = position.astype(int)

    outputs: list[pd.DataFrame] = []
    audit: dict[str, Any] = {}
    for head, cal_probability, eval_probability in (
        ("hurdle", hurdle_cal, hurdle_eval),
        ("ordinal_multiclass", ordinal_cal, ordinal_eval),
    ):
        for objective, macro in (("pooled_f1_threshold", False), ("route_macro_f1_threshold", True)):
            threshold, calibration_score = _select_threshold(
                calibration, cal_probability, macro=macro
            )
            frame = point.copy()
            probability = eval_probability[position]
            frame["risk_head"] = head
            frame["threshold_policy"] = objective
            frame["full_probability"] = probability
            frame["full_prediction"] = probability >= threshold
            outputs.append(frame)
            audit[f"{head}__{objective}"] = {
                "threshold": threshold,
                "calibration_f1": calibration_score,
            }
    baseline = point.copy()
    baseline["risk_head"] = "regression_threshold"
    baseline["threshold_policy"] = "prediction_le_0_5"
    baseline["full_probability"] = np.nan
    baseline["full_prediction"] = baseline["prediction"].le(0.5)
    outputs.append(baseline)

    ordinal_low10_truth = calibration["label_seats"].le(10).to_numpy(float)
    audit["ordinal_low10_brier_calibration"] = float(
        np.average(
            (ordinal_low10_cal - ordinal_low10_truth) ** 2,
            weights=event_weights(calibration),
        )
    )
    eval_low10_truth = evaluation["label_seats"].le(10).to_numpy(float)
    audit["ordinal_low10_brier_evaluation"] = float(
        np.average(
            (ordinal_low10_eval - eval_low10_truth) ** 2,
            weights=event_weights(evaluation),
        )
    )
    return pd.concat(outputs, ignore_index=True), audit


def _risk_band(probability: pd.Series) -> pd.Series:
    return pd.cut(
        probability,
        bins=[-np.inf, 0.02, 0.10, 0.30, np.inf],
        labels=["risk_very_low", "risk_low", "risk_medium", "risk_high"],
    ).astype("string")


def _gap_band(gap: pd.Series) -> pd.Series:
    return pd.cut(
        gap,
        bins=[0, 2, 5, 10, np.inf],
        labels=["gap_1_2", "gap_3_5", "gap_6_10", "gap_11_plus"],
    ).astype("string")


@dataclass
class HierarchicalAdjustments:
    global_adjustment: float
    levels: list[tuple[tuple[str, ...], dict[tuple[str, ...], float]]]


def _fit_hierarchical_adjustments(
    calibration: pd.DataFrame,
    *,
    alpha: float = 0.10,
    min_events: int = 30,
) -> HierarchicalAdjustments:
    frame = calibration.copy()
    frame["risk_band"] = _risk_band(frame["low_seat_probability"])
    frame["gap_band"] = _gap_band(frame["target_stop_gap"])
    y = frame["label_seats"].to_numpy(float)
    conformity = np.maximum(
        frame["interval_lower_90"].to_numpy(float) - y,
        y - frame["interval_upper_90"].to_numpy(float),
    )
    weights = event_weights(frame)
    global_adjustment = weighted_quantile(conformity, weights, 1.0 - alpha)
    specifications = [
        ("route_id", "risk_band", "gap_band"),
        ("route_id", "risk_band"),
        ("route_id",),
        ("risk_band", "gap_band"),
    ]
    levels: list[tuple[tuple[str, ...], dict[tuple[str, ...], float]]] = []
    for columns in specifications:
        values: dict[tuple[str, ...], float] = {}
        grouper: str | list[str] = columns[0] if len(columns) == 1 else list(columns)
        for keys, group in frame.groupby(grouper, observed=True, sort=False):
            key = keys if isinstance(keys, tuple) else (keys,)
            if group["event_id"].nunique() < min_events:
                continue
            mask = frame.index.isin(group.index)
            values[tuple(str(item) for item in key)] = weighted_quantile(
                conformity[mask], weights[mask], 1.0 - alpha
            )
        levels.append((columns, values))
    return HierarchicalAdjustments(global_adjustment=global_adjustment, levels=levels)


def _hierarchical_adjustment(
    bundle: HierarchicalAdjustments,
    evaluation: pd.DataFrame,
) -> np.ndarray:
    frame = evaluation.copy()
    frame["risk_band"] = _risk_band(frame["low_seat_probability"])
    frame["gap_band"] = _gap_band(frame["target_stop_gap"])
    output = np.full(len(frame), np.nan)
    for columns, mapping in bundle.levels:
        missing = ~np.isfinite(output)
        if not missing.any():
            break
        keys = frame.loc[missing, list(columns)].astype(str).apply(tuple, axis=1)
        found = keys.map(mapping).to_numpy(float)
        positions = np.flatnonzero(missing)
        valid = np.isfinite(found)
        output[positions[valid]] = found[valid]
    output[~np.isfinite(output)] = bundle.global_adjustment
    return output


def _interval_metrics(data: pd.DataFrame) -> dict[str, float | int]:
    y = data["label_seats"].to_numpy(float)
    lower = data["interval_lower_90"].to_numpy(float)
    upper = data["interval_upper_90"].to_numpy(float)
    weights = event_weights(data)
    covered = (lower <= y) & (y <= upper)
    width = upper - lower
    score = width.copy()
    below, above = y < lower, y > upper
    score[below] += 20.0 * (lower[below] - y[below])
    score[above] += 20.0 * (y[above] - upper[above])
    low = y <= 10
    return {
        "rows": int(len(data)),
        "events": int(data["event_id"].nunique()),
        "interval_90_coverage": float(np.average(covered, weights=weights)),
        "interval_90_mean_width": float(np.average(width, weights=weights)),
        "interval_90_score": float(np.average(score, weights=weights)),
        "low_0_10_events": int(data.loc[low, "event_id"].nunique()),
        "low_0_10_coverage": float(np.average(covered[low], weights=weights[low])) if low.any() else np.nan,
        "low_0_10_mean_width": float(np.average(width[low], weights=weights[low])) if low.any() else np.nan,
    }


def interval_recalibration_experiment(existing: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    candidate = "state_profile__delta_per_stop"
    calibration = existing.loc[
        existing["candidate"].eq(candidate) & existing["date"].eq(CALIBRATION_DATE)
    ].copy()
    evaluation = existing.loc[
        existing["candidate"].eq(candidate) & existing["date"].eq(EVALUATION_DATE)
    ].copy()
    hierarchy = _fit_hierarchical_adjustments(calibration)
    y = calibration["label_seats"].to_numpy(float)
    conformity = np.maximum(
        calibration["interval_lower_90"].to_numpy(float) - y,
        y - calibration["interval_upper_90"].to_numpy(float),
    )
    global_adjustment = weighted_quantile(conformity, event_weights(calibration), 0.90)
    methods = {
        "existing_group_conformal": np.zeros(len(evaluation)),
        "global_forward_recalibration": np.full(len(evaluation), global_adjustment),
        "hierarchical_route_risk_gap": _hierarchical_adjustment(hierarchy, evaluation),
    }
    frames: list[pd.DataFrame] = []
    for method, adjustment in methods.items():
        frame = evaluation.copy()
        frame["interval_method"] = method
        frame["interval_lower_90"] = np.clip(
            frame["interval_lower_90"].to_numpy(float) - adjustment,
            0.0,
            frame["capacity"].to_numpy(float),
        )
        frame["interval_upper_90"] = np.clip(
            frame["interval_upper_90"].to_numpy(float) + adjustment,
            0.0,
            frame["capacity"].to_numpy(float),
        )
        frames.append(frame)
    predictions = pd.concat(frames, ignore_index=True)
    pooled = pd.DataFrame(
        [
            {"interval_method": method, **_interval_metrics(frame)}
            for method, frame in predictions.groupby("interval_method", sort=False)
        ]
    )
    route_rows = []
    for (method, route_id, route_name), frame in predictions.groupby(
        ["interval_method", "route_id", "route_name"], sort=False
    ):
        route_rows.append(
            {
                "interval_method": method,
                "route_id": route_id,
                "route_name": route_name,
                **_interval_metrics(frame),
            }
        )
    by_route = pd.DataFrame(route_rows)
    audit = {
        "global_forward_adjustment": global_adjustment,
        "hierarchical_global_fallback": hierarchy.global_adjustment,
        "hierarchical_levels": [
            {
                "columns": columns,
                "groups": {"|".join(key): value for key, value in mapping.items()},
            }
            for columns, mapping in hierarchy.levels
        ],
    }
    return pooled, by_route, audit


def main() -> int:
    parser = argparse.ArgumentParser(description="Frozen forward model-correction experiment")
    parser.add_argument(
        "--database", type=Path, default=ROOT / "data/gbis_api_cache.sqlite3"
    )
    parser.add_argument(
        "--featured-cache",
        type=Path,
        default=ROOT / "data/analysis_cache/state_profile_main_features.pkl",
    )
    parser.add_argument("--frozen-manifest", type=Path, default=FROZEN_MANIFEST)
    parser.add_argument(
        "--existing-predictions",
        type=Path,
        default=ROOT / "analysis/confidence_interval_delta_lower_results/predictions.pkl",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=ROOT / "analysis/model_correction_layer_results",
    )
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    load_embedded_modules()
    frozen_args = SimpleNamespace(
        refresh=False,
        rebuild_features=False,
        frozen_manifest=args.frozen_manifest,
        featured_cache=args.featured_cache,
        database=args.database,
    )
    frozen_data, route_metadata, manifest = load_frozen_features(frozen_args)
    data = prepare_comparison_data(frozen_data)
    existing = pd.read_pickle(args.existing_predictions)
    required_dates = {CALIBRATION_DATE, EVALUATION_DATE}
    if not required_dates.issubset(set(existing["date"].astype(str))):
        raise ValueError("existing predictions do not contain both forward dates")

    correction_predictions, correction_audit = residual_correction_experiment(
        existing, seed=args.seed
    )
    correction_pooled, correction_routes, correction_macro = _metric_tables(
        correction_predictions,
        candidate_columns=("base_model", "correction_method"),
    )

    risk_predictions, risk_audit = risk_head_experiment(
        data, existing, seed=args.seed + 100
    )
    risk_pooled, risk_routes, risk_macro = _metric_tables(
        risk_predictions,
        candidate_columns=("risk_head", "threshold_policy"),
    )

    interval_pooled, interval_routes, interval_audit = interval_recalibration_experiment(existing)
    interval_macro = (
        interval_routes.groupby("interval_method", sort=False)[
            [
                "interval_90_coverage",
                "interval_90_mean_width",
                "interval_90_score",
                "low_0_10_coverage",
                "low_0_10_mean_width",
            ]
        ]
        .mean()
        .reset_index()
    )
    interval_macro.insert(1, "routes", len(DEFAULT_ROUTES))

    args.output_dir.mkdir(parents=True, exist_ok=True)
    correction_predictions[
        [column for column in PREDICTION_EXPORT_COLUMNS if column in correction_predictions]
    ].to_pickle(args.output_dir / "correction_predictions.pkl.gz", compression="gzip")
    correction_pooled.to_csv(args.output_dir / "correction_metrics_pooled.csv", index=False)
    correction_routes.to_csv(args.output_dir / "correction_metrics_by_route.csv", index=False)
    correction_macro.to_csv(args.output_dir / "correction_metrics_route_macro.csv", index=False)
    risk_predictions[
        [column for column in PREDICTION_EXPORT_COLUMNS if column in risk_predictions]
    ].to_pickle(args.output_dir / "risk_head_predictions.pkl.gz", compression="gzip")
    risk_pooled.to_csv(args.output_dir / "risk_head_metrics_pooled.csv", index=False)
    risk_routes.to_csv(args.output_dir / "risk_head_metrics_by_route.csv", index=False)
    risk_macro.to_csv(args.output_dir / "risk_head_metrics_route_macro.csv", index=False)
    interval_pooled.to_csv(args.output_dir / "interval_metrics_pooled.csv", index=False)
    interval_routes.to_csv(args.output_dir / "interval_metrics_by_route.csv", index=False)
    interval_macro.to_csv(args.output_dir / "interval_metrics_route_macro.csv", index=False)

    summary = {
        "protocol": {
            "source_cutoff": manifest["source_cutoff"],
            "source_fingerprint": manifest["source_fingerprint"],
            "calibration_date": CALIBRATION_DATE,
            "evaluation_date": EVALUATION_DATE,
            "correction_training_rule": "only prior-date OOF residuals calibrate each base model",
            "risk_head_training_rule": "weekdays before calibration date; calibration date selects probability calibration and thresholds",
            "interval_rule": "prior-date OOF interval conformity; hierarchical route x risk x gap with sparse-group fallback",
            "included_routes": DEFAULT_ROUTES,
            "excluded_routes": "all non-active routes; no complete pooled frozen coverage",
            "sealed_partial_date": manifest["sealed_partial_date"],
            "sealed_data_accessed": False,
        },
        "correction_audit": correction_audit,
        "risk_head_audit": risk_audit,
        "interval_audit": interval_audit,
        "correction_metrics_pooled": correction_pooled.to_dict(orient="records"),
        "correction_metrics_route_macro": correction_macro.to_dict(orient="records"),
        "risk_head_metrics_pooled": risk_pooled.to_dict(orient="records"),
        "risk_head_metrics_route_macro": risk_macro.to_dict(orient="records"),
        "interval_metrics_pooled": interval_pooled.to_dict(orient="records"),
        "interval_metrics_route_macro": interval_macro.to_dict(orient="records"),
        "route_cache_metadata": route_metadata,
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(_json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("\nResidual correction: pooled")
    print(correction_pooled.to_string(index=False))
    print("\nResidual correction: route macro")
    print(correction_macro.to_string(index=False))
    print("\nRisk heads: pooled")
    print(risk_pooled.to_string(index=False))
    print("\nRisk heads: route macro")
    print(risk_macro.to_string(index=False))
    print("\nIntervals: pooled")
    print(interval_pooled.to_string(index=False))
    print("\nIntervals: route macro")
    print(interval_macro.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/model_feasibility.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/model_feasibility.py
from __future__ import annotations

import argparse
import json
import math
import sqlite3
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


DEFAULT_ROUTE_ID = "219000013"
DEFAULT_ROUTE_NAME = "1000"
PEAK_HOURS = set(range(6, 11)) | set(range(16, 22))
QUALITY_ORDER = {"A": 0, "B": 1, "C": 2}


@dataclass(frozen=True)
class FeatureSet:
    name: str
    numeric: tuple[str, ...]
    categorical: tuple[str, ...]

    @property
    def columns(self) -> list[str]:
        return [*self.numeric, *self.categorical]


PLANNING = FeatureSet(
    name="planning",
    numeric=(
        "time_sin",
        "time_cos",
        "route_progress",
        "x",
        "y",
    ),
    categorical=(
        "station_seq_cat",
        "direction",
        "day_of_week",
        "center_yn",
    ),
)

REALTIME = FeatureSet(
    name="realtime_3stops",
    numeric=(
        *PLANNING.numeric,
        "capacity",
        "upstream_load_ratio_3",
        "upstream_load_ratio_5",
        "upstream_stop_distance_3",
        "upstream_age_minutes_3",
        "load_change_5_to_3",
        "previous_bus_load_ratio",
        "headway_minutes",
    ),
    categorical=(*PLANNING.categorical, "low_plate_cat"),
)


def _as_records(frame: pd.DataFrame) -> list[dict[str, Any]]:
    result: list[dict[str, Any]] = []
    for record in frame.to_dict(orient="records"):
        cleaned: dict[str, Any] = {}
        for key, value in record.items():
            if pd.isna(value):
                cleaned[key] = None
            elif isinstance(value, (np.integer,)):
                cleaned[key] = int(value)
            elif isinstance(value, (np.floating,)):
                cleaned[key] = float(value)
            else:
                cleaned[key] = value
        result.append(cleaned)
    return result


def load_data(db_path: Path, route_id: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    uri = f"file:{db_path.resolve()}?mode=ro"
    with sqlite3.connect(uri, uri=True) as connection:
        locations = pd.read_sql_query(
            """
            SELECT run_id, observed_at_kst, query_time, route_id, vehicle_id,
                   plate_no, station_id, station_seq, remaining_seats,
                   low_plate, state_code
            FROM bus_locations
            WHERE route_id = ?
            ORDER BY vehicle_id, observed_at_kst, run_id
            """,
            connection,
            params=(route_id,),
        )
        stations = pd.read_sql_query(
            """
            SELECT route_id, station_id, station_seq, station_name, mobile_no,
                   region_name, x, y, center_yn
            FROM route_stations
            WHERE route_id = ?
            ORDER BY station_seq
            """,
            connection,
            params=(route_id,),
        )
    if locations.empty:
        raise ValueError(f"노선 {route_id}의 차량 위치 데이터가 없습니다.")
    if stations.empty:
        raise ValueError(f"노선 {route_id}의 정류장 메타데이터가 없습니다.")
    locations["observed_at"] = pd.to_datetime(locations["observed_at_kst"])
    return locations, stations


def infer_turnaround_seq(stations: pd.DataFrame) -> int:
    ordered = stations.sort_values("station_seq")
    valid = ordered.dropna(subset=["x", "y"])
    if len(valid) < 3:
        return int(round(float(ordered["station_seq"].max()) / 2))
    origin = valid.iloc[0]
    # GBIS x/y는 경도/위도다. 한 노선 안에서 최대 직선거리 정류장은 회차점의 좋은 근사다.
    lat0 = math.radians(float(origin["y"]))
    lon0 = math.radians(float(origin["x"]))
    lat = np.radians(valid["y"].astype(float).to_numpy())
    lon = np.radians(valid["x"].astype(float).to_numpy())
    dlat = lat - lat0
    dlon = lon - lon0
    hav = np.sin(dlat / 2) ** 2 + np.cos(lat0) * np.cos(lat) * np.sin(dlon / 2) ** 2
    return int(valid.iloc[int(np.argmax(hav))]["station_seq"])


def build_visits(locations: pd.DataFrame, stations: pd.DataFrame) -> pd.DataFrame:
    rows = locations.sort_values(["vehicle_id", "observed_at", "run_id"]).copy()
    grouped = rows.groupby("vehicle_id", sort=False)
    rows["previous_station_seq"] = grouped["station_seq"].shift()
    rows["previous_observed_at"] = grouped["observed_at"].shift()
    gap_minutes = (rows["observed_at"] - rows["previous_observed_at"]).dt.total_seconds() / 60
    rows["new_visit"] = (
        rows["previous_station_seq"].isna()
        | rows["station_seq"].ne(rows["previous_station_seq"])
        | gap_minutes.gt(20)
    )
    rows["visit_no"] = rows.groupby("vehicle_id", sort=False)["new_visit"].cumsum()
    keys = ["vehicle_id", "visit_no"]

    summary = rows.groupby(keys, sort=False).agg(
        route_id=("route_id", "first"),
        plate_no=("plate_no", "first"),
        station_id=("station_id", "first"),
        station_seq=("station_seq", "first"),
        first_seen=("observed_at", "first"),
        last_seen=("observed_at", "last"),
        first_state=("state_code", "first"),
        first_seats=("remaining_seats", "first"),
        last_state=("state_code", "last"),
        last_seats=("remaining_seats", "last"),
        low_plate=("low_plate", "first"),
        samples=("run_id", "size"),
    ).reset_index()

    departure = (
        rows.loc[(rows["state_code"] == 2) & (rows["remaining_seats"] >= 0)]
        .groupby(keys, sort=False)
        .tail(1)[[*keys, "remaining_seats", "observed_at"]]
        .rename(
            columns={
                "remaining_seats": "departure_seats",
                "observed_at": "departure_seen",
            }
        )
    )
    arrival = (
        rows.loc[(rows["state_code"] == 1) & (rows["remaining_seats"] >= 0)]
        .groupby(keys, sort=False)
        .head(1)[[*keys, "remaining_seats", "observed_at"]]
        .rename(
            columns={
                "remaining_seats": "observed_arrival_seats",
                "observed_at": "arrival_seen",
            }
        )
    )
    visits = summary.merge(departure, on=keys, how="left").merge(
        arrival, on=keys, how="left"
    )
    visits = visits.sort_values(["vehicle_id", "visit_no"]).reset_index(drop=True)

    by_vehicle = visits.groupby("vehicle_id", sort=False)
    visits["next_station_seq"] = by_vehicle["station_seq"].shift(-1)
    visits["next_first_seen"] = by_vehicle["first_seen"].shift(-1)
    visits["next_first_state"] = by_vehicle["first_state"].shift(-1)
    visits["next_first_seats"] = by_vehicle["first_seats"].shift(-1)
    next_gap = (visits["next_first_seen"] - visits["last_seen"]).dt.total_seconds() / 60

    visits["label_quality"] = pd.Series(pd.NA, index=visits.index, dtype="string")
    visits["label_seats"] = np.nan
    is_a = visits["departure_seats"].notna()
    is_b = (~is_a) & visits["last_state"].eq(0) & visits["last_seats"].ge(0)
    is_c = (
        (~is_a)
        & (~is_b)
        & visits["next_station_seq"].eq(visits["station_seq"] + 1)
        & next_gap.le(10)
        & visits["next_first_state"].isin([0, 1])
        & visits["next_first_seats"].ge(0)
    )
    # 문자 등급과 수치 좌석을 한 번에 대입하면 NumPy가 전체를 문자열로
    # 승격할 수 있으므로 열별로 대입해 label_seats의 수치형을 보존한다.
    visits.loc[is_a, "label_quality"] = "A"
    visits.loc[is_a, "label_seats"] = visits.loc[is_a, "departure_seats"]
    visits.loc[is_b, "label_quality"] = "B"
    visits.loc[is_b, "label_seats"] = visits.loc[is_b, "last_seats"]
    # C는 좌석 수 회귀에는 부적합하지만 만차/비만차 분류 라벨에는 검증상 안정적이다.
    visits.loc[is_c, "label_quality"] = "C"
    visits.loc[is_c, "label_seats"] = visits.loc[is_c, "next_first_seats"]
    visits["is_full"] = np.where(visits["label_quality"].notna(), visits["label_seats"].eq(0), np.nan)

    previous_seq = by_vehicle["station_seq"].shift()
    previous_time = by_vehicle["last_seen"].shift()
    trip_gap = (visits["first_seen"] - previous_time).dt.total_seconds() / 60
    new_trip = previous_seq.isna() | visits["station_seq"].le(previous_seq) | trip_gap.gt(30)
    visits["trip_no"] = new_trip.groupby(visits["vehicle_id"], sort=False).cumsum().astype(int)
    visits["trip_id"] = visits["vehicle_id"].astype(str) + "-" + visits["trip_no"].astype(str)

    # 사용자가 정류장에서 실제로 마주치는 좌석은 승객을 태운 뒤의 출발 좌석이
    # 아니라 태우기 전 도착 좌석이다. stateCd=1 직접 관측을 우선하고, 이것이
    # 빠졌을 때만 같은 운행의 바로 전 정류장 출발 좌석으로 보완한다.
    by_vehicle = visits.groupby("vehicle_id", sort=False)
    visits["previous_visit_station_seq"] = by_vehicle["station_seq"].shift()
    visits["previous_visit_trip_id"] = by_vehicle["trip_id"].shift()
    visits["previous_departure_seats"] = by_vehicle["departure_seats"].shift()
    visits["previous_departure_seen"] = by_vehicle["departure_seen"].shift()
    previous_gap = (
        visits["first_seen"] - visits["previous_departure_seen"]
    ).dt.total_seconds() / 60
    direct_arrival = visits["observed_arrival_seats"].notna()
    inferred_arrival = (
        (~direct_arrival)
        & visits["previous_visit_trip_id"].eq(visits["trip_id"])
        & visits["station_seq"].eq(visits["previous_visit_station_seq"] + 1)
        & visits["previous_departure_seats"].ge(0)
        & previous_gap.between(0, 10, inclusive="both")
    )
    visits["arrival_label_quality"] = pd.Series(
        pd.NA, index=visits.index, dtype="string"
    )
    visits["arrival_seats"] = np.nan
    visits["arrival_event_time"] = visits["arrival_seen"].where(direct_arrival)
    visits.loc[direct_arrival, "arrival_label_quality"] = "A"
    visits.loc[direct_arrival, "arrival_seats"] = visits.loc[
        direct_arrival, "observed_arrival_seats"
    ]
    visits.loc[inferred_arrival, "arrival_label_quality"] = "B"
    visits.loc[inferred_arrival, "arrival_seats"] = visits.loc[
        inferred_arrival, "previous_departure_seats"
    ]
    # B등급은 실제 도착 상태가 누락됐으므로 목표 정류장의 최초 관측시각을
    # 도착시각 근사로 사용한다.
    visits.loc[inferred_arrival, "arrival_event_time"] = visits.loc[
        inferred_arrival, "first_seen"
    ]
    visits["arrival_is_full"] = np.where(
        visits["arrival_label_quality"].notna(), visits["arrival_seats"].eq(0), np.nan
    )

    visits = visits.merge(stations, on=["route_id", "station_id", "station_seq"], how="left")
    visits["is_pass_node"] = visits["station_name"].str.contains(r"\(경유\)", na=False)
    return visits


def add_upstream_features(visits: pd.DataFrame, horizon: int) -> pd.DataFrame:
    output = visits.copy()
    seat_signal = np.where(
        output["departure_seats"].notna(),
        output["departure_seats"],
        np.where(
            output["last_state"].eq(0) & output["last_seats"].ge(0),
            output["last_seats"],
            np.nan,
        ),
    )
    output["seat_signal"] = seat_signal.astype(float)
    result_seats = np.full(len(output), np.nan)
    result_seq = np.full(len(output), np.nan)
    result_time = np.full(len(output), np.datetime64("NaT", "ns"), dtype="datetime64[ns]")

    for _, indices in output.groupby("trip_id", sort=False).groups.items():
        idx = np.asarray(list(indices), dtype=int)
        group = output.loc[idx]
        candidates = group.loc[group["seat_signal"].notna()]
        if candidates.empty:
            continue
        candidate_seq = candidates["station_seq"].to_numpy(dtype=int)
        candidate_seats = candidates["seat_signal"].to_numpy(dtype=float)
        candidate_time = candidates["last_seen"].dt.tz_localize(None).to_numpy(dtype="datetime64[ns]")
        target_seq = group["station_seq"].to_numpy(dtype=int) - horizon
        positions = np.searchsorted(candidate_seq, target_seq, side="right") - 1
        valid = positions >= 0
        result_seats[idx[valid]] = candidate_seats[positions[valid]]
        result_seq[idx[valid]] = candidate_seq[positions[valid]]
        result_time[idx[valid]] = candidate_time[positions[valid]]

    output[f"upstream_seats_{horizon}"] = result_seats
    output[f"upstream_seq_{horizon}"] = result_seq
    output[f"upstream_time_{horizon}"] = pd.to_datetime(result_time).tz_localize("Asia/Seoul")
    output[f"upstream_stop_distance_{horizon}"] = output["station_seq"] - output[f"upstream_seq_{horizon}"]
    output[f"upstream_age_minutes_{horizon}"] = (
        output["first_seen"] - output[f"upstream_time_{horizon}"]
    ).dt.total_seconds() / 60
    return output


def nominal_capacity(low_plate: pd.Series) -> pd.Series:
    # 현재 표본의 정상 차량 최대 빈자리 45, 2층버스 최대 빈자리 70과 일치한다.
    return pd.Series(np.where(low_plate.eq(2), 70.0, 45.0), index=low_plate.index)


def build_model_table(
    visits: pd.DataFrame,
    stations: pd.DataFrame,
    *,
    label_target: str = "departure",
) -> tuple[pd.DataFrame, int]:
    if label_target not in {"departure", "arrival"}:
        raise ValueError("label_target은 departure 또는 arrival이어야 합니다.")
    featured = add_upstream_features(visits, 3)
    featured = add_upstream_features(featured, 5)
    turnaround_seq = infer_turnaround_seq(stations)
    max_seq = int(stations["station_seq"].max())

    featured["direction"] = np.where(featured["station_seq"] <= turnaround_seq, "to_city", "return")
    to_city_progress = featured["station_seq"] / max(turnaround_seq, 1)
    return_denominator = max(max_seq - turnaround_seq, 1)
    return_progress = (max_seq - featured["station_seq"]) / return_denominator
    featured["route_progress"] = np.where(
        featured["direction"].eq("to_city"), to_city_progress, return_progress
    ).clip(0, 1)

    if label_target == "arrival":
        featured["label_quality"] = featured["arrival_label_quality"]
        featured["label_seats"] = featured["arrival_seats"]
        featured["is_full"] = featured["arrival_is_full"]
        featured["event_time"] = featured["arrival_event_time"].fillna(
            featured["first_seen"]
        )
    else:
        featured["event_time"] = featured["first_seen"]
    featured["date"] = featured["event_time"].dt.date.astype(str)
    featured["hour"] = featured["event_time"].dt.hour
    featured["minute_of_day"] = featured["hour"] * 60 + featured["event_time"].dt.minute
    angle = 2 * np.pi * featured["minute_of_day"] / (24 * 60)
    featured["time_sin"] = np.sin(angle)
    featured["time_cos"] = np.cos(angle)
    featured["day_of_week"] = featured["event_time"].dt.dayofweek.astype(str)
    featured["time_bin_30"] = (featured["minute_of_day"] // 30).astype(int)
    featured["is_peak"] = featured["hour"].isin(PEAK_HOURS)
    featured["station_seq_cat"] = featured["station_seq"].astype(str)
    featured["low_plate_cat"] = featured["low_plate"].astype("Int64").astype(str)
    featured["center_yn"] = featured["center_yn"].fillna("unknown").astype(str)
    featured["capacity"] = nominal_capacity(featured["low_plate"])
    featured["upstream_load_ratio_3"] = 1 - featured["upstream_seats_3"] / featured["capacity"]
    featured["upstream_load_ratio_5"] = 1 - featured["upstream_seats_5"] / featured["capacity"]
    featured["load_change_5_to_3"] = (
        featured["upstream_load_ratio_3"] - featured["upstream_load_ratio_5"]
    )

    actual_stops = featured.loc[~featured["is_pass_node"]].copy()
    actual_stops = actual_stops.sort_values(["station_seq", "event_time", "vehicle_id"])
    at_stop = actual_stops.groupby("station_seq", sort=False)
    # 이전 버스 피처에는 검증된 정류장 출발/교차로 관측(A/B)만 쓴다.
    # C는 만차 여부에는 안정적이지만 정확한 좌석 수에는 오차가 있어 제외한다.
    actual_stops["exact_label_seats"] = actual_stops["label_seats"].where(
        actual_stops["label_quality"].isin(["A", "B"])
    )
    actual_stops["previous_bus_seats"] = at_stop["exact_label_seats"].shift()
    actual_stops["previous_bus_time"] = at_stop["event_time"].shift()
    actual_stops["headway_minutes"] = (
        actual_stops["event_time"] - actual_stops["previous_bus_time"]
    ).dt.total_seconds() / 60
    actual_stops.loc[~actual_stops["headway_minutes"].between(1, 120), "headway_minutes"] = np.nan
    actual_stops["previous_bus_load_ratio"] = 1 - actual_stops["previous_bus_seats"] / actual_stops["capacity"]
    actual_stops = actual_stops.sort_values("event_time").reset_index(drop=True)
    return actual_stops, turnaround_seq


def make_preprocessor(feature_set: FeatureSet) -> ColumnTransformer:
    numeric = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        [("numeric", numeric, list(feature_set.numeric)), ("categorical", categorical, list(feature_set.categorical))],
        remainder="drop",
    )


def model_factories(feature_set: FeatureSet, seed: int) -> dict[str, Pipeline]:
    preprocessor = make_preprocessor(feature_set)
    return {
        "logistic": Pipeline(
            [
                ("features", clone(preprocessor)),
                (
                    "classifier",
                    LogisticRegression(
                        class_weight="balanced",
                        C=0.5,
                        max_iter=2_000,
                        random_state=seed,
                    ),
                ),
            ]
        ),
        "random_forest": Pipeline(
            [
                ("features", clone(preprocessor)),
                (
                    "classifier",
                    RandomForestClassifier(
                        n_estimators=300,
                        max_depth=8,
                        min_samples_leaf=15,
                        max_features="sqrt",
                        class_weight="balanced_subsample",
                        n_jobs=-1,
                        random_state=seed,
                    ),
                ),
            ]
        ),
        "hist_gradient_boosting": Pipeline(
            [
                ("features", clone(preprocessor)),
                (
                    "classifier",
                    HistGradientBoostingClassifier(
                        learning_rate=0.05,
                        max_iter=200,
                        max_leaf_nodes=15,
                        min_samples_leaf=30,
                        l2_regularization=1.0,
                        random_state=seed,
                    ),
                ),
            ]
        ),
    }


def balanced_weights(y: np.ndarray) -> np.ndarray:
    positives = max(int(y.sum()), 1)
    negatives = max(int((1 - y).sum()), 1)
    return np.where(y == 1, len(y) / (2 * positives), len(y) / (2 * negatives))


def fit_model(model: Pipeline, x: pd.DataFrame, y: np.ndarray, name: str) -> Pipeline:
    if name == "hist_gradient_boosting":
        model.fit(x, y, classifier__sample_weight=balanced_weights(y))
    else:
        model.fit(x, y)
    return model


def platt_fit(probabilities: np.ndarray, y: np.ndarray, seed: int) -> LogisticRegression | None:
    if len(np.unique(y)) < 2:
        return None
    clipped = np.clip(probabilities, 1e-6, 1 - 1e-6)
    logits = np.log(clipped / (1 - clipped)).reshape(-1, 1)
    calibrator = LogisticRegression(C=1.0, max_iter=1_000, random_state=seed)
    calibrator.fit(logits, y)
    return calibrator


def platt_predict(calibrator: LogisticRegression | None, probabilities: np.ndarray) -> np.ndarray:
    if calibrator is None:
        return probabilities
    clipped = np.clip(probabilities, 1e-6, 1 - 1e-6)
    logits = np.log(clipped / (1 - clipped)).reshape(-1, 1)
    return calibrator.predict_proba(logits)[:, 1]


def expected_calibration_error(y: np.ndarray, probabilities: np.ndarray, bins: int = 10) -> float:
    if len(y) == 0:
        return float("nan")
    frame = pd.DataFrame({"y": y, "p": probabilities})
    try:
        frame["bin"] = pd.qcut(frame["p"], q=min(bins, frame["p"].nunique()), duplicates="drop")
    except ValueError:
        return float(abs(frame["y"].mean() - frame["p"].mean()))
    grouped = frame.groupby("bin", observed=True)
    total = len(frame)
    return float(
        sum(len(group) / total * abs(group["y"].mean() - group["p"].mean()) for _, group in grouped)
    )


def choose_threshold(y: np.ndarray, probabilities: np.ndarray, target_recall: float = 0.8) -> float:
    if y.sum() == 0:
        return 0.5
    precision, recall, thresholds = precision_recall_curve(y, probabilities)
    candidates = np.flatnonzero(recall[:-1] >= target_recall)
    if len(candidates) == 0:
        return 0.5
    best = candidates[int(np.argmax(precision[candidates]))]
    return float(thresholds[best])


def metric_row(
    y: np.ndarray,
    probabilities: np.ndarray,
    *,
    threshold: float,
    model: str,
    feature_set: str,
    split: str,
) -> dict[str, Any]:
    probabilities = np.clip(probabilities, 1e-6, 1 - 1e-6)
    predicted = probabilities >= threshold
    positives = int(y.sum())
    top_n = max(1, int(math.ceil(len(y) * 0.05)))
    top_indices = np.argsort(probabilities)[-top_n:]
    top_recall = float(y[top_indices].sum() / positives) if positives else float("nan")
    prevalence = float(y.mean()) if len(y) else float("nan")
    top_precision = float(y[top_indices].mean())
    return {
        "split": split,
        "feature_set": feature_set,
        "model": model,
        "rows": int(len(y)),
        "positives": positives,
        "prevalence": prevalence,
        "average_precision": float(average_precision_score(y, probabilities)) if positives else float("nan"),
        "roc_auc": float(roc_auc_score(y, probabilities)) if positives and positives < len(y) else float("nan"),
        "brier": float(brier_score_loss(y, probabilities)),
        "log_loss": float(log_loss(y, probabilities, labels=[0, 1])),
        "ece": expected_calibration_error(y, probabilities),
        "threshold": float(threshold),
        "precision_at_threshold": float(y[predicted].mean()) if predicted.any() else 0.0,
        "recall_at_threshold": float(y[predicted].sum() / positives) if positives else float("nan"),
        "alert_rate": float(predicted.mean()),
        "top_5pct_precision": top_precision,
        "top_5pct_recall": top_recall,
        "top_5pct_lift": float(top_precision / prevalence) if prevalence > 0 else float("nan"),
    }


def historical_rate_predict(train: pd.DataFrame, target: pd.DataFrame, alpha: float = 20.0) -> np.ndarray:
    global_rate = float(train["is_full"].mean())
    exact = train.groupby(["station_seq", "direction", "time_bin_30"])["is_full"].agg(["sum", "count"])
    station = train.groupby(["station_seq", "direction"])["is_full"].agg(["sum", "count"])
    predictions = []
    for row in target.itertuples(index=False):
        key = (row.station_seq, row.direction, row.time_bin_30)
        fallback_key = (row.station_seq, row.direction)
        if key in exact.index:
            stats = exact.loc[key]
        elif fallback_key in station.index:
            stats = station.loc[fallback_key]
        else:
            predictions.append(global_rate)
            continue
        predictions.append((float(stats["sum"]) + alpha * global_rate) / (float(stats["count"]) + alpha))
    return np.asarray(predictions, dtype=float)


def prepare_subset(table: pd.DataFrame) -> pd.DataFrame:
    subset = table.loc[
        table["is_peak"]
        & table["label_quality"].notna()
        & table["date"].between("2026-08-04", "2026-08-09")
    ].copy()
    subset["is_full"] = subset["is_full"].astype(int)
    return subset.sort_values("event_time").reset_index(drop=True)


def evaluate_final_split(
    data: pd.DataFrame,
    feature_sets: Iterable[FeatureSet],
    seed: int,
    threshold_target_recall: float = 0.8,
) -> tuple[pd.DataFrame, dict[tuple[str, str], Pipeline], dict[tuple[str, str], np.ndarray]]:
    train = data.loc[data["date"].isin(["2026-08-04", "2026-08-05"])]
    calibration = data.loc[data["date"].eq("2026-08-06")]
    test = data.loc[data["date"].eq("2026-08-07")]
    if min(train["is_full"].sum(), calibration["is_full"].sum(), test["is_full"].sum()) == 0:
        raise ValueError("최종 시간분할 중 하나에 만차 사례가 없습니다.")

    rows: list[dict[str, Any]] = []
    fitted: dict[tuple[str, str], Pipeline] = {}
    test_predictions: dict[tuple[str, str], np.ndarray] = {}

    baseline_cal = historical_rate_predict(train, calibration)
    baseline_test = historical_rate_predict(train, test)
    baseline_calibrator = platt_fit(baseline_cal, calibration["is_full"].to_numpy(), seed)
    baseline_test_calibrated = platt_predict(baseline_calibrator, baseline_test)
    baseline_threshold = choose_threshold(
        calibration["is_full"].to_numpy(),
        platt_predict(baseline_calibrator, baseline_cal),
        target_recall=threshold_target_recall,
    )
    baseline_row = metric_row(
            test["is_full"].to_numpy(),
            baseline_test_calibrated,
            threshold=baseline_threshold,
            model="historical_rate",
            feature_set="planning",
            split="test_2026-08-07",
        )
    baseline_row["selection_average_precision"] = float(
        average_precision_score(calibration["is_full"].to_numpy(), baseline_cal)
    )
    rows.append(baseline_row)
    test_predictions[("planning", "historical_rate")] = baseline_test_calibrated

    for feature_set in feature_sets:
        x_train = train[feature_set.columns]
        x_cal = calibration[feature_set.columns]
        x_test = test[feature_set.columns]
        y_train = train["is_full"].to_numpy()
        y_cal = calibration["is_full"].to_numpy()
        y_test = test["is_full"].to_numpy()
        for model_name, model in model_factories(feature_set, seed).items():
            fitted_model = fit_model(model, x_train, y_train, model_name)
            raw_cal = fitted_model.predict_proba(x_cal)[:, 1]
            raw_test = fitted_model.predict_proba(x_test)[:, 1]
            calibrator = platt_fit(raw_cal, y_cal, seed)
            calibrated_cal = platt_predict(calibrator, raw_cal)
            calibrated_test = platt_predict(calibrator, raw_test)
            threshold = choose_threshold(
                y_cal, calibrated_cal, target_recall=threshold_target_recall
            )
            result_row = metric_row(
                    y_test,
                    calibrated_test,
                    threshold=threshold,
                    model=model_name,
                    feature_set=feature_set.name,
                    split="test_2026-08-07",
                )
            # 최종 테스트를 보지 않고 보정일의 순위화 성능으로 후보를 선택한다.
            result_row["selection_average_precision"] = float(
                average_precision_score(y_cal, raw_cal)
            )
            rows.append(result_row)
            key = (feature_set.name, model_name)
            fitted[key] = fitted_model
            test_predictions[key] = calibrated_test
    return pd.DataFrame(rows), fitted, test_predictions


def cluster_bootstrap_summary(
    test: pd.DataFrame,
    probabilities: np.ndarray,
    baseline_probabilities: np.ndarray,
    seed: int,
    repeats: int = 2_000,
) -> dict[str, Any]:
    """운행(trip) 단위 재표집으로 연속 정류장 관측의 상관을 보존한다."""
    y = test["is_full"].to_numpy()
    groups = test.groupby("trip_id", sort=False).indices
    trip_ids = np.asarray(list(groups), dtype=object)
    rng = np.random.default_rng(seed)
    values: list[tuple[float, float, float]] = []
    for _ in range(repeats):
        selected = rng.choice(trip_ids, size=len(trip_ids), replace=True)
        positions = np.concatenate([groups[trip_id] for trip_id in selected])
        sampled_y = y[positions]
        if sampled_y.sum() == 0:
            continue
        model_ap = average_precision_score(sampled_y, probabilities[positions])
        baseline_ap = average_precision_score(sampled_y, baseline_probabilities[positions])
        values.append(
            (
                float(model_ap),
                float(brier_score_loss(sampled_y, probabilities[positions])),
                float(model_ap - baseline_ap),
            )
        )
    array = np.asarray(values)

    def interval(column: int) -> dict[str, float]:
        lower, median, upper = np.quantile(array[:, column], [0.025, 0.5, 0.975])
        return {"lower_95": float(lower), "median": float(median), "upper_95": float(upper)}

    return {
        "unit": "trip_id",
        "test_trips": int(len(trip_ids)),
        "valid_repeats": int(len(values)),
        "average_precision": interval(0),
        "brier": interval(1),
        "average_precision_delta_vs_historical_rate": interval(2),
    }


def trip_level_diagnostics(
    test: pd.DataFrame,
    probabilities: np.ndarray,
    threshold: float,
) -> dict[str, Any]:
    scored = test.copy()
    scored["probability"] = probabilities
    full = scored.loc[scored["is_full"].eq(1)].sort_values("event_time")
    first_full = full.groupby("trip_id", sort=False).head(1)
    positive_trips = int(first_full["trip_id"].nunique())
    any_alert = full.groupby("trip_id", sort=False)["probability"].max().ge(threshold)
    return {
        "positive_trips": positive_trips,
        "first_full_stop_alerted": int(first_full["probability"].ge(threshold).sum()),
        "first_full_stop_recall": float(first_full["probability"].ge(threshold).mean()),
        "any_full_stop_alerted_trips": int(any_alert.sum()),
        "any_full_stop_trip_recall": float(any_alert.mean()),
        "observed_full_stop_rows": int(len(full)),
    }


def evaluate_forward_chaining(
    data: pd.DataFrame,
    feature_sets: Iterable[FeatureSet],
    seed: int,
) -> pd.DataFrame:
    dates = ["2026-08-05", "2026-08-06", "2026-08-07"]
    rows: list[dict[str, Any]] = []
    for test_date in dates:
        train = data.loc[data["date"] < test_date]
        test = data.loc[data["date"] == test_date]
        if train["is_full"].sum() == 0 or test["is_full"].sum() == 0:
            continue
        baseline = historical_rate_predict(train, test)
        rows.append(
            metric_row(
                test["is_full"].to_numpy(),
                baseline,
                threshold=0.5,
                model="historical_rate",
                feature_set="planning",
                split=f"forward_{test_date}",
            )
        )
        for feature_set in feature_sets:
            y_train = train["is_full"].to_numpy()
            y_test = test["is_full"].to_numpy()
            for model_name, model in model_factories(feature_set, seed).items():
                fitted_model = fit_model(model, train[feature_set.columns], y_train, model_name)
                probabilities = fitted_model.predict_proba(test[feature_set.columns])[:, 1]
                rows.append(
                    metric_row(
                        y_test,
                        probabilities,
                        threshold=0.5,
                        model=model_name,
                        feature_set=feature_set.name,
                        split=f"forward_{test_date}",
                    )
                )
    return pd.DataFrame(rows)


def summarize_eda(data: pd.DataFrame) -> dict[str, Any]:
    labeled = data.loc[data["label_quality"].notna()].copy()
    by_date = (
        labeled.groupby("date")["is_full"]
        .agg(rows="size", full="sum", full_rate="mean")
        .reset_index()
    )
    by_quality = (
        labeled.groupby("label_quality")["is_full"]
        .agg(rows="size", full="sum", full_rate="mean")
        .reset_index()
    )
    by_direction = (
        labeled.groupby("direction")["is_full"]
        .agg(rows="size", full="sum", full_rate="mean")
        .reset_index()
    )
    top_stops = (
        labeled.groupby(["station_seq", "station_name", "direction"])["is_full"]
        .agg(rows="size", full="sum", full_rate="mean")
        .reset_index()
        .sort_values(["full", "full_rate"], ascending=False)
        .head(12)
    )
    full_rows = labeled.loc[labeled["is_full"].eq(1)]
    return {
        "rows": int(len(labeled)),
        "full_events": int(full_rows.shape[0]),
        "full_rate": float(labeled["is_full"].mean()),
        "full_trips": int(full_rows["trip_id"].nunique()),
        "full_vehicles": int(full_rows["vehicle_id"].nunique()),
        "date_range": [str(labeled["date"].min()), str(labeled["date"].max())],
        "upstream_3_coverage": float(labeled["upstream_load_ratio_3"].notna().mean()),
        "previous_bus_coverage": float(labeled["previous_bus_load_ratio"].notna().mean()),
        "by_date": _as_records(by_date),
        "by_quality": _as_records(by_quality),
        "by_direction": _as_records(by_direction),
        "top_stops": _as_records(top_stops),
    }


def make_eda_plot(data: pd.DataFrame, output_path: Path) -> None:
    labeled = data.loc[data["label_quality"].notna()].copy()
    hourly = labeled.groupby("hour")["is_full"].agg(rate="mean", full="sum", rows="size").reset_index()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].bar(hourly["hour"], hourly["rate"] * 100, color="#2563eb")
    axes[0].set_title("Full-bus rate by hour")
    axes[0].set_xlabel("Hour (KST)")
    axes[0].set_ylabel("Full rate (%)")
    axes[0].set_xticks(range(0, 24, 2))

    stop = (
        labeled.groupby(["station_seq", "direction"])["is_full"]
        .agg(rate="mean", full="sum", rows="size")
        .reset_index()
    )
    for direction, group in stop.groupby("direction"):
        axes[1].plot(group["station_seq"], group["rate"] * 100, marker="o", ms=3, label=direction)
    axes[1].set_title("Full-bus rate along route")
    axes[1].set_xlabel("Station sequence")
    axes[1].set_ylabel("Full rate (%)")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def permutation_table(
    model: Pipeline,
    test: pd.DataFrame,
    feature_set: FeatureSet,
    seed: int,
) -> pd.DataFrame:
    result = permutation_importance(
        model,
        test[feature_set.columns],
        test["is_full"].to_numpy(),
        scoring="average_precision",
        n_repeats=12,
        random_state=seed,
        n_jobs=-1,
    )
    return (
        pd.DataFrame(
            {
                "feature": feature_set.columns,
                "importance_mean": result.importances_mean,
                "importance_std": result.importances_std,
            }
        )
        .sort_values("importance_mean", ascending=False)
        .reset_index(drop=True)
    )


def json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, list):
        return [json_ready(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return None if not math.isfinite(float(value)) else float(value)
    return value


def main() -> int:
    parser = argparse.ArgumentParser(description="광역버스 만차 예측 가능성 분석")
    parser.add_argument("--db", type=Path, default=Path("data/gbis.sqlite3"))
    parser.add_argument("--route-id", default=DEFAULT_ROUTE_ID)
    parser.add_argument("--route-name", default=DEFAULT_ROUTE_NAME)
    parser.add_argument("--output-dir", type=Path, default=Path("analysis/results"))
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    args.output_dir.mkdir(parents=True, exist_ok=True)
    locations, stations = load_data(args.db, args.route_id)
    visits = build_visits(locations, stations)
    table, turnaround_seq = build_model_table(visits, stations)
    data = prepare_subset(table)

    final_metrics, fitted, predictions = evaluate_final_split(data, [PLANNING, REALTIME], args.seed)
    forward_metrics = evaluate_forward_chaining(data, [PLANNING, REALTIME], args.seed)
    test = data.loc[data["date"].eq("2026-08-07")].copy()

    candidates = final_metrics.loc[final_metrics["model"] != "historical_rate"].copy()
    best_row = candidates.sort_values(
        ["selection_average_precision", "brier"], ascending=[False, True]
    ).iloc[0]
    best_key = (str(best_row["feature_set"]), str(best_row["model"]))
    best_feature_set = REALTIME if best_key[0] == REALTIME.name else PLANNING
    importance = permutation_table(fitted[best_key], test, best_feature_set, args.seed)

    strict_mask = test["label_quality"].eq("A")
    strict_metrics = metric_row(
        test.loc[strict_mask, "is_full"].to_numpy(),
        predictions[best_key][strict_mask.to_numpy()],
        threshold=float(best_row["threshold"]),
        model=best_key[1],
        feature_set=best_key[0],
        split="test_2026-08-07_quality_A_only",
    )
    strict_data = data.loc[data["label_quality"].eq("A")].copy()
    strict_retrained_metrics, _, _ = evaluate_final_split(
        strict_data, [PLANNING, REALTIME], args.seed
    )
    strict_selected = (
        strict_retrained_metrics.loc[strict_retrained_metrics["model"].ne("historical_rate")]
        .sort_values(["selection_average_precision", "brier"], ascending=[False, True])
        .iloc[0]
    )

    baseline_key = ("planning", "historical_rate")
    bootstrap = cluster_bootstrap_summary(
        test,
        predictions[best_key],
        predictions[baseline_key],
        args.seed,
    )
    trip_diagnostics = trip_level_diagnostics(
        test, predictions[best_key], float(best_row["threshold"])
    )
    calibration_diagnostics = {
        "observed_rate": float(test["is_full"].mean()),
        "mean_predicted_probability": float(predictions[best_key].mean()),
        "ratio_predicted_to_observed": float(
            predictions[best_key].mean() / test["is_full"].mean()
        ),
    }

    weekend = data.loc[data["date"].isin(["2026-08-08", "2026-08-09"])]
    weekend_summary = {
        "rows": int(len(weekend)),
        "positives": int(weekend["is_full"].sum()),
    }

    final_metrics.to_csv(args.output_dir / "final_metrics.csv", index=False)
    forward_metrics.to_csv(args.output_dir / "forward_metrics.csv", index=False)
    strict_retrained_metrics.to_csv(
        args.output_dir / "strict_A_retrained_metrics.csv", index=False
    )
    importance.to_csv(args.output_dir / "permutation_importance.csv", index=False)
    make_eda_plot(data, args.output_dir / "eda_full_patterns.png")

    result = {
        "route_id": args.route_id,
        "route_name": args.route_name,
        "turnaround_seq": turnaround_seq,
        "source_rows": int(len(locations)),
        "visit_rows": int(len(visits)),
        "model_rows_peak_labeled": int(len(data)),
        "eda": summarize_eda(data),
        "split": {
            "train": ["2026-08-04", "2026-08-05"],
            "calibration": ["2026-08-06"],
            "test": ["2026-08-07"],
            "weekend_sanity": ["2026-08-08", "2026-08-09"],
        },
        "best_model": {
            "feature_set": best_key[0],
            "model": best_key[1],
            "test_metrics": json_ready(best_row.to_dict()),
            "strict_label_sensitivity": json_ready(strict_metrics),
            "strict_A_retrained_same_selected_model": json_ready(
                strict_retrained_metrics.loc[
                    (strict_retrained_metrics["feature_set"].eq(best_key[0]))
                    & (strict_retrained_metrics["model"].eq(best_key[1]))
                ].iloc[0].to_dict()
            ),
            "strict_A_retrained_calibration_selected_model": json_ready(
                strict_selected.to_dict()
            ),
            "trip_level_diagnostics": trip_diagnostics,
            "calibration_diagnostics": calibration_diagnostics,
            "cluster_bootstrap": bootstrap,
        },
        "weekend_sanity": weekend_summary,
        "feature_importance": json_ready(_as_records(importance.head(12))),
        "limitations": [
            "수집 기간이 6일이고 최종 테스트가 1개 평일이라 운영 성능의 확정치가 아니다.",
            "노선 1000만 만차 양성 사례가 있어 노선 간 일반화는 검증하지 못했다.",
            "날씨·교통·행사·공휴일 피처는 현재 DB에 없어 효과를 검증하지 못했다.",
            "C 등급 라벨은 정확한 잔여좌석 수가 아니라 만차/비만차 분류에만 사용해야 한다.",
            "실시간 배차간격은 실제 목표 정류장 관측시각으로 계산한 근사치라 운영 시 ETA로 대체해야 한다.",
            "만차는 승차 실패와 같지 않으며 대기열 데이터가 없어서 보내야 할 차량 수는 아직 검증할 수 없다.",
        ],
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(result), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps(json_ready(result), ensure_ascii=False, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/pooled_main_model_overfit_ablation.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/pooled_main_model_overfit_ablation.py
from __future__ import annotations

import argparse
import json
import sqlite3
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from hypothesis_model_search import add_observed_capacity_features
from latest_main_model_feature_recheck import (
    LATEST_COMPLETE_DATES,
    LATEST_PARTIAL_DATE,
)
from latest_main_model_overfit_ablation import (
    COORDINATE_FEATURES,
    OBSERVED_CEILING_FEATURES,
    apply_candidate,
    train_schema,
)
from linear_feature_experiment import (
    add_strict_prior_flow_features,
    add_strict_prior_low_rate_features,
    json_ready,
)
from main_model_feature_augmentation import feature_sets, required_metrics
from model_feasibility import FeatureSet
from route_specific_feature_experiment import DEFAULT_ROUTES, cache_paths


SOURCE_CUTOFF = "2026-08-13 20:26:03+09:00"
PRIMARY_NAME = "pooled_40_legacy_ceiling_features"


def pooled_candidates() -> dict[str, tuple[FeatureSet, bool]]:
    primary = feature_sets()["importance_pruned"]

    def schema(name: str, removed: tuple[str, ...]) -> FeatureSet:
        return FeatureSet(
            name=name,
            numeric=tuple(
                column for column in primary.numeric if column not in removed
            ),
            categorical=(*primary.categorical, "route_code"),
        )

    return {
        # The Route-1000-only 44/70 output clip cannot be applied project-wide:
        # active pooled routes contain vehicle category 5. Keep its three
        # engineered inputs only as the legacy baseline and use nominal support.
        PRIMARY_NAME: (schema("pooled_40", ()), False),
        "pooled_37_no_ceiling": (
            schema("pooled_37", OBSERVED_CEILING_FEATURES),
            False,
        ),
        "pooled_38_no_coordinates": (
            schema("pooled_38", COORDINATE_FEATURES),
            False,
        ),
        "pooled_35_no_ceiling_coordinates": (
            schema(
                "pooled_35",
                (*OBSERVED_CEILING_FEATURES, *COORDINATE_FEATURES),
            ),
            False,
        ),
    }


def active_route_audit(database: Path, cutoff: str) -> pd.DataFrame:
    with sqlite3.connect(f"file:{database.resolve()}?mode=ro", uri=True) as connection:
        routes = pd.read_sql_query(
            """
            SELECT route_id, station_count, observation_count,
                   first_collected_at, last_collected_at
            FROM routes
            ORDER BY route_id
            """,
            connection,
        )
    active_ids = set(DEFAULT_ROUTES)
    routes["route_name"] = routes["route_id"].map(DEFAULT_ROUTES)
    routes["included"] = routes["route_id"].isin(active_ids)
    routes["reason"] = np.select(
        [routes["included"], routes["station_count"].eq(0)],
        [
            "actively collected through source cutoff",
            "no cached station metadata; historical collection only",
        ],
        default="historical short collection only; not active at source cutoff",
    )
    routes["experiment_source_cutoff"] = cutoff
    return routes


def prepare_pooled_data(
    cache_dir: Path,
    *,
    source_cutoff: str,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    frames: list[pd.DataFrame] = []
    metadata: dict[str, Any] = {}
    for route_id, route_name in DEFAULT_ROUTES.items():
        snapshot_path, flow_path, metadata_path = cache_paths(cache_dir, route_id)
        route_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if route_metadata.get("source_cutoff") != source_cutoff:
            raise ValueError(
                f"노선 {route_name} cache cutoff가 다릅니다: "
                f"{route_metadata.get('source_cutoff')}"
            )
        snapshots = pd.read_pickle(snapshot_path)
        flows = pd.read_pickle(flow_path)
        print(f"[{route_name}] strict-prior 피처 생성", flush=True)
        featured = add_strict_prior_low_rate_features(snapshots)
        featured = add_strict_prior_flow_features(featured, flows)
        featured = add_observed_capacity_features(featured)
        featured["event_id"] = route_id + "::" + featured["event_id"].astype(str)
        featured["trip_id"] = route_id + "::" + featured["trip_id"].astype(str)
        featured["route_id"] = route_id
        featured["route_name"] = route_name
        featured["route_code"] = route_id
        frames.append(featured)
        metadata[route_id] = {
            "route_name": route_name,
            **route_metadata,
        }
    pooled = pd.concat(frames, ignore_index=True)
    pooled["route_code"] = pooled["route_code"].astype("string")
    return pooled, metadata


def rolling_folds(data: pd.DataFrame) -> list[tuple[str, pd.DataFrame, pd.DataFrame]]:
    folds: list[tuple[str, pd.DataFrame, pd.DataFrame]] = []
    for validation_date in (*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE):
        train = data.loc[
            data["date"].lt(validation_date)
            & pd.to_datetime(data["date"]).dt.dayofweek.lt(5)
        ].copy()
        validation = data.loc[data["date"].eq(validation_date)].copy()
        missing_routes = sorted(set(DEFAULT_ROUTES) - set(validation["route_id"]))
        if train.empty or validation.empty or missing_routes:
            raise ValueError(
                f"통합 rolling fold가 불완전합니다: {validation_date}, "
                f"missing_routes={missing_routes}"
            )
        folds.append((validation_date, train, validation))
    return folds


def scoped_metrics(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    partitions = {
        "latest_complete_08_11_12": predictions["date"].isin(
            LATEST_COMPLETE_DATES
        ),
        "latest_partial_08_13": predictions["date"].eq(LATEST_PARTIAL_DATE),
        "latest_all_08_11_13": predictions["date"].isin(
            (*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE)
        ),
    }
    pooled_rows: list[dict[str, Any]] = []
    route_rows: list[dict[str, Any]] = []
    for partition, mask in partitions.items():
        selected = predictions.loc[mask]
        for candidate, frame in selected.groupby("candidate", sort=False):
            pooled_rows.append(
                {
                    "partition": partition,
                    "candidate": candidate,
                    "core_feature_count": int(frame["core_feature_count"].iloc[0]),
                    "total_feature_count": int(frame["total_feature_count"].iloc[0]),
                    **required_metrics(frame),
                }
            )
        for (route_id, route_name, candidate), frame in selected.groupby(
            ["route_id", "route_name", "candidate"], sort=False
        ):
            route_rows.append(
                {
                    "partition": partition,
                    "route_id": route_id,
                    "route_name": route_name,
                    "candidate": candidate,
                    **required_metrics(frame),
                }
            )
    pooled = pd.DataFrame(pooled_rows)
    by_route = pd.DataFrame(route_rows)
    metric_columns = [
        "event_balanced_mae",
        "low_0_10_mae",
        "full_accuracy",
        "full_recall",
        "full_precision",
        "full_f1",
    ]
    macro = (
        by_route.groupby(["partition", "candidate"], sort=False)[metric_columns]
        .mean()
        .reset_index()
    )
    macro.insert(2, "routes", len(DEFAULT_ROUTES))
    return pooled, by_route, macro


def route_stratified_bootstrap(
    predictions: pd.DataFrame,
    candidate: str,
    *,
    low_only: bool,
    repeats: int = 2_000,
    seed: int = 42,
) -> dict[str, Any]:
    selected = predictions.loc[
        predictions["candidate"].isin([PRIMARY_NAME, candidate])
        & predictions["date"].isin(LATEST_COMPLETE_DATES)
    ].copy()
    if low_only:
        selected = selected.loc[selected["label_seats"].le(10)].copy()
    selected["absolute_error"] = (
        selected["label_seats"] - selected["prediction"]
    ).abs()
    event_errors = (
        selected.groupby(
            ["route_id", "trip_id", "event_id", "candidate"], observed=True
        )["absolute_error"]
        .mean()
        .unstack("candidate")
        .dropna(subset=[PRIMARY_NAME, candidate])
        .reset_index()
    )
    event_errors["improvement"] = (
        event_errors[PRIMARY_NAME] - event_errors[candidate]
    )
    route_trip_values: dict[str, list[np.ndarray]] = {}
    for route_id, route in event_errors.groupby("route_id", sort=False):
        route_trip_values[str(route_id)] = [
            trip["improvement"].to_numpy(dtype=float)
            for _, trip in route.groupby("trip_id", sort=False)
        ]
    rng = np.random.default_rng(seed)
    draws = np.empty(repeats, dtype=float)
    for index in range(repeats):
        route_means: list[float] = []
        for trips in route_trip_values.values():
            sampled = rng.integers(0, len(trips), size=len(trips))
            route_means.append(
                float(np.concatenate([trips[item] for item in sampled]).mean())
            )
        draws[index] = float(np.mean(route_means))
    observed = float(
        event_errors.groupby("route_id")["improvement"].mean().mean()
    )
    return {
        "baseline": PRIMARY_NAME,
        "candidate": candidate,
        "metric": "macro_low_0_10_mae" if low_only else "macro_event_balanced_mae",
        "candidate_improvement": observed,
        "ci_95_lower": float(np.quantile(draws, 0.025)),
        "ci_95_upper": float(np.quantile(draws, 0.975)),
        "probability_candidate_better": float((draws > 0).mean()),
        "routes": len(route_trip_values),
        "trips": int(event_errors["trip_id"].nunique()),
        "events": int(len(event_errors)),
    }


def main() -> int:
    parser = argparse.ArgumentParser(
        description="전체 활성 노선 통합 40/37/38/35피처 앙상블 ablation"
    )
    parser.add_argument(
        "--database", type=Path, default=Path("data/gbis_api_cache.sqlite3")
    )
    parser.add_argument(
        "--cache-dir",
        type=Path,
        default=Path("data/analysis_cache/route_specific_features"),
    )
    parser.add_argument("--source-cutoff", default=SOURCE_CUTOFF)
    parser.add_argument(
        "--featured-cache",
        type=Path,
        default=Path("data/analysis_cache/pooled_main_model_features_latest.pkl"),
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("analysis/pooled_main_model_overfit_ablation_results"),
    )
    parser.add_argument("--rebuild-features", action="store_true")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    if args.featured_cache.is_file() and not args.rebuild_features:
        pooled = pd.read_pickle(args.featured_cache)
        if pooled.attrs.get("source_cutoff") != args.source_cutoff:
            raise ValueError("통합 피처 cache cutoff가 요청 cutoff와 다릅니다.")
        route_metadata = pooled.attrs["route_metadata"]
    else:
        pooled, route_metadata = prepare_pooled_data(
            args.cache_dir, source_cutoff=args.source_cutoff
        )
        pooled.attrs["source_cutoff"] = args.source_cutoff
        pooled.attrs["route_metadata"] = route_metadata
        args.featured_cache.parent.mkdir(parents=True, exist_ok=True)
        pooled.to_pickle(args.featured_cache)

    folds = rolling_folds(pooled)
    candidates = pooled_candidates()
    predictions: list[pd.DataFrame] = []
    schema_predictions: dict[tuple[str, ...], pd.DataFrame] = {}
    for name, (features, empirical_cap) in candidates.items():
        key = tuple(features.columns)
        if key not in schema_predictions:
            schema_predictions[key] = train_schema(
                features.name, features, folds, seed=args.seed
            )
        prediction, _ = apply_candidate(
            name, features, empirical_cap, schema_predictions[key]
        )
        prediction["route_id"] = prediction["event_id"].str.split(
            "::", n=1
        ).str[0]
        prediction["route_name"] = prediction["route_id"].map(DEFAULT_ROUTES)
        prediction["core_feature_count"] = len(features.columns) - 1
        prediction["total_feature_count"] = len(features.columns)
        predictions.append(prediction)
    prediction_table = pd.concat(predictions, ignore_index=True)
    pooled_metrics, route_metrics, macro_metrics = scoped_metrics(prediction_table)
    bootstrap = pd.DataFrame(
        [
            route_stratified_bootstrap(
                prediction_table, candidate, low_only=low_only, seed=args.seed
            )
            for candidate in candidates
            if candidate != PRIMARY_NAME
            for low_only in (False, True)
        ]
    )
    route_audit = active_route_audit(args.database, args.source_cutoff)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    prediction_table.to_pickle(args.output_dir / "oof_predictions.pkl")
    pooled_metrics.to_csv(args.output_dir / "metrics_pooled.csv", index=False)
    route_metrics.to_csv(args.output_dir / "metrics_by_route.csv", index=False)
    macro_metrics.to_csv(args.output_dir / "metrics_route_macro.csv", index=False)
    bootstrap.to_csv(args.output_dir / "route_stratified_bootstrap.csv", index=False)
    route_audit.to_csv(args.output_dir / "route_inclusion_audit.csv", index=False)
    summary = {
        "protocol": {
            "source_cutoff": args.source_cutoff,
            "included_routes": DEFAULT_ROUTES,
            "validation_dates": [*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE],
            "complete_selection_dates": LATEST_COMPLETE_DATES,
            "partial_check_date": LATEST_PARTIAL_DATE,
            "training_rule": "pooled route-aware model; each fold uses all routes' earlier weekdays only",
            "feature_history_rule": "strictly earlier calendar dates within each route",
            "route_identity_rule": "route_code categorical feature is added to every pooled candidate",
            "metric_rule": "report pooled, per-route, and unweighted route-macro metrics",
        },
        "route_metadata": route_metadata,
        "candidates": {
            name: {
                "core_feature_count": len(features.columns) - 1,
                "total_feature_count_with_route_code": len(features.columns),
                "empirical_cap": empirical_cap,
                "numeric": features.numeric,
                "categorical": features.categorical,
            }
            for name, (features, empirical_cap) in candidates.items()
        },
        "pooled_metrics": pooled_metrics.to_dict(orient="records"),
        "route_macro_metrics": macro_metrics.to_dict(orient="records"),
        "bootstrap": bootstrap.to_dict(orient="records"),
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("\npooled")
    print(pooled_metrics.to_string(index=False))
    print("\nroute macro")
    print(macro_metrics.to_string(index=False))
    print("\nbootstrap")
    print(bootstrap.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/queue_boarding_policy.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/queue_boarding_policy.py
"""Primary queue-boarding service policy contract."""
from __future__ import annotations

import numpy as np


MODEL_ID = "queue-boarding/v1.0.0"
PRIMARY_METHOD = "hybrid_point_mix_0.75"
PRIMARY_HYBRID_WEIGHT = 0.75
PRIMARY_POINT_WEIGHT = 0.25
PRIMARY_QUEUE_GRID = (0, 1, 3, 5, 10, 15, 20, 30, 40, 60, 100)
PRIMARY_MAX_VISIBLE_BUSES = 3
PRIMARY_DECODER = "argmax"


def sent_class_probabilities(board_by_probability: np.ndarray) -> np.ndarray:
    """Convert monotone cumulative boarding probabilities into classes 0/1/2/3+."""
    board = np.asarray(board_by_probability, dtype=float)
    if board.shape[-1] != PRIMARY_MAX_VISIBLE_BUSES:
        raise ValueError("primary policy requires probabilities for exactly three visible buses")
    board = np.maximum.accumulate(np.clip(board, 0.0, 1.0), axis=-1)
    sent = np.concatenate(
        [board[..., :1], np.diff(board, axis=-1), 1.0 - board[..., -1:]],
        axis=-1,
    )
    sent = np.clip(sent, 0.0, 1.0)
    return sent / np.maximum(sent.sum(axis=-1, keepdims=True), 1e-12)


def decode_sent_class(board_by_probability: np.ndarray) -> np.ndarray:
    """Primary decision rule: unweighted argmax over sent-count probabilities."""
    return np.argmax(sent_class_probabilities(board_by_probability), axis=-1)


### `analysis/queue_boarding_simulation_experiment.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/queue_boarding_simulation_experiment.py
"""Hybrid seat-PMF and user-supplied queue boarding simulation.

The service question is evaluated counterfactually: given ``q`` passengers
ahead of the user and no new arrivals, how many of the next visible buses must
pass before cumulative free seats reach ``q + 1``?  Predicted probabilities are
compared with the same calculation using observed arrival seats.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from confidence_interval_comparison import (
    FROZEN_MANIFEST,
    candidate_features,
    load_frozen_features,
    prepare_comparison_data,
)
from model_correction_layer_experiment import _select_threshold
from queue_boarding_policy import PRIMARY_QUEUE_GRID
from uncertainty_heads import _model_pipeline, event_weights
from yeonwu_peer_feature_experiment import DEFAULT_ROUTES, ROOT, load_embedded_modules


CALIBRATION_DATE = "2026-08-13"
EVALUATION_DATE = "2026-08-14"
POINT_CANDIDATE = "state_profile__delta_per_stop"
HYBRID_CLASSES = 14
MAX_SEAT = 70
DEFAULT_QUEUES = PRIMARY_QUEUE_GRID
DEFAULT_MAX_BUSES = 3


def seat_bucket(values: pd.Series | np.ndarray) -> np.ndarray:
    seats = np.asarray(values, dtype=float)
    return np.select(
        [seats <= 10, seats <= 20, seats <= 30],
        [np.clip(seats, 0, 10).astype(int), 11, 12],
        default=13,
    ).astype(int)


def _full_probability_matrix(probability: np.ndarray, classes: np.ndarray) -> np.ndarray:
    output = np.zeros((len(probability), HYBRID_CLASSES), dtype=float)
    output[:, np.asarray(classes, dtype=int)] = probability
    return output


def fit_hybrid_head(
    train: pd.DataFrame,
    calibration: pd.DataFrame,
    evaluation: pd.DataFrame,
    *,
    seed: int,
    n_estimators: int,
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    feature_set = candidate_features()["state_profile"]
    features = list(feature_set.columns)
    model = _model_pipeline(
        tuple(feature_set.numeric),
        tuple(feature_set.categorical),
        lgb.LGBMClassifier(
            objective="multiclass",
            num_class=HYBRID_CLASSES,
            n_estimators=n_estimators,
            learning_rate=0.035,
            num_leaves=31,
            min_child_samples=25,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
        ),
    )
    model.fit(
        train[features],
        seat_bucket(train["label_seats"]),
        model__sample_weight=event_weights(train),
    )
    cal_raw = np.clip(model.predict_proba(calibration[features]), 1e-9, 1.0)
    eval_raw = np.clip(model.predict_proba(evaluation[features]), 1e-9, 1.0)
    calibrator = LogisticRegression(C=10.0, max_iter=2000, random_state=seed + 1)
    calibrator.fit(
        np.log(cal_raw),
        seat_bucket(calibration["label_seats"]),
        sample_weight=event_weights(calibration),
    )
    cal_probability = _full_probability_matrix(
        calibrator.predict_proba(np.log(cal_raw)), calibrator.classes_
    )
    eval_probability = _full_probability_matrix(
        calibrator.predict_proba(np.log(eval_raw)), calibrator.classes_
    )
    audit = {
        "classes_in_training": sorted(np.unique(seat_bucket(train["label_seats"])).tolist()),
        "classes_in_calibration": sorted(np.unique(seat_bucket(calibration["label_seats"])).tolist()),
        "calibrator_classes": calibrator.classes_.astype(int).tolist(),
        "n_estimators": n_estimators,
        "feature_count": len(features),
    }
    return cal_probability, eval_probability, audit


def build_expansion_matrices(train: pd.DataFrame) -> tuple[dict[str, np.ndarray], np.ndarray]:
    """Map hybrid bucket probabilities to integer-seat PMFs using prior data."""
    weights = event_weights(train)
    table = train[["route_id", "label_seats"]].copy()
    table["weight"] = weights
    table["seat"] = table["label_seats"].round().clip(0, MAX_SEAT).astype(int)
    table["bucket"] = seat_bucket(table["seat"])

    def matrix_for(frame: pd.DataFrame, fallback: np.ndarray | None = None) -> np.ndarray:
        matrix = np.zeros((HYBRID_CLASSES, MAX_SEAT + 1), dtype=float)
        for bucket in range(HYBRID_CLASSES):
            subset = frame.loc[frame["bucket"].eq(bucket)]
            if len(subset):
                counts = subset.groupby("seat", sort=False)["weight"].sum()
                matrix[bucket, counts.index.to_numpy(int)] = counts.to_numpy(float)
            elif fallback is not None:
                matrix[bucket] = fallback[bucket]
            if matrix[bucket].sum() <= 0:
                representative = bucket if bucket <= 10 else {11: 15, 12: 25, 13: 40}[bucket]
                matrix[bucket, representative] = 1.0
            matrix[bucket] /= matrix[bucket].sum()
        return matrix

    global_matrix = matrix_for(table)
    route_matrices = {
        str(route_id): matrix_for(route, fallback=global_matrix)
        for route_id, route in table.groupby("route_id", sort=False)
    }
    return route_matrices, global_matrix


def expand_bucket_probabilities(
    data: pd.DataFrame,
    bucket_probability: np.ndarray,
    route_matrices: dict[str, np.ndarray],
    global_matrix: np.ndarray,
) -> np.ndarray:
    output = np.zeros((len(data), MAX_SEAT + 1), dtype=np.float32)
    routes = data["route_id"].astype(str).to_numpy()
    for route_id in np.unique(routes):
        mask = routes == route_id
        output[mask] = bucket_probability[mask] @ route_matrices.get(route_id, global_matrix)
    capacities = data["capacity"].round().clip(0, MAX_SEAT).astype(int).to_numpy()
    for capacity in np.unique(capacities):
        mask = capacities == capacity
        output[np.ix_(mask, np.arange(capacity + 1, MAX_SEAT + 1))] = 0.0
    normalizer = output.sum(axis=1, keepdims=True)
    output /= np.maximum(normalizer, 1e-12)
    return output


def distribution_metrics(
    data: pd.DataFrame,
    prediction: np.ndarray,
    full_probability: np.ndarray,
    full_threshold: float,
) -> dict[str, float | int]:
    truth = data["label_seats"].to_numpy(float)
    weights = event_weights(data)
    low = truth <= 10
    full_true = truth == 0
    full_pred = full_probability >= full_threshold
    return {
        "rows": int(len(data)),
        "events": int(data["event_id"].nunique()),
        "low_0_10_events": int(data.loc[low, "event_id"].nunique()),
        "full_events": int(data.loc[full_true, "event_id"].nunique()),
        "event_balanced_mae": float(np.average(np.abs(truth - prediction), weights=weights)),
        "low_0_10_mae": float(np.average(np.abs(truth[low] - prediction[low]), weights=weights[low])),
        "full_accuracy": float(accuracy_score(full_true, full_pred, sample_weight=weights)),
        "full_recall": float(recall_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
        "full_precision": float(precision_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
        "full_f1": float(f1_score(full_true, full_pred, sample_weight=weights, zero_division=0)),
        "full_brier": float(np.average((full_probability - full_true.astype(float)) ** 2, weights=weights)),
    }


def convolve_boarding_probabilities(
    seat_pmfs: list[np.ndarray], queue_ahead: int
) -> tuple[np.ndarray, np.ndarray]:
    """Return cumulative boarding probabilities and censored sent-count PMF."""
    cumulative = np.array([1.0], dtype=float)
    board: list[float] = []
    required = int(queue_ahead) + 1
    for pmf in seat_pmfs:
        cumulative = np.convolve(cumulative, np.asarray(pmf, dtype=float))
        board.append(float(cumulative[required:].sum()) if required < len(cumulative) else 0.0)
    board_array = np.maximum.accumulate(np.clip(np.asarray(board), 0.0, 1.0))
    sent = np.empty(len(board_array) + 1, dtype=float)
    sent[0] = board_array[0]
    if len(board_array) > 1:
        sent[1:-1] = np.diff(board_array)
    sent[-1] = 1.0 - board_array[-1]
    sent = np.clip(sent, 0.0, 1.0)
    sent /= max(sent.sum(), 1e-12)
    return board_array, sent


def deterministic_pmf(value: float, capacity: float) -> np.ndarray:
    output = np.zeros(MAX_SEAT + 1, dtype=float)
    seat = int(np.clip(np.rint(value), 0, min(int(round(capacity)), MAX_SEAT)))
    output[seat] = 1.0
    return output


def build_query_scenarios(data: pd.DataFrame, max_buses: int) -> pd.DataFrame:
    frame = data.copy()
    frame["_row_position"] = np.arange(len(frame))
    frame["snapshot_time"] = pd.to_datetime(frame["snapshot_time"])
    frame["event_time"] = pd.to_datetime(frame["event_time"])
    rows: list[dict[str, Any]] = []
    keys = ["route_id", "route_name", "direction", "station_seq_cat", "snapshot_time"]
    for identity, group in frame.groupby(keys, observed=True, sort=False):
        upcoming = (
            group.loc[group["event_time"].gt(group["snapshot_time"])]
            .sort_values("event_time")
            .drop_duplicates("trip_id")
            .head(max_buses)
        )
        if len(upcoming) < max_buses:
            continue
        route_id, route_name, direction, target_station, query_time = identity
        rows.append(
            {
                "route_id": str(route_id),
                "route_name": str(route_name),
                "direction": str(direction),
                "target_station": str(target_station),
                "query_time": query_time,
                "query_bin_15": pd.Timestamp(query_time).floor("15min"),
                "positions": tuple(upcoming["_row_position"].astype(int)),
                "event_ids": tuple(upcoming["event_id"].astype(str)),
                "actual_seats": tuple(upcoming["label_seats"].astype(float)),
            }
        )
    scenarios = pd.DataFrame(rows)
    if scenarios.empty:
        raise ValueError("no query states have enough simultaneously visible buses")
    # Avoid counting the same target and nearly identical bus sequence every few
    # minutes.  Retain one query per 15-minute service state.
    scenarios = (
        scenarios.sort_values("query_time")
        .drop_duplicates(
            ["route_id", "direction", "target_station", "query_bin_15"], keep="first"
        )
        .reset_index(drop=True)
    )
    scenarios.insert(0, "scenario_id", np.arange(len(scenarios)))
    return scenarios


def simulate_queries(
    scenarios: pd.DataFrame,
    seat_pmfs: np.ndarray,
    point_predictions: np.ndarray,
    capacities: np.ndarray,
    *,
    queues: tuple[int, ...],
    max_buses: int,
    hybrid_weights: tuple[float, ...] = (0.0, 1.0),
) -> pd.DataFrame:
    output: list[dict[str, Any]] = []
    for scenario in scenarios.itertuples(index=False):
        positions = np.asarray(scenario.positions, dtype=int)
        actual = np.asarray(scenario.actual_seats, dtype=float)
        hybrid = [seat_pmfs[position] for position in positions]
        deterministic = [
            deterministic_pmf(point_predictions[position], capacities[position])
            for position in positions
        ]
        methods: dict[str, list[np.ndarray]] = {}
        for weight in hybrid_weights:
            if np.isclose(weight, 0.0):
                name = "deterministic_state_profile"
            elif np.isclose(weight, 1.0):
                name = "hybrid_pmf"
            else:
                name = f"hybrid_point_mix_{weight:.2f}"
            methods[name] = [
                weight * hybrid_pmf + (1.0 - weight) * point_pmf
                for hybrid_pmf, point_pmf in zip(hybrid, deterministic, strict=True)
            ]
        for queue in queues:
            actual_board = np.cumsum(actual) >= queue + 1
            actual_sent = int((~actual_board).sum())
            actual_class = min(actual_sent, max_buses)
            for method, pmfs in methods.items():
                board, sent = convolve_boarding_probabilities(pmfs, queue)
                expected_sent = float(np.sum(1.0 - board))
                predicted_class = int(np.argmax(sent))
                row = {
                    "scenario_id": int(scenario.scenario_id),
                    "route_id": scenario.route_id,
                    "route_name": scenario.route_name,
                    "direction": scenario.direction,
                    "target_station": scenario.target_station,
                    "query_time": scenario.query_time,
                    "queue_ahead": int(queue),
                    "method": method,
                    "expected_sent_buses_truncated": expected_sent,
                    "actual_sent_buses_truncated": actual_sent,
                    "predicted_sent_class": predicted_class,
                    "actual_sent_class": actual_class,
                    "tail_probability_after_max_buses": float(sent[-1]),
                    "actual_tail_after_max_buses": int(not actual_board[-1]),
                    "sent_class_probability": float(sent[actual_class]),
                }
                for index in range(max_buses):
                    row[f"board_by_{index + 1}_probability"] = float(board[index])
                    row[f"actual_board_by_{index + 1}"] = int(actual_board[index])
                    row[f"sent_{index}_probability"] = float(sent[index])
                row[f"sent_{max_buses}_plus_probability"] = float(sent[-1])
                output.append(row)
    return pd.DataFrame(output)


def simulation_metric_row(data: pd.DataFrame, max_buses: int) -> dict[str, float | int]:
    expected = data["expected_sent_buses_truncated"].to_numpy(float)
    actual = data["actual_sent_buses_truncated"].to_numpy(float)
    predicted_class = data["predicted_sent_class"].to_numpy(int)
    actual_class = data["actual_sent_class"].to_numpy(int)
    cumulative_errors = []
    for index in range(1, max_buses + 1):
        cumulative_errors.append(
            (
                data[f"board_by_{index}_probability"].to_numpy(float)
                - data[f"actual_board_by_{index}"].to_numpy(float)
            )
            ** 2
        )
    probability = np.clip(data["sent_class_probability"].to_numpy(float), 1e-12, 1.0)
    return {
        "scenarios": int(data["scenario_id"].nunique()),
        "expected_sent_mae": float(np.mean(np.abs(expected - actual))),
        "expected_sent_bias": float(np.mean(expected - actual)),
        "sent_class_accuracy": float(np.mean(predicted_class == actual_class)),
        "sent_class_within_1": float(np.mean(np.abs(predicted_class - actual_class) <= 1)),
        "cumulative_boarding_brier": float(np.mean(np.column_stack(cumulative_errors))),
        "sent_class_log_loss": float(-np.mean(np.log(probability))),
        "mean_tail_probability": float(data["tail_probability_after_max_buses"].mean()),
        "actual_tail_rate": float(data["actual_tail_after_max_buses"].mean()),
    }


def simulation_metric_tables(
    predictions: pd.DataFrame, max_buses: int
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    pooled_rows, route_rows = [], []
    for (method, queue), frame in predictions.groupby(["method", "queue_ahead"], sort=False):
        pooled_rows.append(
            {"method": method, "queue_ahead": queue, **simulation_metric_row(frame, max_buses)}
        )
        for (route_id, route_name), route in frame.groupby(["route_id", "route_name"], sort=False):
            route_rows.append(
                {
                    "method": method,
                    "queue_ahead": queue,
                    "route_id": route_id,
                    "route_name": route_name,
                    **simulation_metric_row(route, max_buses),
                }
            )
    pooled, by_route = pd.DataFrame(pooled_rows), pd.DataFrame(route_rows)
    metric_columns = [
        "expected_sent_mae",
        "expected_sent_bias",
        "sent_class_accuracy",
        "sent_class_within_1",
        "cumulative_boarding_brier",
        "sent_class_log_loss",
        "mean_tail_probability",
        "actual_tail_rate",
    ]
    macro = (
        by_route.groupby(["method", "queue_ahead"], sort=False)[metric_columns]
        .mean()
        .reset_index()
    )
    macro.insert(2, "routes", len(DEFAULT_ROUTES))
    return pooled, by_route, macro


def _align_point_predictions(
    evaluation: pd.DataFrame, existing: pd.DataFrame, *, date: str
) -> np.ndarray:
    point = existing.loc[
        existing["candidate"].eq(POINT_CANDIDATE) & existing["date"].eq(date)
    ].copy()
    keys = ["event_id", "snapshot_time"]
    lookup = point.set_index(keys)["prediction"]
    index = pd.MultiIndex.from_frame(evaluation[keys])
    aligned = lookup.reindex(index).to_numpy(float)
    if np.isnan(aligned).any():
        raise ValueError("state-profile point predictions do not align with evaluation rows")
    return aligned


def _json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): _json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def main() -> int:
    parser = argparse.ArgumentParser(description="Hybrid PMF queue boarding simulation")
    parser.add_argument("--database", type=Path, default=ROOT / "data/gbis_api_cache.sqlite3")
    parser.add_argument(
        "--featured-cache",
        type=Path,
        default=ROOT / "data/analysis_cache/state_profile_main_features.pkl",
    )
    parser.add_argument("--frozen-manifest", type=Path, default=FROZEN_MANIFEST)
    parser.add_argument(
        "--existing-predictions",
        type=Path,
        default=ROOT / "analysis/confidence_interval_delta_lower_results/predictions.pkl",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=ROOT / "analysis/queue_boarding_extended_grid_results",
    )
    parser.add_argument("--queues", nargs="+", type=int, default=list(DEFAULT_QUEUES))
    parser.add_argument("--max-buses", type=int, default=DEFAULT_MAX_BUSES)
    parser.add_argument("--estimators", type=int, default=250)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    if args.max_buses < 1:
        raise ValueError("max-buses must be positive")
    if min(args.queues) < 0:
        raise ValueError("queue lengths must be non-negative")

    load_embedded_modules()
    frozen_args = SimpleNamespace(
        refresh=False,
        rebuild_features=False,
        frozen_manifest=args.frozen_manifest,
        featured_cache=args.featured_cache,
        database=args.database,
    )
    frozen, route_metadata, manifest = load_frozen_features(frozen_args)
    data = prepare_comparison_data(frozen)
    existing = pd.read_pickle(args.existing_predictions)
    weekdays = pd.to_datetime(data["date"]).dt.dayofweek.lt(5)
    train = data.loc[data["date"].lt(CALIBRATION_DATE) & weekdays].copy()
    calibration = data.loc[data["date"].eq(CALIBRATION_DATE)].copy().reset_index(drop=True)
    evaluation = data.loc[data["date"].eq(EVALUATION_DATE)].copy().reset_index(drop=True)
    for name, frame in (("train", train), ("calibration", calibration), ("evaluation", evaluation)):
        missing = sorted(set(DEFAULT_ROUTES) - set(frame["route_id"].astype(str)))
        if frame.empty or missing:
            raise ValueError(f"{name} split invalid; missing_routes={missing}")

    print("[hybrid] fit and calibrate", flush=True)
    cal_bucket, eval_bucket, model_audit = fit_hybrid_head(
        train,
        calibration,
        evaluation,
        seed=args.seed,
        n_estimators=args.estimators,
    )
    route_matrices, global_matrix = build_expansion_matrices(train)
    cal_seat_pmf = expand_bucket_probabilities(
        calibration, cal_bucket, route_matrices, global_matrix
    )
    eval_seat_pmf = expand_bucket_probabilities(
        evaluation, eval_bucket, route_matrices, global_matrix
    )
    cal_mean = cal_seat_pmf @ np.arange(MAX_SEAT + 1)
    eval_mean = eval_seat_pmf @ np.arange(MAX_SEAT + 1)
    full_threshold, calibration_full_f1 = _select_threshold(
        calibration, cal_seat_pmf[:, 0], macro=False
    )
    point_prediction = _align_point_predictions(
        evaluation, existing, date=EVALUATION_DATE
    )
    model_rows = [
        {
            "candidate": "state_profile_point_and_regression_full",
            **distribution_metrics(
                evaluation,
                point_prediction,
                (point_prediction <= 0.5).astype(float),
                0.5,
            ),
        },
        {
            "candidate": "hybrid_pmf_expectation_and_full_head",
            **distribution_metrics(
                evaluation,
                eval_mean,
                eval_seat_pmf[:, 0],
                full_threshold,
            ),
        },
        {
            "candidate": "state_profile_point_plus_hybrid_full_head",
            **distribution_metrics(
                evaluation,
                point_prediction,
                eval_seat_pmf[:, 0],
                full_threshold,
            ),
        },
    ]
    model_metrics = pd.DataFrame(model_rows)

    candidate_weights = (0.0, 0.10, 0.25, 0.50, 0.75, 1.0)
    print("[simulation] select hybrid/point mixture on calibration date", flush=True)
    calibration_point = _align_point_predictions(
        calibration, existing, date=CALIBRATION_DATE
    )
    calibration_scenarios = build_query_scenarios(calibration, args.max_buses)
    calibration_simulation = simulate_queries(
        calibration_scenarios,
        cal_seat_pmf,
        calibration_point,
        calibration["capacity"].to_numpy(float),
        queues=tuple(args.queues),
        max_buses=args.max_buses,
        hybrid_weights=candidate_weights,
    )
    calibration_metrics, _, _ = simulation_metric_tables(
        calibration_simulation, args.max_buses
    )
    calibration_average = (
        calibration_metrics.groupby("method", sort=False)[
            ["expected_sent_mae", "cumulative_boarding_brier", "sent_class_log_loss"]
        ]
        .mean()
        .reset_index()
    )
    brier_method = str(
        calibration_average.sort_values(
            ["cumulative_boarding_brier", "expected_sent_mae"]
        ).iloc[0]["method"]
    )
    mae_method = str(
        calibration_average.sort_values(
            ["expected_sent_mae", "cumulative_boarding_brier"]
        ).iloc[0]["method"]
    )
    method_to_weight = {
        "deterministic_state_profile": 0.0,
        "hybrid_pmf": 1.0,
        **{f"hybrid_point_mix_{weight:.2f}": weight for weight in candidate_weights[1:-1]},
    }
    selected_weights = tuple(
        # Keep 0.75 as the frozen comparison anchor even when an expanded queue
        # grid selects a different mixture for its aggregate calibration score.
        dict.fromkeys([0.0, 1.0, 0.75, method_to_weight[brier_method], method_to_weight[mae_method]])
    )

    print("[simulation] build simultaneous visible-bus query states", flush=True)
    scenarios = build_query_scenarios(evaluation, args.max_buses)
    print(f"[simulation] {len(scenarios)} query states", flush=True)
    simulation = simulate_queries(
        scenarios,
        eval_seat_pmf,
        point_prediction,
        evaluation["capacity"].to_numpy(float),
        queues=tuple(args.queues),
        max_buses=args.max_buses,
        hybrid_weights=selected_weights,
    )
    pooled, by_route, macro = simulation_metric_tables(simulation, args.max_buses)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    model_metrics.to_csv(args.output_dir / "hybrid_model_metrics.csv", index=False)
    calibration_average.to_csv(
        args.output_dir / "mixture_selection_metrics.csv", index=False
    )
    scenarios.drop(columns=["positions"]).to_pickle(
        args.output_dir / "query_scenarios.pkl.gz", compression="gzip"
    )
    calibration_scenarios.drop(columns=["positions"]).to_pickle(
        args.output_dir / "calibration_query_scenarios.pkl.gz", compression="gzip"
    )
    calibration_simulation.to_pickle(
        args.output_dir / "calibration_simulation_predictions.pkl.gz", compression="gzip"
    )
    simulation.to_pickle(args.output_dir / "simulation_predictions.pkl.gz", compression="gzip")
    pooled.to_csv(args.output_dir / "simulation_metrics_pooled.csv", index=False)
    by_route.to_csv(args.output_dir / "simulation_metrics_by_route.csv", index=False)
    macro.to_csv(args.output_dir / "simulation_metrics_route_macro.csv", index=False)

    summary = {
        "protocol": {
            "source_cutoff": manifest["source_cutoff"],
            "source_fingerprint": manifest["source_fingerprint"],
            "training_rule": "complete weekdays strictly before calibration date",
            "calibration_date": CALIBRATION_DATE,
            "evaluation_date": EVALUATION_DATE,
            "included_routes": DEFAULT_ROUTES,
            "excluded_routes": "all non-active routes; no complete pooled frozen coverage",
            "sealed_partial_date": manifest["sealed_partial_date"],
            "sealed_data_accessed": False,
            "queue_definition": "number of passengers ahead of the user",
            "counterfactual_assumptions": [
                "no new passenger arrivals after query time",
                "all passengers board in FIFO order whenever seats are available",
                "only the next simultaneously visible buses are considered",
                "bus seat predictions are independent during convolution",
            ],
            "queues": args.queues,
            "max_visible_buses": args.max_buses,
            "expectation_contract": "E[min(actual buses sent, max_visible_buses)]; tail probability is reported separately",
            "query_downsampling": "one state per route x direction x target station x 15-minute bin",
        },
        "hybrid_buckets": [
            *[str(value) for value in range(11)],
            "11-20",
            "21-30",
            "31+",
        ],
        "coarse_bucket_expansion": "strict-prior route empirical integer-seat distribution with pooled fallback",
        "model_audit": model_audit,
        "full_threshold": full_threshold,
        "calibration_full_f1": calibration_full_f1,
        "query_scenarios": int(len(scenarios)),
        "calibration_query_scenarios": int(len(calibration_scenarios)),
        "mixture_selection": {
            "candidate_hybrid_weights": candidate_weights,
            "brier_selected_method": brier_method,
            "mae_selected_method": mae_method,
            "evaluation_weights": selected_weights,
            "calibration_average_metrics": calibration_average.to_dict(orient="records"),
        },
        "hybrid_model_metrics": model_metrics.to_dict(orient="records"),
        "simulation_metrics_pooled": pooled.to_dict(orient="records"),
        "simulation_metrics_route_macro": macro.to_dict(orient="records"),
        "route_cache_metadata": route_metadata,
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(_json_ready(summary), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("\nHybrid model metrics")
    print(model_metrics.to_string(index=False))
    print("\nSimulation pooled")
    print(pooled.to_string(index=False))
    print("\nSimulation route macro")
    print(macro.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/queue_boarding_v1_inference.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/queue_boarding_v1_inference.py
"""Portable inference API for ``queue-boarding/v1.0.0``."""
from __future__ import annotations

import argparse
import hashlib
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd

from queue_boarding_policy import MODEL_ID
from yeonwu_peer_feature_experiment import ROOT


DEFAULT_ARTIFACT = (
    ROOT
    / "analysis/model_registry/queue-boarding/v1.0.0/queue_boarding_v1_0_0.pkl"
)
DEFAULT_REGISTRY = ROOT / "analysis/model_registry/queue-boarding/registry.json"


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _lookup(profile: dict[str, Any], target: pd.DataFrame) -> pd.DataFrame:
    output = pd.DataFrame(index=target.index)
    blocks: dict[str, pd.DataFrame] = {}
    for name in ("cell", "fallback"):
        block = profile[name]
        keys = list(block["keys"])
        index = pd.MultiIndex.from_frame(target[keys])
        blocks[name] = pd.DataFrame(
            {
                "mean": block["mean"].reindex(index).to_numpy(),
                "count": block["count"].reindex(index).to_numpy(),
                "low10": block["low10"].reindex(index).to_numpy(),
            },
            index=target.index,
        )
    mean = blocks["cell"]["mean"].fillna(blocks["fallback"]["mean"])
    mean = mean.fillna(float(profile["global_mean"]))
    count = blocks["cell"]["count"].fillna(blocks["fallback"]["count"])
    count = count.fillna(0.0)
    low = blocks["cell"]["low10"].fillna(blocks["fallback"]["low10"])
    low = low.fillna(float(profile["global_low10"]))
    current = target["snapshot_remaining_seats"].to_numpy(float)
    output["snapshot_state_expected_seats"] = mean.to_numpy(float)
    output["snapshot_state_seat_deviation"] = current - mean.to_numpy(float)
    output["snapshot_state_low10_rate"] = low.to_numpy(float)
    output["snapshot_state_log_count"] = np.log1p(count.to_numpy(float))
    return output


def _decode_target(
    raw: np.ndarray, data: pd.DataFrame, kind: str
) -> np.ndarray:
    current = data["snapshot_remaining_seats"].to_numpy(float)
    gap = np.maximum(data["target_stop_gap"].to_numpy(float), 1.0)
    if kind == "delta_per_stop":
        prediction = current + np.asarray(raw, float) * gap
    elif kind == "delta_per_sqrt_stop":
        prediction = current + np.asarray(raw, float) * np.sqrt(gap)
    else:
        raise ValueError(f"unsupported point target kind: {kind}")
    return np.clip(prediction, 0, data["capacity"].to_numpy(float))


def _full_probability_matrix(
    probability: np.ndarray, classes: np.ndarray, class_count: int
) -> np.ndarray:
    output = np.zeros((len(probability), class_count), dtype=float)
    output[:, np.asarray(classes, dtype=int)] = probability
    return output


def _deterministic_pmf(value: float, capacity: float, max_seat: int) -> np.ndarray:
    output = np.zeros(max_seat + 1, dtype=float)
    seat = int(np.clip(np.rint(value), 0, min(int(round(capacity)), max_seat)))
    output[seat] = 1.0
    return output


def _convolve(
    seat_pmfs: list[np.ndarray], queue_ahead: int
) -> tuple[np.ndarray, np.ndarray]:
    cumulative = np.asarray([1.0], dtype=float)
    required = int(queue_ahead) + 1
    board: list[float] = []
    for pmf in seat_pmfs:
        cumulative = np.convolve(cumulative, np.asarray(pmf, dtype=float))
        board.append(
            float(cumulative[required:].sum()) if required < len(cumulative) else 0.0
        )
    board_probability = np.maximum.accumulate(
        np.clip(np.asarray(board, dtype=float), 0.0, 1.0)
    )
    sent = np.concatenate(
        [
            board_probability[:1],
            np.diff(board_probability),
            1.0 - board_probability[-1:],
        ]
    )
    sent = np.clip(sent, 0.0, 1.0)
    sent /= max(float(sent.sum()), 1e-12)
    return board_probability, sent


@dataclass
class QueueBoardingV1:
    bundle: dict[str, Any]
    artifact_path: Path

    @classmethod
    def load(
        cls,
        artifact_path: Path = DEFAULT_ARTIFACT,
        *,
        registry_path: Path = DEFAULT_REGISTRY,
        verify_registry: bool = True,
    ) -> "QueueBoardingV1":
        artifact_path = Path(artifact_path)
        if verify_registry:
            registry = json.loads(Path(registry_path).read_text(encoding="utf-8"))
            version = str(registry["primary_version"])
            record = registry["versions"][version]
            declared = record.get("portable_artifact")
            expected_hash = record.get("portable_artifact_sha256")
            if not declared or not expected_hash:
                raise ValueError("registry does not declare a portable artifact")
            expected_path = (Path(registry_path).parent / declared).resolve()
            if artifact_path.resolve() != expected_path:
                raise ValueError("artifact path differs from primary registry")
            if file_sha256(artifact_path) != expected_hash:
                raise ValueError("portable artifact SHA-256 mismatch")
        bundle = joblib.load(artifact_path)
        if bundle.get("model_id") != MODEL_ID or bundle.get("format_version") != 1:
            raise ValueError("unsupported queue-boarding artifact")
        return cls(bundle=bundle, artifact_path=artifact_path)

    def prepare_features(self, rows: pd.DataFrame) -> pd.DataFrame:
        output = rows.copy()
        runtime_missing = sorted({"route_id", "capacity"} - set(output.columns))
        if runtime_missing:
            raise ValueError(f"missing inference runtime columns: {runtime_missing}")
        missing = sorted(
            set(self.bundle["required_prepared_input_features"]) - set(output.columns)
        )
        if missing:
            raise ValueError(f"missing prepared snapshot features: {missing}")
        supported = set(self.bundle["routes"])
        route_values = set(output["route_id"].astype(str))
        unknown = sorted(route_values - supported)
        if unknown:
            raise ValueError(f"unsupported routes: {unknown}")
        output["route_id"] = output["route_id"].astype(str)
        output["route_code"] = output["route_code"].astype("string")
        for column in self.bundle["categorical_features"]:
            output[column] = output[column].astype("string")
        state = _lookup(self.bundle["state_profile"], output)
        output[list(state.columns)] = state
        return output

    def predict_seat_distributions(
        self, rows: pd.DataFrame
    ) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
        prepared = self.prepare_features(rows)
        columns = list(self.bundle["feature_columns"])
        point = np.zeros(len(prepared), dtype=float)
        for name, model in self.bundle["point"]["models"].items():
            raw = model.predict(prepared[columns])
            decoded = _decode_target(
                raw, prepared, self.bundle["point"]["target_kinds"][name]
            )
            point += float(self.bundle["point"]["weights"][name]) * decoded
        point = np.clip(point, 0, prepared["capacity"].to_numpy(float))

        distribution = self.bundle["distribution"]
        raw_probability = np.clip(
            distribution["classifier"].predict_proba(prepared[columns]), 1e-9, 1.0
        )
        calibrator = distribution["calibrator"]
        bucket_probability = _full_probability_matrix(
            calibrator.predict_proba(np.log(raw_probability)),
            calibrator.classes_,
            int(distribution["hybrid_classes"]),
        )
        max_seat = int(distribution["max_seat"])
        hybrid = np.zeros((len(prepared), max_seat + 1), dtype=float)
        routes = prepared["route_id"].astype(str).to_numpy()
        global_matrix = distribution["global_expansion_matrix"]
        for route_id in np.unique(routes):
            mask = routes == route_id
            matrix = distribution["route_expansion_matrices"].get(
                route_id, global_matrix
            )
            hybrid[mask] = bucket_probability[mask] @ matrix
        capacities = prepared["capacity"].round().clip(0, max_seat).astype(int).to_numpy()
        for index, capacity in enumerate(capacities):
            hybrid[index, capacity + 1 :] = 0.0
        hybrid /= np.maximum(hybrid.sum(axis=1, keepdims=True), 1e-12)
        weight = float(self.bundle["boarding_policy"]["hybrid_weight"])
        mixed = np.empty_like(hybrid)
        for index, (value, capacity) in enumerate(zip(point, capacities, strict=True)):
            deterministic = _deterministic_pmf(value, capacity, max_seat)
            mixed[index] = weight * hybrid[index] + (1.0 - weight) * deterministic
        mixed /= np.maximum(mixed.sum(axis=1, keepdims=True), 1e-12)
        return point, mixed, prepared

    def predict_queries(self, rows: pd.DataFrame) -> pd.DataFrame:
        required = {"query_id", "bus_order", "queue_ahead"}
        missing = sorted(required - set(rows.columns))
        if missing:
            raise ValueError(f"missing query columns: {missing}")
        point, pmfs, prepared = self.predict_seat_distributions(rows)
        prepared = prepared.copy()
        prepared["_position"] = np.arange(len(prepared))
        prepared["point_arrival_seats"] = point
        output: list[dict[str, Any]] = []
        expected_buses = int(self.bundle["boarding_policy"]["max_visible_buses"])
        for query_id, group in prepared.groupby("query_id", sort=False):
            ordered = group.sort_values("bus_order")
            if ordered["bus_order"].astype(int).tolist() != list(
                range(1, expected_buses + 1)
            ):
                raise ValueError(
                    f"query {query_id!r} must contain bus_order 1..{expected_buses}"
                )
            queues = ordered["queue_ahead"].astype(int).unique()
            if len(queues) != 1 or queues[0] < 0:
                raise ValueError(f"query {query_id!r} has invalid queue_ahead")
            positions = ordered["_position"].to_numpy(int)
            board, sent = _convolve([pmfs[position] for position in positions], int(queues[0]))
            row: dict[str, Any] = {
                "query_id": query_id,
                "queue_ahead": int(queues[0]),
                "predicted_sent_class": int(np.argmax(sent)),
                "expected_sent_buses_truncated": float(np.sum(1.0 - board)),
                "tail_probability_after_3_buses": float(sent[-1]),
                "model_id": self.bundle["model_id"],
            }
            for index in range(expected_buses):
                row[f"board_by_{index + 1}_probability"] = float(board[index])
                row[f"sent_{index}_probability"] = float(sent[index])
                row[f"bus_{index + 1}_point_arrival_seats"] = float(
                    point[positions[index]]
                )
            row["sent_3_plus_probability"] = float(sent[-1])
            output.append(row)
        return pd.DataFrame(output)


def main() -> int:
    parser = argparse.ArgumentParser(
        description="Run queue-boarding/v1.0.0 portable inference"
    )
    parser.add_argument("--input-csv", type=Path, required=True)
    parser.add_argument("--output-csv", type=Path, required=True)
    parser.add_argument("--artifact", type=Path, default=DEFAULT_ARTIFACT)
    parser.add_argument("--registry", type=Path, default=DEFAULT_REGISTRY)
    args = parser.parse_args()
    model = QueueBoardingV1.load(
        args.artifact, registry_path=args.registry, verify_registry=True
    )
    result = model.predict_queries(pd.read_csv(args.input_csv))
    args.output_csv.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(args.output_csv, index=False)
    print(f"{model.bundle['model_id']}: {len(result)} queries")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/queue_boarding_v1_training.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/queue_boarding_v1_training.py
"""Train and export the portable ``queue-boarding/v1.0.0`` bundle.

The final-evaluation date is intentionally not configurable.  Every fitted
object is derived from the frozen development cache and the pre-registered
training/calibration chronology in the model manifest.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from confidence_interval_comparison import (
    FROZEN_MANIFEST,
    candidate_features,
    load_frozen_features,
    prepare_comparison_data,
)
from main_model_feature_augmentation import component_candidates, make_model
from hypothesis_model_search import candidate_weights, encode_target
from queue_boarding_policy import (
    MODEL_ID,
    PRIMARY_DECODER,
    PRIMARY_HYBRID_WEIGHT,
    PRIMARY_MAX_VISIBLE_BUSES,
    PRIMARY_METHOD,
    PRIMARY_POINT_WEIGHT,
    PRIMARY_QUEUE_GRID,
)
from queue_boarding_simulation_experiment import (
    HYBRID_CLASSES,
    MAX_SEAT,
    _full_probability_matrix,
    build_expansion_matrices,
    seat_bucket,
)
from uncertainty_heads import _model_pipeline, event_weights
from yeonwu_peer_feature_experiment import (
    DEFAULT_ROUTES,
    ROOT,
    STATE_FALLBACK,
    STATE_GROUP,
    load_embedded_modules,
)


TRAIN_END_EXCLUSIVE = "2026-08-13"
CALIBRATION_DATE = "2026-08-13"
POINT_TRAIN_END_EXCLUSIVE = "2026-08-14"
PROFILE_END_INCLUSIVE = "2026-08-16"
SEED = 42
N_ESTIMATORS = 250
DEFAULT_OUTPUT = (
    ROOT
    / "analysis/model_registry/queue-boarding/v1.0.0/queue_boarding_v1_0_0.pkl"
)


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): _json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def build_state_profile(history: pd.DataFrame) -> dict[str, Any]:
    """Freeze the latest allowed historical state lookup for inference."""
    source = history.loc[
        history["date"].le(PROFILE_END_INCLUSIVE)
        & pd.to_datetime(history["date"]).dt.dayofweek.lt(5)
    ].copy()
    value = "snapshot_remaining_seats"

    def statistics(keys: list[str]) -> dict[str, Any]:
        grouped = source.groupby(keys, observed=True)[value]
        mean = grouped.mean()
        count = grouped.count()
        low = (
            source.assign(_low=source[value].le(10).astype(float))
            .groupby(keys, observed=True)["_low"]
            .mean()
        )
        return {"keys": list(keys), "mean": mean, "count": count, "low10": low}

    return {
        "cell": statistics(list(STATE_GROUP)),
        "fallback": statistics(list(STATE_FALLBACK)),
        "global_mean": float(source[value].mean()),
        "global_low10": float(source[value].le(10).mean()),
        "profile_end_inclusive": PROFILE_END_INCLUSIVE,
        "weekday_history_only": True,
        "rows": int(len(source)),
        "events": int(source["event_id"].nunique()),
    }


def fit_hybrid_objects(
    train: pd.DataFrame,
    calibration: pd.DataFrame,
    feature_columns: list[str],
    numeric: tuple[str, ...],
    categorical: tuple[str, ...],
) -> tuple[Any, Any, dict[str, Any]]:
    classifier = _model_pipeline(
        numeric,
        categorical,
        lgb.LGBMClassifier(
            objective="multiclass",
            num_class=HYBRID_CLASSES,
            n_estimators=N_ESTIMATORS,
            learning_rate=0.035,
            num_leaves=31,
            min_child_samples=25,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        ),
    )
    classifier.fit(
        train[feature_columns],
        seat_bucket(train["label_seats"]),
        model__sample_weight=event_weights(train),
    )
    calibration_raw = np.clip(
        classifier.predict_proba(calibration[feature_columns]), 1e-9, 1.0
    )
    calibrator = LogisticRegression(
        C=10.0, max_iter=2000, random_state=SEED + 1
    )
    calibrator.fit(
        np.log(calibration_raw),
        seat_bucket(calibration["label_seats"]),
        sample_weight=event_weights(calibration),
    )
    calibrated = _full_probability_matrix(
        calibrator.predict_proba(np.log(calibration_raw)), calibrator.classes_
    )
    audit = {
        "training_classes": sorted(
            np.unique(seat_bucket(train["label_seats"])).astype(int).tolist()
        ),
        "calibration_classes": sorted(
            np.unique(seat_bucket(calibration["label_seats"])).astype(int).tolist()
        ),
        "calibrator_classes": calibrator.classes_.astype(int).tolist(),
        "calibration_probability_row_sum_max_error": float(
            np.abs(calibrated.sum(axis=1) - 1.0).max()
        ),
    }
    return classifier, calibrator, audit


def train_bundle() -> dict[str, Any]:
    load_embedded_modules()
    frozen_args = SimpleNamespace(
        refresh=False,
        rebuild_features=False,
        frozen_manifest=FROZEN_MANIFEST,
        featured_cache=ROOT / "data/analysis_cache/state_profile_main_features.pkl",
        database=ROOT / "data/gbis_api_cache.sqlite3",
    )
    frozen, route_metadata, frozen_manifest = load_frozen_features(frozen_args)
    prepared = prepare_comparison_data(frozen)
    weekdays = pd.to_datetime(prepared["date"]).dt.dayofweek.lt(5)
    distribution_train = prepared.loc[
        prepared["date"].lt(TRAIN_END_EXCLUSIVE) & weekdays
    ].copy()
    calibration = prepared.loc[
        prepared["date"].eq(CALIBRATION_DATE)
    ].copy()
    point_train = prepared.loc[
        prepared["date"].lt(POINT_TRAIN_END_EXCLUSIVE) & weekdays
    ].copy()
    for name, frame in (
        ("distribution_train", distribution_train),
        ("calibration", calibration),
        ("point_train", point_train),
    ):
        missing = sorted(set(DEFAULT_ROUTES) - set(frame["route_id"].astype(str)))
        if frame.empty or missing:
            raise ValueError(f"{name} split invalid; missing_routes={missing}")
    forbidden = prepared["date"].gt(CALIBRATION_DATE)
    if forbidden.any() and (
        distribution_train["date"].gt(CALIBRATION_DATE).any()
        or calibration["date"].gt(CALIBRATION_DATE).any()
        or point_train["date"].gt(CALIBRATION_DATE).any()
    ):
        raise RuntimeError("post-calibration labels entered fitted model data")

    features = candidate_features()["state_profile"]
    columns = list(features.columns)
    point_models: dict[str, Any] = {}
    target_kinds: dict[str, str] = {}
    for candidate in component_candidates():
        print(f"[point] {candidate.name}", flush=True)
        model = make_model(candidate, features, SEED)
        model.fit(
            point_train[columns],
            encode_target(point_train, candidate.target_kind),
            regressor__sample_weight=candidate_weights(
                point_train,
                candidate.low_weight,
                weighting_kind=candidate.weighting_kind,
                gap_weight_power=candidate.gap_weight_power,
            ),
        )
        point_models[candidate.name] = model
        target_kinds[candidate.name] = candidate.target_kind

    print("[distribution] hybrid bucket classifier + calibration", flush=True)
    classifier, calibrator, hybrid_audit = fit_hybrid_objects(
        distribution_train,
        calibration,
        columns,
        tuple(features.numeric),
        tuple(features.categorical),
    )
    route_matrices, global_matrix = build_expansion_matrices(distribution_train)
    profile = build_state_profile(frozen)
    return {
        "format_version": 1,
        "model_id": MODEL_ID,
        "model_family": "queue-boarding",
        "version": "v1.0.0",
        "created_from_frozen_contract": True,
        "source_cutoff": frozen_manifest["source_cutoff"],
        "source_fingerprint": frozen_manifest["source_fingerprint"],
        "routes": dict(DEFAULT_ROUTES),
        "feature_columns": columns,
        "numeric_features": list(features.numeric),
        "categorical_features": list(features.categorical),
        "required_prepared_input_features": [
            column for column in columns if not column.startswith("snapshot_state_")
        ],
        "state_profile": profile,
        "point": {
            "models": point_models,
            "target_kinds": target_kinds,
            "weights": {"hgb": 0.4, "extra_trees": 0.4, "lightgbm": 0.2},
            "training_end_exclusive": POINT_TRAIN_END_EXCLUSIVE,
            "training_rows": int(len(point_train)),
            "training_events": int(point_train["event_id"].nunique()),
        },
        "distribution": {
            "classifier": classifier,
            "calibrator": calibrator,
            "route_expansion_matrices": route_matrices,
            "global_expansion_matrix": global_matrix,
            "hybrid_classes": HYBRID_CLASSES,
            "max_seat": MAX_SEAT,
            "training_end_exclusive": TRAIN_END_EXCLUSIVE,
            "calibration_date": CALIBRATION_DATE,
            "training_rows": int(len(distribution_train)),
            "training_events": int(distribution_train["event_id"].nunique()),
            "calibration_rows": int(len(calibration)),
            "calibration_events": int(calibration["event_id"].nunique()),
            "audit": hybrid_audit,
        },
        "boarding_policy": {
            "method": PRIMARY_METHOD,
            "point_weight": PRIMARY_POINT_WEIGHT,
            "hybrid_weight": PRIMARY_HYBRID_WEIGHT,
            "queue_grid": list(PRIMARY_QUEUE_GRID),
            "max_visible_buses": PRIMARY_MAX_VISIBLE_BUSES,
            "decoder": PRIMARY_DECODER,
            "classes": {
                0: "board first bus",
                1: "send one bus",
                2: "send two buses",
                3: "three or more / not within visible horizon",
            },
        },
        "input_contract": {
            "unit": "one prepared pre-arrival snapshot row per visible bus",
            "batch_group_columns": ["query_id", "bus_order", "queue_ahead"],
            "bus_order": "1, 2, 3 in expected arrival order",
            "queue_ahead": "non-negative integer passengers ahead of the user",
            "feature_history": "base strict-prior features must use only observations available before inference",
        },
        "training_policy": {
            "development_cache_refresh": False,
            "final_evaluation_labels_used": False,
            "fitted_label_latest_date": CALIBRATION_DATE,
            "seed": SEED,
        },
        "route_cache_metadata": route_metadata,
    }


def main() -> int:
    parser = argparse.ArgumentParser(
        description="Export portable queue-boarding/v1.0.0 model bundle"
    )
    parser.add_argument("--output", type=Path, default=DEFAULT_OUTPUT)
    args = parser.parse_args()
    if args.output.exists():
        raise FileExistsError(f"refusing to overwrite immutable artifact: {args.output}")
    bundle = train_bundle()
    args.output.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(bundle, args.output, compress=3)
    metadata = {
        key: value
        for key, value in bundle.items()
        if key not in {"state_profile", "point", "distribution"}
    }
    metadata["artifact"] = {
        "path": str(args.output.relative_to(ROOT)),
        "sha256": file_sha256(args.output),
        "bytes": args.output.stat().st_size,
    }
    metadata["point"] = {
        key: value for key, value in bundle["point"].items() if key != "models"
    }
    metadata["distribution"] = {
        key: value
        for key, value in bundle["distribution"].items()
        if key
        not in {
            "classifier",
            "calibrator",
            "route_expansion_matrices",
            "global_expansion_matrix",
        }
    }
    metadata["state_profile"] = {
        key: value
        for key, value in bundle["state_profile"].items()
        if key not in {"cell", "fallback"}
    }
    metadata_path = args.output.with_suffix(".metadata.json")
    metadata_path.write_text(
        json.dumps(_json_ready(metadata), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(
        json.dumps(
            {
                "model_id": bundle["model_id"],
                "artifact": str(args.output),
                "sha256": metadata["artifact"]["sha256"],
                "bytes": metadata["artifact"]["bytes"],
            },
            ensure_ascii=False,
            indent=2,
        )
    )
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/route_specific_feature_experiment.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/route_specific_feature_experiment.py
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

import pandas as pd

from all_prearrival_seat_regression import build_all_prearrival_table
from hypothesis_model_search import load_analysis_data
from lightgbm_feature_experiment import run_rolling_origin
from linear_feature_experiment import (
    add_preceding_bus_segment_features,
    add_strict_prior_flow_features,
    add_strict_prior_low_rate_features,
    cumulative_feature_sets,
    json_ready,
)
from model_feasibility import build_model_table, build_visits
from tminus_feasibility import prepare_raw_locations


DEFAULT_ROUTES = {
    "219000013": "1000",
    "222000074": "1100",
    "219000016": "1200",
    "218000010": "1500",
    "222000075": "2000",
    "200000104": "3000",
}
VALIDATION_DATES = ("2026-08-11", "2026-08-12", "2026-08-13")
DEFAULT_SOURCE_CUTOFF = "2026-08-13 20:26:03+09:00"


def cache_paths(cache_dir: Path, route_id: str) -> tuple[Path, Path, Path]:
    return (
        cache_dir / f"{route_id}_snapshots.pkl",
        cache_dir / f"{route_id}_flows.pkl",
        cache_dir / f"{route_id}_metadata.json",
    )


def build_route_snapshots(
    database: Path,
    route_id: str,
    *,
    source_cutoff: str,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    locations, stations = load_analysis_data(database, route_id)
    cutoff = pd.Timestamp(source_cutoff)
    locations = locations.loc[locations["observed_at"].le(cutoff)].copy()
    if locations.empty:
        raise ValueError(f"노선 {route_id}에 cutoff 이전 관측이 없습니다.")
    visits = build_visits(locations, stations)
    table, turnaround_seq = build_model_table(
        visits, stations, label_target="arrival"
    )
    source = table.loc[
        table["is_peak"]
        & table["label_quality"].eq("A")
        & table["label_seats"].notna()
    ].copy()
    if source.empty:
        raise ValueError(f"노선 {route_id}에 peak A급 도착 라벨이 없습니다.")
    _, by_vehicle = prepare_raw_locations(locations, turnaround_seq)
    snapshots = build_all_prearrival_table(
        source,
        visits,
        by_vehicle,
        turnaround_seq=turnaround_seq,
        max_seq=int(stations["station_seq"].max()),
    )

    flows = visits.loc[
        (~visits["is_pass_node"])
        & visits["observed_arrival_seats"].ge(0)
        & visits["departure_seats"].ge(0)
        & visits["arrival_seen"].notna()
        & visits["departure_seen"].notna()
    ].copy()
    flows["direction"] = "return"
    flows.loc[flows["station_seq"].le(turnaround_seq), "direction"] = "to_city"
    flows["date"] = flows["arrival_seen"].dt.date.astype(str)
    flows["time_bin_2h"] = (flows["arrival_seen"].dt.hour // 2 * 2).astype(int)
    flows["stop_net"] = (
        flows["departure_seats"] - flows["observed_arrival_seats"]
    ).astype(float)
    flows = flows[
        [
            "vehicle_id",
            "trip_id",
            "station_seq",
            "direction",
            "date",
            "arrival_seen",
            "departure_seen",
            "time_bin_2h",
            "observed_arrival_seats",
            "departure_seats",
            "stop_net",
        ]
    ].sort_values(["arrival_seen", "station_seq", "vehicle_id"])
    flows.attrs["pass_node_sequences"] = sorted(
        visits.loc[visits["is_pass_node"], "station_seq"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )
    metadata = {
        "route_id": route_id,
        "source_cutoff": source_cutoff,
        "source_rows": int(len(locations)),
        "source_observed_at_min": locations["observed_at"].min().isoformat(),
        "source_observed_at_max": locations["observed_at"].max().isoformat(),
        "turnaround_seq": int(turnaround_seq),
        "snapshot_rows": int(len(snapshots)),
        "events": int(snapshots["event_id"].nunique()),
        "flow_rows": int(len(flows)),
        "dates": sorted(snapshots["date"].unique().tolist()),
    }
    return snapshots, flows, metadata


def load_or_build_route_cache(
    database: Path,
    cache_dir: Path,
    route_id: str,
    *,
    source_cutoff: str,
    rebuild: bool,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    snapshot_path, flow_path, metadata_path = cache_paths(cache_dir, route_id)
    if not rebuild and snapshot_path.is_file() and flow_path.is_file() and metadata_path.is_file():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata.get("source_cutoff") == source_cutoff:
            return pd.read_pickle(snapshot_path), pd.read_pickle(flow_path), {
                **metadata,
                "cache_hit": True,
            }

    snapshots, flows, metadata = build_route_snapshots(
        database, route_id, source_cutoff=source_cutoff
    )
    cache_dir.mkdir(parents=True, exist_ok=True)
    snapshots.to_pickle(snapshot_path)
    flows.to_pickle(flow_path)
    metadata_path.write_text(
        json.dumps(json_ready(metadata), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return snapshots, flows, {**metadata, "cache_hit": False}


def feature_sets() -> dict[str, tuple[str, ...]]:
    cumulative = cumulative_feature_sets()
    return {
        "current_only": cumulative["current_only"],
        "path_features": cumulative["path_flow"],
        "all_features": cumulative["preceding_bus"],
    }


def comparison_rows(
    route_id: str,
    route_name: str,
    metrics: pd.DataFrame,
    metadata: dict[str, Any],
    preceding_audit: dict[str, Any],
) -> list[dict[str, Any]]:
    indexed = metrics.set_index("candidate")
    baseline = indexed.loc["current_only"]
    rows: list[dict[str, Any]] = []
    for candidate in ("path_features", "all_features"):
        candidate_row = indexed.loc[candidate]
        rows.append(
            {
                "route_id": route_id,
                "route_name": route_name,
                "candidate": candidate,
                "snapshot_rows": metadata["snapshot_rows"],
                "events": metadata["events"],
                "validation_events": int(candidate_row["events"]),
                "full_events": int(candidate_row["full_events"]),
                "low_0_10_events": int(candidate_row["low_0_10_events"]),
                "preceding_bus_coverage": float(preceding_audit["coverage"]),
                "baseline_mae": float(baseline["event_balanced_mae"]),
                "candidate_mae": float(candidate_row["event_balanced_mae"]),
                "mae_improvement": float(
                    baseline["event_balanced_mae"]
                    - candidate_row["event_balanced_mae"]
                ),
                "baseline_low_0_10_mae": float(baseline["low_0_10_mae"]),
                "candidate_low_0_10_mae": float(candidate_row["low_0_10_mae"]),
                "low_0_10_mae_improvement": float(
                    baseline["low_0_10_mae"] - candidate_row["low_0_10_mae"]
                ),
                "baseline_full_accuracy": float(baseline["full_accuracy"]),
                "candidate_full_accuracy": float(candidate_row["full_accuracy"]),
                "baseline_full_recall": float(baseline["full_recall"]),
                "candidate_full_recall": float(candidate_row["full_recall"]),
                "baseline_full_precision": float(baseline["full_precision"]),
                "candidate_full_precision": float(candidate_row["full_precision"]),
                "baseline_full_f1": float(baseline["full_f1"]),
                "candidate_full_f1": float(candidate_row["full_f1"]),
                "full_f1_improvement": float(
                    candidate_row["full_f1"] - baseline["full_f1"]
                ),
            }
        )
    return rows


def main() -> int:
    parser = argparse.ArgumentParser(
        description="사용자 제안 피처의 노선별 독립 LightGBM 실험"
    )
    parser.add_argument(
        "--database",
        type=Path,
        default=Path("data/gbis_api_cache.sqlite3"),
    )
    parser.add_argument(
        "--routes",
        nargs="*",
        default=list(DEFAULT_ROUTES),
    )
    parser.add_argument("--source-cutoff", default=DEFAULT_SOURCE_CUTOFF)
    parser.add_argument(
        "--cache-dir",
        type=Path,
        default=Path("data/analysis_cache/route_specific_features"),
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("analysis/route_specific_feature_results"),
    )
    parser.add_argument("--rebuild", action="store_true")
    args = parser.parse_args()

    unknown = sorted(set(args.routes) - set(DEFAULT_ROUTES))
    if unknown:
        raise ValueError(f"노선 이름 매핑이 없는 route_id입니다: {unknown}")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    all_metrics: list[pd.DataFrame] = []
    all_daily: list[pd.DataFrame] = []
    all_comparisons: list[dict[str, Any]] = []
    route_summaries: dict[str, Any] = {}

    for route_id in args.routes:
        route_name = DEFAULT_ROUTES[route_id]
        print(f"[{route_name}] snapshot/flow 준비", flush=True)
        snapshots, flows, metadata = load_or_build_route_cache(
            args.database,
            args.cache_dir,
            route_id,
            source_cutoff=args.source_cutoff,
            rebuild=args.rebuild,
        )
        missing_validation = sorted(set(VALIDATION_DATES) - set(snapshots["date"]))
        if missing_validation:
            raise ValueError(
                f"노선 {route_name}에 검증일이 없습니다: {missing_validation}"
            )
        print(f"[{route_name}] strict-prior 피처 생성", flush=True)
        featured = add_strict_prior_low_rate_features(snapshots)
        featured = add_strict_prior_flow_features(featured, flows)
        featured, preceding_audit = add_preceding_bus_segment_features(
            featured, flows
        )
        development_dates = tuple(
            date
            for date in sorted(featured["date"].unique())
            if date <= max(VALIDATION_DATES)
        )
        print(f"[{route_name}] rolling-origin LightGBM", flush=True)
        metrics, daily, _ = run_rolling_origin(
            featured,
            feature_sets=feature_sets(),
            development_dates=development_dates,
            validation_dates=VALIDATION_DATES,
        )
        metrics.insert(0, "route_name", route_name)
        metrics.insert(0, "route_id", route_id)
        daily.insert(0, "route_name", route_name)
        daily.insert(0, "route_id", route_id)
        all_metrics.append(metrics)
        all_daily.append(daily)
        all_comparisons.extend(
            comparison_rows(
                route_id,
                route_name,
                metrics,
                metadata,
                preceding_audit,
            )
        )
        route_summaries[route_id] = {
            "route_name": route_name,
            "cache": metadata,
            "development_dates": development_dates,
            "validation_dates": VALIDATION_DATES,
            "preceding_bus_segment_audit": preceding_audit,
            "metrics": metrics.to_dict(orient="records"),
        }

    metrics_table = pd.concat(all_metrics, ignore_index=True)
    daily_table = pd.concat(all_daily, ignore_index=True)
    comparison_table = pd.DataFrame(all_comparisons)
    metrics_table.to_csv(args.output_dir / "metrics_by_route.csv", index=False)
    daily_table.to_csv(args.output_dir / "daily_metrics_by_route.csv", index=False)
    comparison_table.to_csv(args.output_dir / "improvement_by_route.csv", index=False)
    summary = {
        "protocol": {
            "source_cutoff": args.source_cutoff,
            "validation_dates": VALIDATION_DATES,
            "training_rule": "each route separately; all route-local dates strictly before each validation date",
            "feature_history_rule": "strictly earlier calendar dates within the same route",
            "feature_sets": feature_sets(),
        },
        "routes": route_summaries,
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(summary), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(comparison_table.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/state_profile_bus_model_2_horizon5_experiment.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/state_profile_bus_model_2_horizon5_experiment.py
"""State-profile evaluation under Bus_model_2's short-horizon protocol.

Uses Yeonwu's arrival labels, restricted to 1..5 remaining stops.  Estimator,
log target, Huber loss, and <=3-seat sample weighting match Bus_model_2.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor

from yeonwu_peer_feature_experiment import (
    DEFAULT_ROUTES,
    ROOT,
    add_strict_prior_state_features,
    cache_observation_cutoff,
    load_embedded_modules,
    load_fresh_features,
    refresh_cache,
)


FEATURES = (
    "route_code", "snapshot_station_seq_cat", "station_seq_cat",
    "snapshot_time_sin", "snapshot_time_cos", "snapshot_day_of_week",
    "snapshot_remaining_seats", "target_stop_gap", "seats_per_remaining_stop",
    "snapshot_state_expected_seats", "snapshot_state_seat_deviation",
    "snapshot_state_low10_rate", "snapshot_state_log_count",
)
CATEGORICAL = ("route_code", "snapshot_station_seq_cat", "station_seq_cat", "snapshot_day_of_week")


def encode(train: pd.DataFrame, target: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    left, right = train.loc[:, FEATURES].copy(), target.loc[:, FEATURES].copy()
    for column in CATEGORICAL:
        categories = pd.Index(left[column].astype("string").dropna().unique())
        left[column] = pd.Categorical(left[column].astype("string"), categories=categories).codes
        right[column] = pd.Categorical(right[column].astype("string"), categories=categories).codes
    for column in set(FEATURES) - set(CATEGORICAL):
        left[column] = pd.to_numeric(left[column], errors="coerce").astype("float64")
        right[column] = pd.to_numeric(right[column], errors="coerce").astype("float64")
    return left, right


def predict_bus2(
    train: pd.DataFrame, target: pd.DataFrame, *, critical_threshold: int,
    critical_weight: float, seed: int,
) -> pd.DataFrame:
    x_train, x_target = encode(train, target)
    model = LGBMRegressor(
        objective="huber", alpha=0.9, n_estimators=300, learning_rate=0.05,
        random_state=seed, n_jobs=-1, verbosity=-1,
    )
    weights = np.where(
        train["label_seats"].to_numpy(float) <= critical_threshold,
        critical_weight, 1.0,
    )
    model.fit(
        x_train, np.log1p(train["label_seats"].to_numpy(float)),
        sample_weight=weights, categorical_feature=list(CATEGORICAL),
    )
    output = target.copy()
    output["prediction"] = np.clip(
        np.expm1(model.predict(x_target)), 0, target["capacity"].to_numpy(float)
    )
    return output


def metric_tables(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    from main_model_feature_augmentation import required_metrics

    pooled_rows, route_rows = [], []
    for partition, frame in predictions.groupby("partition", sort=False):
        pooled_rows.append({"partition": partition, "candidate": "state_profile_bus2_horizon_1_5", **required_metrics(frame)})
        for (route_id, route_name), route in frame.groupby(["route_id", "route_name"], sort=False):
            route_rows.append({"partition": partition, "route_id": route_id, "route_name": route_name, "candidate": "state_profile_bus2_horizon_1_5", **required_metrics(route)})
    pooled, by_route = pd.DataFrame(pooled_rows), pd.DataFrame(route_rows)
    columns = ["event_balanced_mae", "low_0_10_mae", "full_accuracy", "full_recall", "full_precision", "full_f1"]
    macro = by_route.groupby(["partition", "candidate"], sort=False)[columns].mean().reset_index()
    macro.insert(2, "routes", len(DEFAULT_ROUTES))
    return pooled, by_route, macro


def main() -> int:
    parser = argparse.ArgumentParser(description="State profile, Bus_model_2 1~5-stop evaluation")
    parser.add_argument("--database", type=Path, default=ROOT / "data/gbis_api_cache.sqlite3")
    parser.add_argument("--cache-dir", type=Path, default=ROOT / "data/analysis_cache/state_profile_bus2_routes")
    parser.add_argument("--featured-cache", type=Path, default=ROOT / "data/analysis_cache/state_profile_bus2_features.pkl")
    parser.add_argument("--output-dir", type=Path, default=ROOT / "analysis/state_profile_bus_model_2_horizon5_results")
    parser.add_argument("--refresh", action="store_true")
    parser.add_argument("--dotenv", type=Path, default=ROOT / ".env")
    parser.add_argument("--api-base-url", default="https://161.33.212.6")
    parser.add_argument("--api-key-env", default="GBIS_API_KEY")
    parser.add_argument("--rebuild-features", action="store_true")
    parser.add_argument("--critical-threshold", type=int, default=3)
    parser.add_argument("--critical-weight", type=float, default=5.0)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    load_embedded_modules()
    refresh_cache(args)
    cutoff = cache_observation_cutoff(args.database)
    print(f"source cutoff: {cutoff}", flush=True)
    data, route_metadata = load_fresh_features(args, cutoff)
    data = add_strict_prior_state_features(data)
    data = data.loc[data["target_stop_gap"].between(1, 5)].copy()
    data["route_id"] = data["event_id"].str.split("::", n=1).str[0]
    data["route_name"] = data["route_id"].map(DEFAULT_ROUTES)

    # Latest calendar day is incomplete at run time. Never select a model on it.
    partial_date = str(data["date"].max())
    complete = data.loc[data["date"].lt(partial_date)].copy()
    event_order = complete.groupby("event_id", sort=False)["snapshot_time"].min().sort_values()
    split_at = int(len(event_order) * 0.8)
    train_events = set(event_order.index[:split_at])
    train = complete.loc[complete["event_id"].isin(train_events)].copy()
    selection = complete.loc[~complete["event_id"].isin(train_events)].copy()
    missing = sorted(set(DEFAULT_ROUTES) - set(selection["route_id"]))
    if train.empty or selection.empty or missing:
        raise ValueError(f"invalid temporal split; missing test routes={missing}")
    print(f"[complete 80:20] train={len(train):,}; test={len(selection):,}; horizon=1..5", flush=True)
    selected_prediction = predict_bus2(
        train, selection, critical_threshold=args.critical_threshold,
        critical_weight=args.critical_weight, seed=args.seed,
    )
    selected_prediction["partition"] = "complete_time_split_80_20"

    # Separate monitor only. Refit on all complete days, then score incomplete day.
    partial = data.loc[data["date"].eq(partial_date)].copy()
    predictions = [selected_prediction]
    if not partial.empty:
        print(f"[partial monitor {partial_date}] train={len(complete):,}; test={len(partial):,}", flush=True)
        partial_prediction = predict_bus2(
            complete, partial, critical_threshold=args.critical_threshold,
            critical_weight=args.critical_weight, seed=args.seed,
        )
        partial_prediction["partition"] = "partial_day_check"
        predictions.append(partial_prediction)
    scored = pd.concat(predictions, ignore_index=True)
    pooled, by_route, macro = metric_tables(scored)

    args.output_dir.mkdir(parents=True, exist_ok=True)
    scored.to_pickle(args.output_dir / "oof_predictions.pkl")
    pooled.to_csv(args.output_dir / "metrics_pooled.csv", index=False)
    by_route.to_csv(args.output_dir / "metrics_by_route.csv", index=False)
    macro.to_csv(args.output_dir / "metrics_route_macro.csv", index=False)
    (args.output_dir / "summary.json").write_text(json.dumps({
        "source_cutoff": cutoff, "included_routes": DEFAULT_ROUTES,
        "excluded_routes": "all non-active routes; no current complete-data coverage",
        "horizon_rule": "target_stop_gap 1 through 5 inclusive",
        "selection_rule": "event-grouped temporal 80:20 split on completed calendar days",
        "partial_day_check": partial_date,
        "state_profile_rule": "route × snapshot station × 30-minute time bin, strictly earlier weekday dates only",
        "model": "Bus_model_2 LightGBM Huber, log1p target, 300 trees, learning_rate=0.05",
        "critical_weighting": {"threshold": args.critical_threshold, "weight": args.critical_weight},
        "features": list(FEATURES), "route_metadata": route_metadata,
        "pooled_metrics": pooled.to_dict(orient="records"),
        "route_macro_metrics": macro.to_dict(orient="records"),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\npooled metrics")
    print(pooled.to_string(index=False))
    print("\nroute-macro metrics")
    print(macro.to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/tminus_feasibility.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/tminus_feasibility.py
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from high_risk_feasibility import high_risk_mask
from model_feasibility import (
    PLANNING,
    FeatureSet,
    _as_records,
    build_model_table,
    build_visits,
    evaluate_final_split,
    infer_turnaround_seq,
    json_ready,
    load_data,
    nominal_capacity,
    prepare_subset,
)


ROUTE_ID = "219000013"
ROUTE_NAME = "1000"
DEFAULT_HORIZONS = (5, 10, 15, 20, 30)
TARGET_FRESHNESS_MINUTES = 5.0
ROUTE_FRESHNESS_MINUTES = 3.0
RECENT_LOOKBACK_MINUTES = 5


FIXED_3STOP = FeatureSet(
    name="fixed_3stop_target",
    numeric=(
        *PLANNING.numeric,
        "capacity",
        "upstream_load_ratio_3",
        "upstream_stop_distance_3",
        "upstream_age_minutes_3",
    ),
    categorical=(*PLANNING.categorical, "low_plate_cat"),
)


def tminus_feature_sets(horizon: int) -> tuple[FeatureSet, FeatureSet]:
    target = FeatureSet(
        name=f"tminus_{horizon}_target",
        numeric=(
            *PLANNING.numeric,
            "snapshot_capacity",
            "target_load_ratio",
            "target_stop_gap",
            "target_observation_age",
            "target_load_change_5m",
            "target_seq_change_5m",
        ),
        categorical=(
            *PLANNING.categorical,
            "snapshot_low_plate_cat",
            "target_state_cat",
        ),
    )
    route = FeatureSet(
        name=f"tminus_{horizon}_route_context",
        numeric=(
            *target.numeric,
            "lead_load_ratio",
            "lead_gap_stops",
            "lead_relative_to_target_stop",
            "lead_observation_age",
            "lead_passed_target_stop",
            "following_load_ratio",
            "following_gap_stops",
            "following_observation_age",
            "route_vehicle_count",
            "route_full_vehicle_count",
            "route_mean_load_ratio",
            "route_load_std",
        ),
        categorical=target.categorical,
    )
    return target, route


def prepare_raw_locations(
    locations: pd.DataFrame,
    turnaround_seq: int,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    raw = locations.loc[
        locations["station_seq"].notna() & locations["remaining_seats"].ge(0)
    ].copy()
    raw["station_seq"] = raw["station_seq"].astype(int)
    raw["direction"] = np.where(
        raw["station_seq"].le(turnaround_seq), "to_city", "return"
    )
    raw["capacity"] = nominal_capacity(raw["low_plate"])
    raw["load_ratio"] = (1 - raw["remaining_seats"] / raw["capacity"]).clip(0, 1)
    raw = raw.sort_values(["vehicle_id", "observed_at", "run_id"]).reset_index(drop=True)
    by_vehicle = {
        str(vehicle_id): group.reset_index(drop=True)
        for vehicle_id, group in raw.groupby("vehicle_id", sort=False)
    }
    return raw, by_vehicle


def latest_row_before(
    rows: pd.DataFrame,
    when: pd.Timestamp,
    *,
    not_before: pd.Timestamp | None = None,
) -> pd.Series | None:
    # pandas 3 may preserve the source timestamp at microsecond resolution, while
    # Timestamp.value is nanoseconds. Series.searchsorted keeps the units aligned.
    position = int(rows["observed_at"].searchsorted(when, side="right") - 1)
    if position < 0:
        return None
    row = rows.iloc[position]
    if not_before is not None and row["observed_at"] < not_before:
        return None
    return row


def vehicle_snapshot(
    by_vehicle: dict[str, pd.DataFrame],
    when: pd.Timestamp,
    freshness_minutes: float,
) -> pd.DataFrame:
    rows: list[pd.Series] = []
    for vehicle_rows in by_vehicle.values():
        row = latest_row_before(vehicle_rows, when)
        if row is None:
            continue
        age = (when - row["observed_at"]).total_seconds() / 60
        if 0 <= age <= freshness_minutes:
            copied = row.copy()
            copied["observation_age"] = age
            rows.append(copied)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows)


def nearest_vehicle_features(
    candidates: pd.DataFrame,
    target_current_seq: int,
    target_stop_seq: int,
) -> dict[str, float]:
    result = {
        "lead_load_ratio": np.nan,
        "lead_gap_stops": np.nan,
        "lead_relative_to_target_stop": np.nan,
        "lead_observation_age": np.nan,
        "lead_passed_target_stop": np.nan,
        "following_load_ratio": np.nan,
        "following_gap_stops": np.nan,
        "following_observation_age": np.nan,
    }
    lead = candidates.loc[candidates["station_seq"].gt(target_current_seq)].sort_values(
        ["station_seq", "observed_at"], ascending=[True, False]
    )
    if not lead.empty:
        row = lead.iloc[0]
        result.update(
            {
                "lead_load_ratio": float(row["load_ratio"]),
                "lead_gap_stops": float(row["station_seq"] - target_current_seq),
                "lead_relative_to_target_stop": float(
                    row["station_seq"] - target_stop_seq
                ),
                "lead_observation_age": float(row["observation_age"]),
                "lead_passed_target_stop": float(
                    row["station_seq"] >= target_stop_seq
                ),
            }
        )

    following = candidates.loc[candidates["station_seq"].lt(target_current_seq)].sort_values(
        ["station_seq", "observed_at"], ascending=[False, False]
    )
    if not following.empty:
        row = following.iloc[0]
        result.update(
            {
                "following_load_ratio": float(row["load_ratio"]),
                "following_gap_stops": float(target_current_seq - row["station_seq"]),
                "following_observation_age": float(row["observation_age"]),
            }
        )
    return result


def build_tminus_table(
    events: pd.DataFrame,
    visits: pd.DataFrame,
    by_vehicle: dict[str, pd.DataFrame],
    horizon: int,
) -> pd.DataFrame:
    trip_starts = visits.groupby("trip_id", sort=False)["first_seen"].min()
    output_rows: list[dict[str, Any]] = []

    for event in events.itertuples(index=False):
        snapshot_time = event.event_time - pd.Timedelta(minutes=horizon)
        trip_start = trip_starts.get(event.trip_id)
        target_rows = by_vehicle.get(str(event.vehicle_id))
        if target_rows is None or pd.isna(trip_start):
            continue

        target = latest_row_before(target_rows, snapshot_time, not_before=trip_start)
        if target is None:
            continue
        target_age = (snapshot_time - target["observed_at"]).total_seconds() / 60
        if not 0 <= target_age <= TARGET_FRESHNESS_MINUTES:
            continue
        if int(target["station_seq"]) > int(event.station_seq):
            continue

        previous_time = snapshot_time - pd.Timedelta(minutes=RECENT_LOOKBACK_MINUTES)
        previous = latest_row_before(target_rows, previous_time, not_before=trip_start)
        previous_age = (
            (previous_time - previous["observed_at"]).total_seconds() / 60
            if previous is not None
            else np.nan
        )
        previous_is_fresh = previous is not None and 0 <= previous_age <= TARGET_FRESHNESS_MINUTES

        snapshot = vehicle_snapshot(
            by_vehicle, snapshot_time, freshness_minutes=ROUTE_FRESHNESS_MINUTES
        )
        if snapshot.empty:
            same_direction = pd.DataFrame(
                columns=[
                    "station_seq", "observed_at", "remaining_seats",
                    "load_ratio", "observation_age",
                ]
            )
        else:
            same_direction = snapshot.loc[
                snapshot["direction"].eq(event.direction)
                & snapshot["vehicle_id"].ne(event.vehicle_id)
            ].copy()

        record = event._asdict()
        record.update(
            {
                "horizon_minutes": horizon,
                "snapshot_time": snapshot_time,
                "snapshot_capacity": float(target["capacity"]),
                "snapshot_low_plate_cat": str(int(target["low_plate"])),
                "target_state_cat": str(int(target["state_code"])),
                "target_load_ratio": float(target["load_ratio"]),
                "target_stop_gap": float(event.station_seq - target["station_seq"]),
                "target_observation_age": float(target_age),
                "target_load_change_5m": (
                    float(target["load_ratio"] - previous["load_ratio"])
                    if previous_is_fresh
                    else np.nan
                ),
                "target_seq_change_5m": (
                    float(target["station_seq"] - previous["station_seq"])
                    if previous_is_fresh
                    else np.nan
                ),
                "route_vehicle_count": float(len(same_direction) + 1),
                "route_full_vehicle_count": float(
                    same_direction["remaining_seats"].eq(0).sum()
                    + int(target["remaining_seats"] == 0)
                ),
                "route_mean_load_ratio": float(
                    pd.concat(
                        [
                            same_direction["load_ratio"],
                            pd.Series([float(target["load_ratio"])])
                        ],
                        ignore_index=True,
                    ).mean()
                ),
                "route_load_std": float(
                    pd.concat(
                        [
                            same_direction["load_ratio"],
                            pd.Series([float(target["load_ratio"])])
                        ],
                        ignore_index=True,
                    ).std(ddof=0)
                ),
            }
        )
        record.update(
            nearest_vehicle_features(
                same_direction,
                int(target["station_seq"]),
                int(event.station_seq),
            )
        )
        output_rows.append(record)

    if not output_rows:
        return pd.DataFrame(columns=[*events.columns, "horizon_minutes", "snapshot_time"])
    return pd.DataFrame(output_rows).sort_values("event_time").reset_index(drop=True)


def coverage_table(source: pd.DataFrame, snapshots: dict[int, pd.DataFrame]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for horizon, table in snapshots.items():
        for date in ["2026-08-04", "2026-08-05", "2026-08-06", "2026-08-07"]:
            source_day = source.loc[source["date"].eq(date)]
            snapshot_day = table.loc[table["date"].eq(date)]
            source_positive = int(source_day["is_full"].sum())
            snapshot_positive = int(snapshot_day["is_full"].sum())
            rows.append(
                {
                    "horizon_minutes": horizon,
                    "date": date,
                    "source_rows": int(len(source_day)),
                    "snapshot_rows": int(len(snapshot_day)),
                    "row_coverage": (
                        float(len(snapshot_day) / len(source_day)) if len(source_day) else np.nan
                    ),
                    "source_positives": source_positive,
                    "snapshot_positives": snapshot_positive,
                    "positive_coverage": (
                        float(snapshot_positive / source_positive)
                        if source_positive
                        else np.nan
                    ),
                    "median_target_observation_age": (
                        float(snapshot_day["target_observation_age"].median())
                        if len(snapshot_day)
                        else np.nan
                    ),
                    "median_target_stop_gap": (
                        float(snapshot_day["target_stop_gap"].median())
                        if len(snapshot_day)
                        else np.nan
                    ),
                    "recent_change_coverage": (
                        float(snapshot_day["target_load_change_5m"].notna().mean())
                        if len(snapshot_day)
                        else np.nan
                    ),
                    "lead_vehicle_coverage": (
                        float(snapshot_day["lead_load_ratio"].notna().mean())
                        if len(snapshot_day)
                        else np.nan
                    ),
                    "following_vehicle_coverage": (
                        float(snapshot_day["following_load_ratio"].notna().mean())
                        if len(snapshot_day)
                        else np.nan
                    ),
                }
            )
    return pd.DataFrame(rows)


def selected_metric(metrics: pd.DataFrame, feature_set: str) -> pd.Series:
    return (
        metrics.loc[metrics["feature_set"].eq(feature_set)]
        .sort_values(["selection_average_precision", "brier"], ascending=[False, True])
        .iloc[0]
    )


def make_plot(selected: pd.DataFrame, output: Path) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    colors = {
        "fixed_3stop_target": "#94a3b8",
        "target": "#2563eb",
        "route_context": "#f97316",
    }
    for family, group in selected.groupby("family", sort=False):
        group = group.sort_values("horizon_minutes")
        label = family.replace("_", " ")
        color = colors.get(family, "#334155")
        axes[0].plot(
            group["horizon_minutes"], group["average_precision"] * 100,
            marker="o", label=label, color=color,
        )
        axes[1].plot(
            group["horizon_minutes"], group["precision_at_threshold"] * 100,
            marker="o", label=label, color=color,
        )
        axes[2].plot(
            group["horizon_minutes"], group["recall_at_threshold"] * 100,
            marker="o", label=label, color=color,
        )
    axes[0].set_title("Average precision")
    axes[1].set_title("Precision at selected threshold")
    axes[2].set_title("Recall at selected threshold")
    axes[1].axhline(30, color="#dc2626", linestyle="--", linewidth=1)
    axes[2].axhline(50, color="#dc2626", linestyle="--", linewidth=1)
    for axis in axes:
        axis.set_xlabel("Minutes before arrival")
        axis.set_ylabel("Percent")
        axis.set_xticks(sorted(selected["horizon_minutes"].unique()))
        axis.grid(alpha=0.2)
    axes[0].legend()
    fig.tight_layout()
    fig.savefig(output, dpi=160)
    plt.close(fig)


def main() -> int:
    parser = argparse.ArgumentParser(description="1000번 버스 T-minus 만차 예측 검증")
    parser.add_argument("--db", type=Path, default=Path("data/gbis.sqlite3"))
    parser.add_argument(
        "--output-dir", type=Path, default=Path("analysis/tminus_results")
    )
    parser.add_argument(
        "--horizons", type=int, nargs="+", default=list(DEFAULT_HORIZONS)
    )
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    args.output_dir.mkdir(parents=True, exist_ok=True)

    locations, stations = load_data(args.db, ROUTE_ID)
    visits = build_visits(locations, stations)
    model_table, turnaround_seq = build_model_table(visits, stations)
    source = prepare_subset(model_table)
    source = source.loc[high_risk_mask(source)].copy().reset_index(drop=True)
    _, by_vehicle = prepare_raw_locations(locations, turnaround_seq)

    snapshots: dict[int, pd.DataFrame] = {}
    all_metrics: list[pd.DataFrame] = []
    selected_rows: list[dict[str, Any]] = []
    for horizon in sorted(set(args.horizons)):
        snapshot = build_tminus_table(source, visits, by_vehicle, horizon)
        snapshots[horizon] = snapshot
        target_features, route_features = tminus_feature_sets(horizon)
        metrics, _, _ = evaluate_final_split(
            snapshot,
            [FIXED_3STOP, target_features, route_features],
            args.seed,
            threshold_target_recall=0.50,
        )
        metrics.insert(0, "horizon_minutes", horizon)
        all_metrics.append(metrics)

        for feature_set, family in [
            (FIXED_3STOP.name, "fixed_3stop_target"),
            (target_features.name, "target"),
            (route_features.name, "route_context"),
        ]:
            selected = selected_metric(metrics, feature_set).to_dict()
            selected["family"] = family
            selected_rows.append(selected)

    metrics_table = pd.concat(all_metrics, ignore_index=True)
    selected = pd.DataFrame(selected_rows)
    coverage = coverage_table(source, snapshots)
    route_candidates = selected.loc[selected["family"].eq("route_context")]
    best = route_candidates.sort_values(
        ["selection_average_precision", "brier"], ascending=[False, True]
    ).iloc[0]
    dynamic_candidates = selected.loc[selected["family"].isin(["target", "route_context"])]
    best_dynamic = dynamic_candidates.sort_values(
        ["selection_average_precision", "brier"], ascending=[False, True]
    ).iloc[0]

    metrics_table.to_csv(args.output_dir / "all_metrics.csv", index=False)
    selected.to_csv(args.output_dir / "selected_by_horizon.csv", index=False)
    coverage.to_csv(args.output_dir / "snapshot_coverage.csv", index=False)
    make_plot(selected, args.output_dir / "horizon_comparison.png")

    result = {
        "route_id": ROUTE_ID,
        "route_name": ROUTE_NAME,
        "scope": "high_risk_gate",
        "horizons_minutes": sorted(snapshots),
        "snapshot_definition": {
            "anchor": "actual target-stop first-seen time minus horizon",
            "target_max_observation_age_minutes": TARGET_FRESHNESS_MINUTES,
            "route_vehicle_max_observation_age_minutes": ROUTE_FRESHNESS_MINUTES,
            "leakage_rule": "all feature observations are at or before snapshot_time",
        },
        "split": {
            "train": ["2026-08-04", "2026-08-05"],
            "calibration": ["2026-08-06"],
            "untouched_test": ["2026-08-07"],
        },
        "coverage_test": json_ready(
            _as_records(coverage.loc[coverage["date"].eq("2026-08-07")])
        ),
        "selected_by_horizon": json_ready(_as_records(selected)),
        "best_dynamic_selected_on_calibration": json_ready(best_dynamic.to_dict()),
        "best_route_context_selected_on_calibration": json_ready(best.to_dict()),
        "product_kpi": {"minimum_precision": 0.30, "minimum_recall": 0.50},
        "threshold_selection": (
            "calibration date에서 recall >= 0.50인 임계값 중 precision 최대"
        ),
        "best_route_context_passes_test_kpi": bool(
            best["precision_at_threshold"] >= 0.30
            and best["recall_at_threshold"] >= 0.50
        ),
        "best_dynamic_passes_test_kpi": bool(
            best_dynamic["precision_at_threshold"] >= 0.30
            and best_dynamic["recall_at_threshold"] >= 0.50
        ),
        "limitations": [
            "실제 도착시각으로 T-minus 기준점을 만들었으므로 운영 시 ETA 오차가 추가된다.",
            "수집 기간이 짧고 최종 테스트가 하루라 수치는 가능성 검증용이다.",
            "앞·뒤 차량은 정류장 순번 기준이며 실제 도로 거리와 차량 간 시간 간격은 아니다.",
            "3정거장 전 기준선은 예측 기준시각이 달라 T-minus 모델과 완전히 동일한 제품 조건은 아니다.",
        ],
    }
    (args.output_dir / "summary.json").write_text(
        json.dumps(json_ready(result), ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps(json_ready(result), ensure_ascii=False, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `analysis/uncertainty_heads.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/uncertainty_heads.py
"""Shared prediction-interval and low-seat probability heads.

The point estimator remains model-specific.  These heads make uncertainty
comparable across models by using the same q05/q50/q95 LightGBM policy,
event-balanced split-conformal calibration, and P(Y <= threshold) classifier.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder


DEFAULT_ALPHA = 0.10
DEFAULT_LOW_THRESHOLD = 3
QUANTILES = (0.05, 0.50, 0.95)
LOWER_QUANTILE = 0.10
TARGET_KINDS = ("absolute", "delta", "delta_per_stop")


def event_weights(data: pd.DataFrame) -> np.ndarray:
    """Give every arrival event total weight one, independent of snapshots."""
    counts = data.groupby("event_id", sort=False)["event_id"].transform("size")
    return 1.0 / counts.to_numpy(dtype=float)


def weighted_quantile(values: np.ndarray, weights: np.ndarray, q: float) -> float:
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        raise ValueError("weighted quantile has no finite positive-weight values")
    values, weights = values[valid], weights[valid]
    order = np.argsort(values, kind="stable")
    cumulative = np.cumsum(weights[order])
    position = min(float(q), 1.0) * cumulative[-1]
    index = min(int(np.searchsorted(cumulative, position, side="left")), len(order) - 1)
    return float(values[order[index]])


def _preprocessor(numeric: tuple[str, ...], categorical: tuple[str, ...]) -> ColumnTransformer:
    return ColumnTransformer(
        [
            ("numeric", SimpleImputer(strategy="median", add_indicator=True), list(numeric)),
            (
                "categorical",
                Pipeline(
                    [
                        ("impute", SimpleImputer(strategy="most_frequent")),
                        (
                            "ordinal",
                            OrdinalEncoder(
                                handle_unknown="use_encoded_value",
                                unknown_value=-1,
                                encoded_missing_value=-1,
                            ),
                        ),
                    ]
                ),
                list(categorical),
            ),
        ],
        remainder="drop",
    )


def _model_pipeline(
    numeric: tuple[str, ...], categorical: tuple[str, ...], estimator: Any
) -> Pipeline:
    return Pipeline([("features", _preprocessor(numeric, categorical)), ("model", estimator)])


def conformal_groups(data: pd.DataFrame) -> pd.Series:
    seat_band = pd.cut(
        data["snapshot_remaining_seats"],
        bins=[-np.inf, 5, 10, 20, np.inf],
        labels=["seats_0_5", "seats_6_10", "seats_11_20", "seats_21_plus"],
    )
    stop_band = pd.cut(
        data["target_stop_gap"],
        bins=[0, 2, 5, 10, np.inf],
        labels=["stops_1_2", "stops_3_5", "stops_6_10", "stops_11_plus"],
    )
    return seat_band.astype("string") + "|" + stop_band.astype("string")


def _clip(values: np.ndarray, data: pd.DataFrame) -> np.ndarray:
    return np.clip(values, 0.0, data["capacity"].to_numpy(dtype=float))


@dataclass
class UncertaintyBundle:
    quantile_models: dict[float, Pipeline]
    lower_bound_model: Pipeline
    low_seat_model: Pipeline
    probability_calibrator: LogisticRegression | None
    constant_probability: float | None
    conformal_adjustments: dict[str, float]
    global_adjustment: float
    lower_conformal_adjustments: dict[str, float]
    lower_global_adjustment: float
    numeric_features: tuple[str, ...]
    categorical_features: tuple[str, ...]
    alpha: float
    low_threshold: int
    target_kind: str


def uncertainty_bundle_from_dict(payload: dict[str, Any]) -> UncertaintyBundle:
    """Restore the portable dictionary stored in the state-profile artifact."""
    return UncertaintyBundle(**payload)


def encode_uncertainty_target(data: pd.DataFrame, target_kind: str) -> np.ndarray:
    if target_kind not in TARGET_KINDS:
        raise ValueError(f"unknown uncertainty target: {target_kind}")
    target = data["label_seats"].to_numpy(dtype=float)
    if target_kind == "absolute":
        return target
    delta = target - data["snapshot_remaining_seats"].to_numpy(dtype=float)
    if target_kind == "delta":
        return delta
    return delta / np.maximum(data["target_stop_gap"].to_numpy(dtype=float), 1.0)


def decode_uncertainty_target(
    prediction: np.ndarray, data: pd.DataFrame, target_kind: str
) -> np.ndarray:
    prediction = np.asarray(prediction, dtype=float)
    if target_kind == "absolute":
        decoded = prediction
    elif target_kind == "delta":
        decoded = data["snapshot_remaining_seats"].to_numpy(dtype=float) + prediction
    elif target_kind == "delta_per_stop":
        decoded = (
            data["snapshot_remaining_seats"].to_numpy(dtype=float)
            + prediction * np.maximum(data["target_stop_gap"].to_numpy(dtype=float), 1.0)
        )
    else:
        raise ValueError(f"unknown uncertainty target: {target_kind}")
    return _clip(decoded, data)


def fit_uncertainty_bundle(
    train: pd.DataFrame,
    calibration: pd.DataFrame,
    *,
    numeric: tuple[str, ...],
    categorical: tuple[str, ...],
    seed: int = 42,
    alpha: float = DEFAULT_ALPHA,
    low_threshold: int = DEFAULT_LOW_THRESHOLD,
    n_estimators: int = 400,
    min_group_events: int = 30,
    target_kind: str = "absolute",
) -> UncertaintyBundle:
    if train.empty or calibration.empty:
        raise ValueError("proper training and calibration data must both be non-empty")
    features = [*numeric, *categorical]
    train_weight, cal_weight = event_weights(train), event_weights(calibration)
    encoded_target = encode_uncertainty_target(train, target_kind)
    quantile_models: dict[float, Pipeline] = {}
    cal_columns: list[np.ndarray] = []
    for offset, quantile in enumerate(QUANTILES):
        model = _model_pipeline(
            numeric,
            categorical,
            LGBMRegressor(
                objective="quantile",
                alpha=quantile,
                n_estimators=n_estimators,
                learning_rate=0.04,
                num_leaves=31,
                min_child_samples=20,
                subsample=0.85,
                subsample_freq=1,
                colsample_bytree=0.85,
                random_state=seed + offset,
                n_jobs=-1,
                verbosity=-1,
            ),
        )
        model.fit(train[features], encoded_target, model__sample_weight=train_weight)
        quantile_models[quantile] = model
        cal_columns.append(
            decode_uncertainty_target(
                model.predict(calibration[features]), calibration, target_kind
            )
        )

    lower_bound_model = _model_pipeline(
        numeric,
        categorical,
        LGBMRegressor(
            objective="quantile",
            alpha=LOWER_QUANTILE,
            n_estimators=n_estimators,
            learning_rate=0.04,
            num_leaves=31,
            min_child_samples=20,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.85,
            random_state=seed + 20,
            n_jobs=-1,
            verbosity=-1,
        ),
    )
    lower_bound_model.fit(
        train[features], encoded_target, model__sample_weight=train_weight
    )
    cal_lower_raw = decode_uncertainty_target(
        lower_bound_model.predict(calibration[features]), calibration, target_kind
    )

    cal_matrix = np.sort(np.column_stack(cal_columns), axis=1)
    cal_y = calibration["label_seats"].to_numpy(dtype=float)
    conformity = np.maximum.reduce(
        [cal_matrix[:, 0] - cal_y, cal_y - cal_matrix[:, 2], np.zeros(len(calibration))]
    )
    # Weighted split conformal.  Group-specific values fall back to global when
    # their calibration support is too small.
    global_adjustment = weighted_quantile(conformity, cal_weight, 1.0 - alpha)
    groups = conformal_groups(calibration)
    adjustments: dict[str, float] = {}
    for group_name in groups.dropna().unique():
        mask = groups.eq(group_name).to_numpy()
        if calibration.loc[mask, "event_id"].nunique() >= min_group_events:
            adjustments[str(group_name)] = weighted_quantile(
                conformity[mask], cal_weight[mask], 1.0 - alpha
            )

    # For a lower prediction bound L, the nonconformity score is L-Y.  Unlike
    # the two-sided expansion score it may be negative; retaining that sign is
    # required for a sharp, valid one-sided split-conformal bound.
    lower_conformity = cal_lower_raw - cal_y
    lower_global_adjustment = weighted_quantile(
        lower_conformity, cal_weight, 1.0 - alpha
    )
    lower_adjustments: dict[str, float] = {}
    for group_name in groups.dropna().unique():
        mask = groups.eq(group_name).to_numpy()
        if calibration.loc[mask, "event_id"].nunique() >= min_group_events:
            lower_adjustments[str(group_name)] = weighted_quantile(
                lower_conformity[mask], cal_weight[mask], 1.0 - alpha
            )

    classifier = _model_pipeline(
        numeric,
        categorical,
        LGBMClassifier(
            objective="binary",
            n_estimators=n_estimators,
            learning_rate=0.04,
            num_leaves=31,
            min_child_samples=20,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.85,
            random_state=seed + 10,
            n_jobs=-1,
            verbosity=-1,
        ),
    )
    train_binary = train["label_seats"].le(low_threshold).astype(int).to_numpy()
    if np.unique(train_binary).size < 2:
        raise ValueError("proper training split has only one low-seat class")
    classifier.fit(train[features], train_binary, model__sample_weight=train_weight)
    raw_cal = np.clip(classifier.predict_proba(calibration[features])[:, 1], 1e-6, 1 - 1e-6)
    cal_binary = calibration["label_seats"].le(low_threshold).astype(int).to_numpy()
    calibrator: LogisticRegression | None = None
    constant_probability: float | None = None
    if np.unique(cal_binary).size == 2:
        cal_logit = np.log(raw_cal / (1.0 - raw_cal)).reshape(-1, 1)
        calibrator = LogisticRegression(C=1000.0, max_iter=1000, random_state=seed + 11)
        calibrator.fit(cal_logit, cal_binary, sample_weight=cal_weight)
    else:
        constant_probability = float(np.average(cal_binary, weights=cal_weight))

    return UncertaintyBundle(
        quantile_models=quantile_models,
        lower_bound_model=lower_bound_model,
        low_seat_model=classifier,
        probability_calibrator=calibrator,
        constant_probability=constant_probability,
        conformal_adjustments=adjustments,
        global_adjustment=global_adjustment,
        lower_conformal_adjustments=lower_adjustments,
        lower_global_adjustment=lower_global_adjustment,
        numeric_features=numeric,
        categorical_features=categorical,
        alpha=alpha,
        low_threshold=low_threshold,
        target_kind=target_kind,
    )


def predict_uncertainty(bundle: UncertaintyBundle, data: pd.DataFrame) -> pd.DataFrame:
    features = [*bundle.numeric_features, *bundle.categorical_features]
    matrix = np.sort(
        np.column_stack(
            [
                decode_uncertainty_target(
                    bundle.quantile_models[q].predict(data[features]),
                    data,
                    bundle.target_kind,
                )
                for q in QUANTILES
            ]
        ),
        axis=1,
    )
    adjustment = (
        conformal_groups(data)
        .map(bundle.conformal_adjustments)
        .fillna(bundle.global_adjustment)
        .to_numpy(dtype=float)
    )
    lower = _clip(matrix[:, 0] - adjustment, data)
    upper = _clip(matrix[:, 2] + adjustment, data)
    lower_raw = decode_uncertainty_target(
        bundle.lower_bound_model.predict(data[features]), data, bundle.target_kind
    )
    lower_adjustment = (
        conformal_groups(data)
        .map(bundle.lower_conformal_adjustments)
        .fillna(bundle.lower_global_adjustment)
        .to_numpy(dtype=float)
    )
    lower_bound = _clip(lower_raw - lower_adjustment, data)
    raw_probability = np.clip(
        bundle.low_seat_model.predict_proba(data[features])[:, 1], 1e-6, 1 - 1e-6
    )
    if bundle.probability_calibrator is not None:
        logits = np.log(raw_probability / (1.0 - raw_probability)).reshape(-1, 1)
        probability = bundle.probability_calibrator.predict_proba(logits)[:, 1]
    else:
        probability = np.full(len(data), float(bundle.constant_probability))
    return pd.DataFrame(
        {
            "quantile_05": matrix[:, 0],
            "quantile_50": matrix[:, 1],
            "quantile_95": matrix[:, 2],
            "interval_lower_90": lower,
            "interval_upper_90": upper,
            "lower_bound_90": lower_bound,
            "low_seat_probability": probability,
        },
        index=data.index,
    )


def uncertainty_metrics(data: pd.DataFrame) -> dict[str, float | int]:
    y = data["label_seats"].to_numpy(dtype=float)
    lower = data["interval_lower_90"].to_numpy(dtype=float)
    upper = data["interval_upper_90"].to_numpy(dtype=float)
    probability = data["low_seat_probability"].to_numpy(dtype=float)
    weights = event_weights(data)
    covered = (lower <= y) & (y <= upper)
    width = upper - lower
    alpha = DEFAULT_ALPHA
    score = width.copy()
    below, above = y < lower, y > upper
    score[below] += (2.0 / alpha) * (lower[below] - y[below])
    score[above] += (2.0 / alpha) * (y[above] - upper[above])
    binary = y <= DEFAULT_LOW_THRESHOLD
    lower_bound = data["lower_bound_90"].to_numpy(dtype=float)
    lower_covered = y >= lower_bound
    lower_error = y - lower_bound
    lower_pinball = np.maximum(
        DEFAULT_ALPHA * lower_error,
        (1.0 - DEFAULT_ALPHA) * (-lower_error),
    )
    point_distance = np.maximum(
        data["prediction"].to_numpy(dtype=float) - lower_bound, 0.0
    )
    return {
        "interval_90_coverage": float(np.average(covered, weights=weights)),
        "interval_90_mean_width": float(np.average(width, weights=weights)),
        "interval_90_score": float(np.average(score, weights=weights)),
        "low_3_brier": float(np.average((probability - binary) ** 2, weights=weights)),
        "lower_bound_90_coverage": float(np.average(lower_covered, weights=weights)),
        "lower_bound_90_mean_point_distance": float(np.average(point_distance, weights=weights)),
        "lower_bound_90_pinball": float(np.average(lower_pinball, weights=weights)),
        "point_below_lower_bound_rate": float(
            np.average(data["prediction"].to_numpy(dtype=float) < lower_bound, weights=weights)
        ),
    }


def reliability_table(data: pd.DataFrame, bins: int = 10) -> pd.DataFrame:
    probability = data["low_seat_probability"].to_numpy(dtype=float)
    actual = data["label_seats"].le(DEFAULT_LOW_THRESHOLD).to_numpy(dtype=float)
    weights = event_weights(data)
    index = np.minimum((np.clip(probability, 0, 1) * bins).astype(int), bins - 1)
    rows: list[dict[str, float | int]] = []
    for bin_index in range(bins):
        mask = index == bin_index
        if not mask.any():
            continue
        rows.append(
            {
                "probability_bin": f"{bin_index / bins:.1f}-{(bin_index + 1) / bins:.1f}",
                "rows": int(mask.sum()),
                "events": int(data.loc[mask, "event_id"].nunique()),
                "mean_predicted_probability": float(np.average(probability[mask], weights=weights[mask])),
                "actual_low_3_rate": float(np.average(actual[mask], weights=weights[mask])),
            }
        )
    return pd.DataFrame(rows)


### `analysis/yeonwu_peer_feature_experiment.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/yeonwu_peer_feature_experiment.py
"""Pooled, leak-free peer-feature experiment for Yeonwu arrival-seat model.

Reuses exact Yeonwu source bundle embedded in its standalone Colab notebook.
It deliberately does not copy peer notebooks' unsafe row-lag or full-data mean
features.  All state benchmarks use only strictly earlier calendar dates.
"""
from __future__ import annotations

import argparse
import base64
import hashlib
import importlib.abc
import importlib.util
import io
import json
import os
import shutil
import sqlite3
import sys
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


ROOT = Path(__file__).resolve().parents[1]
NOTEBOOK = ROOT / "colab/yeonwu/arrival_seat_model_training.ipynb"
DEFAULT_ROUTES = {
    "219000013": "1000", "222000074": "1100", "219000016": "1200",
    "218000010": "1500", "222000075": "2000", "200000104": "3000",
}


def load_embedded_modules() -> None:
    """Load exact Yeonwu analysis modules from notebook without checkout drift."""
    notebook = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
    cell = next(
        "".join(item["source"])
        for item in notebook["cells"]
        if item.get("cell_type") == "code"
        and "SOURCE_BUNDLE_B64" in "".join(item["source"])
    )
    encoded = cell.split("SOURCE_BUNDLE_B64 = ", 1)[1].split("\n", 1)[0]
    bundle = base64.b64decode(encoded.strip().strip('"'))
    with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
        modules = {name: archive.read(name) for name in archive.namelist()}

    class Finder(importlib.abc.MetaPathFinder, importlib.abc.Loader):
        def path_for(self, fullname: str) -> str | None:
            path = fullname.replace(".", "/")
            choices = (
                (f"{path}/__init__.py", f"{path}.py")
                if fullname.startswith("gbis_client")
                else (f"analysis/{path}.py",)
            )
            return next((item for item in choices if item in modules), None)

        def find_spec(self, fullname, path=None, target=None):  # type: ignore[no-untyped-def]
            filename = self.path_for(fullname)
            return None if filename is None else importlib.util.spec_from_loader(
                fullname, self, is_package=filename.endswith("/__init__.py")
            )

        def create_module(self, spec):  # type: ignore[no-untyped-def]
            return None

        def exec_module(self, module):  # type: ignore[no-untyped-def]
            filename = self.path_for(module.__name__)
            assert filename is not None
            module.__file__ = f"<yeonwu-embedded:{filename}>"
            if filename.endswith("/__init__.py"):
                module.__path__ = []
            exec(compile(modules[filename], module.__file__, "exec"), module.__dict__)

    sys.meta_path.insert(0, Finder())


def cache_observation_cutoff(database: Path) -> str:
    with sqlite3.connect(f"file:{database.resolve()}?mode=ro", uri=True) as conn:
        value = conn.execute("SELECT MAX(observed_at) FROM location_history").fetchone()[0]
    if value is None:
        raise ValueError(f"no history in {database}")
    return str(value)


def cache_fingerprint(database: Path, cutoff: str) -> str:
    stat = database.stat()
    payload = f"{database.resolve()}|{stat.st_size}|{stat.st_mtime_ns}|{cutoff}"
    return hashlib.sha256(payload.encode()).hexdigest()


def refresh_cache(args: argparse.Namespace) -> None:
    if not args.refresh:
        return
    # Project .env also contains comma-separated server-side key lists.  Those
    # are valid application config but not valid zsh assignments, so parse only
    # two scalar client settings instead of asking callers to `source .env`.
    if args.dotenv.is_file():
        for raw in args.dotenv.read_text(encoding="utf-8").splitlines():
            if raw.startswith("GBIS_API_KEY=") and not os.environ.get(args.api_key_env):
                os.environ[args.api_key_env] = raw.split("=", 1)[1].strip().strip("\"'")
            elif raw.startswith("GBIS_API_BASE_URL=") and args.api_base_url == "https://161.33.212.6":
                args.api_base_url = raw.split("=", 1)[1].strip().strip("\"'")
    api_key = os.environ.get(args.api_key_env)
    if not api_key:
        raise RuntimeError(f"--refresh requires environment variable {args.api_key_env}")
    from gbis_client import GBISApiCache

    print("[refresh] routes", flush=True)
    cache = GBISApiCache(
        base_url=args.api_base_url, api_key=api_key, cache_path=args.database
    )
    try:
        cache.refresh_routes()
        for position, route_id in enumerate(DEFAULT_ROUTES, start=1):
            print(f"[refresh {position}/6] {DEFAULT_ROUTES[route_id]}", flush=True)
            cache.refresh_stations(route_id)
            cache.refresh_full_history(route_id)
        print("[refresh] latest locations", flush=True)
        cache.refresh_latest()
    finally:
        cache.close()


def rebuild_feature_cache(args: argparse.Namespace, cutoff: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    from pooled_main_model_overfit_ablation import prepare_pooled_data
    from route_specific_feature_experiment import build_route_snapshots, cache_paths

    route_metadata: dict[str, Any] = {}
    for route_id in DEFAULT_ROUTES:
        snapshots, flows, metadata = build_route_snapshots(
            args.database, route_id, source_cutoff=cutoff
        )
        snapshot_path, flow_path, metadata_path = cache_paths(args.cache_dir, route_id)
        args.cache_dir.mkdir(parents=True, exist_ok=True)
        snapshots.to_pickle(snapshot_path)
        flows.to_pickle(flow_path)
        metadata_path.write_text(json.dumps(metadata, ensure_ascii=False), encoding="utf-8")
        route_metadata[route_id] = metadata
    pooled, prepared_metadata = prepare_pooled_data(args.cache_dir, source_cutoff=cutoff)
    pooled.attrs.update(source_cutoff=cutoff, route_metadata=prepared_metadata)
    pooled.to_pickle(args.featured_cache)
    return pooled, prepared_metadata


def load_fresh_features(args: argparse.Namespace, cutoff: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    # Any source-cache refresh/revision forces every dependent cache rebuild.
    expected_fingerprint = cache_fingerprint(args.database, cutoff)
    metadata_path = args.featured_cache.with_suffix(".metadata.json")
    reusable = args.featured_cache.is_file() and metadata_path.is_file() and not args.rebuild_features
    if reusable:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        reusable = (
            metadata.get("source_cutoff") == cutoff
            and metadata.get("source_fingerprint") == expected_fingerprint
        )
    if reusable:
        data = pd.read_pickle(args.featured_cache)
        return data, data.attrs["route_metadata"]
    data, route_metadata = rebuild_feature_cache(args, cutoff)
    metadata_path.write_text(json.dumps({
        "source_cutoff": cutoff, "source_fingerprint": expected_fingerprint,
        "route_metadata": route_metadata,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    return data, route_metadata


STATE_GROUP = ["route_code", "direction", "snapshot_station_seq_cat", "snapshot_time_bin_30"]
STATE_FALLBACK = ["route_code", "direction", "snapshot_time_bin_30"]


def _state_values(history: pd.DataFrame, target: pd.DataFrame) -> pd.DataFrame:
    """Historical occupancy at current station. History must precede target date."""
    output = pd.DataFrame(index=target.index)
    value = "snapshot_remaining_seats"
    global_mean = float(history[value].mean()) if len(history) else float(target[value].median())
    global_rate = float(history[value].le(10).mean()) if len(history) else 0.0
    for keys, suffix in ((STATE_GROUP, "cell"), (STATE_FALLBACK, "fallback")):
        stats = history.groupby(keys, observed=True)[value].agg(["mean", "count"])
        low = history.assign(_low=history[value].le(10).astype(float)).groupby(keys, observed=True)["_low"].mean()
        lookup = target.set_index(keys).index
        output[f"_state_mean_{suffix}"] = stats["mean"].reindex(lookup).to_numpy()
        output[f"_state_count_{suffix}"] = stats["count"].reindex(lookup).to_numpy()
        output[f"_state_low_{suffix}"] = low.reindex(lookup).to_numpy()
    mean = output["_state_mean_cell"].fillna(output["_state_mean_fallback"]).fillna(global_mean)
    count = output["_state_count_cell"].fillna(output["_state_count_fallback"]).fillna(0.0)
    low = output["_state_low_cell"].fillna(output["_state_low_fallback"]).fillna(global_rate)
    output["snapshot_state_expected_seats"] = mean
    output["snapshot_state_seat_deviation"] = target[value].to_numpy(float) - mean.to_numpy(float)
    output["snapshot_state_low10_rate"] = low
    output["snapshot_state_log_count"] = np.log1p(count.to_numpy(float))
    return output[["snapshot_state_expected_seats", "snapshot_state_seat_deviation", "snapshot_state_low10_rate", "snapshot_state_log_count"]]


def add_strict_prior_state_features(data: pd.DataFrame) -> pd.DataFrame:
    """Cross-fit by date: no current/future date can define its own state profile."""
    output = data.copy()
    columns = ["snapshot_state_expected_seats", "snapshot_state_seat_deviation", "snapshot_state_low10_rate", "snapshot_state_log_count"]
    output[columns] = np.nan
    dates = sorted(output["date"].unique())
    weekdays = pd.to_datetime(output["date"]).dt.dayofweek.lt(5)
    for date in dates:
        mask = output["date"].eq(date)
        history = output.loc[output["date"].lt(date) & weekdays]
        output.loc[mask, columns] = _state_values(history, output.loc[mask]).to_numpy()
    return output


INTERACTION_COLUMNS = (
    "snapshot_seats_x_gap", "target_stop_gap_sq", "recent_change_x_gap",
)
STATE_COLUMNS = (
    "snapshot_state_expected_seats", "snapshot_state_seat_deviation",
    "snapshot_state_low10_rate", "snapshot_state_log_count",
)


def add_peer_features(data: pd.DataFrame) -> pd.DataFrame:
    output = add_strict_prior_state_features(data)
    gap = output["target_stop_gap"].to_numpy(float)
    output["snapshot_seats_x_gap"] = output["snapshot_remaining_seats"].to_numpy(float) * gap
    output["target_stop_gap_sq"] = gap ** 2
    output["recent_change_x_gap"] = output["seat_delta_previous_stop"].fillna(0).to_numpy(float) * gap
    return output


def candidates() -> dict[str, Any]:
    from model_feasibility import FeatureSet
    from pooled_main_model_overfit_ablation import pooled_candidates

    baseline, _ = pooled_candidates()["pooled_40_legacy_ceiling_features"]
    return {
        "baseline_exact": baseline,
        "state_profile": FeatureSet("state_profile", (*baseline.numeric, *STATE_COLUMNS), baseline.categorical),
        "peer_interactions": FeatureSet("peer_interactions", (*baseline.numeric, *INTERACTION_COLUMNS), baseline.categorical),
        "state_plus_interactions": FeatureSet("state_plus_interactions", (*baseline.numeric, *STATE_COLUMNS, *INTERACTION_COLUMNS), baseline.categorical),
    }


def score_partitions(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    from main_model_feature_augmentation import required_metrics
    from latest_main_model_feature_recheck import LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE

    parts = {
        "complete_selection": predictions.date.isin(LATEST_COMPLETE_DATES),
        "partial_day_check": predictions.date.eq(LATEST_PARTIAL_DATE),
        "all_oof": predictions.date.isin((*LATEST_COMPLETE_DATES, LATEST_PARTIAL_DATE)),
    }
    pooled, by_route = [], []
    for partition, mask in parts.items():
        for candidate, frame in predictions.loc[mask].groupby("candidate", sort=False):
            pooled.append({"partition": partition, "candidate": candidate, **required_metrics(frame)})
        for (route_id, route_name, candidate), frame in predictions.loc[mask].groupby(["route_id", "route_name", "candidate"], sort=False):
            by_route.append({"partition": partition, "route_id": route_id, "route_name": route_name, "candidate": candidate, **required_metrics(frame)})
    pooled_df, route_df = pd.DataFrame(pooled), pd.DataFrame(by_route)
    metric_cols = ["event_balanced_mae", "low_0_10_mae", "full_accuracy", "full_recall", "full_precision", "full_f1"]
    macro = route_df.groupby(["partition", "candidate"], sort=False)[metric_cols].mean().reset_index()
    macro.insert(2, "routes", len(DEFAULT_ROUTES))
    return pooled_df, route_df, macro


def main() -> int:
    parser = argparse.ArgumentParser(description="Yeonwu pooled peer-feature experiment; strict-prior only")
    parser.add_argument("--database", type=Path, default=ROOT / "data/gbis_api_cache.sqlite3")
    parser.add_argument("--cache-dir", type=Path, default=ROOT / "data/analysis_cache/peer_feature_routes")
    parser.add_argument("--featured-cache", type=Path, default=ROOT / "data/analysis_cache/yeonwu_peer_features.pkl")
    parser.add_argument("--output-dir", type=Path, default=ROOT / "analysis/yeonwu_peer_feature_results")
    parser.add_argument("--refresh", action="store_true", help="safe incremental GBIS source refresh before rebuilding features")
    parser.add_argument("--dotenv", type=Path, default=ROOT / ".env")
    parser.add_argument("--api-base-url", default="https://161.33.212.6")
    parser.add_argument("--api-key-env", default="GBIS_API_KEY")
    parser.add_argument("--rebuild-features", action="store_true")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    for path in (args.database, NOTEBOOK):
        if not path.is_file():
            raise FileNotFoundError(path)
    load_embedded_modules()
    refresh_cache(args)
    cutoff = cache_observation_cutoff(args.database)
    print(f"source cutoff: {cutoff}")
    data, route_metadata = load_fresh_features(args, cutoff)
    data = add_peer_features(data)

    from latest_main_model_overfit_ablation import apply_candidate, train_schema
    from pooled_main_model_overfit_ablation import rolling_folds

    folds = rolling_folds(data)
    frames = []
    for name, features in candidates().items():
        print(f"[{name}] exact HGB/ExtraTrees/LightGBM ensemble", flush=True)
        oof = train_schema(features.name, features, folds, seed=args.seed)
        prediction, _ = apply_candidate(name, features, False, oof)
        prediction["route_id"] = prediction["event_id"].str.split("::", n=1).str[0]
        prediction["route_name"] = prediction["route_id"].map(DEFAULT_ROUTES)
        frames.append(prediction)
    predictions = pd.concat(frames, ignore_index=True)
    pooled, by_route, macro = score_partitions(predictions)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    predictions.to_pickle(args.output_dir / "oof_predictions.pkl")
    pooled.to_csv(args.output_dir / "metrics_pooled.csv", index=False)
    by_route.to_csv(args.output_dir / "metrics_by_route.csv", index=False)
    macro.to_csv(args.output_dir / "metrics_route_macro.csv", index=False)
    (args.output_dir / "summary.json").write_text(json.dumps({
        "source_cutoff": cutoff, "included_routes": DEFAULT_ROUTES,
        "excluded_routes": "all non-active routes; no current complete-data coverage",
        "complete_selection_dates": ["2026-08-11", "2026-08-12"],
        "partial_day_check": "2026-08-13; not model-selection input",
        "strict_prior_rule": "state profile uses only earlier weekday calendar dates, never same/future date",
        "existing_yeonwu_features_not_duplicated": ["recent station-visit change", "delta targets", "seats per remaining stop"],
        "candidate_features": {name: list(item.columns) for name, item in candidates().items()},
        "route_metadata": route_metadata,
        "pooled_metrics": pooled.to_dict(orient="records"),
        "route_macro_metrics": macro.to_dict(orient="records"),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\ncomplete-day pooled metrics")
    print(pooled.loc[pooled.partition.eq("complete_selection")].to_string(index=False))
    print("\ncomplete-day route-macro metrics")
    print(macro.loc[macro.partition.eq("complete_selection")].to_string(index=False))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


### `gbis_client/__init__.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/gbis_client/__init__.py
"""GBIS 읽기 전용 API의 로컬 SQLite 캐시 클라이언트."""

from .cache import GBISApiCache, GBISClientError

__all__ = ["GBISApiCache", "GBISClientError"]


### `gbis_client/cache.py`


In [ ]:
%%writefile /content/queue_boarding_runtime/gbis_client/cache.py
from __future__ import annotations

import os
import sqlite3
import time
from contextlib import contextmanager
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Callable, Iterator, Mapping

import httpx
import pandas as pd


DEFAULT_CACHE_PATH = Path("data/gbis_api_cache.sqlite3")


SCHEMA = """
PRAGMA journal_mode = WAL;
PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS cache_metadata (
    resource TEXT PRIMARY KEY,
    refreshed_at_utc TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS routes (
    route_id TEXT PRIMARY KEY,
    station_count INTEGER NOT NULL,
    observation_count INTEGER NOT NULL,
    first_collected_at TEXT,
    last_collected_at TEXT,
    cached_at_utc TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS route_stations (
    route_id TEXT NOT NULL,
    station_id TEXT NOT NULL,
    station_seq INTEGER NOT NULL,
    station_name TEXT,
    mobile_no TEXT,
    region_name TEXT,
    x REAL,
    y REAL,
    center_yn TEXT,
    synced_at_kst TEXT,
    cached_at_utc TEXT NOT NULL,
    PRIMARY KEY (route_id, station_seq)
);

CREATE INDEX IF NOT EXISTS idx_api_cache_stations_id
    ON route_stations(station_id);

CREATE TABLE IF NOT EXISTS latest_locations (
    route_id TEXT NOT NULL,
    vehicle_id TEXT NOT NULL,
    observed_at TEXT NOT NULL,
    query_time TEXT,
    plate_no TEXT,
    route_type_code INTEGER,
    station_id TEXT,
    station_seq INTEGER,
    station_name TEXT,
    remaining_seats INTEGER,
    crowded INTEGER,
    low_plate INTEGER,
    state_code INTEGER,
    tagless_code INTEGER,
    cached_at_utc TEXT NOT NULL,
    PRIMARY KEY (route_id, vehicle_id)
);

CREATE INDEX IF NOT EXISTS idx_api_cache_latest_station
    ON latest_locations(route_id, station_seq);

CREATE TABLE IF NOT EXISTS location_history (
    route_id TEXT NOT NULL,
    vehicle_id TEXT NOT NULL,
    observed_at TEXT NOT NULL,
    query_time TEXT,
    plate_no TEXT,
    route_type_code INTEGER,
    station_id TEXT,
    station_seq INTEGER,
    station_name TEXT,
    remaining_seats INTEGER,
    crowded INTEGER,
    low_plate INTEGER,
    state_code INTEGER,
    tagless_code INTEGER,
    cached_at_utc TEXT NOT NULL,
    PRIMARY KEY (route_id, observed_at, vehicle_id)
);

CREATE INDEX IF NOT EXISTS idx_api_cache_history_time
    ON location_history(observed_at);
CREATE INDEX IF NOT EXISTS idx_api_cache_history_route_time
    ON location_history(route_id, observed_at);

CREATE TABLE IF NOT EXISTS history_sync_state (
    route_id TEXT PRIMARY KEY,
    source_max_observed_at TEXT,
    refreshed_at_utc TEXT NOT NULL
);

PRAGMA user_version = 2;
"""


LOCATION_COLUMNS = (
    "observed_at",
    "query_time",
    "route_id",
    "vehicle_id",
    "plate_no",
    "route_type_code",
    "station_id",
    "station_seq",
    "station_name",
    "remaining_seats",
    "crowded",
    "low_plate",
    "state_code",
    "tagless_code",
)


class GBISClientError(RuntimeError):
    """API 호출이나 응답 처리에 실패했을 때 발생합니다."""


def _utc_now() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def _route_id(value: str) -> str:
    normalized = str(value).strip()
    if not normalized.isdigit():
        raise ValueError("route_id는 숫자 문자열이어야 합니다.")
    return normalized


def _datetime_param(value: datetime | str | None) -> str | None:
    if value is None:
        return None
    if isinstance(value, datetime):
        return value.isoformat(timespec="seconds")
    normalized = value.strip()
    if not normalized:
        raise ValueError("날짜/시간 값은 비어 있을 수 없습니다.")
    return normalized


def _parse_datetime(value: str, *, default_timezone: Any = None) -> datetime:
    try:
        parsed = datetime.fromisoformat(value)
    except ValueError as exc:
        raise ValueError(f"날짜/시간 형식이 올바르지 않습니다: {value}") from exc
    if parsed.tzinfo is None and default_timezone is not None:
        parsed = parsed.replace(tzinfo=default_timezone)
    return parsed


def _read_env_file(path: Path) -> dict[str, str]:
    if not path.is_file():
        return {}

    values: dict[str, str] = {}
    lines = path.read_text(encoding="utf-8").splitlines()
    for line_number, raw_line in enumerate(lines, 1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[7:].lstrip()
        if "=" not in line:
            raise ValueError(f"{path}:{line_number}: KEY=VALUE 형식이 아닙니다.")

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if not key:
            raise ValueError(f"{path}:{line_number}: 환경변수 이름이 비어 있습니다.")
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '"'}:
            value = value[1:-1]
        values[key] = value
    return values


def _setting(name: str, file_values: Mapping[str, str]) -> str:
    if name in os.environ:
        return os.environ[name]
    return file_values.get(name, "")


class GBISApiCache:
    """GBIS API 응답을 로컬 SQLite에 저장하고 DataFrame으로 읽습니다."""

    def __init__(
        self,
        *,
        base_url: str,
        api_key: str,
        cache_path: Path | str = DEFAULT_CACHE_PATH,
        timeout_seconds: float = 20.0,
        max_rate_limit_retries: int = 6,
        rate_limit_wait_seconds: float = 1.1,
        transport: httpx.BaseTransport | None = None,
        sleep: Callable[[float], None] = time.sleep,
    ) -> None:
        normalized_url = base_url.strip().rstrip("/")
        if not normalized_url:
            raise ValueError("base_url은 비어 있을 수 없습니다.")
        if not api_key.strip():
            raise ValueError("api_key는 비어 있을 수 없습니다.")
        if timeout_seconds <= 0:
            raise ValueError("timeout_seconds는 0보다 커야 합니다.")
        if max_rate_limit_retries < 0:
            raise ValueError("max_rate_limit_retries는 0 이상이어야 합니다.")
        if rate_limit_wait_seconds <= 0:
            raise ValueError("rate_limit_wait_seconds는 0보다 커야 합니다.")

        self.base_url = normalized_url
        self.cache_path = Path(cache_path).expanduser()
        self.max_rate_limit_retries = max_rate_limit_retries
        self.rate_limit_wait_seconds = rate_limit_wait_seconds
        self._sleep = sleep
        self._http = httpx.Client(
            base_url=self.base_url,
            headers={"Authorization": f"Bearer {api_key.strip()}"},
            timeout=timeout_seconds,
            transport=transport,
        )
        self._initialize()

    @classmethod
    def from_env(
        cls,
        *,
        env_file: Path | str = Path(".env"),
        cache_path: Path | str | None = None,
        timeout_seconds: float = 20.0,
        transport: httpx.BaseTransport | None = None,
    ) -> "GBISApiCache":
        env_path = Path(env_file).expanduser()
        file_values = _read_env_file(env_path)
        base_url = _setting("GBIS_API_BASE_URL", file_values).strip()
        api_key = _setting("GBIS_API_KEY", file_values).strip()
        if not base_url:
            raise ValueError(
                f"GBIS_API_BASE_URL이 환경변수 또는 {env_path}에 없습니다."
            )
        if not api_key:
            raise ValueError(f"GBIS_API_KEY가 환경변수 또는 {env_path}에 없습니다.")
        configured_cache_path = _setting("GBIS_API_CACHE_PATH", file_values).strip()
        resolved_cache_path = cache_path or configured_cache_path or DEFAULT_CACHE_PATH
        return cls(
            base_url=base_url,
            api_key=api_key,
            cache_path=resolved_cache_path,
            timeout_seconds=timeout_seconds,
            transport=transport,
        )

    def close(self) -> None:
        self.checkpoint()
        self._http.close()

    def __enter__(self) -> "GBISApiCache":
        return self

    def __exit__(self, *_: object) -> None:
        self.close()

    @contextmanager
    def _connect(self) -> Iterator[sqlite3.Connection]:
        self.cache_path.parent.mkdir(parents=True, exist_ok=True)
        connection = sqlite3.connect(self.cache_path, timeout=30)
        connection.row_factory = sqlite3.Row
        connection.execute("PRAGMA busy_timeout = 30000")
        connection.execute("PRAGMA foreign_keys = ON")
        try:
            yield connection
            connection.commit()
        except Exception:
            connection.rollback()
            raise
        finally:
            connection.close()

    def _initialize(self) -> None:
        with self._connect() as connection:
            connection.executescript(SCHEMA)
            route_columns = {
                row["name"]
                for row in connection.execute("PRAGMA table_info(routes)").fetchall()
            }
            if "first_collected_at" not in route_columns:
                connection.execute(
                    "ALTER TABLE routes ADD COLUMN first_collected_at TEXT"
                )

    def checkpoint(self) -> None:
        """Drive 등으로 복사하기 전에 WAL 내용을 기본 DB 파일에 반영합니다."""
        if not self.cache_path.is_file():
            return
        with sqlite3.connect(self.cache_path, timeout=30) as connection:
            connection.execute("PRAGMA busy_timeout = 30000")
            connection.execute("PRAGMA wal_checkpoint(TRUNCATE)")

    def _get(
        self,
        path: str,
        *,
        params: Mapping[str, Any] | None = None,
    ) -> dict[str, Any]:
        response: httpx.Response | None = None
        try:
            for attempt in range(self.max_rate_limit_retries + 1):
                response = self._http.get(path, params=params)
                if response.status_code != 429 or attempt == self.max_rate_limit_retries:
                    break
                retry_after = response.headers.get("Retry-After", "").strip()
                try:
                    wait_seconds = float(retry_after)
                except ValueError:
                    wait_seconds = min(
                        self.rate_limit_wait_seconds * (2**attempt),
                        30.0,
                    )
                self._sleep(max(wait_seconds, self.rate_limit_wait_seconds))
            assert response is not None
            response.raise_for_status()
        except httpx.HTTPStatusError as exc:
            detail: Any = None
            try:
                body = exc.response.json()
                if isinstance(body, dict):
                    detail = body.get("detail")
            except ValueError:
                pass
            message = f"GBIS API가 HTTP {exc.response.status_code}를 반환했습니다."
            if detail:
                message = f"{message} {detail}"
            raise GBISClientError(message) from exc
        except httpx.RequestError as exc:
            raise GBISClientError(f"GBIS API에 연결할 수 없습니다: {exc}") from exc

        try:
            payload = response.json()
        except ValueError as exc:
            raise GBISClientError("GBIS API 응답이 JSON 형식이 아닙니다.") from exc
        if not isinstance(payload, dict):
            raise GBISClientError("GBIS API 응답의 최상위 값이 객체가 아닙니다.")
        return payload

    @staticmethod
    def _items(payload: Mapping[str, Any]) -> list[dict[str, Any]]:
        items = payload.get("items")
        if not isinstance(items, list) or any(
            not isinstance(item, dict) for item in items
        ):
            raise GBISClientError("GBIS API 응답의 items 형식이 올바르지 않습니다.")
        return items

    @staticmethod
    def _mark_refreshed(
        connection: sqlite3.Connection,
        resource: str,
        refreshed_at: str,
    ) -> None:
        connection.execute(
            """
            INSERT INTO cache_metadata (resource, refreshed_at_utc)
            VALUES (?, ?)
            ON CONFLICT(resource) DO UPDATE SET
                refreshed_at_utc = excluded.refreshed_at_utc
            """,
            (resource, refreshed_at),
        )

    def refresh_routes(self) -> int:
        items = self._items(self._get("/v1/routes"))
        cached_at = _utc_now()
        rows = [
            (
                str(item["route_id"]),
                int(item.get("station_count") or 0),
                int(item.get("observation_count") or 0),
                item.get("first_collected_at"),
                item.get("last_collected_at"),
                cached_at,
            )
            for item in items
        ]
        with self._connect() as connection:
            connection.execute("DELETE FROM routes")
            connection.executemany(
                """
                INSERT INTO routes (
                    route_id, station_count, observation_count,
                    first_collected_at, last_collected_at, cached_at_utc
                ) VALUES (?, ?, ?, ?, ?, ?)
                """,
                rows,
            )
            self._mark_refreshed(connection, "routes", cached_at)
        return len(rows)

    def refresh_stations(self, route_id: str) -> int:
        normalized_route_id = _route_id(route_id)
        payload = self._get(f"/v1/routes/{normalized_route_id}/stations")
        items = self._items(payload)
        cached_at = _utc_now()
        rows = [
            (
                normalized_route_id,
                str(item["station_id"]),
                int(item["station_seq"]),
                item.get("station_name"),
                item.get("mobile_no"),
                item.get("region_name"),
                item.get("x"),
                item.get("y"),
                item.get("center_yn"),
                item.get("synced_at_kst"),
                cached_at,
            )
            for item in items
        ]
        with self._connect() as connection:
            connection.execute(
                "DELETE FROM route_stations WHERE route_id = ?",
                (normalized_route_id,),
            )
            connection.executemany(
                """
                INSERT INTO route_stations (
                    route_id, station_id, station_seq, station_name, mobile_no,
                    region_name, x, y, center_yn, synced_at_kst, cached_at_utc
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                rows,
            )
            self._mark_refreshed(
                connection, f"stations:{normalized_route_id}", cached_at
            )
        return len(rows)

    @staticmethod
    def _location_rows(
        items: list[dict[str, Any]],
        cached_at: str,
    ) -> list[tuple[Any, ...]]:
        return [
            tuple(item.get(column) for column in LOCATION_COLUMNS) + (cached_at,)
            for item in items
        ]

    def refresh_latest(self, route_id: str | None = None) -> int:
        normalized_route_id = _route_id(route_id) if route_id is not None else None
        params = {"route_id": normalized_route_id} if normalized_route_id else None
        items = self._items(self._get("/v1/locations/latest", params=params))
        cached_at = _utc_now()
        rows = self._location_rows(items, cached_at)

        with self._connect() as connection:
            if normalized_route_id:
                connection.execute(
                    "DELETE FROM latest_locations WHERE route_id = ?",
                    (normalized_route_id,),
                )
            else:
                connection.execute("DELETE FROM latest_locations")
            connection.executemany(
                """
                INSERT INTO latest_locations (
                    observed_at, query_time, route_id, vehicle_id, plate_no,
                    route_type_code, station_id, station_seq, station_name,
                    remaining_seats, crowded, low_plate, state_code, tagless_code,
                    cached_at_utc
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                rows,
            )
            resource = f"latest:{normalized_route_id or 'all'}"
            self._mark_refreshed(connection, resource, cached_at)
        return len(rows)

    def _last_history_timestamp(self, route_id: str) -> str | None:
        with self._connect() as connection:
            row = connection.execute(
                """
                SELECT source_max_observed_at
                FROM history_sync_state
                WHERE route_id = ?
                """,
                (route_id,),
            ).fetchone()
        return None if row is None else row["source_max_observed_at"]

    def refresh_history(
        self,
        route_id: str,
        *,
        from_at: datetime | str | None = None,
        to_at: datetime | str | None = None,
        page_size: int = 500,
    ) -> int:
        normalized_route_id = _route_id(route_id)
        if not 1 <= page_size <= 500:
            raise ValueError("page_size는 1~500 범위여야 합니다.")

        previous_checkpoint = self._last_history_timestamp(normalized_route_id)
        start = _datetime_param(from_at)
        if start is None:
            start = previous_checkpoint
        end = _datetime_param(to_at)
        params: dict[str, Any] = {
            "route_id": normalized_route_id,
            "limit": page_size,
        }
        if start is not None:
            params["from"] = start
        if end is not None:
            params["to"] = end

        total = 0
        max_observed_at = previous_checkpoint
        cursor: str | None = None
        seen_cursors: set[str] = set()
        while True:
            if cursor is not None:
                params["cursor"] = cursor
            payload = self._get("/v1/locations", params=params)
            items = self._items(payload)
            cached_at = _utc_now()
            rows = self._location_rows(items, cached_at)
            with self._connect() as connection:
                connection.executemany(
                    """
                    INSERT INTO location_history (
                        observed_at, query_time, route_id, vehicle_id, plate_no,
                        route_type_code, station_id, station_seq, station_name,
                        remaining_seats, crowded, low_plate, state_code, tagless_code,
                        cached_at_utc
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                    ON CONFLICT(route_id, observed_at, vehicle_id) DO UPDATE SET
                        query_time = excluded.query_time,
                        plate_no = excluded.plate_no,
                        route_type_code = excluded.route_type_code,
                        station_id = excluded.station_id,
                        station_seq = excluded.station_seq,
                        station_name = excluded.station_name,
                        remaining_seats = excluded.remaining_seats,
                        crowded = excluded.crowded,
                        low_plate = excluded.low_plate,
                        state_code = excluded.state_code,
                        tagless_code = excluded.tagless_code,
                        cached_at_utc = excluded.cached_at_utc
                    """,
                    rows,
                )
            total += len(rows)
            observed_values = [
                str(item["observed_at"])
                for item in items
                if item.get("observed_at") is not None
            ]
            if observed_values:
                page_max = max(observed_values)
                max_observed_at = max(
                    value for value in (max_observed_at, page_max) if value is not None
                )

            next_cursor = payload.get("next_cursor")
            if next_cursor is None:
                break
            if not isinstance(next_cursor, str) or not next_cursor:
                raise GBISClientError("GBIS API의 next_cursor 형식이 올바르지 않습니다.")
            if next_cursor in seen_cursors:
                raise GBISClientError("GBIS API가 동일한 cursor를 반복해서 반환했습니다.")
            seen_cursors.add(next_cursor)
            cursor = next_cursor

        with self._connect() as connection:
            refreshed_at = _utc_now()
            connection.execute(
                """
                INSERT INTO history_sync_state (
                    route_id, source_max_observed_at, refreshed_at_utc
                ) VALUES (?, ?, ?)
                ON CONFLICT(route_id) DO UPDATE SET
                    source_max_observed_at = excluded.source_max_observed_at,
                    refreshed_at_utc = excluded.refreshed_at_utc
                """,
                (normalized_route_id, max_observed_at, refreshed_at),
            )
            self._mark_refreshed(
                connection,
                f"history:{normalized_route_id}",
                refreshed_at,
            )
        return total

    def _history_start(self, route_id: str) -> str:
        checkpoint = self._last_history_timestamp(route_id)
        if checkpoint is not None:
            return checkpoint

        with self._connect() as connection:
            row = connection.execute(
                "SELECT first_collected_at FROM routes WHERE route_id = ?",
                (route_id,),
            ).fetchone()
        if row is None or row["first_collected_at"] is None:
            self.refresh_routes()
            with self._connect() as connection:
                row = connection.execute(
                    "SELECT first_collected_at FROM routes WHERE route_id = ?",
                    (route_id,),
                ).fetchone()
        if row is None:
            raise GBISClientError(f"노선 {route_id}을 서버에서 찾을 수 없습니다.")
        if row["first_collected_at"] is None:
            raise GBISClientError(
                "서버가 first_collected_at을 제공하지 않습니다. "
                "Oracle 서버의 API 코드를 먼저 업데이트하세요."
            )
        return str(row["first_collected_at"])

    def refresh_full_history(
        self,
        route_id: str,
        *,
        to_at: datetime | str | None = None,
        window_days: int = 30,
        page_size: int = 500,
    ) -> int:
        """최초에는 전체 이력을, 이후에는 마지막 성공 시각부터 동기화합니다."""
        normalized_route_id = _route_id(route_id)
        if not 1 <= window_days <= 30:
            raise ValueError("window_days는 1~30 범위여야 합니다.")

        start = _parse_datetime(self._history_start(normalized_route_id))
        end_value = _datetime_param(to_at)
        if end_value is None:
            end = datetime.now(start.tzinfo or timezone.utc)
        else:
            end = _parse_datetime(end_value, default_timezone=start.tzinfo)
        if start.tzinfo is None and end.tzinfo is not None:
            start = start.replace(tzinfo=end.tzinfo)
        if start > end:
            return 0

        total = 0
        window_start = start
        window_size = timedelta(days=window_days)
        while window_start <= end:
            window_end = min(window_start + window_size, end)
            total += self.refresh_history(
                normalized_route_id,
                from_at=window_start,
                to_at=window_end,
                page_size=page_size,
            )
            if window_end >= end:
                break
            window_start = window_end
        return total

    def refresh_all(
        self,
        route_ids: list[str] | tuple[str, ...] | None = None,
    ) -> dict[str, int]:
        counts: dict[str, int] = {"routes": self.refresh_routes()}
        if route_ids is None:
            routes = self.routes_df()
            normalized_route_ids = [
                str(row.route_id)
                for row in routes.itertuples()
                if int(row.station_count) > 0
            ]
        else:
            normalized_route_ids = [_route_id(value) for value in route_ids]

        counts["stations"] = sum(
            self.refresh_stations(route_id) for route_id in normalized_route_ids
        )
        counts["latest_locations"] = self.refresh_latest()
        return counts

    def _read_dataframe(
        self,
        query: str,
        params: tuple[Any, ...] = (),
    ) -> pd.DataFrame:
        with self._connect() as connection:
            return pd.read_sql_query(query, connection, params=params)

    def routes_df(self) -> pd.DataFrame:
        return self._read_dataframe("SELECT * FROM routes ORDER BY route_id")

    def stations_df(self, route_id: str | None = None) -> pd.DataFrame:
        if route_id is None:
            return self._read_dataframe(
                "SELECT * FROM route_stations ORDER BY route_id, station_seq"
            )
        normalized_route_id = _route_id(route_id)
        return self._read_dataframe(
            """
            SELECT * FROM route_stations
            WHERE route_id = ?
            ORDER BY station_seq
            """,
            (normalized_route_id,),
        )

    def latest_locations_df(self, route_id: str | None = None) -> pd.DataFrame:
        if route_id is None:
            return self._read_dataframe(
                """
                SELECT * FROM latest_locations
                ORDER BY route_id, station_seq, vehicle_id
                """
            )
        normalized_route_id = _route_id(route_id)
        return self._read_dataframe(
            """
            SELECT * FROM latest_locations
            WHERE route_id = ?
            ORDER BY station_seq, vehicle_id
            """,
            (normalized_route_id,),
        )

    def history_df(
        self,
        route_id: str | None = None,
        *,
        from_at: datetime | str | None = None,
        to_at: datetime | str | None = None,
    ) -> pd.DataFrame:
        conditions: list[str] = []
        params: list[Any] = []
        if route_id is not None:
            conditions.append("route_id = ?")
            params.append(_route_id(route_id))
        start = _datetime_param(from_at)
        if start is not None:
            conditions.append("observed_at >= ?")
            params.append(start)
        end = _datetime_param(to_at)
        if end is not None:
            conditions.append("observed_at <= ?")
            params.append(end)
        where = f"WHERE {' AND '.join(conditions)}" if conditions else ""
        return self._read_dataframe(
            f"""
            SELECT * FROM location_history
            {where}
            ORDER BY observed_at, route_id, vehicle_id
            """,
            tuple(params),
        )

    def cache_status_df(self) -> pd.DataFrame:
        return self._read_dataframe(
            "SELECT * FROM cache_metadata ORDER BY resource"
        )


### `analysis/frozen_development_manifest.json`


In [ ]:
%%writefile /content/queue_boarding_runtime/analysis/frozen_development_manifest.json
{
  "policy": "frozen_development_dataset",
  "adopted_at_kst": "2026-08-17",
  "source_cache": "data/gbis_api_cache.sqlite3",
  "source_cutoff": "2026-08-16T22:40:03+09:00",
  "source_fingerprint": "e07461510b382b819b3cf1a34fd7be585403e5b10674c07307cee9a2de872afb",
  "feature_cache": "data/analysis_cache/state_profile_main_features.pkl",
  "feature_metadata": "data/analysis_cache/state_profile_main_features.metadata.json",
  "sealed_partial_date": "2026-08-16",
  "included_routes": {
    "219000013": "1000",
    "222000074": "1100",
    "219000016": "1200",
    "218000010": "1500",
    "222000075": "2000",
    "200000104": "3000"
  }
}


## 2. 런타임 로드

모든 원본 소스를 기록한 뒤 import 경로를 연결합니다. 기존 학습기의 `load_embedded_modules()`는
과거 노트북 압축 번들을 읽기 위한 호환 계층이므로, 여기서는 이미 기록된 원본 모듈을 직접
사용하도록 비활성화합니다. 실제 학습 로직은 `queue_boarding_v1_training.py` 그대로입니다.


In [ ]:
sys.path.insert(0, str(RUNTIME_ROOT / 'analysis'))
sys.path.insert(0, str(RUNTIME_ROOT))

import queue_boarding_v1_training as training
training.load_embedded_modules = lambda: None
from queue_boarding_v1_inference import QueueBoardingV1

print('training source:', training.__file__)
print('inference source:', sys.modules['queue_boarding_v1_inference'].__file__)


### 선택적 재학습

정확한 재학습에는 다음 동결 개발 파일을 같은 상대 경로로 업로드해야 합니다.

- `/content/queue_boarding_runtime/data/gbis_api_cache.sqlite3`
- `/content/queue_boarding_runtime/data/analysis_cache/state_profile_main_features.pkl`
- `/content/queue_boarding_runtime/data/analysis_cache/state_profile_main_routes/`

최종 평가일 자료는 이 위치에 섞지 마세요. 재학습 결과는 등록 PKL을 덮어쓰지 않고 별도 파일로
저장합니다.


In [ ]:
RETRAIN = False
if RETRAIN:
    rebuilt = training.train_bundle()
    rebuild_path = Path('/content/queue_boarding_v1_0_0_rebuild.pkl')
    joblib.dump(rebuilt, rebuild_path, compress=3)
    print('saved:', rebuild_path)


## 3. 모델 PKL 업로드 및 검증

`queue_boarding_v1_0_0.pkl`을 Colab 세션에 업로드하세요. 메타데이터 JSON과 registry는 없어도
됩니다. 대신 아래에서 등록된 SHA-256을 직접 확인합니다.


In [ ]:
EXPECTED_ARTIFACT_SHA256 = '8ec2c8bfd356d2edd32f83ec00755dac59fd50e6577104efced64789faa12a9e'
ARTIFACT = Path('/content/queue_boarding_v1_0_0.pkl')

if not ARTIFACT.is_file():
    try:
        from google.colab import files
        uploaded = files.upload()
        if 'queue_boarding_v1_0_0.pkl' not in uploaded:
            raise FileNotFoundError('queue_boarding_v1_0_0.pkl을 업로드해야 합니다.')
    except ImportError:
        raise FileNotFoundError(f'{ARTIFACT}에 모델 PKL을 놓아주세요.')

actual_sha256 = hashlib.sha256(ARTIFACT.read_bytes()).hexdigest()
assert actual_sha256 == EXPECTED_ARTIFACT_SHA256, (actual_sha256, EXPECTED_ARTIFACT_SHA256)
model = QueueBoardingV1.load(ARTIFACT, verify_registry=False)
display({
    'model_id': model.bundle['model_id'],
    'source_cutoff': model.bundle['source_cutoff'],
    'source_fingerprint': model.bundle['source_fingerprint'],
    'boarding_policy': model.bundle['boarding_policy'],
    'required_features': len(model.bundle['required_prepared_input_features']),
})


## 4. 내장 예제로 추론

한 질의는 다음 버스 3대에 해당하는 3행으로 구성합니다. `bus_order`는 1·2·3이고 같은
`query_id`의 `queue_ahead`는 동일해야 합니다.


In [ ]:
from io import StringIO

EXAMPLE_INPUT_CSV = 'route_id,capacity,snapshot_time_sin,snapshot_time_cos,route_progress,snapshot_route_progress,x,y,snapshot_capacity,target_load_ratio,target_stop_gap,snapshot_remaining_seats,seat_delta_previous_stop,seat_change_per_stop,rolling_seat_change_per_stop_3,minutes_since_previous_stop,rolling_minutes_per_stop_3,estimated_minutes_to_arrival,projected_arrival_seats,seats_per_remaining_stop,load_gap_interaction,currently_low_5,currently_low_10,observed_ceiling_capacity,observed_ceiling_load_ratio,observed_ceiling_load_gap,target_low_10_rate,target_low_rate_log_count,path_low_10_mean,path_low_10_sum,path_flow_mean,path_flow_sum,path_flow_std,path_flow_fallback_share,previous_bus_departure_age_minutes,station_seq_cat,direction,snapshot_day_of_week,snapshot_low_plate_cat,target_state_cat,snapshot_station_seq_cat,snapshot_time_bin_30,route_code,query_id,bus_order,queue_ahead\n218000010,45.0,0.9600498543859287,0.27982901403099203,1.0,0.38,126.9029667,37.51835,45.0,0.28888888888888886,31.0,32.0,-6.0,-1.5,-0.7888888888888889,5.0,1.25,38.75,7.544444444444444,1.032258064516129,8.955555555555554,0.0,0.0,44.0,0.2727272727272727,8.454545454545453,0.0018059829341007885,0.0,0.03165054820137805,0.9811669942427194,-0.526051942651833,-11.573142738340326,3.219930476660466,1.0,,50,to_city,1,0,2,19,9,218000010,example-final-eval,1,30\n218000010,45.0,0.9600498543859287,0.27982901403099203,1.0,0.22,126.9029667,37.51835,45.0,0.0888888888888889,39.0,41.0,0.0,0.0,0.0,5.0,1.3333333333333335,52.00000000000001,41.0,1.0512820512820513,3.4666666666666672,0.0,0.0,44.0,0.06818181818181823,2.659090909090911,0.0018059829341007885,0.0,0.02629932058671481,1.0256735028818775,-0.6257064332522909,-18.77119299756873,2.7688225295873434,1.0,,50,to_city,1,0,2,11,9,218000010,example-final-eval,2,30\n218000010,45.0,0.9600498543859287,0.27982901403099203,1.0,0.02,126.9029667,37.51835,45.0,0.0888888888888889,49.0,41.0,,,,,,,,0.8367346938775511,4.355555555555556,0.0,0.0,44.0,0.06818181818181823,3.3409090909090935,0.0018059829341007885,0.0,0.02433135701837451,1.1922364939003511,-0.5868526488714267,-21.126695359371364,2.534882631492004,1.0,,50,to_city,1,0,0,1,9,218000010,example-final-eval,3,30\n'
example = pd.read_csv(StringIO(EXAMPLE_INPUT_CSV))
display(example[['query_id', 'bus_order', 'queue_ahead', 'route_id', 'snapshot_remaining_seats', 'target_stop_gap']])

prediction = model.predict_queries(example)
display(prediction.T)

probability_columns = [f'board_by_{index}_probability' for index in (1, 2, 3)]
assert prediction[probability_columns].notna().all().all()
assert prediction[probability_columns].apply(lambda column: column.between(0, 1).all()).all()
assert prediction.loc[0, probability_columns].is_monotonic_increasing


## 5. 사용자 CSV 배치 추론

사용자 CSV도 예제와 같은 prepared feature schema여야 합니다. 원시 GBIS 응답을 바로 넣는
인터페이스가 아니라, 질의 시점 이전 정보만으로 만든 도착 전 피처를 입력하는 인터페이스입니다.


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    input_name = next(name for name in uploaded if name.lower().endswith('.csv'))
    batch = pd.read_csv(io.BytesIO(uploaded[input_name]))
    batch_prediction = model.predict_queries(batch)
    output_path = Path('/content/boarding_predictions.csv')
    batch_prediction.to_csv(output_path, index=False)
    display(batch_prediction)
    files.download(str(output_path))
except ImportError:
    print('Colab이 아니면 batch DataFrame에 입력을 넣고 model.predict_queries(batch)를 호출하세요.')


## 파일 역할

- **필수:** `queue_boarding_v1_0_0.pkl`
- **선택:** `queue_boarding_v1_0_0.metadata.json` — 사람이 읽는 감사·요약 정보
- **선택:** registry/manifest — 운영 환경에서 primary alias와 파일 SHA를 fail-closed 검증할 때 사용
- PKL 자체에 모델 계약과 학습된 객체가 들어 있으므로 이 노트북은 PKL SHA를 직접 검증합니다.

`predicted_sent_class=3`은 정확히 세 대가 아니라 **다음 3대 안에 못 타거나 3대 이상을 보내야 함**을
뜻합니다.
